# CoFormer — Co-Trained Autoformalization for Faithful Neuro-Symbolic Reasoning

Reproducible experimental notebook. Single-GPU (A100/H100), `Run All`.

---

## 1. Problem

A neuro-symbolic reasoning pipeline has two stages: **translation** of natural
language into a symbolic formula, and **symbolic reasoning** over that formula.
Given a correct formula the logic layer is a sound oracle, so pipeline accuracy
is governed by the translation (autoformalization) step. In existing systems
that step is a frozen, one-shot language model that never receives a learning
signal from the reasoning stage.

**Thesis under test.** Co-training the translator with the gradient
back-propagated from a differentiable logic layer improves translation
faithfulness, and that gain transfers to end-task accuracy and to
out-of-distribution robustness.

## 2. Method

1. **Translator `f_theta`** — encodes each premise sentence independently; one
   attention query per variable emits that variable's 3-way distribution over
   {negative, absent, positive}.
2. **Discrete–continuous bridge** — straight-through Gumbel-Softmax: a hard
   {−1, 0, +1} clause matrix in the forward pass, soft gradients backward.
3. **Differentiable logic layer `L`** — relaxed soft-MaxSAT that refines a
   continuous assignment. The final `refine_backprop_steps` iterations stay in
   the autograd graph (truncated backpropagation through the solver).

**Objective** = anchored task loss + `lambda_faith · L_faith` + anti-degeneracy
terms (class-balanced faithfulness NLL, one-sided density hinge, logit L2).

The task term is **anchored**: it solves the *predicted* CNF and scores the
resulting assignment against the **gold** clauses, so the objective cannot be
satisfied by emitting permissive clauses.

**Scope of the relaxation.** It cannot certify unsatisfiability; the
completeness probe measures the rate. Every reported *decision* therefore comes
from Z3, identically for the proposed method and for every baseline. The
relaxation is used only as a training signal.

## 3. Conditions

All four are trained on identical data with identical seeds and compute, all
carry the anti-degeneracy terms, and exactly one mechanism differs:

| condition | per-slot `L_faith` | solver-routed gold signal | bridge |
|---|---|---|---|
| `frozen` | yes | **detached** | straight-through |
| `coformer` | yes | yes | straight-through |
| `no_faith` | no | yes | straight-through |
| `no_stgs` | yes | yes | plain soft relaxation |

Because the task loss is anchored to the gold clauses, `no_faith` is not "no
faithfulness supervision": it is the condition whose gold supervision arrives
only through the solver, never per slot.

## 4. Data and metrics

Synthetic satisfiable k-SAT with a **gold CNF**, so faithfulness is measurable
exactly; instances are verbalised as templated English. OOD = paraphrase
(held-out templates) and scale (wider instances). Real benchmarks: FOLIO and
ProofWriter, decided by Z3.

Metrics: density-controlled clause fidelity, chance-normalised gold-clause
satisfaction, exact solve, refinement iterations, wall clock, error-type
decomposition.

## 5. Analysis

Hypotheses, metrics and decision rules are pre-registered before any model is
trained. Contrasts are paired over seeds, bootstrapped, and Holm-corrected
within family. Every run is audited for degeneracy, and the reporting layer
refuses to declare a hypothesis supported when either compared condition is
flagged.

## 6. Running it

`Runtime → Run all` on an A100. Set `FULL_SCALE=True` for the full
configuration; results are checkpointed per (condition, seed) and resume across
sessions.


## Revision v2 — changes required before the reported re-run

| ID | Change | Cells |
|---|---|---|
| M1 | Faithfulness NLL evaluated on `log_softmax(logits)` for all conditions (synthetic and real). v1 evaluated it on the straight-through sample, which blocked corrective gradients. | 8, 9, C3 |
| — | `LOSS_VERSION` added to every fingerprint: all v1 checkpoints, calibrations, sweeps and real-data results are archived, never resumed. | 3c |
| M2 | Epoch calibration re-runs under v2; `switch_probe` already asserts an identical budget for every family. | 9c, 10c |
| M3 | Real data: `soft_frozen` system added to complete the factorial. | C3, C5 |
| M5 | H7–H9 marked exploratory and analysed outside the Holm family. | 3b, 11b |
| M6 | Optional larger family C (3-SAT, n=40), gated by `RUN_LARGE_FAMILY`. | 3 |
| M7 | Gumbel-Rao straight-through estimator (conditions `grs_frozen`, `grs_routed`) in a separate extension cell. | 9, 10e |
| M8 | Sensitivity overrides applied after `scaled()`; refinement-truncation test on `no_stgs` (S = 50/100/200); ratio-sweep chance level re-measured; negative control on `no_stgs` and `coformer`; ST-bias probe with task-only objective. | 9b, 10, 13b, 13c, 14 |
| — | Duplicate `lambda_*` fields removed from `Config`. | 3 |

## Revision v3 — execution safety and reporting integrity

| Change | Cells |
|---|---|
| Family-B results file was unversioned (`results_probeB.csv`): a stale v1 matrix counted as complete and **B would never have been re-run**. Probe files now carry the run fingerprint. | 10c, 14 |
| `per_example_synth.json` was unversioned: v1 and v2 per-example scores could mix in the cluster bootstrap. Now fingerprinted. | 10, 11b |
| Session-budget guard (`ensure_budget`) before every heavy unit; the run stops cleanly at a checkpoint boundary. | 4, 9c, 10, 10c, 10e, 13c, 13e, C5, C6 |
| Sensitivity runs cached one by one (v2 cached only after all 11 runs). | 13c |
| Completeness probe uses the trained step size (2.0) instead of the default (0.5). | U |
| Claim sentences in the health check are derived from the analysis (no hard-coded "unsupported" / "anti-parallel" text). | 14 |
| `run_metadata.json` rewritten at the end with the calibrated configuration. | 14 |
| C7 reports the real-data seeds actually used; HF_TOKEN warning corrected (token optional). | 4, C7 |
| Epoch calibration on BOTH bridges (`no_stgs`, `coformer`); shared budget = maximum of the per-condition budgets; every outcome cached (v2 recalibrated in every session when no plateau was found). | 9c |
| Phase-2 ordering respects `ABLATION_SEEDS` (latent bug when it differs from `FACTORIAL_MIN_SEEDS`). | 10 |
| New Cell 4b archives all v1 checkpoints and outputs to `<BASE>_v1_archive/` once (keeps dataset, benchmark and LLM-generation caches). | 4b |
| All checkpoint writes (results, per-example scores, calibration, diagnostics, LLM generation cache, real-data results) are atomic, so a disconnect during a Drive write cannot corrupt a checkpoint. | 4, 9b, 9c, 10, 10c, 10e, B2, C2, C5 |

## Revision v4 - stages that answer the questions the first run could not

| Change | Cells |
|---|---|
| `faith_fraction` argument: per-slot supervision is given to a fixed random subset of training instances, the rest are trained through the solver alone. | 8, 9 |
| Pre-specified plan II (E-family: confirmatory estimator replication on seeds 12-23; P-family: solver gradient under partial supervision at each fraction). | 3b |
| Cell 10f: partial-supervision matrix, 4 conditions x 3 fractions x 12 seeds, checkpointed and resumable. | 10f |
| Cell 10g: estimator replication on fresh seeds. | 10g |
| Cell 11f: Holm-corrected tests for both new families, plus the supervision curve figure. | 11f |
| Stage flags `RUN_PARTIAL_SUPERVISION`, `PARTIAL_SEEDS`, `RUN_ESTIMATOR_CONFIRMATORY`. | 3 |

The previously stored results are reused: the configuration is unchanged, so the checkpoint
guard adopts them and only the new stages are executed.

## Revision v5 - storage resilience

| Problem observed | Fix |
|---|---|
| Colab's Drive mount failed with "credential propagation was unsuccessful" and the notebook silently continued in an empty ephemeral directory, which would retrain every stage and lose the result at the end of the session. | The mount is retried (`DRIVE_MOUNT_ATTEMPTS`, with `force_remount` from the second attempt) and, if it still fails, the run stops with an explanatory error unless `ALLOW_LOCAL_FALLBACK = True`. |
| `IN_COLAB` was set only inside the mount `try`, so a failed mount made the run metadata claim the session was not on Colab. | Colab is detected independently of the mount. |
| No way to resume a Drive-less session. | `restore_state(zip path or URL)` and `snapshot_state()`; `STATE_ARCHIVE` restores automatically into an empty local run, and the final cell snapshots the state when the session is not persistent. |
| The budget-stop message did not mention that an ephemeral session must be snapshotted. | The message now says so when `PERSISTENT` is false. |

The Drive mount failure itself is an authentication problem on Google's side and cannot be
repaired from the notebook; what the notebook now guarantees is that it never wastes GPU time
on state it cannot keep.

| Change | Cells |
|---|---|
| `pull_state()` accepts any zip in the state repository, so it can be seeded by dragging the Drive backup into the Hugging Face web interface. | 4 |
| `ALLOW_FULL_RECOMPUTE` guard: if no previous state is found anywhere, the run stops rather than silently restarting a 100+ GPU-hour computation. | 3, 4 |


## Revision v9 - Drive only, with an API fallback for the mount

| Change | Cells |
|---|---|
| The Hugging Face state layer was removed. | 3, 4 |
| When `drive.mount` fails, the notebook authenticates with `google.colab.auth` and uses the **Drive REST API** instead: the state (`checkpoints/` + `outputs/`) is kept as a single zip in `My Drive/CoFormer_state`. The mount and the API use different authentication paths, so the API frequently works when the mount does not. | 4 |
| `push_state` / `pull_state` / `maybe_push_state` now talk to Drive; autosave after every finished run, throttled to `DRIVE_STATE_AUTOSAVE_MIN` minutes, plus forced uploads at the budget stop and at the end of the run. | 4, 10, 10c, 10e, 10f, 10g, C5, 14 |
| If neither the mount nor the API works and no local state is present, the run stops instead of recomputing 100+ GPU-hours. | 4 |

## Revision v10 - runtime kernel fallback

| Problem | Fix |
|---|---|
| On some Colab images `torch.prod` fails with `nvrtc: error: failed to open libnvrtc-builtins.so`, because the runtime compiler that builds the fused product kernel is incomplete. The soft-MaxSAT layer calls it in every refinement step, so training aborts. | The clause product falls back to `exp(sum(log(.)))`, which is identical here since the factors are clamped to `[1e-7, 1]`. A probe at import time selects the path once, and an in-flight `RuntimeError` also switches it. | 

## Revision v11 - partial-supervision grid moved below the ceiling

The first probe of stage 10f showed that `f = 0.05` already reaches the ceiling
(`excess = 0.906`, identical to full supervision), so the original grid would have measured the
saturated regime three more times. The grid is now `(0.001, 0.005, 0.02)`, i.e. 20, 100 and 400
supervised instances out of 20,000, and rows from superseded fractions are archived
automatically instead of being mixed into the curve.


## Revision v12 — Plan III and Run-All safety

This revision adds the pre-specified **Plan III** experiments and makes a Run All over the stored
study safe. No stored result is retrained or overwritten.

| Change | Cells |
|---|---|
| Stage flags `RUN_PLAN_III_*` (all `False` by default), `STRICT_FINGERPRINT_GUARD`, `RECOMPUTE_REPORTED_DIAGNOSTICS`. | 3 |
| Cell 3d: the Plan III specification is written to `outputs/plan_III.json` with its SHA-256 **before** any Plan III run. It also defines the fingerprint guard and the reuse guard. | 3d |
| Fingerprint guard. Every checkpoint fingerprint is compared with the value of the reported study before any archiving code runs. A mismatch stops the run instead of archiving results. | 9b, 9c, 10, 10e, C5 |
| Reported diagnostics that were recomputed on every Run All are now reused when their output exists: ST-bias probe, objective anchoring, inference latency, contamination probe, FOLIO faithfulness validation, completeness probe. | 13b, 13d, C2b, C6, U |
| `requirements.txt` / `run_metadata.json` of the reported run are never overwritten; later sessions write `*.session-<UTC>.*` files. | 4, 14 |
| Cell 10h: Plan III synthetic experiments (answer supervision, truncation horizon, task-loss weight, training budget). They use their own fingerprinted checkpoint file and never touch the stored matrices. | 10h |
| Cell 11g: Plan III analysis (exact sign-flip tests, Holm, non-superiority). | 11g |
| Plan III family D: direct-answering and chain-of-thought baselines added to the natural-language comparison. Only the new (system, seed) rows run. | C5 |
| Cell 4b (v1 archive) disabled: it would move the reported checkpoints if its marker file were missing. | 4b |
| v12.1: library versions are read without importing them; Part C skips (instead of running with mixed versions) if transformers is not the pinned 4.46.2; C1 no longer overwrites `requirements.txt`; `llm_generate` accepts the transformers 5 return type. | 3d, C1, C2 |
| Captions in the output bundle corrected to match the final analysis. The contamination verdict text is corrected as well. | 14, C2b |

**Procedure.**
1. Back up the `CoFormer` state folder.
2. Run All with every `RUN_PLAN_III_*` flag set to `False`. This verifies the fingerprints, retrains nothing, and writes `outputs/plan_III.json`.
3. Deposit `plan_III.json` with a public time stamp (for example on OSF).
4. Set the flags to `True` in the priority order A2 → A1 → KW → BUDGET → NL, then Run All again. Every stage resumes from its checkpoint.


## Revision v14 — Family H: a linguistically harder benchmark (pre-specified)

Adds one pre-specified study. Everything from v13 (follow-up study 2) is unchanged; stored runs are
never retrained, and family H writes only to `results_famH_<fp>.csv` and `outputs/*famH*`.

* **Formulas:** the stored family-A formulas, instance by instance; only the text changes.
* **Language:** 3 aliases per entity and 3 synonyms per property, subject ellipsis,
  polarity as the parity of up to *level* stacked negations, and *level − 1* distractor sub-clauses.
  A symbolic parser proves that every sentence determines its clause.
* **Pilot:** `frozen`, levels 1–3, seed 1000. A fixed rule selects the lowest level whose ID excess
  lies between 20% and 95% of its maximum (fallbacks fixed in Cell 3f).
* **Main:** `coformer` vs `frozen` at the selected level, 24 epochs:
  full supervision (16 seeds) and f = 0.05 (12 seeds).
  Exact one-sided sign-flip tests, Holm over the two.

| Stage | Runs | Approx. A100 time |
|---|---|---|
| Pilot | 3 | ~2 h |
| H_full | 32 | ~13–27 h |
| H_scarce | 24 | ~10–20 h |

Times are rough (family A's 13.4-min median at 12 epochs, scaled to 24 epochs and to the sentence
length of the selected level; lower figure level 1, upper level 3).

Cell 3f fixes the specification (`outputs/followup_h.json`, SHA-256, e-mailed) before any run;
Cell 10j runs the stages and pauses cleanly at the session budget; Cell 11i applies the rules.


## Revision v13 — Follow-up study 2 (referee-requested experiments)

Adds three pre-specified experiments on family A. No stored result is retrained or overwritten;
the new runs write to their own checkpoint file `results_fs2_<fp>.csv`.

| Design | What it tests | Runs | Approx. A100 time |
|---|---|---|---|
| C3 | A2 with **shuffled** answer labels: is the A2 harm caused by the answer content? | 12 | ~4.5 h |
| C2 (Q=1) | **Non-differentiable** answer feedback: RLOO with the exact solver's labels + per-slot supervision on 100 instances | 12 | ~3 h |
| C1 | **Training to convergence**: `coformer` vs `frozen` at f = 0.005 for 48 epochs, 12 seeds | 24 | ~21 h |
| C2 (Q=16) | RLOO with one query per variable, with and without per-slot supervision | 12 | ~3 h |

Cell 3e fixes the specification (`outputs/followup2.json`, SHA-256, e-mailed) before any run;
Cell 10i runs the stages in the order above and pauses cleanly at the session budget; Cell 11h
applies the decision rules (exact one-sided sign-flip tests, Holm over C1-C3).
Exact entailment labels are computed by enumerating all 2^16 assignments on the GPU; on the gold
formulas this reproduces the DPLL labels of the follow-up study exactly (checked at run time).


## Cell 2 — Dependency setup (idempotent, 2026+ runtime friendly)
Installs only lightweight, version-floored packages. We deliberately do **not** reinstall
`torch`: Colab ships a CUDA build matched to the A100 driver, and forcing a version is the most
common way to break *Run all* on a fresh 2026 runtime.

In [ ]:
# === Cell 2: dependency setup (idempotent, version-pinned) ===
import subprocess, sys, importlib

PINS = ["numpy==2.1.3", "pandas==2.2.3", "matplotlib==3.9.2", "scipy==1.14.1",
        "tqdm==4.67.1", "requests==2.32.3", "statsmodels==0.14.4"]

def _pip(args, hard=False):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        msg = f"pip install {' '.join(args)} FAILED:\n{r.stderr[-1500:]}"
        if hard:
            raise RuntimeError(msg)
        print("WARNING:", msg)
    return r.returncode == 0

# torch is assumed present (Colab GPU runtime ships a CUDA build matched to the driver;
# forcing a version is the most common way to break Run-All on a fresh runtime).
_pip(["--upgrade", *PINS], hard=False)

print("Resolved environment:")
_versions = {}
for mod in ("torch", "numpy", "pandas", "matplotlib", "scipy", "statsmodels", "requests"):
    try:
        m = importlib.import_module(mod)
        _versions[mod] = getattr(m, "__version__", "?")
    except Exception as e:
        _versions[mod] = f"MISSING ({e})"
    print(f"  {mod:12s} {_versions[mod]}")
print("\nDependency setup complete.")


## Cell 3 — *Hard-GPU Only* assertion, A100/H100 check, AMP setup, global config
Hard-asserts a CUDA GPU and stops with an actionable message otherwise. Warns (does not crash) if
the GPU is not A100/H100, since timings and the AMP dtype are tuned for those. Enables TF32 and
selects **bf16** autocast on Ampere/Hopper (numerically safer than fp16 for this iterative solver).
All experiment knobs live here so readers see the full configuration in one place.

In [ ]:
# === Cell 3: device gate, AMP, and global configuration ===
import os, random, time, math, json, dataclasses, traceback, platform
from typing import Optional
from dataclasses import dataclass
import numpy as np
import torch

# Run-All safe: do NOT crash when there is no GPU. Part C requires one; Part A can run at small
# scale on CPU. No fabricated result is ever produced on either path.
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    DEVICE = torch.device("cuda")
    GPU_NAME = torch.cuda.get_device_name(0)
    _cc_major, _cc_minor = torch.cuda.get_device_capability(0)
    IS_AMPERE_OR_NEWER = _cc_major >= 8                   # A100=8.0, H100=9.0
    print(f"GPU: {GPU_NAME} (compute capability {_cc_major}.{_cc_minor})")
    if not IS_AMPERE_OR_NEWER:
        print("WARNING: GPU older than A100; bf16/TF32 paths and timings are tuned for A100/H100.")
else:
    DEVICE = torch.device("cpu")
    GPU_NAME = "CPU (no CUDA)"; IS_AMPERE_OR_NEWER = False
    print("WARNING: no CUDA GPU. Part C (LLM co-training) will SKIP. "
          "Part A can still run on CPU at small scale (FULL_SCALE=False).")
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | DEVICE={DEVICE} "
      f"| {platform.platform()}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
AMP_DTYPE = torch.bfloat16 if IS_AMPERE_OR_NEWER else torch.float16
print(f"AMP dtype: {AMP_DTYPE} (autocast enabled on GPU only)")

@dataclass
class Config:
    FULL_SCALE: bool = True         # full SCALE (see the cost note printed below)
    USE_LLM_BACKBONE: bool = True   # Part C (real LoRA LLM co-training)
    USE_AMP: bool = True
    LOCAL_MODE: bool = False        # R2-FIX-A2: run outside Colab (no Drive, local paths)
    seeds: tuple = (0, 1, 2)

    # --- synthetic instance generator ---------------------------------------------------
    n_vars: int = 10
    sat_ratio: float = 4.0          # m/n; random 3-SAT phase transition is ~4.26
    clause_len: int = 3
    use_planted: bool = False       # keep the old easy generator available for comparison
    n_train: int = 2000
    n_eval: int = 400
    ood_n_vars: int = 15

    # --- optimisation --------------------------------------------------------------------
    epochs: int = 8
    batch_size: int = 64            # sequences are real sentences now -> longer than v1
    lr: float = 3e-3
    d_model: int = 128
    n_layers: int = 2
    n_heads: int = 4
    refine_steps: int = 20
    refine_step_size: float = 2.0
    refine_backprop_steps: int = 10   # final steps kept in the autograd graph
    probe: str = "A"          # "A" = 3-SAT, 16 vars; "B" = 4-SAT, 12 vars, disjoint lexicon
    partc_seeds: int = 4
    partc_epochs: int = 2
    gumbel_tau_start: float = 2.0
    gumbel_tau_end: float = 0.5
    lambda_faith: float = 1.0
    lambda_logit: float = 1e-4    # anti-saturation L2 on emitted logits
    lambda_density: float = 0.5   # anti-collapse density prior (swept in 9b)
    # Set once by the epoch calibration. `scaled()` RAISES the preset fields and
    # never lowers them, so without this marker every later Config(...).scaled()
    # silently restores the uncalibrated epoch count. That defect appeared at
    # three separate call sites; carrying the calibrated value on the config
    # itself fixes every present and future caller at the source.
    calibrated_epochs: Optional[int] = None
    RESPECT_POWER: bool = True

    @property
    def n_clauses(self):
        return int(round(self.n_vars * self.sat_ratio))

    def scaled(self):
        """- FULL_SCALE retuned for one Colab Pro+ A100 session.

        The v4 preset (n_vars=20 -> 80 clauses, n_train=20000, epochs=25, batch=32) was written
        before the per-clause encoder. With the old flat-stream model it implied a ~3528-token
        sequence at batch 32 -- roughly 6 GB of attention per layer on a 40 GB card, i.e. an
        almost certain OOM, and an extrapolated 16-48 h for Part A alone.

        The per-clause encoder removes the quadratic term: cost is now n_clauses * L^2 with
        L ~ 41, so instance width is cheap and the binding constraint is the number of optimiser
        steps. The preset therefore buys STEPS and DATA instead of sequence length:
            n_vars 10 -> 16 (64 clauses, still well past the phase transition at ratio 4)
            n_train 2000 -> 20000, epochs 8 -> 40  => 12500 optimiser steps (was 250)
        Measured VRAM and wall clock are printed by Cell 10 before training starts.
        """
        if self.probe == "C":
            # REVISION v2 (M6): optional larger 3-SAT family; enabled via RUN_LARGE_FAMILY.
            self.clause_len = 3
            self.sat_ratio = 4.0
        if self.probe == "B":
            # A genuinely different instance family: longer clauses, fewer variables, a higher
            # clause/variable ratio, and a disjoint surface realisation. What is NOT varied is
            # the mechanism under test -- the four conditions and the bridge are identical.
            self.clause_len = 4
            self.sat_ratio = 9.0          # near the 4-SAT phase transition
        if self.FULL_SCALE:
            self.n_vars, self.ood_n_vars = {"A": (16, 24), "B": (12, 18), "C": (40, 60)}[self.probe]
            self.n_train, self.n_eval = 20000, 2000
            self.epochs = 40
            self.batch_size = 64
            self.lr = 1e-3
            self.d_model, self.n_layers, self.n_heads = 256, 4, 8
            self.refine_steps = 50
        # Probe B previously ran at batch 32. That constraint predates the
        # per-clause encoder: with a flat token stream the sequence was ~1764
        # tokens and batch 64 would not fit. Encoding clauses independently
        # makes the cost n_clauses * L^2 with L = 60, so batch 64 needs ~26 GB
        # of a 40 GB card -- inside the 80% ceiling the pre-flight enforces.
        #
        # Keeping batch 32 doubled probe B's optimiser steps (25 000 vs 12 480)
        # for the same 40 epochs, so probe B cost twice probe A for no reason
        # and the two families were optimised under DIFFERENT budgets, which is
        # a comparability defect as well as a waste. Equalising the batch size
        # equalises the step count, which is what the cross-probe comparison
        # assumes. Small cards fall back to 32 via the autopilot preset.
        if self.calibrated_epochs is not None:
            self.epochs = int(self.calibrated_epochs)
        return self

# External validity requires more than one instance family. PROBES lists the
# families the study reports; the matrix is run once per probe and each probe
# writes its own fingerprinted results file, so a finding claimed for "the
# method" is a finding reproduced across clause width, clause/variable ratio,
# instance width and surface lexicon -- not a property of one generator.
# --- Storage resilience (v5) ---
# Colab's Drive mount occasionally fails with "credential propagation was unsuccessful".
# That is an authentication failure on Google's side, not in this notebook. What the
# notebook controls is what happens next, and silently continuing in an empty ephemeral
# directory is the worst option: it retrains everything and loses the result when the
# session ends. The storage cell therefore retries, and then STOPS unless the fallback is
# explicitly allowed below.
ALLOW_LOCAL_FALLBACK = True    # Drive is unavailable on this account; run on local disk
ALLOW_FULL_RECOMPUTE = False   # safety: never silently retrain 100+ GPU-hours from scratch
DRIVE_MOUNT_ATTEMPTS = 3
STATE_ARCHIVE = ""             # path or URL of a state zip to restore into an empty local run

# --- Drive state synchronisation (v9) ---
# The FUSE mount (`drive.mount`) is Colab's own authentication path and it is the part that
# fails with "credential propagation was unsuccessful". The Drive REST API uses a different
# path (`google.colab.auth.authenticate_user`), and in practice it often works when the mount
# does not. When the mount fails, the notebook therefore keeps using Drive through the API:
# checkpoints and outputs are synchronised as a single zip in a Drive folder.
USE_DRIVE_API_FALLBACK = True
DRIVE_STATE_FOLDER = "CoFormer_state"   # created in My Drive if absent
DRIVE_STATE_AUTOSAVE_MIN = 20           # minimum minutes between automatic uploads

# --- Stage flags for the v4 extensions (see the revision log at the top) ---
RUN_PARTIAL_SUPERVISION   = True    # M1: curve over the fraction of per-slot supervision
# REVISION v11: the first probe showed that f = 0.05 (1,000 of 20,000 instances) already
# saturates the task (excess 0.906, the ceiling). The curve therefore has to be sampled far
# lower, between the collapse at f = 0 and the ceiling: 0.001/0.005/0.02 are 20/100/400
# supervised instances. The endpoints f = 0 and f = 1 are already in the stored matrices.
PARTIAL_FRACTIONS         = (0.001, 0.005, 0.02)
PARTIAL_CONDITIONS        = ("frozen", "coformer", "soft_frozen", "no_stgs")
PARTIAL_SEEDS             = 12      # lower to 8 to save ~11 GPU-hours
RUN_ESTIMATOR_CONFIRMATORY = True   # M2: pre-specified replication on fresh seeds
ESTIMATOR_CONF_CONDITIONS = ("frozen", "coformer", "grs_frozen", "grs_routed")
ESTIMATOR_CONF_SEEDS      = tuple(range(12, 24))

# --- REVISION v12: Plan III -- AUTO variant: every stage enabled; Run All needs no edits
RUN_PLAN_III_A2     = True    # priority 1: answers through the solver + per-slot on 100 instances (24 runs)
RUN_PLAN_III_A1     = True    # priority 2: answers only, no per-slot supervision (12 runs)
RUN_PLAN_III_KW     = True    # priority 3: truncation horizon K=T and task-loss weight 5 at f=0.005 (24 runs)
RUN_PLAN_III_BUDGET = True    # priority 4: 2x epoch budget at f=0.005 with learning curves (12 runs, 2x cost)
RUN_PLAN_III_NL     = True    # priority 5: direct-answering and chain-of-thought baselines (Part C)
# --- REVISION v13: follow-up study 2 (specified in Cell 3e)
RUN_FS2_C1 = True   # training to convergence at f=0.005 (24 runs, ~21 A100-h)
RUN_FS2_C2 = True   # exact-solver answer feedback via RLOO (24 runs, ~6 A100-h)
RUN_FS2_C3 = True   # A2 with shuffled answer labels (12 runs, ~4.5 A100-h)
# --- REVISION v14: family H (specified in Cell 3f)
RUN_FAMH = True   # harder-language benchmark: pilot + 56 runs (~25-50 A100-h, rough)
STRICT_FINGERPRINT_GUARD = True        # stop instead of archiving stored results
RECOMPUTE_REPORTED_DIAGNOSTICS = False # reuse reported values of non-checkpointed diagnostics

RUN_LARGE_FAMILY = False     # REVISION v2 (M6): set True to add family C (~30 GPU-h)
PROBES = ("A", "B", "C") if RUN_LARGE_FAMILY else ("A", "B")

CFG = Config().scaled()
_cfg_dump = {**dataclasses.asdict(CFG), "n_clauses (derived)": CFG.n_clauses}
print("\nConfiguration:\n" + json.dumps(_cfg_dump, indent=2, default=str))

def set_all_seeds(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
set_all_seeds(CFG.seeds[0])
print("\nSeeding + determinism configured.")

if not HAS_GPU:
    CFG.USE_AMP = False
    print("CPU mode: CFG.USE_AMP=False (autocast disabled).")
AMP_DEVICE = "cuda" if HAS_GPU else "cpu"

# --- stage gate flags, declared HERE so Run-All degrades instead of crashing --
# Each later stage sets its own flag once its dependencies are known good. They
# are declared up front because a stage that fails to install must leave the
# downstream cells able to SKIP cleanly; if the flag itself is undefined, every
# consumer raises NameError instead, which is the one failure mode the "never
# fabricate, always skip" design is meant to exclude. Declaring them costs
# nothing and makes the skip path reachable from any starting state.
Z3_OK = False                 # set True by Cell B1 once z3 imports
BACKBONE_READY = False        # set True by Cell C1 once the LLM stack loads
REAL_AVAILABLE = False        # set True by Cell B2 once a corpus is parsed
skip_reason = "stage has not run yet"
real = {"folio": [], "proofwriter": [], "sources": {}}
real_df = None                # Cell B5 replaces this with a DataFrame
real_comparison_df = None     # Cell C5 replaces this with a DataFrame

def _empty_cache():
    """Guarded CUDA cache flush: a no-op off-GPU so Part C can be dry-run on CPU."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Optional extra diagnostics that TRAIN EXTRA MODELS (lambda sweep, negative control,
# hyper-parameter sensitivity, qualitative case dump, faithfulness validation). Gated so a
# default Run-All stays fast and predictable.
RUN_EXTRA_DIAGNOSTICS = True     # full artifact set: sweeps, negative control, case dump

# =========================================================================
# AUTOPILOT -- the notebook sizes and sequences its own work.
#
# A Run All must complete the study without anyone editing a constant. The
# three decisions that previously needed a human are made here from measured
# hardware and a declared wall-clock budget:
#
#   1. SCALE   -- FULL_SCALE follows the device. A 40 GB+ accelerator runs the
#                 full configuration; a smaller GPU runs a reduced one; CPU runs
#                 the demo preset. Nothing is silently downscaled: the chosen
#                 preset is printed and recorded in run_metadata.json.
#   2. PROBES  -- both instance families are executed in one session, in order
#                 of scientific priority, rather than requiring a second Run All
#                 with CFG.probe edited by hand.
#   3. EXTRAS  -- sweeps and the negative control run in the SAME session,
#                 immediately after the matrix they depend on, budget allowing.
#
# Work is ordered so that a session cut short still leaves the most valuable
# evidence on disk, and every stage is checkpointed, so re-running the notebook
# resumes rather than repeats. AUTOPILOT = False restores manual control.
# =========================================================================
import time as _t
SESSION_T0 = _t.time()   # this session's clock, for honest budget checks

AUTOPILOT = True
# REVISION v14: 21 h fits one Colab Pro+ session (24 h maximum) with a margin for the cells
# that follow the training stages; a headless run sets COFORMER_SESSION_HOURS (run_headless.py).
SESSION_BUDGET_HOURS = float(os.environ.get("COFORMER_SESSION_HOURS", "21"))

if AUTOPILOT:
    _vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9
                if HAS_GPU else 0.0)
    if _vram_gb >= 38:
        _preset, CFG.FULL_SCALE = "full", True
    elif _vram_gb >= 14:
        _preset, CFG.FULL_SCALE = "reduced", True
    else:
        _preset, CFG.FULL_SCALE = "demo", False
    # Rebuild from DEFAULTS, not from the already-scaled CFG: `scaled()` only
    # raises the preset fields, so re-running it on a scaled config can never
    # bring them back down and a "demo" verdict would silently keep full-scale
    # epochs and dataset size.
    _carry = {k: getattr(CFG, k) for k in
              ("probe", "USE_LLM_BACKBONE", "LOCAL_MODE", "USE_AMP",
               "seeds", "RESPECT_POWER")}
    CFG = Config(**{**_carry, "FULL_SCALE": CFG.FULL_SCALE}).scaled()
    if _preset == "reduced":                 # fit a 16 GB card without OOM
        CFG.batch_size = max(8, CFG.batch_size // 2)
        CFG.n_train = min(CFG.n_train, 8000)
        CFG.epochs = min(CFG.epochs, 25)
    print(f"AUTOPILOT: device has {_vram_gb:.0f} GB -> `{_preset}` preset "
          f"(FULL_SCALE={CFG.FULL_SCALE}, batch={CFG.batch_size}, "
          f"n_train={CFG.n_train}, epochs={CFG.epochs})")
    print(f"           session budget {SESSION_BUDGET_HOURS:.0f} h; the plan is "
          f"fitted to it and checkpointed, so a cut-off session resumes.")
else:
    print("AUTOPILOT disabled: using the constants as written in Config.")

_steps_per_run = CFG.epochs * ((CFG.n_train + CFG.batch_size - 1) // CFG.batch_size)
print(f"\nSCALE: FULL_SCALE={CFG.FULL_SCALE} -> n_vars={CFG.n_vars}, n_clauses={CFG.n_clauses}, "
      f"n_train={CFG.n_train}, epochs={CFG.epochs}, d_model={CFG.d_model} x {CFG.n_layers}L")
print(f"       {_steps_per_run} optimiser steps per run "
      f"(the demo preset gives 250 -- the run that learned nothing had 250)")
print(f"       Extra diagnostics: {'ON (full artifact set)' if RUN_EXTRA_DIAGNOSTICS else 'OFF'}")
print(f"       Cell 3c fixes the seed count from the power analysis and prints the run count; "
      f"Cell 10 projects the wall clock from a MEASURED pre-flight run before the matrix starts.")


## Cell 3b — Pre-registration: locked hypotheses, metrics, and decision thresholds
To eliminate any suspicion of p-hacking or post-hoc metric selection, the analysis is
**pre-registered here, before any model is trained.** Each hypothesis is bound to exactly one
primary metric, one comparison, one directional prediction, and one decision rule. The reporting
cells later read these locked definitions rather than choosing favorable metrics after seeing
results. The locked definitions are inspectable in this cell.

In [ ]:
# === Cell 3b: locked pre-registration (read by all later reporting) ===
PREREGISTRATION = {
    "H1": dict(metric="faith_excess_id", comparison=("coformer", "frozen"), direction="greater",
               decision="supported if mean diff > 0 AND Holm-adjusted p < 0.05 AND neither "
                        "condition is flagged degenerate",
               rationale="the logic-layer gradient improves symbolic-translation fidelity "
                         "beyond what gold-CNF supervision alone achieves (measured as EXCESS "
                         "over a density-matched noise baseline; see NOTES_PREREG)"),
    "H2": dict(metric="csat_norm_id", comparison=("coformer", "frozen"), direction="greater",
               decision="supported if mean diff > 0 AND Holm-adjusted p < 0.05",
               rationale="the fidelity gain transfers to end-task reasoning quality"),
    "H3": dict(metric="ood_rel_drop", comparison=("coformer", "frozen"), direction="less",
               decision="supported if CoFormer ood_rel_drop < frozen AND Holm p < 0.05",
               rationale="co-training reduces RELATIVE ID->OOD degradation"),
    "H4": dict(metric="refine_steps", comparison=("coformer", "frozen"), direction="less",
               decision="supported if CoFormer needs fewer refine steps AND Holm p < 0.05",
               rationale="gradient-guided translation converges faster"),
    "H5": dict(metric="error_wrong_translation", comparison=("coformer", "frozen"),
               direction="less", decision="supported if wrong-translation share drops AND Holm p<0.05",
               rationale="co-training shifts errors away from translation"),
}

# --- factorial hypotheses: the contrasts the 2x2 makes available -----------
PREREGISTRATION.update({
    "H7": dict(family="exploratory", metric="faith_excess_id",
               comparison=("no_stgs", "coformer"), direction="greater",
               decision="supported if mean diff > 0 AND Holm-adjusted p < 0.05",
               rationale="MAIN EFFECT OF THE BRIDGE at a routed solver "
                         "gradient: replacing straight-through with the soft "
                         "relaxation improves density-controlled fidelity"),
    "H8": dict(family="exploratory", metric="faith_excess_id",
               comparison=("soft_frozen", "frozen"), direction="greater",
               decision="supported if mean diff > 0 AND Holm-adjusted p < 0.05",
               rationale="MAIN EFFECT OF THE BRIDGE at a DETACHED solver "
                         "gradient. Taken with H7 this separates the bridge "
                         "from co-training: if H8 holds as strongly as H7, the "
                         "bridge alone accounts for the effect and the solver "
                         "gradient contributes nothing"),
    "H9": dict(family="exploratory", metric="faith_excess_id",
               comparison=("no_stgs", "soft_frozen"), direction="greater",
               decision="supported if mean diff > 0 AND Holm-adjusted p < 0.05",
               rationale="EFFECT OF THE SOLVER GRADIENT GIVEN A USABLE BRIDGE. "
                         "This is the co-training claim of H1/H2 re-tested in "
                         "the regime where the translator can actually learn: "
                         "H1 tests it under a bridge measured to be "
                         "anti-parallel to the descent direction, H9 where "
                         "that confound is absent"),
})

# REVISION v2 (M5): H7-H9 were added together with the soft-bridge control cells.
# Without an external time stamp they are analysed as EXPLORATORY, outside the Holm family.

# ---------------------------------------------------------------------------
# Pre-specified plan II (fixed BEFORE the v4 stages are run).
#
# E-family: confirmatory replication of the exploratory estimator finding of the
# first run, on SEEDS THAT WERE NOT USED THERE (12-23). One-sided predictions,
# Holm-corrected within the family, minimum effect 0.03 on the end-task metric.
#
# P-family: the partial-supervision curve. For each bridge and each fraction f of
# instances that carry per-slot supervision, does routing the solver gradient
# improve density-controlled fidelity? This is the regime the first run could not
# reach, because at f = 1 every condition saturates.
PREREGISTRATION_II = {
    "E1": dict(metric="csat_norm_paraphrase", a="grs_routed", b="coformer",
               direction="greater", min_effect=0.03,
               claim="Gumbel-Rao improves paraphrase retention over straight-through (routed)"),
    "E2": dict(metric="csat_norm_paraphrase", a="grs_frozen", b="frozen",
               direction="greater", min_effect=0.03,
               claim="Gumbel-Rao improves paraphrase retention over straight-through (detached)"),
    "E3": dict(metric="ood_rel_drop", a="grs_routed", b="coformer",
               direction="less", min_effect=0.03,
               claim="Gumbel-Rao reduces the relative out-of-distribution drop"),
}

PREREGISTRATION_PARTIAL = {
    "P_ST": dict(metric="faith_excess_id", a="coformer", b="frozen",
                 direction="greater", min_effect=0.03,
                 claim="Under partial supervision the solver gradient helps (straight-through bridge)"),
    "P_SOFT": dict(metric="faith_excess_id", a="no_stgs", b="soft_frozen",
                   direction="greater", min_effect=0.03,
                   claim="Under partial supervision the solver gradient helps (soft bridge)"),
}

FACTORIAL_SPEC = dict(
    name="bridge x solver-gradient factorial",
    cells={("straight_through", "detached"): "frozen",
           ("straight_through", "routed"):   "coformer",
           ("soft",             "detached"): "soft_frozen",
           ("soft",             "routed"):   "no_stgs"},
    decision="report both main effects and the interaction with CIs; an "
             "attribution claim requires one main effect to dominate",
    rationale="A 2x2 is the minimum design that can attribute an effect to one "
              "of two simultaneously varied mechanisms. Without the "
              "soft/detached cell, `no_stgs` vs `frozen` confounds the bridge "
              "with the solver gradient and supports no causal claim.")

H6_SPEC = dict(
    name="ablation dominance",
    claims=[("coformer", "no_faith"), ("coformer", "no_stgs"),
            ("coformer", "frozen"), ("coformer", "soft_frozen"),
            ("coformer", "soft_no_faith")],
    decision="supported if coformer >= each ablation on BOTH faith_excess_id and csat_norm_id "
             "for a per-seed majority (share > 0.5)",
    rationale="STGS makes the logic-layer gradient reachable; L_faith directs it. Removing "
              "either should not help. No total order between the two removals is claimed.")

NOTES_PREREG = {
    "degenerate_solutions": (
        "Both loss terms have a degenerate optimum "
        "that a weak translator reaches before it learns anything:\n"
        "  * L_faith: with clause_len/n_vars sparsity the 'absent' class holds ~70% of slots, so "
        "    emitting an EMPTY clause matrix is a strong local optimum. In an earlier run `frozen` and "
        "    `no_stgs` sat at faith F1 = 0.0000 with ZERO variance across 12 seeds -- the exact "
        "    score of a constant all-absent predictor.\n"
        "  * L_task: emitting a MAXIMALLY DENSE clause matrix makes every clause trivially "
        "    satisfiable, so satisfaction ~ 1 at no cost. In an earlier run `no_faith` sat at 0.2310; a "
        "    constant all-positive predictor scores 0.2315 on the same data.\n"
        "Mitigations now in place: (a) the faithfulness NLL is CLASS-BALANCED by inverse "
        "frequency, so the absent class can no longer dominate; (b) a density prior penalises "
        "clause matrices whose literal density departs from the known gold density; (c) every run "
        "reports pred_density / pred_entropy and carries a `degenerate` flag; (d) Cell 11 and the "
        "completion alert REFUSE to report a hypothesis as supported when either side is "
        "degenerate. A degenerate baseline is not a baseline -- that is the same class of error "
        "as the untrained baseline fixed in earlier development."),
    "h1_amendment": (
        "H1 originally compared raw clause-F1. "
        "That metric is confounded by output density: for an information-free clause matrix at "
        "literal density d against gold density g, expected F1 = g*d/(g+d), which is strictly "
        "increasing in d. Since L_task = -log(satisfaction) is minimised by DENSE clauses, every "
        "condition carrying the solver gradient drifts to a higher density than `frozen`, and raw "
        "F1 rewards it for that alone. The confound follows analytically from the metric's "
        "definition -- it is not a pattern discovered in the data -- so amending H1 is a design "
        "correction, not selective reporting. H1 is now stated on `faith_excess_id`, raw F1 minus "
        "the closed-form noise baseline at the density the model actually emitted. Both raw F1 and "
        "excess are reported in every table so the amendment is auditable.\n"
        "For the record, on the 12-seed run that exposed this: raw F1 gave coformer 0.139 vs "
        "frozen 0.111 (Cohen's d = 1.17), while the closed form predicts 0.141 and 0.114 for pure "
        "noise at those densities. Excess was -0.002 and -0.003. The apparent effect was entirely "
        "density."),
    "ablation_semantics": (
        "R13 anchored the solver objective to the GOLD clauses, which "
        "changes what `no_faith` isolates. Before, its only signal was self-satisfaction, which "
        "consumed no gold information at all; now the anchored task loss reads the gold clauses "
        "too, through the solver rather than per slot. So `no_faith` is no longer 'the condition "
        "with no faithfulness supervision' -- it is 'the condition whose gold supervision arrives "
        "ONLY through the solver, never per slot'. That is a cleaner ablation for the mechanism "
        "under test, but it is a different one, and the H6 rationale must be read accordingly: "
        "coformer vs no_faith now contrasts per-slot supervision ON TOP OF the solver path "
        "against the solver path alone."),
    "endtask_metric": (
        "Primary end-task metric is csat_norm (random-normalised gold-clause "
        "satisfaction); exact full-solve accuracy is secondary and reported alongside. A random "
        "assignment scores csat_norm = 0 by construction, so the metric cannot be gamed by a "
        "degenerate CNF."),
    "role_of_the_relaxation": (
        "The soft-MaxSAT relaxation is a TRAINING SIGNAL, not an inference oracle. Cell U measures "
        "that it essentially never signals unsatisfiability on Z3-certified UNSAT instances, and a "
        "three-way entailment decision requires exactly such a proof for the 'False' class. Every "
        "reported DECISION -- CoFormer's and every baseline's -- is produced by a sound Z3 check."),
    "identical_inference": (
        "CoFormer and all solver baselines share ONE inference procedure: emit a CNF, then decide "
        "entailment with two Z3 SAT queries. No system predicts labels with a learned head, so "
        "none can exploit label priors."),
    "synthetic_task_scope": (
        "Instances are verbalised as templated English over a shared word-level lexicon, NOT as a "
        "one-to-one token encoding of the CNF. Paraphrase-OOD uses held-out templates built "
        "exclusively from words seen in training (asserted programmatically)."),
    "instance_hardness": (
        "Uniform random k-SAT at clause/variable ratio `sat_ratio`, rejection-sampled for "
        "satisfiability with a complete DPLL solver. The ratio is reported with every result and "
        "swept in Cell 9b."),
    "proofwriter_gold_cnf": (
        "ProofWriter gold CNF is derived FROM THE CONTEXT TEXT (facts + universally "
        "quantified rules grounded over the entities), because the reachable distribution exposes "
        "no structured triples/rules. Derivation is SELF-VALIDATING: an example enters the "
        "gold-CNF ('full') regime only if Z3 entailment over the derived CNF reproduces the gold "
        "label. Offline on the 600-example dev set this yields 313 eligible examples (52%) at "
        "96.2% label agreement; the rest run on the proxy regime. The eligible fraction is "
        "reported at run time, never assumed."),
    "scale_ood_operationalization": (
        "Scale-OOD evaluates instances wider than training. The translator emits a fixed "
        "n_vars-wide clause window; at eval we SLOT-TILE it across the full width and score at "
        "FULL width. It does NOT crop gold to the model width."),
    "acc_id_floor": (
        "ood_rel_drop is NaN when the ID anchor is below CSAT_FLOOR (relative retention is "
        "undefined when the anchor collapses) and when it would be negative. Dropped pairwise."),
    "small_n_caveat": (
        "The paired bootstrap resamples SEEDS. Achieved power is computed post hoc in Cell 3c; "
        "real-data contrasts additionally use a CLUSTER bootstrap over (seed, example)."),
}

ALPHA = 0.05
MIN_EFFECT = 0.03
MIN_EFFECT_REAL = 0.05
ACC_ID_FLOOR = 0.05     # legacy exact-solve floor, retained for the secondary metric
CSAT_FLOOR = 0.05       # primary anchor floor for the H3 relative-retention guard

print("Pre-registered hypotheses (locked before training):")
for h, d in PREREGISTRATION.items():
    assert isinstance(d, dict), f"PREREGISTRATION[{h}] must be a dict"
    print(f"  {h}: {d['metric']} | {d['comparison'][0]} vs {d['comparison'][1]} | {d['direction']}")
print(f"  H6 (outside the Holm family): {H6_SPEC['name']}")
print(f"\nFamily-wise alpha = {ALPHA}; minimum meaningful effect = {MIN_EFFECT} (synthetic) / "
      f"{MIN_EFFECT_REAL} (real)")
print("\n" + NOTES_PREREG["ablation_semantics"])
print("\n" + NOTES_PREREG["h1_amendment"])
print("\n" + NOTES_PREREG["degenerate_solutions"])
print("\n" + NOTES_PREREG["endtask_metric"])


## Cell 3c — A-priori power analysis: how many seeds do we need?
Before running, we estimate the number of seeds required to detect a target effect size with
adequate power, so the seed count is justified rather than arbitrary. We use a paired-design
power calculation (the seed-paired comparison of CoFormer vs frozen). If the configured seed count
is below the recommended n, the cell prints a clear recommendation to increase `CFG.seeds`. The seed count is therefore justified rather than arbitrary.

In [ ]:
# === Cell 3c: a-priori AND post-hoc power analysis for the paired seed comparison ===
from statsmodels.stats.power import TTestPower

def recommend_seeds(effect_size_d=0.8, power=0.80, alpha=ALPHA):
    """Paired t-test: minimum number of seeds to detect Cohen's d at the given power."""
    return int(np.ceil(TTestPower().solve_power(effect_size=effect_size_d, power=power,
                                                alpha=alpha, alternative="larger")))

def achieved_power(n_seeds, effect_size_d=0.8, alpha=ALPHA):
    """R2-FIX-A7: post-hoc achieved power, so 'is N seeds enough?' gets a NUMBER, not a shrug."""
    if n_seeds < 2:
        return float("nan")
    return float(TTestPower().power(effect_size=effect_size_d, nobs=n_seeds,
                                    alpha=alpha, alternative="larger"))

print("A-priori requirement (paired design, power=0.80):")
for d in (0.5, 0.8, 1.2):
    print(f"  Cohen's d={d}: need >= {recommend_seeds(d)} seeds")

# The seed cap is the binding constraint on statistical conclusion validity, so
# it is set from the a-priori requirement rather than from convenience. At 12
# seeds the design reaches power 0.80 for d=0.8; below 8 it cannot resolve
# anything but very large effects, and the nulls it produces are uninformative.
MAX_SEEDS = 12 if CFG.FULL_SCALE else 12
n_needed = recommend_seeds(effect_size_d=0.8)
current = len(CFG.seeds)
if getattr(CFG, "RESPECT_POWER", True) and current < n_needed:
    n_use = min(n_needed, MAX_SEEDS)
    CFG.seeds = tuple(range(n_use))
    print(f"\nPower applied: seeds {current} -> {n_use} (d=0.8, power=0.80, alpha={ALPHA}).")
    if n_use < n_needed:
        print(f"  NOTE: capped at MAX_SEEDS={MAX_SEEDS}; full power needs {n_needed}.")
else:
    print(f"\nSeeds={current}; a-priori requirement >= {n_needed} "
          f"(RESPECT_POWER={getattr(CFG,'RESPECT_POWER', True)}).")

POWER_TABLE = {d: achieved_power(len(CFG.seeds), d) for d in (0.5, 0.8, 1.2, 2.0)}
print(f"\nACHIEVED power with n={len(CFG.seeds)} seeds (alpha={ALPHA}, one-sided paired):")
for d, p in POWER_TABLE.items():
    flag = "adequate" if p >= 0.80 else ("marginal" if p >= 0.5 else "UNDERPOWERED")
    print(f"  d={d}: power = {p:.2f}  ({flag})")
print("Report this table in the paper. A non-significant result at d=0.5 with power "
      f"{POWER_TABLE[0.5]:.2f} is an absence of evidence, not evidence of absence.")

# The seed-paired bootstrap is not the only estimator available. Where a metric
# is computed per evaluation example, cluster_bootstrap_paired resamples seeds
# AND examples, so an evaluation set of n_eval examples contributes to the
# interval instead of the design being capped at len(seeds) observations. Both
# are reported: agreement means the finding survives the seed budget,
# disagreement means the seed-paired test is underpowered and is described so.
_p80 = POWER_TABLE[0.8]
if _p80 < 0.80:
    print(f"\nNOTE: at {len(CFG.seeds)} seeds the seed-paired test reaches power "
          f"{_p80:.2f} for d=0.8. Every confirmatory contrast is therefore ALSO "
          f"reported with an example-level cluster bootstrap, which does not "
          f"depend on the seed count for its resolution.")

N_DIAG_RUNS = (4 + 4 + 6 + 1 + 6 + 2) if RUN_EXTRA_DIAGNOSTICS else 0   # lambda/density/ratio/neg/sens/cases
N_MATRIX_RUNS = 4 * len(CFG.seeds)
print(f"\nCOST: {N_MATRIX_RUNS} matrix runs (4 conditions x {len(CFG.seeds)} seeds)"
      + (f" + {N_DIAG_RUNS} diagnostic runs (RUN_EXTRA_DIAGNOSTICS=True)"
         if RUN_EXTRA_DIAGNOSTICS else " (diagnostics OFF)")
      + f" = {N_MATRIX_RUNS + N_DIAG_RUNS} trainings.")
print(f"      Cell 10 projects the wall clock from a MEASURED pre-flight run before the matrix "
      f"starts. The matrix is checkpointed per (condition, seed), so it resumes across Colab "
      f"sessions; the diagnostics in Cell 9b are not, so run them in a session you can finish.")
if RUN_EXTRA_DIAGNOSTICS:
    print(f"      To trim: RUN_EXTRA_DIAGNOSTICS=False drops {N_DIAG_RUNS} runs and only the "
          f"sweep/sensitivity artifacts. Main tables, figures and all H1-H6 tests are unaffected.")
print("NOTE:", NOTES_PREREG["small_n_caveat"])

# --- whole-study cost projection, printed BEFORE anything is trained -------
# The per-run cost is calibrated from a measured A100-40 run at full scale and
# scaled by the optimiser-step count of the configuration in force. It is an
# estimate, not a measurement; Cell 10 replaces it with a measured projection
# from the pre-flight run before the matrix starts.
_N_COND = 6
_REF_SECS_PER_STEP = 5096.0 / (40 * (20000 // 32))     # measured, probe B
def _steps_for(probe):
    _c = Config(**{**dataclasses.asdict(CFG), "probe": probe}).scaled()
    return _c.epochs * ((_c.n_train + _c.batch_size - 1) // _c.batch_size)

_per_probe = {p: _steps_for(p) * _REF_SECS_PER_STEP for p in PROBES}
_n_seeds = len(CFG.seeds)
_matrix_h = sum(v for v in _per_probe.values()) * _N_COND * _n_seeds / 3600
_diag_h = 14 * max(_per_probe.values()) / 3600 if RUN_EXTRA_DIAGNOSTICS else 0.0
_total_h = _matrix_h + _diag_h

print("\n" + "=" * 72)
print("STUDY COST PROJECTION (read this before committing GPU time)")
for p, secs in _per_probe.items():
    print(f"  probe {p}: ~{secs/60:.0f} min per run x {_N_COND} conditions "
          f"x {_n_seeds} seeds = {secs*_N_COND*_n_seeds/3600:.0f} GPU-h")
if _diag_h:
    print(f"  diagnostics (sweeps): ~{_diag_h:.0f} GPU-h")
print(f"  TOTAL: ~{_total_h:.0f} GPU-h "
      f"= ~{_total_h/SESSION_BUDGET_HOURS:.0f} sessions of "
      f"{SESSION_BUDGET_HOURS:.0f} h")
_phase1_h = sum(_per_probe.values()) * 4 * min(4, _n_seeds) / 3600
print(f"  Phase 1 alone (the 4-cell factorial at {min(4, _n_seeds)} seeds, both "
      f"probes): ~{_phase1_h:.0f} GPU-h = ~{_phase1_h/SESSION_BUDGET_HOURS:.0f} "
      f"session(s). This is the smallest run that answers the attribution "
      f"question, and the ordering in Cell 10 completes it first.")
if _total_h > 4 * SESSION_BUDGET_HOURS:
    print(f"\n  This plan spans many sessions. Everything is checkpointed, so "
          f"re-running the notebook resumes and NOTHING needs editing. To "
          f"finish sooner, reduce CFG.seeds (currently {_n_seeds}) or set "
          f"PROBES = ('{list(PROBES)[0]}',) to run one instance family.")
print("=" * 72)

# --- shared statistics helpers -----------------------------------------------------------
def paired_bootstrap(a, b, n_boot=10000, seed=0):
    """Bootstrap the mean paired difference (a - b) by resampling SEEDS."""
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a, float), np.asarray(b, float)
    diff = a - b
    boots = np.array([rng.choice(diff, size=len(diff), replace=True).mean()
                      for _ in range(n_boot)])
    return diff.mean(), min(1.0, 2 * min((boots <= 0).mean(), (boots >= 0).mean()))

def cohens_d_paired(a, b):
    """Paired Cohen's d. a constant difference across seeds used to return +/-inf,
    which then printed as `-inf` in the confirmatory table -- an artifact of a tiny seed budget,
    not an infinitely large effect. Zero variance now yields NaN, which the reporting layer
    renders as 'undefined (zero across-seed variance)'."""
    diff = np.asarray(a, float) - np.asarray(b, float)
    if len(diff) < 2:
        return float("nan")
    sd = diff.std(ddof=1)
    if sd < 1e-9:
        return 0.0 if abs(diff.mean()) < 1e-9 else float("nan")
    return diff.mean() / sd

def boot_ci_d(a, b, n_boot=5000, seed=0):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a, float), np.asarray(b, float)
    ds = [d for _ in range(n_boot)
          for d in [cohens_d_paired(*(lambda i: (a[i], b[i]))(rng.integers(0, len(a), len(a))))]
          if np.isfinite(d)]
    if not ds:
        return float("nan"), float("nan")
    return np.percentile(ds, [2.5, 97.5])


# --- validity guards on the estimator itself ------------------------------
def contrast_testability(a, b, alpha=ALPHA):
    """Can this contrast reach `alpha` AT ALL, given n and the tie structure?

    Two failure modes are invisible in a p-value and must be reported before
    it is interpreted:

      * VACUOUS -- the paired difference vector is identically zero. The
        bootstrap then returns p = 1.000 by construction and no data could
        have produced another value. Such a contrast is not a null result; it
        is not a test, and including it in a Holm family inflates the family
        size and weakens the correction applied to the contrasts that ARE
        testable.
      * FLOORED -- ties among the differences bound the attainable p from
        below. Resampling n differences admits n**n equally likely draws; if
        k of them are <= 0, the all-non-positive resample alone carries mass
        (k/n)**n, so p >= 2*(k/n)**n. With n=4 and two ties that is 0.125,
        above alpha before any correction.

    Returns (status, p_floor, note).
    """
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[~np.isnan(d)]
    n = len(d)
    if n < 2:
        return "insufficient", 1.0, f"n={n}"
    if np.all(np.abs(d) < 1e-12):
        return "vacuous", 1.0, ("identically zero difference vector: the two "
                                "systems produced the same output on every "
                                "unit, so the bootstrap cannot reject")
    k_le = int((d <= 0).sum()); k_ge = int((d >= 0).sum())
    p_floor = float(min(1.0, 2 * min((k_le / n) ** n, (k_ge / n) ** n)))
    if p_floor > alpha:
        return "floored", p_floor, (f"{max(k_le, k_ge)}/{n} differences share a "
                                    f"sign or are tied, so the attainable "
                                    f"p is bounded below by {p_floor:.3f} "
                                    f"> alpha={alpha}")
    return "testable", p_floor, ""


def cluster_bootstrap_paired(per_seed_a, per_seed_b, n_boot=10000, seed=0):
    """Two-stage bootstrap over (seed, unit): resample seeds, then units within
    each drawn seed.

    This is the estimator that recovers power without buying more seeds. The
    seed-paired bootstrap has n = len(seeds) effective observations; this one
    resamples the EVALUATION UNITS as well, so an evaluation set of a few
    thousand examples contributes to the interval. It does not pretend that
    predictions from one trained model are independent draws -- the outer
    stage still resamples whole seeds -- so it is conservative with respect to
    between-run variance while being far more informative than n=4 alone.

    per_seed_*: dict seed -> 1-D array of per-unit scores, paired within seed.
    Returns (observed_diff, lo, hi, p).
    """
    rng = np.random.default_rng(seed)
    seeds = [s for s in per_seed_a
             if s in per_seed_b and len(per_seed_a[s]) and len(per_seed_b[s])]
    if not seeds:
        return (float("nan"),) * 4
    obs = float(np.mean([np.mean(per_seed_a[s]) - np.mean(per_seed_b[s])
                         for s in seeds]))
    draws = np.empty(n_boot)
    for i in range(n_boot):
        picked = rng.choice(len(seeds), len(seeds), replace=True)
        vals = []
        for j in picked:
            s = seeds[j]
            x = np.asarray(per_seed_a[s], float)
            y = np.asarray(per_seed_b[s], float)
            m = min(len(x), len(y))
            idx = rng.integers(0, m, m)
            vals.append(x[idx].mean() - y[idx].mean())
        draws[i] = np.mean(vals)
    lo, hi = np.percentile(draws, [2.5, 97.5])
    p = 2 * min((draws <= 0).mean(), (draws >= 0).mean())
    return obs, float(lo), float(hi), float(min(1.0, p))


print("Validity guards ready (contrast_testability / cluster_bootstrap_paired).")
print("  A contrast is reported as VACUOUS or FLOORED -- and excluded from the")
print("  Holm family -- BEFORE its p-value is interpreted.")

print("\nShared statistics helpers ready (paired_bootstrap / cohens_d_paired / boot_ci_d).")

_FP_FIELDS_SYNTH = ("n_vars", "sat_ratio", "clause_len", "use_planted", "n_train", "n_eval",
                    "ood_n_vars", "epochs", "batch_size", "lr", "d_model", "n_layers", "n_heads",
                    "refine_steps", "gumbel_tau_start", "gumbel_tau_end", "lambda_faith",
                    "lambda_density")
_FP_FIELDS_REAL = ("lr", "d_model", "refine_steps", "gumbel_tau_end", "lambda_faith",
                   "lambda_density", "partc_seeds", "partc_epochs")

# REVISION v2: bumping LOSS_VERSION invalidates EVERY cached result, calibration, sweep and
# real-data checkpoint, so nothing produced with the v1 faithfulness loss can be resumed into.
LOSS_VERSION = "v2-faith-on-logits"

def config_fingerprint(schema=(), extra="", fields=None):
    """Hash of (result schema, experiment config). Any change that alters the MEANING of a stored
    result row must change this, so a checkpoint from an older revision can never be resumed into.

    this lived in Cell 10 while Cell C5 called it -- the same cross-part dependency
    that removed for the bootstrap helpers, and it broke Part C the same way. Utilities
    shared by both parts belong in this cell, which always runs.
    """
    import hashlib as _h
    key = "|".join(sorted(schema)) + "||" + "|".join(
        f"{k}={getattr(CFG, k, None)}"
        for k in (fields if fields is not None else _FP_FIELDS_SYNTH)) + "||" + extra \
        + "||" + LOSS_VERSION
    return _h.md5(key.encode()).hexdigest()[:10]

print("Shared checkpoint fingerprint helper ready (config_fingerprint).")


## Cell 4 — Google Drive mount, persistent paths, and secure email alerts
Mounts Drive and creates a project folder that holds the **dataset cache**, **checkpoints**, and
**publication outputs** so nothing is recomputed or re-downloaded across sessions.

Email alerts use SMTP credentials read from **Colab Secrets** (`Tools → Secrets` / the key icon),
never hard-coded. Add three secrets: `SMTP_USER` (your email), `SMTP_APP_PASSWORD` (a Gmail
*App Password*, not your login password), and optionally `ALERT_TO` (recipient; defaults to
`SMTP_USER`). If secrets are missing, alerts auto-disable and the run continues normally — so the
notebook still works without configuring email.

In [ ]:
# === Cell 4: storage paths (Drive or local) + secure email alerts ===
PROJECT = "CoFormer"

# REVISION v5: Colab detection no longer depends on a successful Drive mount. The previous
# version set IN_COLAB only inside the mount `try`, so a failed mount made the run metadata
# claim the session was not on Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DRIVE_MOUNTED = False
_mount_error = None
if not CFG.LOCAL_MODE and IN_COLAB:
    from google.colab import drive as _gdrive
    for _attempt in range(1, DRIVE_MOUNT_ATTEMPTS + 1):
        try:
            _gdrive.mount("/content/drive", force_remount=(_attempt > 1))
            DRIVE_MOUNTED = True
            break
        except Exception as e:
            _mount_error = e
            print(f"Drive mount attempt {_attempt}/{DRIVE_MOUNT_ATTEMPTS} failed: {e}")
            if _attempt < DRIVE_MOUNT_ATTEMPTS:
                time.sleep(5 * _attempt)

if DRIVE_MOUNTED:
    BASE = f"/content/drive/MyDrive/{PROJECT}"
elif CFG.LOCAL_MODE or ALLOW_LOCAL_FALLBACK or not IN_COLAB:
    BASE = os.path.abspath(f"./{PROJECT}_run")
    if not CFG.LOCAL_MODE and IN_COLAB:
        print("\nWARNING: running on the ephemeral local disk because Drive is unavailable.")
        print("         Nothing here survives the end of the session. Restore your state with")
        print("         restore_state(...) and call snapshot_state() before the session ends.")
else:
    raise RuntimeError(
        "Google Drive could not be mounted after "
        f"{DRIVE_MOUNT_ATTEMPTS} attempts ({_mount_error}).\n"
        "The run was stopped on purpose: continuing would retrain every stage in an empty\n"
        "ephemeral directory and lose the result when the session ends.\n"
        "Options:\n"
        "  1. Fix the mount (allow third-party cookies for [*.]google.com, sign out of all\n"
        "     Google accounts except one, then Runtime > Disconnect and delete runtime).\n"
        "  2. Work without Drive: set ALLOW_LOCAL_FALLBACK = True in Cell 3, then use\n"
        "     restore_state(<zip path or URL>) at the start and snapshot_state() at the end\n"
        "     of every session."
    )

PERSISTENT = DRIVE_MOUNTED
DATA_DIR = os.path.join(BASE, "data_cache")
CKPT_DIR = os.path.join(BASE, "checkpoints")
OUT_DIR  = os.path.join(BASE, "outputs")
for d in (DATA_DIR, CKPT_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

# REVISION v3: atomic checkpoint writes. A Colab disconnect during a write to Drive would
# otherwise leave a truncated CSV/JSON that crashes the next session on resume.
def atomic_csv(frame, path, **kw):
    _tmp = path + ".tmp"
    frame.to_csv(_tmp, **kw)
    os.replace(_tmp, path)

def atomic_json(obj, path, **kw):
    _tmp = path + ".tmp"
    with open(_tmp, "w") as _fh:
        json.dump(obj, _fh, **kw)
    os.replace(_tmp, path)
print("Project dirs ready under:", BASE, "| Colab:", IN_COLAB, "| Drive:", DRIVE_MOUNTED)


# ---- state restore / snapshot, for sessions that run without Drive (REVISION v5) ----
def restore_state(source, dest=None):
    """Restore a state archive (local zip path or URL) into the run directory.

    Accepts the zip produced by snapshot_state() or a Drive download of the project
    folder; the CoFormer directory is located inside the archive automatically.
    """
    import zipfile, shutil, tempfile, urllib.request
    dest = dest or BASE
    local = source
    if str(source).startswith(("http://", "https://")):
        local = os.path.join(tempfile.gettempdir(), "coformer_state.zip")
        print(f"Downloading {source} ...")
        urllib.request.urlretrieve(source, local)
    tmp = tempfile.mkdtemp()
    with zipfile.ZipFile(local) as zf:
        zf.extractall(tmp)
    src = None
    for root, dirs, _files in os.walk(tmp):
        if os.path.basename(root) in (PROJECT, f"{PROJECT}_run") and "checkpoints" in dirs:
            src = root
            break
    if src is None:
        raise FileNotFoundError("no CoFormer directory with a checkpoints/ folder in the archive")
    for sub in ("checkpoints", "outputs", "data_cache"):
        s_dir = os.path.join(src, sub)
        if os.path.isdir(s_dir):
            shutil.copytree(s_dir, os.path.join(dest, sub), dirs_exist_ok=True)
    shutil.rmtree(tmp, ignore_errors=True)
    n_ck = len(os.listdir(os.path.join(dest, "checkpoints")))
    print(f"Restored into {dest}: {n_ck} checkpoint entries; "
          f"results.csv={os.path.exists(os.path.join(dest, 'checkpoints', 'results.csv'))}")
    return dest


def snapshot_state(download=True):
    """Zip the run directory so a Drive-less session can be resumed later."""
    import shutil
    stamp = time.strftime("%Y%m%d-%H%M")
    out = f"/content/{PROJECT}_state_{stamp}" if os.path.isdir("/content") else \
          os.path.abspath(f"./{PROJECT}_state_{stamp}")
    path = shutil.make_archive(out, "zip", os.path.dirname(BASE), os.path.basename(BASE))
    size = os.path.getsize(path) / 1e6
    print(f"State snapshot: {path} ({size:.0f} MB)")
    if download and IN_COLAB:
        try:
            from google.colab import files as _files
            _files.download(path)
        except Exception as e:
            print(f"  automatic download failed ({e}); download it from the file browser")
    return path


if not PERSISTENT and STATE_ARCHIVE and not os.path.exists(
        os.path.join(CKPT_DIR, "results.csv")):
    restore_state(STATE_ARCHIVE)

# Persist the exact environment next to the results, so the artifact is self-describing.
# REVISION v12: the metadata of the reported run is never overwritten; a later session writes
# `<name>.session-<UTC>.<ext>` and keeps writing to that file for the rest of the session.
_META_SESSION = {}
def _meta_path(fname):
    if fname in _META_SESSION:
        return _META_SESSION[fname]
    p = os.path.join(OUT_DIR, fname)
    if os.path.exists(p) and not globals().get("RECOMPUTE_REPORTED_DIAGNOSTICS", False):
        stem, ext = os.path.splitext(fname)
        p = os.path.join(OUT_DIR, f"{stem}.session-{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}{ext}")
    _META_SESSION[fname] = p
    return p

try:
    with open(_meta_path("requirements.txt"), "w") as fh:
        fh.write("\n".join(f"{k}=={v}" for k, v in _versions.items()) + "\n")
    with open(_meta_path("run_metadata.json"), "w") as fh:
        json.dump({"device": str(DEVICE), "gpu": GPU_NAME, "platform": platform.platform(),
                   "versions": _versions, "config": dataclasses.asdict(CFG),
                   "n_clauses": CFG.n_clauses, "colab": IN_COLAB}, fh, indent=2, default=str)
    print("Wrote requirements.txt + run_metadata.json to", OUT_DIR)
except Exception as e:
    print("Could not write environment metadata:", e)

# ---- secure email alerts (optional; absence never blocks a run) ----
import smtplib, ssl
from email.message import EmailMessage
SECRET_DIAG = {}

def _get_secret(name, default=None):
    """Read a Colab secret, then the environment. Records WHY a lookup failed in SECRET_DIAG."""
    try:
        from google.colab import userdata
    except Exception:
        v = os.environ.get(name)
        SECRET_DIAG[name] = "found in environment" if v else "not in environment (not in Colab)"
        return v if v else default
    try:
        v = userdata.get(name)
        if v:
            SECRET_DIAG[name] = "ok (Colab secret)"
            return v
        SECRET_DIAG[name] = "Colab secret exists but is EMPTY"
    except Exception as e:
        _t = type(e).__name__
        if "NotebookAccess" in _t:
            SECRET_DIAG[name] = ("secret exists but NOTEBOOK ACCESS IS OFF -- open the key icon "
                                 "in the left sidebar and enable the toggle for this notebook")
        elif "SecretNotFound" in _t:
            SECRET_DIAG[name] = ("no Colab secret with this exact name (names are CASE-SENSITIVE; "
                                 "`hf_token` and `HUGGINGFACE_TOKEN` will not be found)")
        else:
            SECRET_DIAG[name] = f"Colab secret lookup failed: {_t}: {e}"
    v = os.environ.get(name)
    if v:
        SECRET_DIAG[name] += " -- but found in the environment instead"
        return v
    return default

SMTP_USER = _get_secret("GMAIL_USER") or _get_secret("SMTP_USER")
SMTP_PASS = _get_secret("GMAIL_APP_PASSWORD") or _get_secret("SMTP_APP_PASSWORD")
ALERT_TO  = _get_secret("GMAIL_TO") or _get_secret("ALERT_TO") or SMTP_USER
SMTP_HOST = _get_secret("SMTP_HOST", "smtp.gmail.com")
SMTP_PORT = int(_get_secret("SMTP_PORT", "465"))
EMAIL_ENABLED = bool(SMTP_USER and SMTP_PASS and ALERT_TO)

_hf = _get_secret("HF_TOKEN")
print("\nSecret visibility:")
for _n in ("HF_TOKEN", "GMAIL_USER", "GMAIL_APP_PASSWORD"):
    print(f"  {_n:20s} {SECRET_DIAG.get(_n, 'not looked up')}")
if not _hf and CFG.USE_LLM_BACKBONE:
    print("\n" + "!" * 72)
    print("HF_TOKEN is not visible. This is only a problem for GATED backbones: the declared")
    print("backbone (Qwen2.5-7B-Instruct) is public, so Part C still runs without a token.")
    print(f"  lookup result: {SECRET_DIAG.get('HF_TOKEN', 'not looked up')}")
    print("  MOST LIKELY CAUSE: Colab grants secret access PER NOTEBOOK FILE. If you uploaded a")
    print("  new revision of this notebook, the grant did not come with it. Open the key icon in")
    print("  the left sidebar and re-enable the toggle for HF_TOKEN on THIS notebook.")
    print("  Other causes: the secret is named differently (case-sensitive), or its value has a")
    print("  trailing space. Model-page approval alone is not sufficient.")
    print("  Fix it now -- it takes seconds, and Part C is otherwise skipped after Part A runs.")
    print("  To run Part A only and silence this, set CFG.USE_LLM_BACKBONE = False in Cell 3.")
    print("!" * 72)
elif _hf and CFG.USE_LLM_BACKBONE:
    print("  HF_TOKEN is visible; Part C can run.")
print("Email alerts:", ("ENABLED -> " + str(ALERT_TO)) if EMAIL_ENABLED
      else "DISABLED (set SMTP_USER / SMTP_APP_PASSWORD to enable)")

def send_alert(subject, body):
    """Send an alert email. Never raises: a failed email must not kill the experiment."""
    if not EMAIL_ENABLED:
        print(f"[alert suppressed] {subject}")
        return
    try:
        msg = EmailMessage()
        msg["Subject"] = f"[{PROJECT}] {subject}"
        msg["From"], msg["To"] = SMTP_USER, ALERT_TO
        msg.set_content(body)
        with smtplib.SMTP_SSL(SMTP_HOST, SMTP_PORT,
                              context=ssl.create_default_context(), timeout=30) as s:
            s.login(SMTP_USER, SMTP_PASS); s.send_message(msg)
        print(f"[alert sent] {subject}")
    except Exception as e:
        print(f"[alert FAILED, continuing] {subject} :: {e}")

# REVISION v3: session-budget guard. Heavy stages call ensure_budget() before each unit of
# work; when the remaining session time is shorter than the unit, the run stops CLEANLY at a
# checkpoint boundary instead of being killed by the runtime in the middle of a training run.
class SessionBudgetReached(RuntimeError):
    pass

def budget_left_s():
    return SESSION_BUDGET_HOURS * 3600 - (time.time() - SESSION_T0)

def ensure_budget(stage, need_s):
    left = budget_left_s()
    if left < need_s:
        msg = (f"Session budget reached before `{stage}` (needs ~{need_s/60:.0f} min, "
               f"{max(left, 0)/60:.0f} min left). All finished work is checkpointed. "
               f"Start a new session and Run All again; it resumes here.")
        if not globals().get("PERSISTENT", True):
            push_state(tag="session end", force=True)
        if not globals().get("PERSISTENT", True) and globals().get("DRIVE_API") is None:
            msg += ("\nThis session is NOT writing to Drive: call snapshot_state() now and "
                    "keep the zip, or the work of this session is lost.")
        print("\n" + "=" * 72 + "\n" + msg + "\n" + "=" * 72)
        send_alert("Session budget reached", msg)
        raise SessionBudgetReached(msg)

send_alert("Run started",
           f"GPU={GPU_NAME}\nFULL_SCALE={CFG.FULL_SCALE}\nseeds={CFG.seeds}\nbase={BASE}")


# ---------------------------------------------------------------------------
# REVISION v9: Drive state synchronisation through the Drive REST API.
#
# Tier 1 is the FUSE mount handled above (PERSISTENT == True; nothing below is needed).
# Tier 2, used when the mount fails, authenticates with google.colab.auth and keeps the run
# state in a Drive folder as a single zip: `checkpoints/` and `outputs/` only, because the
# dataset cache is regenerated deterministically from the stored seeds and would otherwise
# make every upload an 80 MB transfer.
STATE_FILE = f"{PROJECT}_state_latest.zip"
_LAST_STATE_PUSH = [0.0]
DRIVE_API = None          # the Drive service object once authenticated
_DRIVE_STATE_IDS = {}     # cached folder and file ids

def _drive_service():
    """Authenticate with the Drive REST API (a different path from the FUSE mount)."""
    global DRIVE_API
    if DRIVE_API is not None:
        return DRIVE_API
    if PERSISTENT or not USE_DRIVE_API_FALLBACK or not IN_COLAB:
        return None
    try:
        from google.colab import auth as _colab_auth
        _colab_auth.authenticate_user()
        from googleapiclient.discovery import build as _build
        DRIVE_API = _build("drive", "v3", cache_discovery=False)
        print("Drive REST API authenticated (the FUSE mount is unavailable, the API is not).")
    except Exception as e:
        print(f"Drive REST API unavailable as well ({type(e).__name__}: {e}).")
        DRIVE_API = None
    return DRIVE_API

def _drive_folder_id(svc):
    if "folder" in _DRIVE_STATE_IDS:
        return _DRIVE_STATE_IDS["folder"]
    q = (f"name = '{DRIVE_STATE_FOLDER}' and mimeType = 'application/vnd.google-apps.folder' "
         f"and trashed = false")
    found = svc.files().list(q=q, spaces="drive", fields="files(id, name)").execute()
    files_ = found.get("files", [])
    if files_:
        fid = files_[0]["id"]
    else:
        meta = {"name": DRIVE_STATE_FOLDER, "mimeType": "application/vnd.google-apps.folder"}
        fid = svc.files().create(body=meta, fields="id").execute()["id"]
        print(f"Created Drive folder `{DRIVE_STATE_FOLDER}` for the run state.")
    _DRIVE_STATE_IDS["folder"] = fid
    return fid

def _drive_state_file_id(svc, name=None):
    name = name or STATE_FILE
    key = f"file:{name}"
    if key in _DRIVE_STATE_IDS:
        return _DRIVE_STATE_IDS[key]
    fid = _drive_folder_id(svc)
    q = f"name = '{name}' and '{fid}' in parents and trashed = false"
    files_ = svc.files().list(q=q, spaces="drive",
                              fields="files(id, name, modifiedTime)").execute().get("files", [])
    _DRIVE_STATE_IDS[key] = files_[0]["id"] if files_ else None
    return _DRIVE_STATE_IDS[key]

def _zip_state(path):
    import zipfile
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as zf:
        for sub in ("checkpoints", "outputs"):
            for root, _dirs, files_ in os.walk(os.path.join(BASE, sub)):
                for fn in files_:
                    if fn.endswith(".tmp"):
                        continue
                    full = os.path.join(root, fn)
                    zf.write(full, os.path.join(PROJECT, os.path.relpath(full, BASE)))
    return path

def push_state(tag="auto", force=False, quiet=False):
    """Upload `checkpoints/` and `outputs/` to the Drive state folder."""
    if PERSISTENT:
        return None                      # the mount writes to Drive directly
    if not force and (time.time() - _LAST_STATE_PUSH[0]) < DRIVE_STATE_AUTOSAVE_MIN * 60:
        return None
    svc = _drive_service()
    if svc is None:
        return None
    try:
        from googleapiclient.http import MediaFileUpload
        tmp = os.path.join("/tmp", STATE_FILE)
        _zip_state(tmp)
        media = MediaFileUpload(tmp, mimetype="application/zip", resumable=False)
        fid = _drive_state_file_id(svc)
        if fid:
            svc.files().update(fileId=fid, media_body=media).execute()
        else:
            meta = {"name": STATE_FILE, "parents": [_drive_folder_id(svc)]}
            new = svc.files().create(body=meta, media_body=media, fields="id").execute()
            _DRIVE_STATE_IDS[f"file:{STATE_FILE}"] = new["id"]
        _LAST_STATE_PUSH[0] = time.time()
        if not quiet:
            print(f"  [state pushed to Drive/{DRIVE_STATE_FOLDER} "
                  f"({os.path.getsize(tmp)/1e6:.1f} MB, {tag})]")
        return tmp
    except Exception as e:
        print(f"  [state push failed: {type(e).__name__}: {e}]")
        return None

def pull_state(force=False):
    """Restore the run state from the Drive state folder (newest zip if several)."""
    if PERSISTENT:
        return True
    if not force and os.path.exists(os.path.join(CKPT_DIR, "results.csv")):
        print("State already present locally; skipping the pull.")
        return True
    svc = _drive_service()
    if svc is None:
        return False
    try:
        import io
        from googleapiclient.http import MediaIoBaseDownload
        fid_folder = _drive_folder_id(svc)
        files_ = svc.files().list(
            q=f"'{fid_folder}' in parents and trashed = false",
            spaces="drive", orderBy="modifiedTime desc",
            fields="files(id, name, modifiedTime, size)").execute().get("files", [])
        zips = [f for f in files_ if f["name"].lower().endswith(".zip")]
        if not zips:
            print(f"Drive folder `{DRIVE_STATE_FOLDER}` holds no state zip yet.")
            return False
        chosen = next((f for f in zips if f["name"] == STATE_FILE), zips[0])
        local = os.path.join("/tmp", chosen["name"])
        with io.FileIO(local, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, svc.files().get_media(fileId=chosen["id"]))
            done = False
            while not done:
                _status, done = downloader.next_chunk()
        print(f"Pulled `{chosen['name']}` from Drive/{DRIVE_STATE_FOLDER}.")
        restore_state(local)
        return True
    except Exception as e:
        print(f"State pull failed ({type(e).__name__}: {e}).")
        return False

def maybe_push_state(tag="auto"):
    """Called after every finished run; uploads at most every DRIVE_STATE_AUTOSAVE_MIN."""
    if not PERSISTENT:
        push_state(tag=tag, quiet=True)

if not PERSISTENT and USE_DRIVE_API_FALLBACK:
    if _drive_service() is not None:
        print(f"Drive state synchronisation: My Drive/{DRIVE_STATE_FOLDER} "
              f"(autosave every {DRIVE_STATE_AUTOSAVE_MIN} min)")
        pull_state()

# Safety: never start a 100+ GPU-hour recomputation by accident.
if (not PERSISTENT and not os.path.exists(os.path.join(CKPT_DIR, "results.csv"))
        and not ALLOW_FULL_RECOMPUTE):
    raise RuntimeError(
        "No previous run state was found.\n"
        f"  local : {CKPT_DIR}\n"
        f"  Drive : My Drive/{DRIVE_STATE_FOLDER} "
        f"({'API authenticated' if DRIVE_API else 'API unavailable'})\n"
        "Starting now would retrain every stage from scratch (>100 GPU-hours).\n"
        "Either restore the state manually in a cell:\n"
        "    restore_state('<path or URL of the CoFormer backup zip>'); push_state(force=True)\n"
        "or fix the Drive mount and run again. Set ALLOW_FULL_RECOMPUTE = True only if you "
        "really intend to recompute everything."
    )


## Cell 4b — Archive v1 artifacts (runs once, idempotent)

Moves every artifact produced by the v1 code into `<BASE>_v1_archive/` instead of deleting it, so the
audit trail of the first run is preserved. Dataset caches, the real-benchmark cache and the LLM
generation cache are kept in place, because they do not depend on the corrected loss.


In [ ]:
# === Cell 4b: archive v1 artifacts (REVISION v3) ===
import shutil as _sh
# REVISION v12: the v1 archive step ran on 15 Sep 2026. It moves results.csv and the real-data
# checkpoints whenever the marker file is missing (e.g. after a state restore that drops hidden
# files), which would archive the REPORTED study. It is therefore disabled, and it also refuses to
# run when any fingerprinted (v3+) checkpoint is present.
ARCHIVE_V1_ARTIFACTS = False
if any(f.startswith(("results_probe", "results_partial_", "results_estimator_confirmatory_"))
       for f in os.listdir(CKPT_DIR)):
    ARCHIVE_V1_ARTIFACTS = False
_ARCH = BASE.rstrip("/") + "_v1_archive"
_MARKER = os.path.join(OUT_DIR, ".v3_run_started")

# exact v1 names; fingerprinted v3 files never match these
_V1_CHECKPOINTS = [
    "results.csv", "results.csv.fingerprint", "results.csv.fingerprint.cfg",
    "results_probeA.csv", "results_probeB.csv", "per_example_synth.json",
    "real_comparison.csv", "real_comparison.csv.fingerprint", "per_example_real.json",
    "epoch_calibration_3e811e83fa.json", "diagnostics_bb4b680910",
]
if ARCHIVE_V1_ARTIFACTS and not os.path.exists(_MARKER):
    _moved = []
    os.makedirs(os.path.join(_ARCH, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(_ARCH, "outputs"), exist_ok=True)
    for _n in _V1_CHECKPOINTS + [f for f in os.listdir(CKPT_DIR) if ".stale-" in f]:
        _src = os.path.join(CKPT_DIR, _n)
        if os.path.exists(_src):
            _sh.move(_src, os.path.join(_ARCH, "checkpoints", _n)); _moved.append(f"checkpoints/{_n}")
    for _n in os.listdir(OUT_DIR):          # every v1 output would otherwise leak into the paper bundle
        _src = os.path.join(OUT_DIR, _n)
        if os.path.getmtime(_src) >= SESSION_T0:   # written by THIS session (requirements, metadata)
            continue
        _sh.move(_src, os.path.join(_ARCH, "outputs", _n)); _moved.append(f"outputs/{_n}")
    open(_MARKER, "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
    print(f"Archived {len(_moved)} v1 artifact(s) to {_ARCH}")
    for _m in _moved: print("  -", _m)
    print("Kept in place: data_cache/*.npz, data_cache/real_benchmarks_v4.json, checkpoints/llm_generations.json")
else:
    print("v1 archive step already done (or disabled); nothing moved.")


## Cell 3d — Plan III: pre-specified follow-up experiments, fingerprint guard, reuse guard
The Plan III specification (hypotheses, designs, metrics, decision rules and seeds) is serialised to
`outputs/plan_III.json` together with its SHA-256. If Plan III results already exist, the cell refuses
to continue after any later change to the specification. The fingerprint guard stops a Run All before
a stored result could be archived. The reuse guard keeps the reported values of diagnostics that are
not checkpointed.


In [ ]:
# === Cell 3d: Plan III specification + fingerprint guard + reuse guard (REVISION v12) ===
import hashlib as _hl, json as _json_iii

# ---------------------------------------------------------------------------------------------
# (1) Fingerprint guard. The values below are those of the reported study (family A, calibrated
#     budget 12 epochs; diagnostics and calibration fingerprints are taken before calibration).
#     A mismatch means that a configuration field, the result schema or LOSS_VERSION changed; the
#     checkpoint code would then ARCHIVE the stored results and retrain them. The guard stops first.
# ---------------------------------------------------------------------------------------------
FP_EXPECTED = {
    "diagnostics": "ee02e0da24",   # Cell 9b  DIAG_DIR
    "epochcal":    "fc9115e27a",   # Cell 9c  epoch-calibration cache
    "config":      "f1e0063f5b",   # Cell 10  results.csv configuration fingerprint
    "probe":       "99f69cb71d",   # Cell 10  per-probe, partial and estimator-replication files
    "estimator":   "0ea071f97e",   # Cell 10e exploratory estimator extension
    "real":        "d9a08fe280",   # Cell C5  natural-language comparison
}

def _study_exists():
    return os.path.exists(os.path.join(CKPT_DIR, "results.csv"))

def assert_fingerprint(name, value):
    exp = FP_EXPECTED.get(name)
    if exp is None or value == exp:
        print(f"  [fingerprint guard] {name}: {value} OK")
        return
    msg = (f"[fingerprint guard] `{name}` fingerprint is {value}, but the stored study was produced "
           f"under {exp}. Continuing would archive the stored results and retrain them. Undo the "
           f"change to the configuration / result schema / LOSS_VERSION (Plan III experiments must "
           f"use a copy of CFG), or set STRICT_FINGERPRINT_GUARD = False if a fresh study is intended.")
    if STRICT_FINGERPRINT_GUARD and _study_exists():
        raise RuntimeError(msg)
    print("  WARNING " + msg)

# Library versions of the reported study (run_metadata.json). New Plan III runs under different
# versions are allowed, but the difference is recorded and must be reported.
REPORTED_VERSIONS = {"torch": "2.11.0+cu128", "numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.14.1",
                     "transformers": "4.46.2", "peft": "0.13.2", "bitsandbytes": "0.50.2"}
def _version_drift():
    # REVISION v12.1: read installed versions WITHOUT importing the packages. Importing
    # transformers here (v12) loaded the runtime's version before Cell C1 pins 4.46.2.
    from importlib import metadata as _md
    out = {}
    for mod, rep in REPORTED_VERSIONS.items():
        try:
            cur = _md.version(mod)
        except Exception:
            continue
        if mod == "torch" and "+" in rep and "+" not in cur:
            rep = rep.split("+")[0]
        if cur != rep:
            out[mod] = (rep, cur)
    return out
VERSION_DRIFT = _version_drift()
if VERSION_DRIFT:
    print("  NOTE: library versions differ from the reported study (reported -> now): "
          + ", ".join(f"{k} {a} -> {b}" for k, (a, b) in VERSION_DRIFT.items())
          + ". Stored results are unaffected; new Plan III runs record the versions in use. "
          "(transformers/peft/accelerate are pinned by Cell C1 before Part C imports them.)")

# ---------------------------------------------------------------------------------------------
# (2) Reuse guard for diagnostics that are recomputed on every Run All (not checkpointed).
# ---------------------------------------------------------------------------------------------
def _reuse_reported(fname):
    p = os.path.join(OUT_DIR, fname)
    ok = (not RECOMPUTE_REPORTED_DIAGNOSTICS) and os.path.exists(p)
    if ok:
        print(f"  [v12] reusing the reported {fname} "
              f"(set RECOMPUTE_REPORTED_DIAGNOSTICS = True to recompute)")
    return ok

# ---------------------------------------------------------------------------------------------
# (3) Plan III. Fixed BEFORE any Plan III run; deposit outputs/plan_III.json publicly first.
# ---------------------------------------------------------------------------------------------
PLAN_III = {
  "version": "III-1",
  "family": "A (3-SAT, n=16, ratio 4.0); all other settings as in the reported study",
  "primary_metric": "faith_excess_id (density-controlled clause-F1 excess, ID split)",
  "comparators": "stored runs of the reported study (partial-supervision matrix at f=0.005, "
                 "main matrix), which share code, data, seeds and supervised subsets",
  "answer_supervision": {
    "labels": "for each training instance one conclusion literal (variable uniform, polarity "
              "uniform; numpy default_rng(0)), labelled True / False / Unknown by the complete "
              "DPLL solver of Cell 5 (F entails c / F entails not-c / neither)",
    "loss": "cross-entropy of kappa*(D, -D, mu - |D|) with D = (m+1)*(S(C+c) - S(C+not c)), where "
            "S is the relaxed layer's mean clause satisfaction on the emitted formula augmented "
            "with the unit clause; kappa = 4, mu = 0.5; class weights = inverse label frequency",
    "rationale": "the natural-language surrogate uses the unscaled difference, which is bounded by "
                 "1/(m+1) and makes Unknown the arg-max by construction; the (m+1) rescaling removes "
                 "that bound. mu was fixed on GOLD formulas of 1,000 training instances before any "
                 "Plan III training (balanced accuracy of the surrogate's arg-max on gold formulas "
                 "0.61 at mu = 0.5, against 0.33 for chance). Label shares on the full training set: True 0.236, "
                 "False 0.243, Unknown 0.521",
    "no_anchored_task_loss": True,
  },
  "confirmatory": {
    "A2_ST":   dict(a="ans_st_ps", b="frozen", faith_fraction=0.005, seeds=list(range(12)),
                    direction="greater", note="answers through the solver on all instances "
                    "+ per-slot supervision on 100 instances, vs per-slot supervision only"),
    "A2_soft": dict(a="ans_soft_ps", b="soft_frozen", faith_fraction=0.005, seeds=list(range(12)),
                    direction="greater", note="as A2_ST with the soft bridge"),
  },
  "confirmatory_rule": "one-sided exact sign-flip permutation test on the per-seed differences "
                       "(all 2^12 sign patterns); Holm over {A2_ST, A2_soft} at alpha = 0.05; "
                       "SUPPORTED iff Holm-adjusted p < 0.05 AND mean difference >= 0.03",
  "secondary_confirmatory": "success indicator (faith_excess_id > 0.30): exact one-sided McNemar; "
                            "non-superiority: a benefit >= 0.03 is excluded iff the upper limit of "
                            "the t-based 90% interval of the mean difference is < 0.03",
  "diagnostic": {
    "A1_ST":   dict(a="ans_st", faith_fraction=0.0, seeds=list(range(6)),
                    compare="no_faith (gold formula through the anchored loss)"),
    "A1_soft": dict(a="ans_soft", faith_fraction=0.0, seeds=list(range(6)),
                    compare="soft_no_faith"),
    "K_ST":    dict(a="coformer[K=T=50]", b="frozen", faith_fraction=0.005, seeds=list(range(6))),
    "K_soft":  dict(a="no_stgs[K=T=50]", b="soft_frozen", faith_fraction=0.005, seeds=list(range(6))),
    "W_ST":    dict(a="coformer[task weight 5]", b="frozen", faith_fraction=0.005, seeds=list(range(6))),
    "W_soft":  dict(a="no_stgs[task weight 5]", b="soft_frozen", faith_fraction=0.005, seeds=list(range(6))),
    "B_ST":    dict(a="coformer[2x epochs]", b="frozen[2x epochs]", faith_fraction=0.005,
                    seeds=list(range(6)), extra="learning curves at every 3rd epoch"),
    "D":       dict(systems=["Direct-frozen", "CoT-frozen"], seeds=[0, 1, 2, 3],
                    compare="SATLM-frozen, LogicLM-frozen, majority floor; accuracy and abstention"),
  },
  "diagnostic_rule": "estimates with 95% t intervals, exact two-sided sign-flip p-values and minimum "
                     "detectable differences; no verdicts (6 seeds cannot reach significance after "
                     "correction)",
  "stage_flags": ["RUN_PLAN_III_A2", "RUN_PLAN_III_A1", "RUN_PLAN_III_KW",
                  "RUN_PLAN_III_BUDGET", "RUN_PLAN_III_NL"],
}
ANSWER_KAPPA, ANSWER_MARGIN = 4.0, 0.5

_plan_blob = _json_iii.dumps(PLAN_III, sort_keys=True, indent=1)
PLAN_III_SHA256 = _hl.sha256(_plan_blob.encode()).hexdigest()
_plan_path = os.path.join(OUT_DIR, "plan_III.json")
PLAN_III_CSV = None   # set in Cell 10h
_any_iii = any(globals().get(f, False) for f in PLAN_III["stage_flags"])
if os.path.exists(_plan_path):
    _stored = _json_iii.load(open(_plan_path))
    if _stored.get("sha256") != PLAN_III_SHA256:
        _iii_results = [f for f in os.listdir(CKPT_DIR) if f.startswith("results_planIII_")]
        if _iii_results:
            raise RuntimeError("Plan III specification changed after Plan III results were written "
                               f"({_iii_results}). Restore the deposited specification.")
        print("  Plan III specification changed before any Plan III run; rewriting plan_III.json.")
        _stored = None
else:
    _stored = None
if _stored is None:
    atomic_json({"sha256": PLAN_III_SHA256,
                 "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                 "plan": PLAN_III}, _plan_path, indent=1)
print(f"Plan III specification: outputs/plan_III.json  SHA-256 {PLAN_III_SHA256}")
if _any_iii:
    # Record the specification outside this session BEFORE any Plan III run: an e-mail alert
    # (if SMTP secrets are set) and a Drive state push carry the hash and the time.
    _written = _json_iii.load(open(_plan_path)).get("written_utc", "?")
    try:
        send_alert("Plan III specification fixed before its runs",
                   f"SHA-256 {PLAN_III_SHA256}\nwritten {_written}\n\n{_plan_blob[:3000]}")
    except Exception as _e:
        print(f"  (alert not sent: {_e})")
    try:
        if not globals().get("PERSISTENT", True):
            push_state(tag="plan_III_spec", force=True)
    except Exception as _e:
        print(f"  (state push failed: {_e})")
if not _any_iii:
    print("  No Plan III stage is enabled. Deposit plan_III.json with a public time stamp, then set "
          "the RUN_PLAN_III_* flags (priority A2 > A1 > KW > BUDGET > NL) and Run All again.")


## Cell 3e — Follow-up study 2: pre-specified referee experiments

In [ ]:
# === Cell 3e: Follow-up study 2 specification (REVISION v13) ===
# Referee-requested experiments. Fixed BEFORE any follow-up-2 run; the SHA-256 and the UTC time
# are written to outputs/followup2.json and sent by e-mail (if SMTP secrets are set).
import hashlib as _hl2, json as _json_fs2

FS2 = {
  "version": "FS2-1",
  "family": "A (3-SAT, n=16, ratio 4.0); all other settings as in the reported study",
  "primary_metric": "faith_excess_id (density-controlled clause-F1 excess, ID split, final model)",
  "designs": {
    "C1_convergence": dict(
        conditions=["coformer", "frozen"], faith_fraction=0.005, seeds=list(range(12)),
        epochs=48, note="4x the calibrated budget (12 epochs); same cosine schedule stretched to 48 "
        "epochs; the FINAL model is evaluated (no early stopping, no model selection); learning "
        "curve of the ID excess on the first 512 ID instances every 6 epochs (reporting only)"),
    "C2_rl": dict(
        estimator="RLOO score-function gradient (4 formula samples per instance from the "
                  "translator's per-slot categorical distribution; leave-one-out baseline); no "
                  "gradient passes through any solver",
        reward="class-balanced agreement between the EXACT entailment labels of the sampled "
               "formula and the gold labels over the instance's queries (label True/False/Unknown "
               "as in the follow-up study; weights = inverse gold-label frequency over the "
               "training set); exact labels by exhaustive enumeration of the 2^16 assignments",
        loss="task_weight * -(advantage * log-probability of the sampled formula, i.e. the sum over "
             "its slots; standard REINFORCE/RLOO), averaged over samples and instances; "
             "task_weight = 1; per-slot loss, density hinge and logit penalty as in `frozen`",
        runs={"rl_ans_ps[Q=1]": dict(faith_fraction=0.005, queries=1, seeds=list(range(12)),
                                     note="same single query per instance as A1/A2"),
              "rl_ans_ps[Q=16]": dict(faith_fraction=0.005, queries=16, seeds=list(range(6)),
                                      note="one query per variable, polarity from default_rng(1)"),
              "rl_ans[Q=16]": dict(faith_fraction=0.0, queries=16, seeds=list(range(6)),
                                   note="answers only, no per-slot supervision")}),
    "C3_shuffled_answers": dict(
        condition="ans_st_ps", faith_fraction=0.005, seeds=list(range(12)),
        labels="the follow-up study's answer labels permuted across training instances "
               "(numpy default_rng(2)); class weights unchanged",
        note="tests whether the A2 harm comes from the answer content or from adding a loss"),
  },
  "confirmatory": {
    "C1": dict(a="coformer[48 ep]", b="frozen[48 ep]", direction="greater",
               claim="with training to convergence, routing the solver gradient improves ID excess "
                     "under scarce per-slot supervision"),
    "C2": dict(a="rl_ans_ps[Q=1]", b="frozen (stored partial-supervision run, f=0.005)",
               direction="greater",
               claim="answer feedback from an exact solver without differentiation improves ID "
                     "excess under scarce per-slot supervision"),
    "C3": dict(a="ans_st_ps with shuffled labels", b="ans_st_ps (stored follow-up run A2)",
               direction="greater",
               claim="the A2 harm is caused by the answer content: shuffled labels harm less"),
  },
  "confirmatory_rule": "one-sided exact sign-flip permutation test on the 12 per-seed differences "
                       "(all 2^12 sign patterns); Holm over {C1, C2, C3} at alpha = 0.05; SUPPORTED "
                       "iff Holm-adjusted p < 0.05 AND mean difference >= 0.03",
  "secondary": "success indicator (faith_excess_id > 0.30): exact one-sided McNemar; hurdle "
               "summary (success counts; mean excess of successful and failed runs); "
               "non-superiority: a benefit >= 0.03 is excluded iff the upper limit of the "
               "t-based 90% interval of the mean difference is < 0.03",
  "convergence_criterion": "C1 is called converged if, averaged over seeds, the ID excess on the "
                           "first 512 ID instances changes by less than 0.02 between epochs 36 "
                           "and 48 in both conditions",
  "diagnostic": "rl_ans_ps[Q=16] vs frozen and vs rl_ans_ps[Q=1] (seeds 0-5); rl_ans[Q=16] vs the "
                "follow-up study's A1 runs (seeds 0-5); estimates with 95% t intervals and exact "
                "two-sided sign-flip p-values, no verdicts",
  "order": "C3, C2 (Q=1), C1, C2 (Q=16 designs)",
  "stage_flags": ["RUN_FS2_C1", "RUN_FS2_C2", "RUN_FS2_C3"],
}
RL_SAMPLES, RL_Q16_SEED, SHUFFLE_SEED = 4, 1, 2

_fs2_blob = _json_fs2.dumps(FS2, sort_keys=True, indent=1)
FS2_SHA256 = _hl2.sha256(_fs2_blob.encode()).hexdigest()
_fs2_path = os.path.join(OUT_DIR, "followup2.json")
if os.path.exists(_fs2_path):
    _fs2_stored = _json_fs2.load(open(_fs2_path))
    if _fs2_stored.get("sha256") != FS2_SHA256:
        _fs2_results = [f for f in os.listdir(CKPT_DIR) if f.startswith("results_fs2_")]
        if _fs2_results:
            raise RuntimeError("Follow-up-2 specification changed after its results were written "
                               f"({_fs2_results}). Restore the deposited specification.")
        print("  Follow-up-2 specification changed before any run; rewriting followup2.json.")
        _fs2_stored = None
else:
    _fs2_stored = None
if _fs2_stored is None:
    atomic_json({"sha256": FS2_SHA256,
                 "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                 "plan": FS2}, _fs2_path, indent=1)
print(f"Follow-up study 2 specification: outputs/followup2.json  SHA-256 {FS2_SHA256}")
_any_fs2 = any(globals().get(f, False) for f in FS2["stage_flags"])
if _any_fs2:
    _w2 = _json_fs2.load(open(_fs2_path)).get("written_utc", "?")
    try:
        send_alert("Follow-up study 2 specification fixed before its runs",
                   f"SHA-256 {FS2_SHA256}\nwritten {_w2}\n\n{_fs2_blob[:3500]}")
    except Exception as _e:
        print(f"  (alert not sent: {_e})")
    try:
        if not globals().get("PERSISTENT", True):
            push_state(tag="followup2_spec", force=True)
    except Exception as _e:
        print(f"  (state push failed: {_e})")
else:
    print("  No follow-up-2 stage is enabled (RUN_FS2_C1/C2/C3 are False).")


## Cell 3f — Family H: pre-specified harder-language benchmark

In [ ]:
# === Cell 3f: Family H specification (REVISION v14) ===
# A linguistically harder benchmark on the SAME formulas as family A. Fixed BEFORE any family-H run:
# the specification (including the SHA-256 of the generator source below) and the UTC time are
# written to outputs/followup_h.json and sent by e-mail (if SMTP secrets are set).
import hashlib as _hlH, json as _jsonH

_FAMH_SRC = r'''
# --- Family H generator (included verbatim in Cell 10j) ---------------------------------------
# Same formula distribution as family A (uniform random 3-SAT, n = 16, m = 64, rejection-sampled
# for satisfiability, identical generator and seeds); only the LANGUAGE changes. Each clause is
# still one text segment, so the per-clause Translator and run_condition are reused unchanged.
#
# Difficulty mechanisms (level l = 1, 2, 3):
#   * variable v = (entity v // 4, property v % 4); 3 aliases per entity, 3 synonyms per property,
#     so the translator must map 9 surface forms onto each variable
#   * subject ellipsis: two positive literals of the same entity -> "E is P1 or P2"
#   * polarity = parity of the number k of stacked negations, k <= l ("is not" counts as one)
#   * l - 1 distractor sub-clauses that MENTION other variables without asserting them
# Paraphrase-OOD split: clause frames and distractor frames never seen in training, built only
# from training words (asserted), so the shift is compositional, not lexical.
import numpy as np

H_ENTITIES = [["alice", "the doctor", "the redhead"],
              ["bob", "the pilot", "the drummer"],
              ["carol", "the teacher", "the singer"],
              ["dave", "the chef", "the painter"]]
H_PROPS = [["tired", "exhausted", "sleepy"],
           ["happy", "cheerful", "glad"],
           ["at home", "indoors", "inside"],
           ["busy", "occupied", "working"]]
H_NVARS = len(H_ENTITIES) * len(H_PROPS)
H_NEG = ["it is false that", "nobody claims that", "it is untrue that"]
H_FRAMES_ID = [("either {L} .", " or "), ("at least one holds : {L} .", " or "),
               ("{L} .", " or else "), ("we are told that {L} .", " or ")]
H_FRAMES_OOD = [("we are told that either {L} .", " or else "),
                ("at least one holds , we are told : {L} .", " or ")]
H_DISTRACT_ID = ["regardless of whether {A} ,", "even though nobody knows if {A} ,",
                 "although it is unclear whether {A} ,"]
H_DISTRACT_OOD = ["even though it is unclear whether {A} ,", "although nobody knows if {A} ,"]


def _h_words(items):
    out = set()
    for it in items:
        s = it[0] + " " + it[1] if isinstance(it, tuple) else it
        out |= set(s.replace("{L}", " ").replace("{A}", " ").split())
    return out


H_WORDS_ID = (_h_words(H_FRAMES_ID) | _h_words(H_DISTRACT_ID) | _h_words(H_NEG)
              | {w for al in H_ENTITIES + H_PROPS for a in al for w in a.split()} | {"is", "not", "or"})
H_WORDS_OOD = _h_words(H_FRAMES_OOD) | _h_words(H_DISTRACT_OOD)
assert H_WORDS_OOD <= H_WORDS_ID, f"OOD frames add unseen words {sorted(H_WORDS_OOD - H_WORDS_ID)}"
assert not ({f for f, _ in H_FRAMES_OOD} & {f for f, _ in H_FRAMES_ID})
assert not (set(H_DISTRACT_OOD) & set(H_DISTRACT_ID))


def _h_atom(v, rng, neg_copula=False):
    e, p = divmod(int(v), len(H_PROPS))
    E = H_ENTITIES[e][int(rng.integers(3))]
    P = H_PROPS[p][int(rng.integers(3))]
    return f"{E} is not {P}" if neg_copula else f"{E} is {P}"


def _h_literal(v, sign, level, rng):
    """Literal (v, sign) with k stacked negations, k <= level, k odd iff sign < 0."""
    ks = [k for k in range(level + 1) if (k % 2 == 1) == (sign < 0)]
    k = int(ks[int(rng.integers(len(ks)))])
    use_cop = k >= 1 and rng.random() < 0.5          # innermost negation as "is not"
    s = _h_atom(v, rng, neg_copula=use_cop)
    for _ in range(k - (1 if use_cop else 0)):
        s = f"{H_NEG[int(rng.integers(len(H_NEG)))]} {s}"
    return s


def verbalize_clause_H(row, *, level, rng, ood=False):
    """ONE clause row (n_vars,) in {-1,0,+1} -> ONE sentence (str)."""
    lits = [(int(v), int(np.sign(row[v]))) for v in np.nonzero(row)[0]]
    order = rng.permutation(len(lits)); lits = [lits[i] for i in order]
    frames = H_FRAMES_OOD if ood else H_FRAMES_ID
    tmpl, sep = frames[int(rng.integers(len(frames)))]
    parts, i = [], 0
    while i < len(lits):
        v, s = lits[i]
        j = next((j for j in range(i + 1, len(lits))
                  if s > 0 and lits[j][1] > 0 and lits[j][0] // 4 == v // 4), None)
        if j is not None and rng.random() < 0.5:     # subject ellipsis, positive literals only
            v2 = lits.pop(j)[0]
            E = H_ENTITIES[v // 4][int(rng.integers(3))]
            parts.append(f"{E} is {H_PROPS[v % 4][int(rng.integers(3))]} or "
                         f"{H_PROPS[v2 % 4][int(rng.integers(3))]}")
        else:
            parts.append(_h_literal(v, s, level, rng))
        i += 1
    body = tmpl.replace("{L}", sep.join(parts))
    n_d = level - 1
    if n_d:
        in_cl = {v for v, _ in lits} | {int(v) for v in np.nonzero(row)[0]}
        pool = np.array([v for v in range(H_NVARS) if v not in in_cl])
        dfr = H_DISTRACT_OOD if ood else H_DISTRACT_ID
        pre = [dfr[int(rng.integers(len(dfr)))].replace(
                   "{A}", _h_atom(int(v), rng, neg_copula=bool(rng.random() < 0.5)))
               for v in rng.choice(pool, size=n_d, replace=False)]
        body = " ".join(pre) + " " + body
    return body


def h_max_clause_tokens(level, clause_len=3):
    """Worst-case sentence length in words (a proven bound, so nothing is ever truncated)."""
    fw = max(len(f.replace("{L}", " ").split()) for f, _ in H_FRAMES_ID + H_FRAMES_OOD)
    sw = max(len(s.split()) for _, s in H_FRAMES_ID + H_FRAMES_OOD)
    ew = max(len(a.split()) for al in H_ENTITIES for a in al)
    pw = max(len(a.split()) for al in H_PROPS for a in al)
    nw = max(len(n.split()) for n in H_NEG)
    atom = ew + 2 + pw                                  # "E is not P"
    lit = max(level * nw + atom - 1, (level - 1) * nw + atom)
    ellip = ew + 1 + pw + 1 + pw                        # covers two literals
    lits = clause_len * lit + (clause_len - 1) * sw
    lits = max(lits, ellip + (clause_len - 2) * lit + (clause_len - 2) * sw)
    dw = max(len(d.replace("{A}", " ").split()) for d in H_DISTRACT_ID + H_DISTRACT_OOD) + atom
    return fw + lits + (level - 1) * dw


# ---------------------------------------------------------------- reference parser (validation)
_H_ENT = {tuple(a.split()): e for e, al in enumerate(H_ENTITIES) for a in al}
_H_PRP = {tuple(a.split()): p for p, al in enumerate(H_PROPS) for a in al}
_H_NEGT = [tuple(n.split()) for n in H_NEG]


def _h_match(w, i, table):
    best = None
    for ph, idx in table.items():
        if tuple(w[i:i + len(ph)]) == ph and (best is None or len(ph) > len(best[0])):
            best = (ph, idx)
    return best


def parse_clause_H(text):
    """Symbolic inverse of verbalize_clause_H: sentence -> {var: sign}. Raises if not parseable.
    Used only to PROVE that every sentence determines its clause uniquely."""
    w = text.split()
    heads = sorted({tuple(d.split("{A}")[0].split()) for d in H_DISTRACT_ID + H_DISTRACT_OOD},
                   key=len, reverse=True)
    i = 0
    while True:                                         # leading distractors
        h = next((h for h in heads if tuple(w[i:i + len(h)]) == h), None)
        if h is None:
            break
        i = w.index(",", i) + 1
    w = w[i:]
    fr = sorted(H_FRAMES_ID + H_FRAMES_OOD, key=lambda f: -len(f[0].split("{L}")[0].split()))
    for f, _ in fr:
        pre, post = (x.split() for x in f.split("{L}"))
        if tuple(w[:len(pre)]) == tuple(pre) and tuple(w[len(w) - len(post):]) == tuple(post):
            w = w[len(pre):len(w) - len(post)]
            break
    else:
        raise ValueError(f"no frame: {text!r}")
    out, i, k = {}, 0, 0
    while i < len(w):
        if w[i] in ("or", "else") and k == 0:
            i += 1; continue
        n = next((n for n in _H_NEGT if tuple(w[i:i + len(n)]) == n), None)
        if n:
            k += 1; i += len(n); continue
        m = _h_match(w, i, _H_ENT)
        if m is None:
            raise ValueError(f"no entity at {' '.join(w[i:i + 4])!r} in {text!r}")
        i += len(m[0])
        assert w[i] == "is", text; i += 1
        if w[i] == "not":
            k += 1; i += 1
        mp = _h_match(w, i, _H_PRP); assert mp, text; i += len(mp[0])
        v = 4 * m[1] + mp[1]; assert v not in out, text
        out[v] = -1 if k % 2 else 1; k = 0
        if i + 1 < len(w) and w[i] == "or" and _h_match(w, i + 1, _H_PRP):
            mp2 = _h_match(w, i + 1, _H_PRP); v2 = 4 * m[1] + mp2[1]
            assert v2 not in out, text
            out[v2] = 1; i += 1 + len(mp2[0])
    return out
'''
exec(compile(_FAMH_SRC, "<famH-generator>", "exec"), globals())
FAMH_GEN_SHA256 = _hlH.sha256(_FAMH_SRC.encode()).hexdigest()

FAMH = {
  "version": "H-1",
  "formulas": "the stored family-A gold clause matrices (training 20,000; ID 2,000; paraphrase 2,000 "
              "instances; 3-SAT, n = 16, m = 64, rejection-sampled for satisfiability), same instances "
              "in the same order; only the text is regenerated. No scale split (the 16-variable "
              "lexicon does not extend); the scale-based fields are set to NaN and not reported.",
  "generator_sha256": FAMH_GEN_SHA256,
  "language": "variable v = (entity v // 4, property v % 4); 3 aliases per entity and 3 synonyms per "
              "property; subject ellipsis for two positive literals of one entity (probability 0.5); "
              "polarity = parity of k stacked negations, k <= level, uniform over the admissible k, "
              "innermost negation as 'is not' with probability 0.5; negation scopes over the "
              "immediately following atom only; level - 1 leading distractor sub-clauses that mention "
              "variables outside the clause without asserting them",
  "paraphrase_split": "clause frames and distractor frames disjoint from training, built only from "
                      "training words (asserted); same negation operators",
  "text_seeds": {"train": "5100 + level", "id": "5200 + level", "paraphrase": "5300 + level"},
  "validation": "a symbolic parser recovers the gold clause from every sampled sentence (checked on "
                "4,000 sentences per split at generation); no sentence is truncated (proven bound)",
  "model_and_training": "as in the reported study (Translator, relaxed MaxSAT layer T = 50, K = 10, "
                        "losses and schedule unchanged) except epochs = 24 (twice the calibrated "
                        "budget) for every family-H run; the FINAL model is evaluated; learning curve "
                        "of the ID excess on the first 512 ID instances every 6 epochs (reporting only)",
  "pilot": dict(condition="frozen", faith_fraction=1.0, seed=1000, levels=[1, 2, 3], epochs=24,
                rule="share = faith_excess_id / (1 - g/2) with g = 3/16 (maximum 0.906). Select the "
                     "LOWEST level with 0.20 <= share <= 0.95. If no level qualifies, select the HIGHEST "
                     "level with share > 0.95; if there is none (all shares < 0.20), select level 1. The "
                     "pilot seed is not used in any confirmatory test, and coformer is not run in the "
                     "pilot, so the selection cannot depend on the contrast under test."),
  "designs": {
    "H_full": dict(conditions=["coformer", "frozen"], faith_fraction=1.0, seeds=list(range(16))),
    "H_scarce": dict(conditions=["coformer", "frozen"], faith_fraction=0.05, seeds=list(range(12))),
  },
  "confirmatory": {
    "H_full": dict(a="coformer", b="frozen", direction="greater", metric="faith_excess_id",
                   claim="on the harder language with full per-slot supervision, routing the solver "
                         "gradient improves ID excess"),
    "H_scarce": dict(a="coformer", b="frozen", direction="greater", metric="faith_excess_id",
                     claim="on the harder language with per-slot supervision on 5% of the training "
                           "instances, routing the solver gradient improves ID excess"),
  },
  "confirmatory_rule": "one-sided exact sign-flip permutation test on the per-seed differences (all "
                       "2^n sign patterns); Holm over {H_full, H_scarce} at alpha = 0.05; SUPPORTED iff "
                       "Holm-adjusted p < 0.05 AND mean difference >= 0.03",
  "testability": "a member is UNTESTABLE (ceiling) if the median frozen share is >= 0.95; it then gets "
                 "no verdict, and the Holm family size stays 2",
  "secondary": "no verdicts: faith_excess_paraphrase, csat_norm_id, csat_norm_paraphrase and the "
               "paraphrase retention csat_norm_paraphrase / csat_norm_id (mean difference, 95% t "
               "interval, exact two-sided sign-flip p); success indicator (faith_excess_id > 0.30) "
               "with exact one-sided McNemar and hurdle means; non-superiority (a benefit >= 0.03 is "
               "excluded iff the upper limit of the 90% t interval is < 0.03); 80% MDE",
  "convergence_criterion": "reported, not used for any verdict: converged if, averaged over seeds, the "
                           "ID excess on the first 512 ID instances changes by less than 0.02 between "
                           "epochs 18 and 24 in both conditions",
  "order": "pilot (levels 1, 2, 3), then for seed s = 0..15: H_full frozen, H_full coformer, and for "
           "s < 12 H_scarce frozen, H_scarce coformer",
  "stage_flags": ["RUN_FAMH"],
}
FAMH_EPOCHS, FAMH_PILOT_SEED = 24, 1000

_famh_blob = _jsonH.dumps(FAMH, sort_keys=True, indent=1)
FAMH_SHA256 = _hlH.sha256(_famh_blob.encode()).hexdigest()
_famh_path = os.path.join(OUT_DIR, "followup_h.json")
_famh_stored = _jsonH.load(open(_famh_path)) if os.path.exists(_famh_path) else None
if _famh_stored is not None and _famh_stored.get("sha256") != FAMH_SHA256:
    if [f for f in os.listdir(CKPT_DIR) if f.startswith("results_famH_")]:
        raise RuntimeError("Family-H specification changed after its results were written. "
                           "Restore the deposited specification.")
    print("  Family-H specification changed before any run; rewriting followup_h.json.")
    _famh_stored = None
if _famh_stored is None:
    atomic_json({"sha256": FAMH_SHA256,
                 "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                 "plan": FAMH, "generator_source": _FAMH_SRC}, _famh_path, indent=1)
print(f"Family H specification: outputs/followup_h.json  SHA-256 {FAMH_SHA256}")
print(f"  generator source SHA-256 {FAMH_GEN_SHA256}")
if globals().get("RUN_FAMH", False):
    _wH = _jsonH.load(open(_famh_path)).get("written_utc", "?")
    try:
        send_alert("Family H specification fixed before its runs",
                   f"SHA-256 {FAMH_SHA256}\nwritten {_wH}\n\n{_famh_blob[:3500]}")
    except Exception as _e:
        print(f"  (alert not sent: {_e})")
    try:
        if not globals().get("PERSISTENT", True):
            push_state(tag="famH_spec", force=True)
    except Exception as _e:
        print(f"  (state push failed: {_e})")
else:
    print("  Family H is not enabled (RUN_FAMH is False).")


## Cell 5 — Synthetic instances: phase-transition random k-SAT, verbalised as English

Instances are **uniform random k-SAT at clause/variable ratio `sat_ratio`**,
rejection-sampled for satisfiability and verified by a complete DPLL solver.
The ratio and the mean number of draws per accepted instance are reported with
every result. `use_planted=True` selects a planted generator instead, which
places instances far below the satisfiability phase transition and is retained
only for a hardness comparison.

Instances are verbalised as **templated English sentences over a word-level
lexicon**: each variable maps to a multi-word predicate phrase (`alice is
tired`), clauses are rendered by sentence templates, and negation has several
lexical realisations. There is no token-to-literal correspondence, so the model
must segment sentences, resolve phrases to variables, and recover polarity from
surface form.

Paraphrase-OOD uses **held-out templates whose vocabulary is asserted,
programmatically, to be a subset of the training vocabulary**, so the shift is
compositional rather than lexical. This is a controlled probe, not naturalistic
text; the naturalistic evidence is the real-benchmark study.


In [ ]:
# === Cell 5: phase-transition random k-SAT, verbalised as PER-CLAUSE sentences ===
import numpy as np, hashlib, itertools

def _dpll(clauses, n_vars):
    """Complete DPLL with unit propagation and pure-literal elimination."""
    clauses = [set(c) for c in clauses]

    def simplify(cls, lit):
        out = []
        for c in cls:
            if lit in c:
                continue
            if -lit in c:
                c2 = c - {-lit}
                if not c2:
                    return None
                out.append(c2)
            else:
                out.append(c)
        return out

    def solve(cls):
        while True:
            units = [next(iter(c)) for c in cls if len(c) == 1]
            if not units:
                break
            for u in units:
                cls = simplify(cls, u)
                if cls is None:
                    return False
            if not cls:
                return True
        if not cls:
            return True
        lits = {l for c in cls for l in c}
        for p in [l for l in lits if -l not in lits]:
            cls = simplify(cls, p)
            if cls is None:
                return False
        if not cls:
            return True
        counts = {}
        for c in cls:
            for l in c:
                counts[l] = counts.get(l, 0) + 1
        lit = max(counts, key=counts.get)
        for choice in (lit, -lit):
            nxt = simplify(cls, choice)
            if nxt is not None and solve(nxt):
                return True
        return False

    return solve(clauses)

def _mat_to_clauses(mat):
    return [{int(np.sign(mat[c, v])) * (v + 1) for v in np.nonzero(mat[c])[0]}
            for c in range(mat.shape[0])]

def _make_planted_instance(n_vars, n_clauses, clause_len, rng):
    planted = rng.integers(0, 2, size=n_vars) * 2 - 1
    clauses = np.zeros((n_clauses, n_vars), dtype=np.int8)
    for c in range(n_clauses):
        vs = rng.choice(n_vars, size=clause_len, replace=False)
        pol = rng.integers(0, 2, size=clause_len) * 2 - 1
        if not np.any(pol == planted[vs]):
            j = rng.integers(0, clause_len); pol[j] = planted[vs[j]]
        clauses[c, vs] = pol
    return clauses

def _make_random_sat_instance(n_vars, n_clauses, clause_len, rng, max_tries=200):
    """Uniform random k-SAT, rejection-sampled for satisfiability (verified by DPLL)."""
    for t in range(max_tries):
        clauses = np.zeros((n_clauses, n_vars), dtype=np.int8)
        for c in range(n_clauses):
            vs = rng.choice(n_vars, size=clause_len, replace=False)
            pol = rng.integers(0, 2, size=clause_len) * 2 - 1
            clauses[c, vs] = pol
        if _dpll(_mat_to_clauses(clauses), n_vars):
            return clauses, t + 1
    return None, max_tries

PROBE = getattr(CFG, "probe", "A")

SUBJECTS_ALL = ["alice", "bob", "carol", "dave", "erin", "frank", "grace", "henry",
            "irene", "jack", "kate", "liam", "mona", "nina", "oscar", "paul",
            "quinn", "rosa", "sam", "tina", "umar", "vera", "wendy", "xena",
            "yara", "zack", "adam", "bella", "cleo", "dino", "elsa", "finn"]
SUBJECTS = SUBJECTS_ALL[:16] if PROBE == "A" else SUBJECTS_ALL[16:]
PREDICATES = ["is tired", "is happy", "is at home", "is on duty", "is ready",
              "is late", "is quiet", "is busy", "is calm", "is early",
              "is available", "is awake"]

def predicate_phrase(v):
    return f"{SUBJECTS[v % len(SUBJECTS)]} {PREDICATES[(v // len(SUBJECTS)) % len(PREDICATES)]}"

# Surface variation is deliberately wide. A translator that generalises has to
# recover the phrase-to-variable mapping and the polarity from form, not from a
# fixed template signature; with only two or three realisations per role a model
# can key on template identity and score perfectly in distribution while failing
# any paraphrase. The paraphrase-OOD banks are DISJOINT from the training banks
# in template but share the training vocabulary exactly (asserted below), so the
# shift stays compositional rather than lexical.
ID_CLAUSE_TEMPLATES = [
    ("either {LITS} .", " or "),
    ("at least one of these holds : {LITS} .", " or "),
    ("we know that {LITS} , as these facts are given .", " or "),
    ("suppose {LITS} .", " or "),
    ("here is what holds : {LITS} .", " or "),
    ("consider the following : {LITS} .", " or "),
    ("{LITS} , and this is given .", " or "),
    ("it holds that {LITS} .", " or "),
]
OOD_CLAUSE_TEMPLATES = [
    ("one of these is true : {LITS} .", " or "),
    ("it is true that at least one of {LITS} holds .", " or "),
    ("these facts are given : either {LITS} .", " or "),
    ("we consider it given that {LITS} .", " or "),
    ("at least one of the following is not false : {LITS} .", " or "),
    ("suppose we know the following holds : {LITS} .", " or "),
]
ID_POS_TEMPLATES  = [
    "{P}", "it is true that {P}", "we know {P}", "it holds that {P}",
    "it is given that {P}", "{P} is true",
]
ID_NEG_TEMPLATES  = [
    "it is not true that {P}", "{P} is not the case", "it is false that {P}",
    "we know {P} is false", "{P} does not hold", "it is not given that {P}",
]
OOD_POS_TEMPLATES = [
    "we know that {P}", "the case is that {P}", "it is not false that {P}",
    "we consider it true that {P}", "{P} , and this holds",
]
OOD_NEG_TEMPLATES = [
    "it is not the case that {P}", "we know that {P} is not true",
    "we consider it false that {P}", "it is given that {P} does not hold",
    "{P} , and this is false",
]

def _tokens_of(ts):
    out = set()
    for t in ts:
        s = t[0] if isinstance(t, tuple) else t
        for w in s.replace("{LITS}", " ").replace("{P}", " ").split():
            out.add(w)
    return out

_ID_WORDS = (_tokens_of(ID_CLAUSE_TEMPLATES) | _tokens_of(ID_NEG_TEMPLATES)
             | _tokens_of(ID_POS_TEMPLATES) | {"or"})
_OOD_WORDS = (_tokens_of(OOD_CLAUSE_TEMPLATES) | _tokens_of(OOD_NEG_TEMPLATES)
              | _tokens_of(OOD_POS_TEMPLATES))
_unseen = _OOD_WORDS - _ID_WORDS
assert not _unseen, (f"paraphrase-OOD templates introduce unseen words {sorted(_unseen)}; "
                     f"the shift would be lexical rather than compositional")
print(f"Paraphrase-OOD template check: {len(_OOD_WORDS)} template words, all seen in ID. OK.")
_id_surface  = len(ID_CLAUSE_TEMPLATES) * len(ID_POS_TEMPLATES) * len(ID_NEG_TEMPLATES)
_ood_surface = len(OOD_CLAUSE_TEMPLATES) * len(OOD_POS_TEMPLATES) * len(OOD_NEG_TEMPLATES)
print(f"Surface-variation budget: {len(ID_CLAUSE_TEMPLATES)} clause x "
      f"{len(ID_POS_TEMPLATES)} positive x {len(ID_NEG_TEMPLATES)} negative "
      f"= {_id_surface} in-distribution realisations per clause shape; "
      f"{_ood_surface} disjoint paraphrase realisations.")
assert not (set(ID_CLAUSE_TEMPLATES) & set(OOD_CLAUSE_TEMPLATES)), \
    "paraphrase templates must be disjoint from training templates"
assert not (set(ID_POS_TEMPLATES) & set(OOD_POS_TEMPLATES)), \
    "paraphrase literal templates must be disjoint from training templates"

PAD_ID = 0
_MAX_VARS = max(CFG.n_vars, CFG.ood_n_vars)

def _build_global_vocab(max_vars):
    words, seen = ["<PAD>"], {"<PAD>"}
    def add(w):
        if w not in seen: seen.add(w); words.append(w)
    for v in range(max_vars):
        for w in predicate_phrase(v).split(): add(w)
    for w in sorted(_ID_WORDS | _OOD_WORDS): add(w)
    return {w: i for i, w in enumerate(words)}

VOCAB = _build_global_vocab(_MAX_VARS)
ID2TOK = {i: w for w, i in VOCAB.items()}
assert VOCAB["<PAD>"] == 0

def _encode(text):
    return [VOCAB[w] for w in text.split()]

def verbalize_clause(clause, *, style, rng):
    """ONE clause -> ONE templated English sentence (token ids)."""
    if style == "paraphrase":
        cl_t, neg_t, pos_t = OOD_CLAUSE_TEMPLATES, OOD_NEG_TEMPLATES, OOD_POS_TEMPLATES
    else:
        cl_t, neg_t, pos_t = ID_CLAUSE_TEMPLATES, ID_NEG_TEMPLATES, ID_POS_TEMPLATES
    tmpl, sep = cl_t[int(rng.integers(len(cl_t)))]
    lits = list(np.nonzero(clause)[0])
    if style == "paraphrase":
        rng.shuffle(lits)                      # unseen literal order inside the sentence
    parts = []
    for v in lits:
        bank = pos_t if clause[v] > 0 else neg_t
        parts.append(bank[int(rng.integers(len(bank)))].replace("{P}", predicate_phrase(int(v))))
    return _encode(tmpl.replace("{LITS}", sep.join(parts)))

def _max_clause_tokens():
    tw = max(len(t[0].split()) - 1 for t in ID_CLAUSE_TEMPLATES + OOD_CLAUSE_TEMPLATES)
    lw = max(len(t.split()) - 1 for t in ID_NEG_TEMPLATES + OOD_NEG_TEMPLATES
             + ID_POS_TEMPLATES + OOD_POS_TEMPLATES)
    pw = max(len(predicate_phrase(v).split()) for v in range(_MAX_VARS))
    return tw + CFG.clause_len * (lw + pw) + (CFG.clause_len - 1) + 2

CLAUSE_TOKENS = _max_clause_tokens()

def build_split(n, n_vars, n_clauses, clause_len, *, style, seed, planted=False):
    """-> X (n, n_clauses, CLAUSE_TOKENS), Yc (n, n_clauses, n_vars) in EMISSION order."""
    rng = np.random.default_rng(seed)
    X = np.zeros((n, n_clauses, CLAUSE_TOKENS), dtype=np.int64)
    Yc = np.zeros((n, n_clauses, n_vars), dtype=np.float32)
    tries_total, truncated = 0, 0
    for i in range(n):
        if planted:
            clauses = _make_planted_instance(n_vars, n_clauses, clause_len, rng); tries_total += 1
        else:
            clauses, t = _make_random_sat_instance(n_vars, n_clauses, clause_len, rng)
            tries_total += t
            if clauses is None:
                clauses = _make_planted_instance(n_vars, n_clauses, clause_len, rng)
        order = list(range(n_clauses))
        if style == "paraphrase":
            rng.shuffle(order)                 # unseen clause ORDER, gold follows (a CNF is a set)
        for j, ci in enumerate(order):
            toks = verbalize_clause(clauses[ci], style=style, rng=rng)
            if len(toks) > CLAUSE_TOKENS:
                truncated += 1; toks = toks[:CLAUSE_TOKENS]
            X[i, j, :len(toks)] = toks
            Yc[i, j] = clauses[ci]
    return X, Yc, dict(mean_tries=tries_total / max(n, 1), truncated=truncated)

def _fingerprint(cfg):
    key = (f"probe{getattr(cfg,'probe','A')}_"
           f"{cfg.n_vars}_{cfg.n_clauses}_{cfg.clause_len}_{cfg.sat_ratio}_{cfg.use_planted}_"
           f"{cfg.n_train}_{cfg.n_eval}_{cfg.ood_n_vars}_perclause_emissionorder_v5")
    return hashlib.md5(key.encode()).hexdigest()[:8]

CACHE_FILE = os.path.join(DATA_DIR, f"dataset_{_fingerprint(CFG)}.npz")

if os.path.exists(CACHE_FILE):
    print("Loading dataset from cache:", CACHE_FILE)
    z = np.load(CACHE_FILE)
    train = (z["Xtr"], z["Ytr"]); eval_id = (z["Xid"], z["Yid"])
    eval_para = (z["Xpa"], z["Ypa"]); eval_scale = (z["Xsc"], z["Ysc"])
    GEN_STATS = json.loads(str(z["stats"]))
else:
    print(f"Generating dataset: uniform random {CFG.clause_len}-SAT at ratio {CFG.sat_ratio} "
          f"(planted={CFG.use_planted}) ...")
    _p = CFG.use_planted
    Xtr, Ytr, s_tr = build_split(CFG.n_train, CFG.n_vars, CFG.n_clauses, CFG.clause_len,
                                 style="canonical", seed=100, planted=_p)
    Xid, Yid, s_id = build_split(CFG.n_eval, CFG.n_vars, CFG.n_clauses, CFG.clause_len,
                                 style="canonical", seed=200, planted=_p)
    Xpa, Ypa, s_pa = build_split(CFG.n_eval, CFG.n_vars, CFG.n_clauses, CFG.clause_len,
                                 style="paraphrase", seed=300, planted=_p)
    n_cl_sc = min(int(round(CFG.ood_n_vars * CFG.sat_ratio)), CFG.n_clauses)
    Xsc, Ysc, s_sc = build_split(CFG.n_eval, CFG.ood_n_vars, n_cl_sc, CFG.clause_len,
                                 style="canonical", seed=400, planted=_p)
    train, eval_id, eval_para, eval_scale = (Xtr, Ytr), (Xid, Yid), (Xpa, Ypa), (Xsc, Ysc)
    GEN_STATS = dict(train=s_tr, id=s_id, paraphrase=s_pa, scale=s_sc,
                     sat_ratio=CFG.sat_ratio, planted=CFG.use_planted)
    np.savez_compressed(CACHE_FILE, Xtr=Xtr, Ytr=Ytr, Xid=Xid, Yid=Yid, Xpa=Xpa, Ypa=Ypa,
                        Xsc=Xsc, Ysc=Ysc, stats=json.dumps(GEN_STATS))
    print("Saved dataset cache:", CACHE_FILE)

VOCAB_SIZE = len(VOCAB)
SEQ_LEN = train[0].shape[2]                    # tokens PER CLAUSE
print(f"Train {train[0].shape} | Eval-ID {eval_id[0].shape} | "
      f"OOD-para {eval_para[0].shape} | OOD-scale {eval_scale[0].shape}")
print(f"Vocab {VOCAB_SIZE} word types (pad=0) | tokens per clause {SEQ_LEN} | "
      f"clause/var ratio {CFG.sat_ratio} | planted={CFG.use_planted}")
_trunc = sum(v["truncated"] for k, v in GEN_STATS.items() if isinstance(v, dict))
print(f"Truncated clause sentences: {_trunc} (must be 0 -- CLAUSE_TOKENS is a worst-case bound)")
print(f"Rejection sampling: mean {GEN_STATS['train']['mean_tries']:.2f} draws per satisfiable "
      f"training instance")
print("\nExample rendered clause (instance 0, clause 0):")
print("  " + " ".join(ID2TOK[t] for t in train[0][0][0] if t != PAD_ID))
print("  gold row:", train[1][0][0].astype(int).tolist())


## Cell 6 — Translator `f_theta` and the straight-through Gumbel-Softmax bridge

Each clause is a single templated English sentence, and sentences are encoded
**independently**: `X` has shape `(n, n_clauses, CLAUSE_TOKENS)` and the encoder
runs over `n_clauses` short sequences rather than one long stream. Sentence
segmentation is supplied as preprocessing, as in SATLM, Logic-LM and LINC.

Inside a sentence, **one attention query per variable** reads the encoded tokens
and a shared `Linear(d, 3)` emits that variable's distribution over {negative,
absent, positive}. The straight-through Gumbel-Softmax bridge then yields a hard
{−1, 0, +1} clause matrix in the forward pass while passing gradients through
the soft distribution — the mechanism by which the logic layer trains the
translator.

Evaluation uses `deterministic_onehot` (plain argmax, no Gumbel noise), so a
trained model scores identically on repeated evaluation.

Per-sentence encoding also removes the quadratic term in instance width: cost is
`n_clauses * L^2` rather than `(n_clauses * L)^2`.


In [ ]:
# === Cell 6: per-clause sentence encoder + ST-GS bridge + deterministic eval ===
import torch.nn as nn, torch.nn.functional as F

class Translator(nn.Module):
    """One templated English sentence -> 3-way logits over {neg, absent, pos} for every variable.

    v2 encoded the whole instance as one ~1764-token stream and asked n_clauses learned
    queries to each locate "the i-th clause". Clause sentences vary in length, so no query could
    key on position; it would have to count delimiters across the stream with two attention
    layers. Empirically it never learned: all four conditions scored exactly the closed-form F1 of
    an information-free CNF.

    Now each clause SENTENCE is encoded independently (segmentation supplied as preprocessing,
    as in SATLM / Logic-LM / LINC). One shared query token pools the sentence; the output row is
    determined by which slice the sentence came from, so no positional counting is required. The
    model still has to solve the actual task: which variable does this predicate phrase denote,
    and is it asserted or negated.

    Compute per instance drops from O(T^2) = 1764^2 to n_clauses * L^2 = 40 * 44^2, about 80x.
    """
    def __init__(self, vocab, clause_tokens, n_clauses, n_vars, cfg):
        super().__init__()
        self.n_clauses, self.n_vars = n_clauses, n_vars
        d = cfg.d_model
        self.embed = nn.Embedding(vocab, d, padding_idx=0)
        self.pos = nn.Parameter(torch.randn(1, clause_tokens, d) * 0.02)
        enc = nn.TransformerEncoderLayer(d, cfg.n_heads, dim_feedforward=d * 4,
                                         batch_first=True, dropout=0.1,
                                         norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc, cfg.n_layers, enable_nested_tensor=False)
        self.var_q = nn.Parameter(torch.randn(1, n_vars, d) * 0.02)
        self.cross = nn.MultiheadAttention(d, cfg.n_heads, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, 3)

    def forward(self, x):                       # x: (B, n_clauses, L)
        B, Cc, L = x.shape
        flat = x.reshape(B * Cc, L)
        pad = (flat == 0)
        # a sentence of only padding would make softmax undefined; keep its first slot visible
        pad[:, 0] = False
        h = self.embed(flat) + self.pos[:, :L]
        h = self.encoder(h, src_key_padding_mask=pad)
        q = self.var_q.expand(B * Cc, -1, -1)          # (B*Cc, n_vars, d)
        c, _ = self.cross(q, h, h, key_padding_mask=pad)
        c = self.norm(c + q)
        return self.head(c).view(B, Cc, self.n_vars, 3)

def st_gumbel_softmax(logits, tau, hard=True):
    """TRAINING path: Gumbel noise + optional straight-through hard forward."""
    y_soft = F.gumbel_softmax(logits, tau=tau, hard=False, dim=-1)
    if not hard:
        return y_soft
    idx = y_soft.argmax(dim=-1, keepdim=True)
    y_hard = torch.zeros_like(y_soft).scatter_(-1, idx, 1.0)
    return y_hard + (y_soft - y_soft.detach())

def deterministic_onehot(logits):
    """EVAL path: plain argmax one-hot, NO Gumbel noise, so evaluation is reproducible."""
    idx = logits.argmax(dim=-1, keepdim=True)
    return torch.zeros_like(logits).scatter_(-1, idx, 1.0)

SIGN = torch.tensor([-1.0, 0.0, 1.0], device=DEVICE)
def to_clause_matrix(onehot):
    return (onehot * SIGN).sum(-1)

def tile_clause_matrix_to_width(cm, target_vars):
    """Scale-OOD: replicate the learned n_vars-wide window across the full instance width."""
    B, Cc, V = cm.shape
    if target_vars <= V:
        return cm[:, :, :target_vars]
    reps = (target_vars + V - 1) // V
    return cm.repeat(1, 1, reps)[:, :, :target_vars]

print("Translator (per-clause sentence encoder) + ST-GS bridge + deterministic eval defined.")


## Cell 7 — Differentiable soft-MaxSAT logic layer `L`

Maintains a continuous assignment in [−1, 1] and refines it for `refine_steps`
gradient steps against a smooth satisfaction surrogate,
`P(clause satisfied) = 1 − prod over present literals of P(literal false)`.

Callers that hold the emitting distribution pass a **soft presence**
(`p_pos + p_neg`) so the absent class stays differentiable and an empty clause
matrix retains a gradient route out; callers that only have a hard CNF —
evaluation, the baselines, the completeness probe — pass nothing and get exact
hard behaviour.

**The backpropagation window (`refine_backprop_steps`).** The final K refinement
steps stay in the autograd graph and earlier steps are detached: truncated
backpropagation through the solver. K is a config field because the trade-off
between gradient magnitude and memory is real.

**Scope of the relaxation.** It cannot certify unsatisfiability. The
completeness probe measures how often it signals UNSAT on Z3-certified UNSAT
instances, which is why every reported decision comes from Z3.


In [ ]:
# === Cell 7: differentiable soft-MaxSAT logic layer ===
_PROD_KERNEL_OK = [True]   # REVISION v10: set to False if nvrtc cannot build the prod kernel


def _check_prod_kernel():
    """One-off probe so the fallback is chosen before training, not inside a run."""
    if DEVICE.type != "cuda":
        return True
    try:
        torch.prod(torch.rand(2, 3, 4, device=DEVICE), dim=-1)
        return True
    except RuntimeError as e:
        _PROD_KERNEL_OK[0] = False
        print("torch.prod is not usable on this runtime "
              f"({str(e).strip().splitlines()[-1][:100]}).")
        print("The soft-MaxSAT layer will use the mathematically equivalent exp-log form.")
        return False


class SoftMaxSATLayer(nn.Module):
    """Relaxed MaxSAT refinement. Differentiable w.r.t. the clause matrix, so gradients reach
    the translator through the ST-GS bridge. Records refinement steps for H4."""

    def __init__(self, refine_steps, step_size=0.5, init_seed=1234, backprop_steps=None):
        super().__init__()
        self.refine_steps = refine_steps
        self.step_size = step_size
        self.init_seed = init_seed          # deterministic starting assignment
        self.backprop_steps = (refine_steps if backprop_steps is None
                               else max(1, min(backprop_steps, refine_steps)))

    def soft_satisfaction(self, clause_mat, assign, present=None):
        """P(clause satisfied) = 1 - prod_{present literals} P(literal false).

        the old expression was

            1.0 - prod(1.0 - lit_true * present + (1 - present))

        For an ABSENT variable (present = 0) the factor evaluates to 1 - 0 + 1 = 2, not 1.
        With V=12 and clause_len=3 that multiplies the product by 2^9 = 512, so the returned
        "satisfaction" left [0,1] entirely (verified: a 3-literal clause returned -11.9).
        Downstream, task_loss did `final_sat.clamp(1e-4, 1-1e-4)`; clamp passes ZERO gradient
        outside its range, so L_task contributed no gradient at all -- the logic layer never
        trained the translator, and the paper's central mechanism was inert. It also made
        Cell U's `final_sat.min() < 0.5` UNSAT signal fire on essentially everything.

        Absent variables must simply not participate in the product.
        """
        lit = clause_mat * assign.unsqueeze(1)               # (B, C, V)
        lit_true = torch.sigmoid(4.0 * lit)                  # P(literal true)
        if present is None:
            present = (clause_mat.abs() > 0).float()
        p_false = 1.0 - lit_true * present                   # absent -> factor exactly 1.0
        # REVISION v10: `torch.prod` over the variable axis is executed by PyTorch's runtime
        # kernel compiler (nvrtc). Some Colab images ship a torch build whose
        # libnvrtc-builtins is missing, and the call then fails with a compilation error
        # instead of a numerical one. The clause product is therefore computed as
        # exp(sum(log(.))) whenever the fused kernel is unavailable; the two are
        # mathematically identical here because the factors are already clamped to
        # [1e-7, 1], and the switch happens once per process.
        p_clamped = p_false.clamp(1e-7, 1.0)
        if _PROD_KERNEL_OK[0]:
            try:
                return 1.0 - torch.prod(p_clamped, dim=-1)
            except RuntimeError as _e:
                _PROD_KERNEL_OK[0] = False
                print("  [torch.prod unavailable on this runtime "
                      f"({str(_e).strip().splitlines()[-1][:80]}); "
                      "switching to the exp-log form for the rest of the run]")
        sat = 1.0 - torch.exp(torch.log(p_clamped).sum(dim=-1))
        return sat

    def forward(self, clause_mat, present=None):
        B, Cc, V = clause_mat.shape
        steps_used = torch.full((B,), float(self.refine_steps),
                                device=clause_mat.device, dtype=torch.float)
        with torch.enable_grad():
            # deterministic, device-local initialisation (was torch.randn -> non-reproducible eval)
            g = torch.Generator(device="cpu").manual_seed(self.init_seed)
            assign = (0.01 * torch.randn(B, V, generator=g)).to(clause_mat.device)
            prev = None
            _track = clause_mat.requires_grad or (present is not None and present.requires_grad)
            _first_kept = self.refine_steps - (self.backprop_steps if _track
                                               else self.refine_steps + 1)
            for s in range(self.refine_steps):
                _keep = _track and s >= _first_kept
                if not _keep:
                    assign = assign.detach().requires_grad_(True)
                elif assign.is_leaf and not assign.requires_grad:
                    assign = assign.requires_grad_(True)
                sat = self.soft_satisfaction(clause_mat, assign, present).mean(dim=1)
                grad = torch.autograd.grad(sat.sum(), assign, create_graph=_keep)[0]
                assign = torch.tanh(assign + self.step_size * grad)
                cur = self.soft_satisfaction(clause_mat, assign, present).mean(dim=1)
                if prev is not None:
                    improved = (cur - prev) > 1e-4
                    steps_used = torch.where(improved, steps_used,
                                  torch.minimum(steps_used, torch.full_like(steps_used, float(s))))
                prev = cur.detach()
        final_sat = self.soft_satisfaction(clause_mat, assign, present)
        return assign, final_sat, steps_used

print("Soft-MaxSAT layer defined; satisfaction now in (0,1) so L_task carries real gradient.")
print("Completeness caveat unchanged: the relaxation cannot certify UNSAT (measured in Cell U).")

_check_prod_kernel()


## Cell 8 — Losses, metrics, and the guards that keep them honest

**The solver objective is anchored, not self-referential.** `anchored_task_loss`
solves the *predicted* CNF and scores the resulting assignment against the
**gold** clauses. Asking the model to satisfy its own clauses is trivially gamed
by emitting permissive ones; the anchoring probe measures the correlation
between fidelity and each objective.

**Faithfulness is reported density-controlled.** For an information-free clause
matrix at density *d* against gold density *g*, expected corpus F1 is
`g·d/(g+d)`, strictly increasing in *d*. Raw F1 therefore pays a model for
emitting denser clauses. `faith_excess` subtracts that closed-form baseline,
computed from the gold and predicted densities **of the tensors the F1 was
scored on**, which differ per split.

**The end-task metric is chance-normalised.** `csat_norm` rescales gold-clause
satisfaction so a random assignment scores 0 and a perfect one 1, with the
chance level *measured* per split rather than assumed — rejection-sampling for
satisfiability lifts it above the analytic 1 − 2^−k.

**Anti-degeneracy guards.** A class-balanced faithfulness NLL, a one-sided
density **hinge** that is exactly zero across a wide band and resists only
genuinely empty or saturated CNFs, and an L2 on the emitted logits that keeps
the model out of the saturated regime.


In [ ]:
# === Cell 8: losses with anti-collapse terms + corpus metrics + collapse diagnostics ===
def task_loss(final_sat):
    """Expected UNSATISFACTION of the emitted clauses: minimise 1 - sat.

    This was `-log(sat.clamp_min(1e-6))`. An empty clause is unsatisfiable by
    definition, so -log(sat) is genuinely singular there, and every attempt to tame it with a
    clamp reintroduced a zero-gradient region at exactly the point the optimiser needs to escape
    from -- twice now, in earlier development and again in earlier development. A linear objective has constant gradient
    -1 with respect to satisfaction, is finite everywhere including sat = 0, needs no clamp, and
    is the standard MaxSAT relaxation objective (maximise the expected number of satisfied
    clauses) rather than a surrogate for it. It is also gentler on nearly-unsatisfied clauses,
    which is what let a single bad clause dominate the batch under -log.
    """
    return (1.0 - final_sat).mean()

GOLD_DENSITY = CFG.clause_len / CFG.n_vars
LAMBDA_DENSITY = 0.5                       # weight of the anti-collapse hinge (swept in Cell 9b)

def _class_weights_from_gold(gold_clause, eps=1e-6):
    """Inverse-frequency weights over {neg, absent, pos}, normalised to mean 1 under the prior."""
    idx = (gold_clause + 1).long().reshape(-1)
    counts = torch.bincount(idx, minlength=3).float().clamp_min(eps)
    freq = counts / counts.sum()
    w = 1.0 / freq
    return w / (freq * w).sum()            # sum_c freq_c * w_c == 1

FAITH_CLASS_WEIGHTS = _class_weights_from_gold(
    torch.from_numpy(train[1][:512])).to(DEVICE)
print(f"Faithfulness class weights [neg, absent, pos] = "
      f"{[round(float(x), 3) for x in FAITH_CLASS_WEIGHTS]} "
      f"(gold prior: absent {1-GOLD_DENSITY:.2f} / pos {GOLD_DENSITY/2:.2f} / "
      f"neg {GOLD_DENSITY/2:.2f})")

LAMBDA_LOGIT = 1e-4        # L2 on emitted logits; swept alongside lambda_density in Cell 9b

def logit_penalty(logits):
    """Keep the emitting distribution out of the saturated regime where every corrective
    gradient is scaled to zero. Not a regulariser for generalisation -- an optimiser guard."""
    return logits.float().pow(2).mean()

def anchored_task_loss(assign, gold_clause, layer):
    """Expected unsatisfaction OF THE GOLD CLAUSES under the assignment the solver derived from
    the model's predicted CNF. Differentiable end to end: gold <- assign <- clause_mat <- theta."""
    return (1.0 - layer.soft_satisfaction(gold_clause, assign)).mean()

def faithfulness_loss(pred_onehot, gold_clause, weights=None, logits=None,
                      instance_mask=None):
    """REVISION v2 (M1). v1 evaluated the NLL on the bridge OUTPUT. Under the straight-through
    bridge that output is exactly 0/1 in the forward pass, and clamp_min(1e-6) passes no gradient
    for the zero entries, so every slot whose sampled class differed from gold received NO
    corrective gradient. The loss is now evaluated on log_softmax(logits) for every condition,
    which makes per-slot supervision identical across bridges; the bridge now only changes the
    path of the solver gradient and the density hinge. `pred_onehot` is kept for backward
    compatibility and used only when logits are not supplied."""
    gold_idx = (gold_clause + 1).long()
    if logits is not None:
        logp = F.log_softmax(logits.float(), dim=-1)
    else:
        logp = torch.log(pred_onehot.clamp_min(1e-6))
    w = FAITH_CLASS_WEIGHTS if weights is None else weights
    if instance_mask is None:
        return F.nll_loss(logp.reshape(-1, 3), gold_idx.reshape(-1), weight=w)
    # REVISION v4 (partial supervision): the class-balanced NLL is averaged over the
    # supervised instances only. Instances outside the supervised subset contribute
    # nothing to L_faith; their only gold information reaches the translator through
    # the solver, which is exactly the regime the curve is meant to probe.
    if instance_mask.sum() == 0:
        return logp.sum() * 0.0
    per_slot = F.nll_loss(logp.reshape(-1, 3), gold_idx.reshape(-1),
                          weight=w, reduction="none").reshape(gold_idx.shape)
    slot_w = w[gold_idx]
    m = instance_mask.to(per_slot.dtype).reshape(-1, *([1] * (per_slot.dim() - 1)))
    return (per_slot * m).sum() / (slot_w * m).sum().clamp_min(1e-8)

DENSITY_LO = 0.35      # hinge lower edge, as a fraction of the gold density
DENSITY_HI = 2.50      # hinge upper edge, as a multiple of the gold density

def density_penalty(onehot, target_density=None):
    """One-sided HINGE against the two degenerate regimes -- NOT a pull toward the gold density.

    The R3 version penalised (density - gold_density)^2, a two-sided pull. Combined with
    the class-balanced NLL it created a constant-output attractor: emit the gold MARGINAL for every
    slot, ignore the input entirely. That satisfies the density term exactly and is a local optimum
    of the balanced NLL for an uninformative predictor, so the model never has to look at the
    sentence. The FULL_SCALE run landed in it and the evidence is unambiguous:

        lambda_density  faith excess     density
              0.0          +0.1907        0.303      <- learned; NOT flagged degenerate
              0.1          -0.0001        0.375
              0.5          +0.0005        0.1875     <- exactly gold 3/16; the attractor
              2.0          +0.0825        0.160

    and at lambda_density=0.5 the SHUFFLED-LABEL negative control produced bit-identical metrics
    to the real run (F1 0.0943, density 0.1875), as did every clause/variable ratio and both the
    frozen and coformer conditions -- six different configurations, one identical output. The
    model had stopped depending on the data at all.

    The penalty now only acts near genuine collapse (a near-empty or near-saturated CNF) and is
    exactly zero across the whole useful range, so it cannot define an attractor.
    """
    t = GOLD_DENSITY if target_density is None else target_density
    lo, hi = DENSITY_LO * t, min(0.95, DENSITY_HI * t)
    dens = (onehot[..., 0] + onehot[..., 2]).mean()
    return (F.relu((lo - dens) / max(lo, 1e-6)) ** 2
            + F.relu((dens - hi) / max(1.0 - hi, 1e-6)) ** 2)

print(f"Anti-collapse hinge: penalty is zero for literal density in "
      f"[{DENSITY_LO*GOLD_DENSITY:.3f}, {min(0.95, DENSITY_HI*GOLD_DENSITY):.3f}] "
      f"(gold {GOLD_DENSITY:.3f}); it guards the empty- and saturated-CNF regimes only.")

@torch.no_grad()
def prediction_diagnostics(onehot, cm):
    """Collapse diagnostics reported for EVERY run."""
    dens = float((cm.abs() > 0).float().mean())
    pos = float((cm > 0).float().sum() / max(float((cm.abs() > 0).float().sum()), 1.0))
    p = onehot.reshape(-1, 3).mean(0).clamp_min(1e-9)
    ent = float(-(p * p.log()).sum() / np.log(3))            # 0 = one class, 1 = uniform
    return dens, pos, ent

# --- corpus-level faithfulness ---------------------------------------------------------------
@torch.no_grad()
def clause_f1_counts(pred_clause, gold_clause):
    p = (pred_clause.abs() > 0); g = (gold_clause.abs() > 0)
    tp = (p & g & (pred_clause == gold_clause)).sum().item()
    return tp, p.sum().item(), g.sum().item()

def f1_from_counts(tp, np_, ng):
    prec = tp / max(np_, 1); rec = tp / max(ng, 1)
    return 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)

# --- continuous, chance-normalised end-task metric ---------------------------------
@torch.no_grad()
def clause_satisfaction(assign, gold_clause):
    """Fraction of GOLD clauses satisfied by the hard assignment (per instance)."""
    hard = torch.sign(assign); hard[hard == 0] = 1.0
    lit = gold_clause * hard.unsqueeze(1)
    present = (gold_clause.abs() > 0).float()
    return (((lit > 0).float() * present).sum(-1) > 0).float().mean(dim=1)      # (B,)

def _measure_chance_csat(gold_np, n_draws=256, seed=0):
    """Chance level MEASURED on the actual gold, not assumed.

    The analytic value for uniform random k-SAT is 1 - 2^-clause_len (0.875 for 3-SAT), but our
    instances are REJECTION-SAMPLED for satisfiability, which skews the clause distribution and
    lifts the chance level above the analytic figure (measured +0.02 at n_vars=6). Normalising by
    the analytic constant would therefore credit a random assignment with a positive score --
    exactly the loophole csat_norm exists to close. We measure it per split instead."""
    g = torch.from_numpy(gold_np)
    gen = torch.Generator().manual_seed(seed)
    vals = []
    for _ in range(n_draws):
        a = torch.sign(torch.randn(g.shape[0], g.shape[2], generator=gen))
        a[a == 0] = 1.0
        vals.append(float(clause_satisfaction(a, g).mean()))
    return float(np.mean(vals))

ANALYTIC_CSAT = 1.0 - 0.5 ** CFG.clause_len
CHANCE_CSAT = {"id": _measure_chance_csat(eval_id[1]),
               "paraphrase": _measure_chance_csat(eval_para[1]),
               "scale": _measure_chance_csat(eval_scale[1])}
print(f"Chance clause satisfaction -- analytic {ANALYTIC_CSAT:.3f}, MEASURED per split: "
      + ", ".join(f"{k} {v:.4f}" for k, v in CHANCE_CSAT.items()))
print("csat_norm rescales each split by its OWN measured chance, so a random assignment scores "
      "0.000 and a perfect one 1.000 on every split.")
print(f"CAVEAT: the rescaling amplifies sampling noise by 1/(1-chance) ~ "
      f"{1/(1-CHANCE_CSAT['id']):.1f}x, so csat_norm needs a reasonably large n_eval "
      f"(currently {CFG.n_eval}) to be stable. Its across-seed sd is reported in every table.")

def csat_normalise(raw, split="id"):
    r = CHANCE_CSAT.get(split, ANALYTIC_CSAT)
    return (raw - r) / max(1.0 - r, 1e-6)

@torch.no_grad()
def end_task_accuracy(assign, gold_clause):
    """Secondary metric: exact full solve (all clauses satisfied)."""
    hard = torch.sign(assign); hard[hard == 0] = 1.0
    lit = gold_clause * hard.unsqueeze(1)
    present = (gold_clause.abs() > 0).float()
    return ((((lit > 0).float() * present).sum(-1) > 0).all(dim=1)).float().mean().item()

@torch.no_grad()
def per_instance_solved(assign, gold_clause):
    hard = torch.sign(assign); hard[hard == 0] = 1.0
    lit = gold_clause * hard.unsqueeze(1)
    present = (gold_clause.abs() > 0).float()
    return (((lit > 0).float() * present).sum(-1) > 0).all(dim=1)

# --- vectorised, order-independent clause-set comparison -------------------------------------
_CLAUSE_HASH_CACHE = {}
def _clause_hash_table(n_vars, device):
    key = (n_vars, str(device))
    if key not in _CLAUSE_HASH_CACHE:
        g = torch.Generator(device="cpu").manual_seed(20240917)
        _CLAUSE_HASH_CACHE[key] = torch.randint(1, 2**62, (n_vars, 3), generator=g,
                                                dtype=torch.int64).to(device)
    return _CLAUSE_HASH_CACHE[key]

@torch.no_grad()
def _clause_signatures(mat):
    B, Cc, V = mat.shape
    tbl = _clause_hash_table(V, mat.device)
    idx = (mat.long() + 1).clamp(0, 2)
    picked = torch.gather(tbl.unsqueeze(0).unsqueeze(0).expand(B, Cc, V, 3), 3,
                          idx.unsqueeze(-1)).squeeze(-1)
    return torch.sort(picked.sum(dim=-1), dim=1).values

@torch.no_grad()
def per_instance_translation_ok(pred_clause, gold_clause, tol=1):
    return (_clause_signatures(pred_clause) != _clause_signatures(gold_clause)).sum(dim=1) <= tol

@torch.no_grad()
def error_decomposition(pred_clause, assign, gold_clause):
    solved = per_instance_solved(assign, gold_clause)
    trans_ok = per_instance_translation_ok(pred_clause, gold_clause)
    return (solved.float().mean().item(),
            ((~solved) & (~trans_ok)).float().mean().item(),
            ((~solved) & trans_ok).float().mean().item())

print("Losses + metrics defined (class-balanced faith NLL, density prior, csat_norm anchor).")

# === parameter-free soft entailment surrogate (Part C training signal) ===
ENTAIL_SHARPNESS = 8.0
ENTAIL_MARGIN = 0.05

def soft_entailment_logits(s_with, s_against, k=ENTAIL_SHARPNESS, margin=ENTAIL_MARGIN):
    """Logits [True, False, Unknown] mirroring the two-query Z3 rule used at inference.
    Nothing here is learned, so no label prior can be absorbed."""
    diff = s_with - s_against
    return torch.stack([k * diff, -k * diff, k * (margin - diff.abs())], dim=-1)

# === real-data faithfulness loss (regime-sensitive, class-balanced, audited) ===
def faithfulness_loss_real(pred_onehot, gold_clauses, regime, n_clauses, n_vars, logits=None):
    """regime='full': class-balanced gold-CNF NLL.
       regime='proxy': NO faithfulness supervision exists -- returns zero.

    the `final_sat` argument became dead when removed the
    self-satisfaction proxy, and it was the ONLY consumer of a 50-step refinement pass
    that Part C still ran on every training example. Dropping it removes one of the
    three solver calls per example -- roughly a third of Part C's training cost -- for
    a quantity nothing reads.
    Returns (loss, used_gold, coverage)."""
    if regime == "full" and gold_clauses:
        gold = torch.zeros(n_clauses, n_vars, device=pred_onehot.device)
        n_lits = n_fit = 0
        for ci, cl in enumerate(gold_clauses[:n_clauses]):
            for lit in cl:
                n_lits += 1
                if abs(lit) <= n_vars:
                    gold[ci, abs(lit) - 1] = float(np.sign(lit)); n_fit += 1
        dropped = max(0, len(gold_clauses) - n_clauses)
        if n_fit > 0:
            w = _class_weights_from_gold(gold).to(pred_onehot.device)
            logp = (F.log_softmax(logits.float(), dim=-1) if logits is not None
                    else torch.log(pred_onehot.clamp_min(1e-6)))      # REVISION v2 (M1)
            loss = F.nll_loss(logp.reshape(-1, 3), (gold + 1).long().reshape(-1), weight=w)
            return loss, True, dict(lits_total=n_lits, lits_fit=n_fit, clauses_dropped=dropped)
    return torch.zeros((), device=pred_onehot.device), False, dict(
        lits_total=0, lits_fit=0, clauses_dropped=0)
print("faithfulness_loss_real defined (class-balanced on the real gold CNF too).")

def expected_f1_random(pred_density, gold_density):
    """Closed-form corpus F1 of an information-free clause matrix at a given literal density."""
    if pred_density <= 0 or gold_density <= 0:
        return 0.0
    return gold_density * pred_density / (gold_density + pred_density)

def faith_excess(f1, pred_density, gold_density=None):
    g = GOLD_DENSITY if gold_density is None else gold_density
    return f1 - expected_f1_random(pred_density, g)

print(f"Density-controlled faithfulness enabled. Noise baseline at gold density "
      f"{GOLD_DENSITY:.2f}: F1(d) = {GOLD_DENSITY:.2f}*d/({GOLD_DENSITY:.2f}+d)  "
      f"-> d=0.15: {expected_f1_random(0.15, GOLD_DENSITY):.3f}, "
      f"d=0.30: {expected_f1_random(0.30, GOLD_DENSITY):.3f}. "
      f"faith_excess = faith_F1 - that value.")


## Cell 9 — Train and evaluate one (condition, seed)

Trains one model under one condition and returns in-distribution and OOD
metrics, wall-clock, refinement steps, and collapse diagnostics. Training uses
`torch.amp` autocast (bf16 on A100/H100) with a `GradScaler` only for fp16; the
logic layer runs in full precision.

**The four conditions.** All are trained, all carry the anti-collapse hinge and
the logit penalty, and exactly one mechanism differs:

| condition | per-slot `L_faith` | solver-routed gold signal | bridge |
|---|---|---|---|
| `frozen` | yes | **detached** | straight-through |
| `coformer` | yes | yes | straight-through |
| `no_faith` | no | yes | straight-through |
| `no_stgs` | yes | yes | plain soft relaxation |

**What `no_faith` isolates.** Since the solver objective is anchored to the gold
clauses, `no_faith` is the condition whose gold supervision arrives **only
through the solver**, never per slot. `coformer` vs `no_faith` therefore
contrasts per-slot supervision on top of the solver path against the solver path
alone.

**Diagnostics recorded per run.** `pred_density` (argmax) beside
`pred_soft_density`, because an empty emitted CNF has two different causes — a
training collapse, or a model that learned the class marginal and is
argmax-"absent" everywhere. `collapse_step` dates a training collapse rather
than leaving it to be inferred.


In [ ]:
# === Cell 9: train + eval one (condition, seed), with collapse diagnostics ===
from torch.utils.data import TensorDataset, DataLoader

def _seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

def make_loader(split, bs, shuffle, seed=0, with_index=False):
    X, Yc = split
    _t = [torch.from_numpy(X), torch.from_numpy(Yc)]
    if with_index:                       # REVISION v4: needed for partial supervision
        _t.append(torch.arange(X.shape[0]))
    ds = TensorDataset(*_t)
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=False,
                      num_workers=2, pin_memory=torch.cuda.is_available(),
                      worker_init_fn=_seed_worker, generator=g)

def gumbel_rao_st(logits, tau, k=10):
    """REVISION v2 (M7): Gumbel-Rao straight-through estimator (Paulus, Maddison & Krause, 2021).
    Forward: a hard one-hot sample. Backward: the average of k relaxed samples whose Gumbel noise
    is drawn CONDITIONALLY on that hard sample, which lowers the variance of the ST gradient."""
    z = logits.float()
    with torch.no_grad():
        g = -torch.log(-torch.log(torch.rand_like(z).clamp(1e-10, 1 - 1e-10)))
        idx = (z + g).argmax(-1, keepdim=True)
        hard = torch.zeros_like(z).scatter_(-1, idx, 1.0)
    lse = torch.logsumexp(z, dim=-1, keepdim=True)
    soft_sum = 0.0
    for _ in range(k):
        u_top = torch.rand_like(lse).clamp(1e-10, 1 - 1e-10)
        top = lse - torch.log(-torch.log(u_top))                 # max ~ Gumbel(logsumexp z)
        u = torch.rand_like(z).clamp(1e-10, 1 - 1e-10)
        gj = z - torch.log(-torch.log(u))                        # Gumbel(z_j)
        trunc = -torch.log(torch.exp(-top) + torch.exp(-gj))     # truncated below `top`
        perturbed = hard * top + (1.0 - hard) * trunc
        # relaxed sample: noise is detached, so the gradient flows through the logits only
        soft_sum = soft_sum + torch.softmax((z + (perturbed - z).detach()) / tau, dim=-1)
    soft = soft_sum / k
    return hard + (soft - soft.detach())

def tau_schedule(ep, total, cfg):
    frac = ep / max(1, total - 1)
    return cfg.gumbel_tau_start + frac * (cfg.gumbel_tau_end - cfg.gumbel_tau_start)

# Condition semantics -- all four TRAINED; exactly one mechanism differs:
#   frozen    L_faith (+ density prior) only; solver gradient DETACHED  [DiLA analogue]
#   coformer  L_task through the logic layer + L_faith + density prior  [ours]
#   no_faith  L_task through the logic layer + density prior only
#   no_stgs   as coformer but plain soft relaxation instead of straight-through
def run_condition(condition, seed, cfg, return_model=False, eval_every=None,
                  eval_at=None, faith_fraction=1.0):
    """`eval_every` is used ONLY by the epoch calibration in Cell 9c. When it is
    None (every matrix run) the training loop is byte-identical to before, so
    calibration cannot perturb a reported result."""
    set_all_seeds(seed)
    model = Translator(VOCAB_SIZE, CLAUSE_TOKENS, cfg.n_clauses, cfg.n_vars, cfg).to(DEVICE)
    logic = SoftMaxSATLayer(cfg.refine_steps,
                            step_size=getattr(cfg, 'refine_step_size', 0.5),
                            backprop_steps=getattr(cfg, 'refine_backprop_steps', None)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
    # Honour cfg.n_train. Without this a caller that reduces the data budget
    # (the sensitivity sweeps) silently trains on the full split, so the
    # reduction saves nothing and the sweep is mispriced. For the matrix
    # cfg.n_train equals len(train[0]), so this slice is a no-op there.
    _tr = (train if cfg.n_train >= len(train[0])
           else (train[0][:cfg.n_train], train[1][:cfg.n_train]))
    # REVISION v4: `faith_fraction` < 1 gives per-slot supervision to a fixed random
    # subset of the training instances; the rest are trained through the solver alone.
    # The subset depends on the seed only, so it is identical across conditions and
    # the comparison stays paired.
    _n_tr = len(_tr[0])
    _partial = faith_fraction < 1.0
    _sup_mask = None
    if _partial:
        _rng_sup = np.random.default_rng(10_000 + seed)
        _k_sup = int(round(faith_fraction * _n_tr))
        _sup_mask = torch.zeros(_n_tr, dtype=torch.bool)
        if _k_sup > 0:
            _sup_mask[torch.from_numpy(
                _rng_sup.choice(_n_tr, size=_k_sup, replace=False))] = True
    loader = make_loader(_tr, cfg.batch_size, True, seed=seed, with_index=_partial)
    use_scaler = cfg.USE_AMP and AMP_DTYPE == torch.float16
    scaler = torch.amp.GradScaler(AMP_DEVICE, enabled=use_scaler)

    # Full 2x2 factorial over (bridge) x (solver gradient), plus the two
    # faithfulness ablations. `use_stgs` selects the straight-through bridge;
    # `solver_grad` selects whether the logic-layer gradient reaches theta.
    #
    #   condition      bridge            solver grad   per-slot L_faith
    #   frozen         straight-through  detached      yes
    #   coformer       straight-through  routed        yes
    #   no_faith       straight-through  routed        no
    #   no_stgs        soft              routed        yes
    #   soft_frozen    soft              detached      yes   <- completes the cell
    #   soft_no_faith  soft              routed        no    <- completes the cell
    #
    # soft_frozen is the decisive control: if it also solves the task, the
    # advantage of no_stgs is attributable to the bridge alone and the solver
    # gradient contributes nothing. If it falls short, the solver gradient
    # contributes GIVEN a usable bridge.
    SPEC = {
        "frozen":        dict(stgs=True,  solver=False, faith=True),
        "coformer":      dict(stgs=True,  solver=True,  faith=True),
        "no_faith":      dict(stgs=True,  solver=True,  faith=False),
        "no_stgs":       dict(stgs=False, solver=True,  faith=True),
        "soft_frozen":   dict(stgs=False, solver=False, faith=True),
        "soft_no_faith": dict(stgs=False, solver=True,  faith=False),
        # REVISION v2 (M7): low-variance straight-through estimator, run in the extension cell
        "grs_frozen":    dict(stgs=True,  solver=False, faith=True, grs=True),
        "grs_routed":    dict(stgs=True,  solver=True,  faith=True, grs=True),
    }
    assert condition in SPEC, f"unknown condition {condition!r}; expected one of {sorted(SPEC)}"
    _sp = SPEC[condition]
    solver_grad, use_faith, use_stgs = _sp["solver"], _sp["faith"], _sp["stgs"]
    use_grs = _sp.get("grs", False)
    lam_dens    = getattr(cfg, "lambda_density", LAMBDA_DENSITY)

    _total = max(1, cfg.epochs * ((len(_tr[0]) + cfg.batch_size - 1) // cfg.batch_size))
    _warm = max(1, int(0.1 * _total))
    def _lr_at(step):
        if step < _warm:
            return (step + 1) / _warm
        prog = (step - _warm) / max(1, _total - _warm)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_at)

    t0 = time.time(); n_grad_steps = 0; collapse_step = None
    epoch_curve = []

    @torch.no_grad()
    def _probe_fidelity(n=512):
        """Cheap density-controlled fidelity on a fixed eval subsample."""
        was_training = model.training
        model.eval()
        X, Y = eval_id[0][:n], eval_id[1][:n]
        tp = np_ = ng = 0; pnz = gnz = slots = 0.0
        for b in range(0, len(X), cfg.batch_size):
            xb_ = torch.from_numpy(X[b:b + cfg.batch_size]).to(DEVICE)
            yb_ = torch.from_numpy(Y[b:b + cfg.batch_size]).to(DEVICE)
            cm_ = to_clause_matrix(deterministic_onehot(model(xb_).float()))
            a_, b_, c_ = clause_f1_counts(cm_, yb_)
            tp += a_; np_ += b_; ng += c_
            pnz += float((cm_.abs() > 0).sum()); gnz += float((yb_.abs() > 0).sum())
            slots += float(cm_.numel())
        if was_training:
            model.train()
        f1 = f1_from_counts(tp, np_, ng)
        return faith_excess(f1, pnz / max(slots, 1.0), gnz / max(slots, 1.0))

    for ep in range(cfg.epochs):
        tau = tau_schedule(ep, cfg.epochs, cfg); model.train()
        for _batch in loader:
            xb, ycb = _batch[0], _batch[1]
            _bmask = (_sup_mask[_batch[2]].to(DEVICE) if _partial else None)
            xb, ycb = xb.to(DEVICE, non_blocking=True), ycb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(AMP_DEVICE, dtype=AMP_DTYPE, enabled=cfg.USE_AMP):
                logits = model(xb)
                onehot = (gumbel_rao_st(logits, tau) if use_grs
                          else st_gumbel_softmax(logits.float(), tau, hard=use_stgs))
                clause_mat = to_clause_matrix(onehot)
            lam_logit = getattr(cfg, "lambda_logit", LAMBDA_LOGIT)
            cm_for_solver = clause_mat if solver_grad else clause_mat.detach()
            # soft presence keeps the empty-CNF escape route differentiable
            present_soft = (onehot[..., 0] + onehot[..., 2]).float() if solver_grad else None
            assign_tr, final_sat, _ = logic(cm_for_solver.float(), present=present_soft)
            loss = (lam_dens * density_penalty(onehot)      # anti-collapse, all conditions
                    + lam_logit * logit_penalty(logits))   # anti-saturation, all conditions
            if solver_grad:
                loss = loss + anchored_task_loss(assign_tr, ycb, logic)
            if use_faith:
                loss = loss + cfg.lambda_faith * faithfulness_loss(
                    onehot, ycb, logits=logits, instance_mask=_bmask)
            if n_grad_steps % 50 == 0:
                with torch.no_grad():
                    _d = float((clause_mat.abs() > 0).float().mean())
                if _d < 1e-6 and collapse_step is None:
                    collapse_step = n_grad_steps          # R12: date the collapse, don't infer it
            if use_scaler:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            sched.step(); n_grad_steps += 1
        _want = ((ep + 1) in eval_at) if eval_at else (
            eval_every and (ep + 1) % eval_every == 0)
        if eval_every and (_want or ep == cfg.epochs - 1):
            epoch_curve.append((ep + 1, float(_probe_fidelity())))
    train_secs = time.time() - t0
    assert n_grad_steps > 0, f"{condition}: no optimisation steps were taken"
    param_norm = sum(float(p.detach().float().norm()) for p in model.parameters())

    @torch.no_grad()
    def evaluate(split, split_key, want_errors=False):
        model.eval()
        tp = np_ = ng = 0; csats, accs, steps = [], [], []
        per_ex_csat = []          # per-example gold-clause satisfaction, for the
                                  # example-level cluster bootstrap in Cell 11b
        wrong_list, refine_list = [], []
        dens_l, pos_l, ent_l = [], [], []
        _pred_nz = _gold_nz = _slots = 0.0; soft_dens_l = []
        for xb, ycb in make_loader(split, cfg.batch_size, False, seed=seed):
            xb, ycb = xb.to(DEVICE), ycb.to(DEVICE)
            _lg = model(xb).float()
            soft_dens_l.append(float(torch.softmax(_lg, -1)[..., [0, 2]].sum(-1).mean()))
            onehot = deterministic_onehot(_lg)
            cm = to_clause_matrix(onehot)
            d, pf, en = prediction_diagnostics(onehot, cm)
            dens_l.append(d); pos_l.append(pf); ent_l.append(en)
            if cm.shape[2] != ycb.shape[2]:
                cm = tile_clause_matrix_to_width(cm, ycb.shape[2])
            if cm.shape[1] != ycb.shape[1]:
                k = min(cm.shape[1], ycb.shape[1]); cm, ycb = cm[:, :k], ycb[:, :k]
            assign, _, su = logic(cm)
            a, b, c = clause_f1_counts(cm, ycb); tp += a; np_ += b; ng += c
            _pred_nz += float((cm.abs() > 0).sum()); _gold_nz += float((ycb.abs() > 0).sum())
            _slots += float(cm.numel())
            _cs = clause_satisfaction(assign, ycb)
            per_ex_csat.append(_cs.detach().cpu().numpy())
            csats.append(float(_cs.mean()))
            accs.append(end_task_accuracy(assign, ycb)); steps.append(su.mean().item())
            if want_errors:
                _, w, r = error_decomposition(cm, assign, ycb)
                wrong_list.append(w); refine_list.append(r)
        _f1 = f1_from_counts(tp, np_, ng)
        _per_ex = np.concatenate(per_ex_csat) if per_ex_csat else np.array([])
        _dens_scored = _pred_nz / max(_slots, 1.0)      # predicted density, as scored
        _gold_dens = _gold_nz / max(_slots, 1.0)        # gold density of THIS split
        out = dict(f1=_f1, excess=faith_excess(_f1, _dens_scored, _gold_dens),
                   gold_density=_gold_dens, scored_density=_dens_scored,
                   soft_density=float(np.mean(soft_dens_l)),
                   csat_raw=float(np.mean(csats)),
                   csat_norm=float(csat_normalise(np.mean(csats), split_key)),
                   acc=float(np.mean(accs)), steps=float(np.mean(steps)),
                   density=float(np.mean(dens_l)), pos_frac=float(np.mean(pos_l)),
                   entropy=float(np.mean(ent_l)))
        if want_errors:
            out.update(wrong=float(np.mean(wrong_list)), refine=float(np.mean(refine_list)))
        out["_per_example_csat"] = _per_ex
        return out

    # Evaluation is a FIXED cost that does not shrink with cfg.n_train, so at a
    # reduced training budget it dominates. Callers that cut n_train get a
    # proportionally cut evaluation set too, otherwise a "cheap" sweep run costs
    # nearly as much as a full one.
    _ev = (lambda sp: sp if cfg.n_eval >= len(sp[0])
           else (sp[0][:cfg.n_eval], sp[1][:cfg.n_eval]))
    e_id = evaluate(_ev(eval_id), "id", want_errors=True)
    e_pa = evaluate(_ev(eval_para), "paraphrase"); e_sc = evaluate(_ev(eval_scale), "scale")

    degenerate, why = False, []
    if e_id["density"] < 0.02:
        degenerate = True
        if e_id["soft_density"] > 0.05:
            why.append(f"argmax-of-the-marginal: emitted CNF is empty (density "
                       f"{e_id['density']:.3f}) but the SOFT density is {e_id['soft_density']:.3f} "
                       f"-- the model learned the class prior, not the instance")
        else:
            why.append(f"near-empty CNF,training collapsed (density {e_id['density']:.3f}, "
                       f"soft {e_id['soft_density']:.3f})")
    if e_id["density"] > 0.95:
        degenerate = True; why.append(f"near-saturated CNF (density {e_id['density']:.3f})")
    if e_id["entropy"] < 0.05:
        degenerate = True; why.append(f"single-class output (entropy {e_id['entropy']:.3f})")
    if abs(e_id["csat_norm"]) < 1e-4 and e_id["f1"] < 1e-6:
        degenerate = True; why.append("chance-level on both metrics")
    if e_id["excess"] < 0.01:
        degenerate = True
        why.append(f"density-matched noise (F1 {e_id['f1']:.3f} = the closed-form value for an "
                   f"information-free CNF at density {e_id['density']:.3f}; excess "
                   f"{e_id['excess']:+.3f})")

    anchor = e_id["csat_norm"]
    if anchor < CSAT_FLOOR:
        ret_pa = ret_sc = ood_rel_drop = float("nan")
    else:
        ret_pa = e_pa["csat_norm"] / anchor
        ret_sc = e_sc["csat_norm"] / anchor
        ood_rel_drop = 1.0 - 0.5 * (ret_pa + ret_sc)
        if ood_rel_drop < 0:
            ood_rel_drop = float("nan")

    rec = dict(condition=condition, seed=seed, faith_fraction=faith_fraction,
               faith_id=e_id["f1"], faith_paraphrase=e_pa["f1"], faith_scale=e_sc["f1"],
               faith_excess_id=e_id["excess"], faith_excess_paraphrase=e_pa["excess"],
               faith_excess_scale=e_sc["excess"],
               gold_density_id=e_id["gold_density"],
               gold_density_scale=e_sc["gold_density"],
               csat_norm_id=e_id["csat_norm"], csat_norm_paraphrase=e_pa["csat_norm"],
               csat_norm_scale=e_sc["csat_norm"], csat_raw_id=e_id["csat_raw"],
               acc_id=e_id["acc"], acc_paraphrase=e_pa["acc"], acc_scale=e_sc["acc"],
               refine_steps=e_id["steps"],
               ood_retention_para=ret_pa, ood_retention_scale=ret_sc, ood_rel_drop=ood_rel_drop,
               error_wrong_translation=e_id["wrong"], error_refine_failure=e_id["refine"],
               pred_density=e_id["density"], pred_soft_density=e_id["soft_density"],
               pred_pos_frac=e_id["pos_frac"],
               pred_entropy=e_id["entropy"],
               degenerate=degenerate, degenerate_why="; ".join(why),
               solver_gradient=solver_grad, faith_term=use_faith, straight_through=use_stgs,
               factor_bridge=("gumbel_rao" if use_grs else ("straight_through" if use_stgs else "soft")),
               factor_solver=("routed" if solver_grad else "detached"),
               factor_faith=("per_slot" if use_faith else "solver_only"),
               grad_steps=n_grad_steps, param_norm=round(param_norm, 2),
               collapse_step=(-1 if collapse_step is None else collapse_step),
               sat_ratio=cfg.sat_ratio, lambda_density=lam_dens,
               lambda_logit=lam_logit,
               refine_step_size=getattr(cfg, "refine_step_size", 0.5),
               refine_backprop_steps=getattr(cfg, "refine_backprop_steps", None),
               train_secs=train_secs)
    # per-example scores travel with the record (underscore keys are stripped
    # before persistence) so Cell 11b can resample examples as well as seeds
    rec["_per_example_csat_id"] = e_id.get("_per_example_csat")
    rec["_epoch_curve"] = epoch_curve
    if return_model:
        rec["_model"], rec["_logic"] = model, logic
    return rec

print("Train/eval routine defined.")
print("  primary end-task metric : csat_norm (chance = 0.000, perfect = 1.000)")
print("  secondary               : acc (exact full solve)")
print(f"  collapse diagnostics     : pred_density (gold {GOLD_DENSITY:.2f}), pred_entropy, "
      f"pred_pos_frac -> `degenerate` flag")


## Cell 9b — Extended controls: sweeps and a shuffled-label negative control

* **λ_faith sweep** — varies the weight of the faithfulness loss, so the result
  is not a knife-edge artifact of one hyper-parameter.
* **λ_density sweep** — documents the transition into and out of the collapse
  regime; `lambda_density = 0` removes the anti-collapse guard entirely.
* **Clause/variable ratio sweep** — instance-hardness curve, `frozen` vs
  `coformer`.
* **Negative control (shuffled labels)** — re-runs the full model with the gold
  structure randomly permuted. A sound method must *lose* its advantage here; if
  it does not, any apparent gain is an artifact.

These train extra models and are gated behind `RUN_EXTRA_DIAGNOSTICS`.


In [ ]:
# === Cell 9b: extended controls -- lambda sweep, DENSITY sweep, ratio sweep, negative control ===
import pandas as pd

DIAG_DIR = os.path.join(CKPT_DIR, f"diagnostics_{config_fingerprint(('diagnostics',))}")
assert_fingerprint("diagnostics", config_fingerprint(('diagnostics',)))   # REVISION v12
os.makedirs(DIAG_DIR, exist_ok=True)
_stale_diag = [d for d in os.listdir(CKPT_DIR)
               if d.startswith("diagnostics") and os.path.join(CKPT_DIR, d) != DIAG_DIR]
if _stale_diag:
    print(f"Diagnostic caches from other configurations are present and will NOT be reused: "
          f"{', '.join(_stale_diag)}")
    print(f"  this run writes to {os.path.basename(DIAG_DIR)}")

def _diag_cached_df(name, fn):
    """Cells 13c and 13e train 8 models between them (~5.6 h at FULL_SCALE) and, unlike
    these sweeps, were never cached -- so every re-run of the notebook, including one whose only
    purpose was to regenerate a table after a reporting fix, paid for them again. They now share
    this fingerprinted cache. Delete checkpoints/diagnostics_* to force recomputation."""
    path = os.path.join(DIAG_DIR, f"{name}.csv")
    if os.path.exists(path):
        print(f"  [cached] {name} -> {path}")
        return pd.read_csv(path)
    out = fn()
    atomic_csv(out, path, index=False)
    return out

def _diag_cached(name, fn):
    """Cache each diagnostic to Drive. At FULL_SCALE this cell trains 15 models; a dropped Colab
    session would otherwise discard every one of them. Keyed by configuration fingerprint."""
    path = os.path.join(DIAG_DIR, f"{name}.csv")
    if os.path.exists(path):
        print(f"  [cached] {name} -> {path}")
        return pd.read_csv(path)
    out = fn()
    atomic_csv(out, path, index=False)
    return out

# The sweeps establish that an ORDERING is stable across a knob, not an
# absolute number, so they do not need the production data budget. Running them
# at a reduced budget cuts their cost roughly threefold while preserving what
# they are read for. The reduction is applied uniformly across every point of a
# sweep, so within-sweep comparisons remain like-for-like, and it is recorded
# on every row so a sweep result can never be mistaken for a matrix result.
SWEEP_SCALE = 0.35            # fraction of the production training budget
SWEEP_EPOCH_SCALE = 0.6

def _cfg_with(sweep=False, **over):
    cfg = Config(**{**dataclasses.asdict(CFG), **over}).scaled()
    cfg.calibrated_epochs = CFG.calibrated_epochs
    cfg.epochs = CFG.epochs
    if sweep:
        cfg.n_eval = max(min(500, cfg.n_eval), int(cfg.n_eval * SWEEP_SCALE))
        # Floors are RELATIVE to the configuration in force. An absolute floor
        # would make a sweep larger than the matrix under a reduced preset,
        # which is how a cost saving turns into a cost increase.
        cfg.n_train = max(min(2000, cfg.n_train), int(cfg.n_train * SWEEP_SCALE))
        cfg.epochs = max(min(8, cfg.epochs), int(cfg.epochs * SWEEP_EPOCH_SCALE))
    return cfg

import contextlib


@contextlib.contextmanager
def _sweep_data(cfg):
    """Temporarily shrink the global splits to the sweep budget.

    `run_condition` reads the global `train` / `eval_*` tensors directly and
    takes only hyper-parameters from cfg, so setting cfg.n_train does NOT make a
    sweep cheaper: it still traverses the full training set every epoch and
    evaluates on the full 3 x n_eval. Slicing the globals here is what makes
    SWEEP_SCALE real -- without it the sweeps cost the same as matrix runs.

    Chance normalisation is unaffected: CHANCE_CSAT is measured once on the full
    splits and is not recomputed here, so csat_norm keeps the matrix scale and
    only becomes noisier. The originals are restored on exit, including on error.
    """
    global train, eval_id, eval_para, eval_scale
    _orig = (train, eval_id, eval_para, eval_scale)
    nt = min(int(cfg.n_train), len(train[0]))
    ne = min(int(cfg.n_eval), len(eval_id[0]))
    train = (train[0][:nt], train[1][:nt])
    eval_id = (eval_id[0][:ne], eval_id[1][:ne])
    eval_para = (eval_para[0][:ne], eval_para[1][:ne])
    eval_scale = (eval_scale[0][:ne], eval_scale[1][:ne])
    try:
        yield
    finally:
        train, eval_id, eval_para, eval_scale = _orig


def run_lambda_sweep(seed=0, lambdas=(0.0, 0.5, 1.0, 2.0)):
    rows = []
    for lam in lambdas:
        _c = _cfg_with(sweep=True, lambda_faith=lam)
        with _sweep_data(_c):
            rec = run_condition("coformer", seed, _c)
        rec["sweep_scale"] = SWEEP_SCALE
        rec["lambda_faith"] = lam
        rows.append({k: v for k, v in rec.items() if not k.startswith("_")})
    return pd.DataFrame(rows)

def run_density_sweep(seed=0, lambdas=(0.0, 0.1, 0.5, 2.0)):
    """shows the result is not an artifact of the anti-collapse regulariser.
    lambda_density = 0 reproduces the run-1 collapse; the sweep documents the transition."""
    rows = []
    for lam in lambdas:
        _c = _cfg_with(sweep=True, lambda_density=lam)
        with _sweep_data(_c):
            rec = run_condition("coformer", seed, _c)
        rec["sweep_scale"] = SWEEP_SCALE
        rec["lambda_density"] = lam
        rows.append({k: v for k, v in rec.items() if not k.startswith("_")})
    return pd.DataFrame(rows)

def run_ratio_sweep(seed=0, ratios=(2.0, 3.0, 4.0)):
    """R5: instance-hardness curve. Regenerates the splits per ratio in the PER-CLAUSE format and
    reports where co-training helps as instances harden."""
    global train, eval_id, eval_para, eval_scale, SEQ_LEN, CHANCE_CSAT
    saved = (train, eval_id, eval_para, eval_scale, SEQ_LEN, CHANCE_CSAT)
    rows = []
    try:
        for r in ratios:
            cfg_r = _cfg_with(sweep=True, sat_ratio=r)
            nc = cfg_r.n_clauses
            train = build_split(cfg_r.n_train, cfg_r.n_vars, nc, cfg_r.clause_len,
                                style="canonical", seed=100)[:2]
            eval_id = build_split(cfg_r.n_eval, cfg_r.n_vars, nc, cfg_r.clause_len,
                                  style="canonical", seed=200)[:2]
            eval_para = build_split(cfg_r.n_eval, cfg_r.n_vars, nc, cfg_r.clause_len,
                                    style="paraphrase", seed=300)[:2]
            eval_scale = build_split(cfg_r.n_eval, cfg_r.ood_n_vars,
                                     min(int(round(cfg_r.ood_n_vars * r)), nc),
                                     cfg_r.clause_len, style="canonical", seed=400)[:2]
            SEQ_LEN = train[0].shape[2]
            # REVISION v2 (M8): chance level re-measured on the regenerated splits
            CHANCE_CSAT = {"id": _measure_chance_csat(eval_id[1]),
                           "paraphrase": _measure_chance_csat(eval_para[1]),
                           "scale": _measure_chance_csat(eval_scale[1])}
            for cond in ("frozen", "coformer"):
                rec = run_condition(cond, seed, cfg_r)
                rec["sweep_ratio"] = r
                rows.append({k: v for k, v in rec.items() if not k.startswith("_")})
    finally:
        train, eval_id, eval_para, eval_scale, SEQ_LEN, CHANCE_CSAT = saved
    return pd.DataFrame(rows)

def run_negative_control(seed=0, condition="no_stgs"):
    """Shuffle the gold clause matrix across instances -> the faithfulness signal becomes noise.
    A valid method must NOT retain its advantage under shuffled supervision."""
    global train
    Xtr, Ytr = train
    perm = np.random.default_rng(seed).permutation(len(Ytr))
    saved = train
    try:
        train = (Xtr, Ytr[perm].copy())
        rec = run_condition(condition, seed, CFG)
    finally:
        train = saved
    # REVISION v2 (M8): the control is run on the condition that LEARNS (no_stgs by default);
    # a control on a near-chance condition cannot discriminate.
    rec["base_condition"] = condition
    rec["condition"] = f"{condition}_shuffled(neg-control)"
    return rec

_matrix_csv = os.path.join(CKPT_DIR, "results.csv")
_matrix_done = False
if os.path.exists(_matrix_csv):
    try:
        _m = pd.read_csv(_matrix_csv)
        _matrix_done = len(_m) >= 6 * len(CFG.seeds)
    except Exception:
        _matrix_done = False

_partc_csv = os.path.join(CKPT_DIR, "real_comparison.csv")
_partc_done = os.path.exists(_partc_csv)
# The SHUFFLED-LABEL NEGATIVE CONTROL IS NEVER GATED. It is the test that an
# apparent advantage is not an artifact of the supervision pipeline, so making
# it conditional on an unrelated stage lets that stage's failure silently
# disarm the study's most important falsification check. It costs one training
# run, and it runs whenever the matrix is complete.
NEGATIVE_CONTROL_ALWAYS = True

_n_cells = (sum(len(seeds_for(c)) for c in CONDITIONS)
            if "seeds_for" in dir() else 6 * len(CFG.seeds))
_gate = None
if RUN_EXTRA_DIAGNOSTICS and not _matrix_done:
    _gate = (f"the main results matrix is not complete "
             f"({len(_m) if os.path.exists(_matrix_csv) else 0}/{_n_cells} runs)")

# The control is INVOKED at the end of Cell 10, immediately after the matrix
# completes. This cell runs before the matrix, so invoking it here would
# defer it to a second Run All -- which is how it came to be skipped before.
neg_df = neg = None

# The sweeps are DEFERRED, not skipped: they depend on the matrix, which this
# cell precedes. Cell 10d runs them in the same session as soon as that
# dependency is met, so a Run All completes them without a second pass.
SWEEPS_PENDING = bool(RUN_EXTRA_DIAGNOSTICS and _gate is not None)

if _gate is not None:
    print(f"Deferred to Cell 10d (same session): {_gate}.")
    print("Cell 10d executes them automatically once the matrix is complete, "
          "budget permitting; nothing needs to be re-run by hand.")
    sweep_df = dens_df = ratio_df = None
elif RUN_EXTRA_DIAGNOSTICS:
    print("Lambda_faith sweep (seed 0)...")
    sweep_df = _diag_cached("lambda_sweep", lambda: run_lambda_sweep(seed=0))
    print(sweep_df[["lambda_faith", "faith_id", "faith_excess_id", "csat_norm_id", "pred_density",
                    "degenerate"]].round(3).to_string(index=False))
    sweep_df.to_csv(os.path.join(OUT_DIR, "lambda_sweep.csv"), index=False)

    print("\nLambda_density sweep (seed 0) -- lambda=0 should REPRODUCE the run-1 collapse:")
    dens_df = _diag_cached("density_sweep", lambda: run_density_sweep(seed=0))
    print(dens_df[["lambda_density", "faith_id", "faith_excess_id", "csat_norm_id", "pred_density",
                   "pred_entropy", "degenerate"]].round(3).to_string(index=False))
    dens_df.to_csv(os.path.join(OUT_DIR, "density_sweep.csv"), index=False)

    print("\nInstance-hardness (clause/variable ratio) sweep, frozen vs coformer:")
    ratio_df = _diag_cached("ratio_sweep", lambda: run_ratio_sweep(seed=0))
    print(ratio_df[["sweep_ratio", "condition", "faith_id", "faith_excess_id", "csat_norm_id",
                    "degenerate"]].round(3).to_string(index=False))
    ratio_df.to_csv(os.path.join(OUT_DIR, "ratio_sweep.csv"), index=False)

else:
    sweep_df = dens_df = ratio_df = None
    print("Skipped: RUN_EXTRA_DIAGNOSTICS is False (trains ~15 extra models; "
          "enable for the full pass).")


## Cell 9c — Epoch calibration

The training budget is the only remaining lever on cost: refinement depth was
measured to have no effect on step time, so total time is
`epochs x n_train / batch_size` multiplied by a fixed per-step cost.

Forty epochs was chosen conservatively, not measured. This cell measures it
once, on the condition that demonstrably learns, by recording
density-controlled fidelity against epoch and locating the plateau. The result
is applied **identically to every condition**, so the equal-budget property the
ablation lattice depends on is preserved: what changes is the size of the shared
budget, not its distribution.

The calibration is deliberately conservative. It takes the first epoch reaching
99% of the best fidelity observed, adds a 30% margin, and never goes below a
floor. If fidelity is still rising at the end of the calibration run, the
original budget is kept: an ambiguous curve must not be read as permission to
train less.


In [ ]:
# === Cell 9c: measure the training budget instead of assuming it ===
AUTO_EPOCHS = True
EPOCH_FLOOR = 12          # never calibrate below this many epochs
EPOCH_MARGIN = 1.30       # safety margin over the measured plateau
PLATEAU_FRAC = 0.99       # "plateau" = first epoch within this of the best

EPOCH_CALIBRATION = None
_cal_path = os.path.join(CKPT_DIR,
                         f"epoch_calibration_{config_fingerprint(('epochcal',))}.json")

assert_fingerprint("epochcal", config_fingerprint(('epochcal',)))   # REVISION v12

if not AUTO_EPOCHS:
    print("AUTO_EPOCHS disabled: using CFG.epochs as written "
          f"({CFG.epochs}).")
elif os.path.exists(_cal_path):
    EPOCH_CALIBRATION = json.load(open(_cal_path))
    CFG.epochs = int(EPOCH_CALIBRATION["epochs"])
    CFG.calibrated_epochs = CFG.epochs
    print(f"Epoch calibration loaded from cache: {CFG.epochs} epochs "
          f"(plateau at epoch {EPOCH_CALIBRATION['plateau_epoch']}).")
else:
    # REVISION v3: calibrate on BOTH bridges. v1 calibrated on `no_stgs` only; once the
    # faithfulness loss is corrected the straight-through conditions may converge more slowly,
    # and a budget fitted to the soft bridge alone would under-train them and re-introduce a
    # bridge confound through the training budget. Each condition's curve is cached on its
    # own, and the shared budget is the MAXIMUM of the per-condition budgets. A condition that
    # never leaves the information-free region within E0 epochs yields no plateau and is
    # ignored with a warning (training longer cannot be justified by a curve that shows no
    # learning at all); a condition still rising at E0 forces E0.
    _orig_epochs = CFG.epochs
    _probe_epochs = sorted({e for e in
                            list(range(1, min(9, _orig_epochs) + 1))
                            + list(range(8, _orig_epochs + 1, max(1, _orig_epochs // 8)))
                            + [_orig_epochs]})
    CAL_CONDITIONS = ("no_stgs", "coformer")

    def _calibrate(cond):
        _p = _cal_path.replace(".json", f"_{cond}.json")
        if os.path.exists(_p):
            return json.load(open(_p))
        ensure_budget(f"epoch calibration [{cond}]", 60 * 60)
        print(f"Calibrating on `{cond}`: {_orig_epochs} epochs, probes at {_probe_epochs}")
        _cal = run_condition(cond, 0, CFG, eval_every=1, eval_at=_probe_epochs)
        _curve = _cal.get("_epoch_curve") or []
        rec = dict(condition=cond, curve=[[int(e), round(float(v), 5)] for e, v in _curve])
        if len(_curve) < 3:
            rec.update(status="too_few_points", epochs=_orig_epochs)
        else:
            _eps = [e for e, _ in _curve]; _vals = [v for _, v in _curve]; _best = max(_vals)
            _k = max(2, len(_vals) // 4)
            _tail = sum(_vals[-_k:]) / _k
            _prev = sum(_vals[-2 * _k:-_k]) / _k if len(_vals) >= 2 * _k else _vals[0]
            if _best <= 0.01:
                rec.update(status="never_learned", epochs=None, best_excess=round(float(_best), 4))
            elif (_tail - _prev) > (1.0 - PLATEAU_FRAC) * _best:
                rec.update(status="still_rising", epochs=_orig_epochs, best_excess=round(float(_best), 4))
            else:
                _thr = PLATEAU_FRAC * _best; _plateau = _eps[-1]
                for _j, (_e, _v) in enumerate(_curve):
                    if all(_vv >= _thr for _, _vv in _curve[_j:]):
                        _plateau = _e; break
                rec.update(status="plateau", plateau_epoch=int(_plateau),
                           best_excess=round(float(_best), 4),
                           epochs=int(min(_orig_epochs, max(EPOCH_FLOOR,
                                                            math.ceil(_plateau * EPOCH_MARGIN)))))
        atomic_json(rec, _p, indent=2)
        print(f"  [{cond}] {rec['status']} -> epochs {rec['epochs']}")
        del _cal; _empty_cache()
        return rec

    _recs = [_calibrate(c) for c in CAL_CONDITIONS]
    _usable = [r["epochs"] for r in _recs if r["epochs"] is not None]
    for r in _recs:
        if r["status"] == "never_learned":
            print(f"  WARNING: `{r['condition']}` never left the information-free region in "
                  f"{_orig_epochs} epochs; it does not constrain the budget. Report this.")
    _new = int(max(_usable)) if _usable else _orig_epochs
    EPOCH_CALIBRATION = dict(original_epochs=_orig_epochs, epochs=_new, margin=EPOCH_MARGIN,
                             floor=EPOCH_FLOOR, per_condition=_recs,
                             plateau_epoch=max([r.get("plateau_epoch", 0) for r in _recs]))
    CFG.epochs = _new
    CFG.calibrated_epochs = _new
    atomic_json(EPOCH_CALIBRATION, _cal_path, indent=2)        # always cached, so later sessions never recalibrate
    send_alert("Epoch budget calibrated", json.dumps(EPOCH_CALIBRATION, indent=1)[:1500])

print(f"\nTraining budget in force: {CFG.epochs} epochs "
      f"= {CFG.epochs * ((CFG.n_train + CFG.batch_size - 1) // CFG.batch_size)} "
      f"optimiser steps per run.")


## Cell 10 — Run the matrix with checkpoint/resume and progress alerts
Iterates over conditions × seeds. **Each finished (condition, seed) is persisted to Drive**, so a
session timeout never loses work: re-running *Run all* reloads completed cells and continues from
the first unfinished one — letting the full study complete across one or several sessions. A
progress email fires every few results, and a completion email summarizes wall-clock. Any
exception triggers a crash alert (with traceback) before re-raising, so failures are never silent.

In [ ]:
# === Cell 10: full matrix with SCHEMA-CHECKED checkpoint/resume + email alerts ===

FACTORIAL_CONDITIONS = ["frozen", "coformer", "soft_frozen", "no_stgs"]
FACTORIAL_MIN_SEEDS = 4

# Seed allocation follows the hypotheses. H7/H8/H9 and the main effects are
# tested on the four factorial cells and need the full seed budget. The two
# faithfulness ablations enter only the H6 DOMINANCE check, which is a per-seed
# win share rather than a mean contrast, and a share is well estimated from a
# smaller sample. Spending the full budget on them buys precision no hypothesis
# uses, at a third of the cost of the whole matrix.
# REVISION v8: the completed run used the full seed budget for the faithfulness ablations,
# because at 4 seeds a contrast is untestable as soon as two differences tie. The stored
# matrix therefore has 6 x 12 rows, and this value must match it.
ABLATION_SEEDS = 12

def seeds_for(condition):
    seeds = list(CFG.seeds)
    if condition in FACTORIAL_CONDITIONS:
        return seeds
    return seeds[:min(ABLATION_SEEDS, len(seeds))]

def order_matrix_cells(conditions, seeds, done):
    """Return the outstanding (condition, seed) cells in priority order."""
    seeds = list(seeds)
    core = [c for c in FACTORIAL_CONDITIONS if c in conditions]
    rest = [c for c in conditions if c not in core]
    head, tail = seeds[:FACTORIAL_MIN_SEEDS], seeds[FACTORIAL_MIN_SEEDS:]
    plan = ([(c, s) for s in head for c in core] +                    # phase 1
            [(c, s) for s in head for c in rest if s in seeds_for(c)] +   # phase 2 (v3: respects ABLATION_SEEDS)
            [(c, s) for s in tail for c in conditions
             if s in seeds_for(c)])                                   # phase 3
    seen = set()
    out = []
    for cell in plan:
        if cell in done or cell in seen:
            continue
        seen.add(cell); out.append(cell)
    return out


import pandas as pd, hashlib, shutil
# Full 2x2 factorial (bridge x solver gradient) plus the two faithfulness
# ablations. `soft_frozen` is the cell that makes the headline contrast
# attributable: it isolates the bridge with the solver gradient detached.
CONDITIONS = ["frozen", "coformer", "no_faith", "no_stgs",
              "soft_frozen", "soft_no_faith"]
FACTORIAL_CELLS = {                      # (bridge, solver gradient) -> condition
    ("straight_through", "detached"): "frozen",
    ("straight_through", "routed"):   "coformer",
    ("soft",             "detached"): "soft_frozen",
    ("soft",             "routed"):   "no_stgs",
}
RESULTS_CSV = os.path.join(CKPT_DIR, "results.csv")

REQUIRED_COLUMNS = [
    "condition", "seed", "factor_bridge", "factor_solver", "factor_faith", "faith_id", "faith_excess_id", "csat_norm_id", "csat_norm_paraphrase", "csat_norm_scale",
    "acc_id", "acc_paraphrase", "acc_scale", "refine_steps", "ood_rel_drop",
    "error_wrong_translation", "error_refine_failure",
    "pred_density", "pred_soft_density", "pred_pos_frac", "pred_entropy", "degenerate",
    "grad_steps", "param_norm", "sat_ratio", "train_secs",
]

import inspect
def _core_code_hash():
    import hashlib as _h
    parts = []
    for obj in (SoftMaxSATLayer.soft_satisfaction, SoftMaxSATLayer.forward,
                task_loss, faithfulness_loss, density_penalty, faith_excess,
                expected_f1_random, clause_satisfaction, csat_normalise,
                Translator.forward, run_condition):
        try:
            parts.append(inspect.getsource(obj))
        except (OSError, TypeError):
            # REVISION v3: repr() contains a memory address, which changes every session and
            # would silently invalidate every checkpoint on resume.
            parts.append(getattr(obj, "__qualname__", type(obj).__name__))
    return _h.md5("".join(parts).encode()).hexdigest()[:8]

CODE_FP = _core_code_hash()
# REVISION v3: stable, configuration-level fingerprint (primary family config + LOSS_VERSION)
# shared by every per-probe and per-example file, so that write and read sites always agree.
PROBE_FP = config_fingerprint(REQUIRED_COLUMNS, extra="probe-files")
RUN_FP = config_fingerprint(REQUIRED_COLUMNS, extra=CODE_FP)
print(f"Fingerprint {RUN_FP} = config + core-code hash {CODE_FP}")
FP_FILE = RESULTS_CSV + ".fingerprint"

CHECKPOINT_ADOPTED = None
FP_CFG_FILE = FP_FILE + ".cfg"


def _checkpoint_is_compatible():
    """Decide whether stored rows may be resumed into.

    The fingerprint combines the experiment CONFIGURATION with a hash of the
    eleven functions that define measurement semantics. Both must guard the
    checkpoint, but they fail differently and must not be treated alike:

      * CONFIG changed  -> the stored rows describe a DIFFERENT experiment.
                           They are worthless here and are archived.
      * CODE changed only -> the rows may still be comparable, because many
                           edits are behaviour-neutral (a new optional
                           argument, a docstring, a print). Discarding tens of
                           GPU-hours for such an edit is as damaging a mistake
                           as resuming into an incompatible checkpoint.

    In the second case the checkpoint is ADOPTED and the decision is recorded
    in run_metadata.json, so the departure from the strict guard is visible in
    the released artefact rather than silent. Adoption requires the stored rows
    to carry the full required schema; anything less is archived.
    """
    global CHECKPOINT_ADOPTED
    if not os.path.exists(RESULTS_CSV):
        return True, ""

    stored = open(FP_FILE).read().strip() if os.path.exists(FP_FILE) else "(none)"
    cfg_now = config_fingerprint(REQUIRED_COLUMNS, extra="")
    cfg_stored = (open(FP_CFG_FILE).read().strip()
                  if os.path.exists(FP_CFG_FILE) else None)
    cols = list(pd.read_csv(RESULTS_CSV, nrows=1).columns)
    missing = [c for c in REQUIRED_COLUMNS if c not in cols]

    if stored == RUN_FP:
        return (False, f"missing columns {missing}") if missing else (True, "")

    if missing:
        return False, f"fingerprint mismatch and missing columns {missing}"

    if cfg_stored is None:
        return False, (f"fingerprint mismatch (stored {stored}, current "
                       f"{RUN_FP}) and no configuration fingerprint was "
                       f"recorded alongside it, so config and code changes "
                       f"cannot be told apart")

    if cfg_stored != cfg_now:
        return False, (f"CONFIGURATION changed (stored {cfg_stored}, current "
                       f"{cfg_now}); the stored rows describe a different "
                       f"experiment")

    n = len(pd.read_csv(RESULTS_CSV))
    print("CHECKPOINT ADOPTED -- configuration identical, core code changed")
    print(f"  stored {stored} -> current {RUN_FP}")
    print(f"  configuration fingerprint {cfg_now} unchanged, so the {n} stored "
          f"rows describe the same experiment; resuming instead of retraining.")
    print("  Recorded in run_metadata.json as `checkpoint_adopted`.")
    open(FP_FILE, "w").write(RUN_FP)
    CHECKPOINT_ADOPTED = dict(previous_fingerprint=stored,
                              current_fingerprint=RUN_FP,
                              config_fingerprint=cfg_now, rows=n)
    return True, ""


assert_fingerprint("config", config_fingerprint(REQUIRED_COLUMNS, extra=""))   # REVISION v12
assert_fingerprint("probe", PROBE_FP)
_ok, _why = _checkpoint_is_compatible()
open(FP_CFG_FILE, "w").write(config_fingerprint(REQUIRED_COLUMNS, extra=""))
if not _ok:
    _archive = RESULTS_CSV.replace(".csv", f".stale-{RUN_FP}.csv")
    shutil.move(RESULTS_CSV, _archive)
    print(f"STALE CHECKPOINT ARCHIVED -> {_archive}\n  reason: {_why}\n"
          f"  The stored rows were written by a different code revision or configuration, so "
          f"resuming from them would silently mix incomparable results. Starting clean.")
    send_alert("Stale checkpoint archived", f"{_why}\narchived to {_archive}")

def estimate_budget(cfg):
    B, Cc, L, d = cfg.batch_size, cfg.n_clauses, CLAUSE_TOKENS, cfg.d_model
    rows = B * Cc                                     # sentences encoded per step
    act_gb = rows * L * d * 4 * (12 * cfg.n_layers + 8) / 1e9     # activations, fp32-equivalent
    attn_gb = rows * cfg.n_heads * L * L * 4 / 1e9                # dense fallback upper bound
    solver_gb = B * Cc * cfg.n_vars * 4 * cfg.refine_steps / 1e9
    steps = cfg.epochs * ((cfg.n_train + B - 1) // B)
    return dict(rows_per_step=rows, act_gb=act_gb, attn_gb=attn_gb, solver_gb=solver_gb,
                peak_gb=act_gb + attn_gb + solver_gb + 1.0, steps=steps)

_b = estimate_budget(CFG)
print(f"BUDGET  {CFG.n_clauses} clauses x {CLAUSE_TOKENS} tokens, batch {CFG.batch_size} "
      f"-> {_b['rows_per_step']} sentences/step, {_b['steps']} optimiser steps/run")
print(f"        estimated peak VRAM ~{_b['peak_gb']:.1f} GB "
      f"(activations {_b['act_gb']:.1f} + attention {_b['attn_gb']:.1f} + solver "
      f"{_b['solver_gb']:.1f} + overhead 1.0)")
if HAS_GPU:
    _total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"        device has {_total_gb:.0f} GB")
    if _b["peak_gb"] > 0.8 * _total_gb:
        raise RuntimeError(
            f"Estimated peak VRAM {_b['peak_gb']:.1f} GB exceeds 80% of the {_total_gb:.0f} GB "
            f"device. Halve CFG.batch_size (currently {CFG.batch_size}) or reduce CFG.n_vars "
            f"(currently {CFG.n_vars}; n_clauses scales with it). Nothing has been trained yet.")
print(f"        matrix = {len(CONDITIONS)} conditions x {len(CFG.seeds)} seeds = "
      f"{len(CONDITIONS)*len(CFG.seeds)} runs (checkpointed after every run; safe to resume "
      f"across Colab sessions)")

done = set()          # the reuse check consults this; the resume block
                      # below re-derives it from the checkpoint that survives.
PREFLIGHT_CAPABILITY = True
PREFLIGHT_EPOCH_MULT = 1 if CFG.FULL_SCALE else 4
PREFLIGHT_CONDITION = "no_stgs"    # the condition that establishes task learnability
PREFLIGHT_FAIL_BELOW = 0.005       # at or under this, the output carries no information
PREFLIGHT_WARN_BELOW = 0.05        # above FAIL but here: learns, but likely too weak for H2/H3

if PREFLIGHT_CAPABILITY and not os.path.exists(RESULTS_CSV):
    _pf_cfg = Config(**{**dataclasses.asdict(CFG),
                        "epochs": CFG.epochs * PREFLIGHT_EPOCH_MULT}).scaled()
    _pf_cfg.epochs = CFG.epochs * PREFLIGHT_EPOCH_MULT       # survive scaled() overriding epochs
    print(f"PRE-FLIGHT: training `{PREFLIGHT_CONDITION}` for {_pf_cfg.epochs} epochs "
          f"(~{_pf_cfg.epochs * (CFG.n_train // CFG.batch_size)} optimiser steps) to test whether "
          f"the architecture can learn the mapping at all ...")
    _pf = run_condition(PREFLIGHT_CONDITION, 0, _pf_cfg)
    _ex, _d, _f1 = _pf["faith_excess_id"], _pf["pred_density"], _pf["faith_id"]
    print(f"  raw F1 {_f1:.3f} at density {_d:.3f}; information-free baseline "
          f"{expected_f1_random(_d, GOLD_DENSITY):.3f}  ->  EXCESS {_ex:+.3f}")
    print(f"  csat_norm {_pf['csat_norm_id']:+.3f} (chance = 0)")
    if _ex < PREFLIGHT_FAIL_BELOW:
        msg = (f"PRE-FLIGHT FAILED: excess {_ex:+.3f} <= {PREFLIGHT_FAIL_BELOW}. With "
               f"{_pf_cfg.epochs} epochs `{PREFLIGHT_CONDITION}` -- the condition that solves "
               f"probe A -- still emits density-matched noise, so the TASK is not learnable by "
               f"any condition and the full matrix would spend {4*len(CFG.seeds)} trainings "
               f"comparing noise against noise.\n"
               f"Address CAPACITY or OPTIMISATION first: more epochs, lower lr, larger d_model / "
               f"n_layers, or fewer variables per instance. Set PREFLIGHT_CAPABILITY=False to "
               f"override.")
        send_alert("PRE-FLIGHT FAILED", msg)
        raise RuntimeError(msg)
    if _ex < PREFLIGHT_WARN_BELOW:
        print(f"  PRE-FLIGHT WARNING: excess {_ex:+.3f} is above noise but weak. The end-task "
              f"metric needs roughly clause-F1 0.3 before csat_norm leaves chance (measured by "
              f"the oracle degradation curve), so H2 and H3 will most likely come back "
              f"uncomputable. Proceeding, but expect a DEGENERATE health verdict.")
        send_alert("Pre-flight WEAK", f"excess {_ex:+.3f}; matrix proceeding but H2/H3 at risk")
    FIDELITY_TARGET = 0.32          # where the oracle curve shows csat_norm leaving chance
    if _pf["faith_id"] < FIDELITY_TARGET:
        print(f"  NOTE: clause-F1 {_pf['faith_id']:.3f} is below the {FIDELITY_TARGET} needed for "
              f"the end-task metric to leave chance. Escalating: one probe at "
              f"{PREFLIGHT_EPOCH_MULT * 3}x epochs to separate optimisation from capacity ...")
        _esc_cfg = Config(**dataclasses.asdict(CFG)).scaled()
        _esc_cfg.epochs = CFG.epochs * 3
        _esc = run_condition(PREFLIGHT_CONDITION, 0, _esc_cfg)
        _gain = _esc["faith_id"] - _pf["faith_id"]
        print(f"  {CFG.epochs} epochs -> F1 {_pf['faith_id']:.3f}, excess {_ex:+.3f}, "
              f"csat_norm {_pf['csat_norm_id']:+.3f}")
        print(f"  {_esc_cfg.epochs} epochs -> F1 {_esc['faith_id']:.3f}, "
              f"excess {_esc['faith_excess_id']:+.3f}, csat_norm {_esc['csat_norm_id']:+.3f}")
        if _gain > 0.05:
            print(f"  VERDICT: fidelity is OPTIMISATION-limited (+{_gain:.3f} from more epochs). "
                  f"Set Config.scaled() epochs to {_esc_cfg.epochs} and re-run; the matrix will "
                  f"then test the mechanism in a regime where it can actually matter.")
        else:
            print(f"  VERDICT: fidelity is CAPACITY-limited (only +{_gain:.3f} from 3x epochs). "
                  f"More training will not help; raise d_model/n_layers, or report the result "
                  f"explicitly scoped to this capacity regime -- which is a defensible finding, "
                  f"not a failure.")
        with open(os.path.join(OUT_DIR, "capacity_probe.txt"), "w") as _fh:
            _fh.write(f"epochs={CFG.epochs} F1={_pf['faith_id']:.4f} "
                      f"excess={_ex:.4f} csat={_pf['csat_norm_id']:.4f}\n"
                      f"epochs={_esc_cfg.epochs} F1={_esc['faith_id']:.4f} "
                      f"excess={_esc['faith_excess_id']:.4f} csat={_esc['csat_norm_id']:.4f}\n"
                      f"gain={_gain:.4f}\n")
        send_alert("Capacity probe", f"{CFG.epochs}ep F1={_pf['faith_id']:.3f} -> "
                                     f"{_esc_cfg.epochs}ep F1={_esc['faith_id']:.3f} "
                                     f"(gain {_gain:+.3f})")
        del _esc, _esc_cfg

    print(f"  PRE-FLIGHT PASSED: the mapping is learnable (excess {_ex:+.3f}).")
    _pf_reusable = (_pf_cfg.epochs == CFG.epochs and PREFLIGHT_EPOCH_MULT == 1
                    and _pf["condition"] in CONDITIONS
                    and (_pf["condition"], _pf["seed"]) not in done)
    if _pf_reusable:
        _seed_row = {k: v for k, v in _pf.items() if not k.startswith("_")}
        _missing_pf = [c for c in REQUIRED_COLUMNS if c not in _seed_row]
        assert not _missing_pf, f"pre-flight record is missing {_missing_pf}; cannot reuse it"
        atomic_csv(pd.DataFrame([_seed_row]), RESULTS_CSV, index=False)
        with open(FP_FILE, "w") as _fh:
            _fh.write(RUN_FP)
        print(f"  Kept as the ({_pf['condition']}, seed {_pf['seed']}) matrix cell "
              f"-- one fewer run to train.")
    elif PREFLIGHT_EPOCH_MULT != 1:
        print(f"  Not reused as a matrix cell: the probe ran {PREFLIGHT_EPOCH_MULT}x the "
              f"production epoch count, so it is not the same experiment.")
    _per_epoch = _pf["train_secs"] / _pf_cfg.epochs
    _run_s = _per_epoch * CFG.epochs
    _diag = (4 + 4 + 6 + 1 + 6 + 2) if RUN_EXTRA_DIAGNOSTICS else 0
    _total_h = _run_s * (sum(len(seeds_for(c)) for c in CONDITIONS) + _diag) / 3600
    print(f"  MEASURED COST: {_per_epoch:.1f}s/epoch -> {_run_s/60:.1f} min per run -> "
          f"{_total_h:.1f} h for {sum(len(seeds_for(c)) for c in CONDITIONS)} matrix runs"
          + (f" + {_diag} diagnostic runs." if _diag else "."))
    BUDGET_HOURS = 20.0
    if _total_h > BUDGET_HOURS:
        _fit_seeds = max(3, int(BUDGET_HOURS * 3600 / (_run_s * len(CONDITIONS)) - _diag / 4))
        _fit_epochs = max(5, int(CFG.epochs * BUDGET_HOURS / _total_h))
        print(f"  This exceeds a {BUDGET_HOURS:.0f} h session. Two ways to fit it:")
        print(f"    (a) do nothing -- every run is checkpointed, so re-running the notebook "
              f"resumes exactly where it stopped. Expect ~{_total_h/BUDGET_HOURS:.1f} sessions.")
        print(f"    (b) CFG.seeds = tuple(range({_fit_seeds})) -> achieved power at d=0.8 becomes "
              f"{achieved_power(_fit_seeds, 0.8):.2f} (currently {POWER_TABLE[0.8]:.2f} with "
              f"{len(CFG.seeds)} seeds), or CFG.epochs = {_fit_epochs}.")
        print(f"    Do NOT cut refine_steps: measured to have no effect on training time, and it "
              f"lowers end-task quality.")
    send_alert("Pre-flight passed",
               f"excess {_ex:+.3f}; projected {_total_h:.1f} h for the full matrix")
    del _pf, _pf_cfg

# REVISION v3: fingerprinted, so per-example scores from another code revision are never mixed in
PER_EXAMPLE_SYNTH_PATH = os.path.join(CKPT_DIR, f"per_example_synth_{PROBE_FP}.json")
try:
    PER_EXAMPLE_SYNTH = (json.load(open(PER_EXAMPLE_SYNTH_PATH))
                         if os.path.exists(PER_EXAMPLE_SYNTH_PATH) else {})
except Exception:
    PER_EXAMPLE_SYNTH = {}

if os.path.exists(RESULTS_CSV):
    done_df = pd.read_csv(RESULTS_CSV)
    done = set(zip(done_df.condition, done_df.seed))
    records = done_df.to_dict("records")
    print(f"Resuming: {len(records)} (condition, seed) results already stored "
          f"(fingerprint {RUN_FP}).")
else:
    done, records = set(), []
    print(f"Fresh run (fingerprint {RUN_FP}).")

# --- work ordering: a truncated session must still leave usable evidence ----
# Running condition-major (all seeds of `frozen`, then all of `coformer`, ...)
# means an interrupted session ends with one condition complete and nothing
# comparable to it. The order below is instead PHASED:
#
#   phase 1  the four factorial cells at the first FACTORIAL_MIN_SEEDS seeds
#            -> a complete, analysable bridge x solver-gradient factorial
#   phase 2  the two faithfulness ablations at those same seeds
#            -> the complete condition lattice at a reduced seed count
#   phase 3  the remaining seeds, seed-major, so the matrix stays balanced
#
# Every phase boundary is a point at which the study is interpretable. The
# matrix is checkpointed per cell, so stopping anywhere loses at most one run.
todo = order_matrix_cells(CONDITIONS, CFG.seeds, done)
_expected_cells = sum(len(seeds_for(c)) for c in CONDITIONS)
print(f"To run this session: {len(todo)} of {_expected_cells} cells "
      f"({len(FACTORIAL_CONDITIONS)} factorial conditions at {len(CFG.seeds)} "
      f"seeds + {len(CONDITIONS) - len(FACTORIAL_CONDITIONS)} ablations at "
      f"{min(ABLATION_SEEDS, len(CFG.seeds))}).")
_p1 = len([1 for c, s in todo if c in FACTORIAL_CONDITIONS
           and s in list(CFG.seeds)[:FACTORIAL_MIN_SEEDS]])
print(f"  ordering: phase 1 = the {len(FACTORIAL_CONDITIONS)}-cell factorial at "
      f"{min(FACTORIAL_MIN_SEEDS, len(CFG.seeds))} seeds ({_p1} runs outstanding), "
      f"then the remaining ablations, then the remaining seeds. Stopping at any "
      f"phase boundary still leaves an interpretable study.")
send_alert("Stage: training matrix",
           f"{len(records)} done, {len(todo)} remaining this session.")

def _persist(recs):
    atomic_csv(pd.DataFrame.from_records(recs), RESULTS_CSV, index=False)
    with open(FP_FILE, "w") as fh:
        fh.write(RUN_FP)

t_start = time.time()
try:
    for k, (cond, sd) in enumerate(todo, 1):
        _unit = (float(pd.DataFrame.from_records(records)["train_secs"].max()) * 1.25 + 180
                 if records else 30 * 60)
        ensure_budget(f"matrix cell {cond}/seed {sd}", _unit)       # REVISION v3
        rec = run_condition(cond, sd, CFG)
        records.append({kk: vv for kk, vv in rec.items() if not kk.startswith("_")})
        _pe = rec.get("_per_example_csat_id")
        if _pe is not None and len(_pe):
            PER_EXAMPLE_SYNTH[f"{cond}|{sd}"] = [round(float(x), 6) for x in _pe]
            atomic_json(PER_EXAMPLE_SYNTH, PER_EXAMPLE_SYNTH_PATH)
        _persist(records)
        maybe_push_state(f"matrix {cond}/{sd}")
        line = (f"[{cond:9s}|seed {sd}] faith_ID={rec['faith_id']:.3f} "
                f"excess={rec['faith_excess_id']:+.3f} "
                f"csat_norm={rec['csat_norm_id']:+.3f} "
                f"density={rec['pred_density']:.2f} "
                f"{'DEGENERATE' if rec['degenerate'] else 'ok'} "
                f"{rec['train_secs']:.1f}s")
        print(line)
        if k % max(1, len(todo) // 4) == 0 or k == len(todo):
            send_alert(f"Progress {k}/{len(todo)}", line)
except SessionBudgetReached:
    raise
except Exception:
    tb = traceback.format_exc()
    send_alert("CRASH", tb); print(tb); raise

df = pd.DataFrame.from_records(records)

# Final schema assertion: fail HERE, with an actionable message, rather than in Cell 11's
# groupby with an opaque KeyError.
_missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
assert not _missing, (
    f"results matrix is missing {_missing}. This means rows from an incompatible revision were "
    f"reused. Delete {RESULTS_CSV} (and its .fingerprint) and re-run this cell.")
_expected_rows = sum(len(seeds_for(c)) for c in CONDITIONS)
assert len(df) >= _expected_rows, (          # REVISION v8: extra seeds are never a problem
    f"expected at least {_expected_rows} rows, got {len(df)}")
if len(df) > _expected_rows:
    print(f"note: the stored matrix has {len(df)} rows, more than the {_expected_rows} this "
          f"configuration would schedule (ABLATION_SEEDS={ABLATION_SEEDS}); all of them are used.")

# --- shuffled-label negative control, run unconditionally once the matrix is
# complete. A method whose advantage survives randomly permuted supervision has
# no advantage; this is the check that distinguishes a result from an artifact,
# and it is deliberately not gated behind any other stage.
if globals().get("NEGATIVE_CONTROL_ALWAYS", True):
    try:
        print("\nShuffled-label negative controls (ungated, seed 0) ...")
        _neg_rows = []
        for _base in ("no_stgs", "coformer"):          # REVISION v2 (M8)
            if not os.path.exists(os.path.join(DIAG_DIR, f"negative_control_{_base}.csv")):
                ensure_budget(f"negative control on {_base}", 20 * 60)   # REVISION v3
            _r = _diag_cached(f"negative_control_{_base}",
                              lambda _b=_base: pd.DataFrame([{k: v for k, v in
                                  run_negative_control(seed=0, condition=_b).items()
                                  if not k.startswith("_")}]))
            _neg_rows.append(_r)
        neg_df = pd.concat(_neg_rows, ignore_index=True)
        for _, neg in neg_df.iterrows():
            _base = neg["base_condition"]
            _real_ref = float(df[df.condition == _base]["faith_id"].mean())
            _gap = abs(float(neg["faith_id"]) - _real_ref)
            print(f"  [{_base}] shuffled F1={neg['faith_id']:.4f} excess={neg['faith_excess_id']:+.4f} "
                  f"csat_norm={neg['csat_norm_id']:+.4f} | real F1={_real_ref:.4f} | gap {_gap:.4f} "
                  f"-> {'PASSED' if _gap >= 1e-3 else 'FAILED'}")
        neg_df.to_csv(os.path.join(OUT_DIR, "negative_control.csv"), index=False)
    except SessionBudgetReached:
        raise
    except Exception as _e:
        neg_df = neg = None
        print(f"  negative control could not run: {type(_e).__name__}: {_e}")

send_alert("Training matrix complete",
           f"All {len(df)} results done in {time.time()-t_start:.0f}s this session.")
print(f"\nAll results ({len(df)} rows, schema {RUN_FP} verified):"); df


## Cell 10c — Multi-probe orchestration (autopilot)

The finding must hold across instance families, not only within one. This cell
rebuilds every probe-dependent global for each remaining probe and runs its
matrix with the same conditions, seeds and analysis, writing one checkpointed
results file per probe.

It is budget-aware and resumable: each (probe, condition, seed) cell is
persisted as it completes, the projected cost of the remaining work is compared
against the session budget before each probe starts, and a session that runs out
of time stops cleanly at a cell boundary. Re-running the notebook continues from
there. No constant needs editing between probes.


In [ ]:
# === Cell 10c: multi-probe orchestration ===
# Every probe-dependent global is rebuilt here, in one place, so that switching
# instance families cannot leave a stale lexicon, vocabulary, chance level or
# class prior behind -- the class of defect that makes two probes silently
# incomparable.

def switch_probe(probe):
    """Rebuild all probe-dependent state in place and return the new config."""
    global CFG, PROBE, SUBJECTS, VOCAB, ID2TOK, VOCAB_SIZE, CLAUSE_TOKENS
    global SEQ_LEN, _MAX_VARS, train, eval_id, eval_para, eval_scale, GEN_STATS
    global GOLD_DENSITY, ANALYTIC_CSAT, CHANCE_CSAT, FAITH_CLASS_WEIGHTS

    # `scaled()` RAISES the preset fields and never lowers them, so calling it
    # on an already-calibrated config silently restores the uncalibrated epoch
    # count. Carry the calibrated budget across the probe switch explicitly.
    # Without this the probes train under DIFFERENT budgets, which is the exact
    # comparability defect the equalised batch size was introduced to remove.
    _keep_epochs = CFG.epochs
    CFG = Config(**{**dataclasses.asdict(CFG), "probe": probe}).scaled()
    if globals().get("EPOCH_CALIBRATION"):
        CFG.epochs = int(EPOCH_CALIBRATION["epochs"])
    else:
        CFG.epochs = min(CFG.epochs, _keep_epochs)
    assert CFG.epochs == _keep_epochs, (
        f"probe switch changed the training budget: {_keep_epochs} -> "
        f"{CFG.epochs}. Every probe must train under the same budget.")
    PROBE = probe
    SUBJECTS = SUBJECTS_ALL[:16] if probe == "A" else SUBJECTS_ALL[16:]
    _MAX_VARS = max(CFG.n_vars, CFG.ood_n_vars)
    VOCAB = _build_global_vocab(_MAX_VARS)
    ID2TOK = {i: w for w, i in VOCAB.items()}
    VOCAB_SIZE = len(VOCAB)
    CLAUSE_TOKENS = _max_clause_tokens()

    cache = os.path.join(DATA_DIR, f"dataset_{_fingerprint(CFG)}.npz")
    if os.path.exists(cache):
        z = np.load(cache)
        train = (z["Xtr"], z["Ytr"]); eval_id = (z["Xid"], z["Yid"])
        eval_para = (z["Xpa"], z["Ypa"]); eval_scale = (z["Xsc"], z["Ysc"])
        GEN_STATS = json.loads(str(z["stats"]))
        print(f"  probe {probe}: dataset loaded from cache")
    else:
        _p = CFG.use_planted
        Xtr, Ytr, s_tr = build_split(CFG.n_train, CFG.n_vars, CFG.n_clauses,
                                     CFG.clause_len, style="canonical", seed=100, planted=_p)
        Xid, Yid, s_id = build_split(CFG.n_eval, CFG.n_vars, CFG.n_clauses,
                                     CFG.clause_len, style="canonical", seed=200, planted=_p)
        Xpa, Ypa, s_pa = build_split(CFG.n_eval, CFG.n_vars, CFG.n_clauses,
                                     CFG.clause_len, style="paraphrase", seed=300, planted=_p)
        n_sc = min(int(round(CFG.ood_n_vars * CFG.sat_ratio)), CFG.n_clauses)
        Xsc, Ysc, s_sc = build_split(CFG.n_eval, CFG.ood_n_vars, n_sc,
                                     CFG.clause_len, style="canonical", seed=400, planted=_p)
        train, eval_id, eval_para, eval_scale = (Xtr, Ytr), (Xid, Yid), (Xpa, Ypa), (Xsc, Ysc)
        GEN_STATS = dict(train=s_tr, id=s_id, paraphrase=s_pa, scale=s_sc,
                         sat_ratio=CFG.sat_ratio, planted=CFG.use_planted)
        np.savez_compressed(cache, Xtr=Xtr, Ytr=Ytr, Xid=Xid, Yid=Yid, Xpa=Xpa,
                            Ypa=Ypa, Xsc=Xsc, Ysc=Ysc, stats=json.dumps(GEN_STATS))
        print(f"  probe {probe}: dataset generated and cached")

    SEQ_LEN = train[0].shape[2]
    # metric anchors depend on the instance family and MUST be recomputed
    GOLD_DENSITY = CFG.clause_len / CFG.n_vars
    ANALYTIC_CSAT = 1.0 - 0.5 ** CFG.clause_len
    CHANCE_CSAT = {"id": _measure_chance_csat(eval_id[1]),
                   "paraphrase": _measure_chance_csat(eval_para[1]),
                   "scale": _measure_chance_csat(eval_scale[1])}
    FAITH_CLASS_WEIGHTS = _class_weights_from_gold(
        torch.from_numpy(train[1][:512])).to(DEVICE)
    print(f"  probe {probe}: {CFG.epochs} epochs "
          f"({CFG.epochs * ((CFG.n_train + CFG.batch_size - 1) // CFG.batch_size)} "
          f"steps), {CFG.clause_len}-SAT, n_vars={CFG.n_vars}, "
          f"{CFG.n_clauses} clauses, ratio {CFG.sat_ratio}, "
          f"gold density {GOLD_DENSITY:.3f}, chance csat {CHANCE_CSAT['id']:.4f}")
    return CFG


def probe_results_path(probe):
    # REVISION v3: v1 used an unversioned name, so a stale family-B matrix counted as
    # "already complete" and was never re-run. The file now carries the run fingerprint.
    return os.path.join(CKPT_DIR, f"results_probe{probe}_{PROBE_FP}.csv")


def run_probe_matrix(probe, budget_s=None):
    """Run (or resume) the full condition x seed matrix for one probe."""
    path = probe_results_path(probe)
    done, records = set(), []
    if os.path.exists(path):
        _d = pd.read_csv(path)
        done = set(zip(_d.condition, _d.seed)); records = _d.to_dict("records")
    todo = order_matrix_cells(CONDITIONS, CFG.seeds, done)
    _n_cells = sum(len(seeds_for(c)) for c in CONDITIONS)
    if not todo:
        print(f"  probe {probe}: matrix already complete "
              f"({len(records)}/{sum(len(seeds_for(c)) for c in CONDITIONS)})")
        return pd.DataFrame.from_records(records), True
    print(f"  probe {probe}: {len(todo)} of {_n_cells} cells this session")
    t0 = time.time()
    for k, (cond, sd) in enumerate(todo, 1):
        _unit = (float(pd.DataFrame.from_records(records)["train_secs"].max()) * 1.25 + 180
                 if records else 45 * 60)
        if budget_left_s() < _unit:                                  # REVISION v3
            print(f"  probe {probe}: session budget reached after {k - 1} cells. "
                  f"Progress is checkpointed; re-run the notebook to continue.")
            return pd.DataFrame.from_records(records), False
        rec = run_condition(cond, sd, CFG)
        records.append({kk: vv for kk, vv in rec.items() if not kk.startswith("_")})
        _pe = rec.get("_per_example_csat_id")
        if _pe is not None and len(_pe):
            PER_EXAMPLE_SYNTH[f"{probe}|{cond}|{sd}"] = [round(float(x), 6) for x in _pe]
            atomic_json(PER_EXAMPLE_SYNTH, PER_EXAMPLE_SYNTH_PATH)
        atomic_csv(pd.DataFrame.from_records(records), path, index=False)
        maybe_push_state(f"probe {probe} {cond}/{sd}")
        print(f"    [{probe}|{cond:13s}|seed {sd}] "
              f"excess={rec['faith_excess_id']:+.3f} "
              f"csat={rec['csat_norm_id']:+.3f} "
              f"{'DEGENERATE' if rec['degenerate'] else 'ok'} "
              f"{rec['train_secs']:.0f}s")
    return pd.DataFrame.from_records(records), True


# --- execute the remaining probes -----------------------------------------
PROBE_RESULTS = {}
_primary = CFG.probe
if os.path.exists(RESULTS_CSV):
    import shutil as _sh
    if not os.path.exists(probe_results_path(_primary)):
        _sh.copy2(RESULTS_CSV, probe_results_path(_primary))
    PROBE_RESULTS[_primary] = pd.read_csv(probe_results_path(_primary))

if AUTOPILOT and len(PROBES) > 1:
    _saved = (CFG, PROBE, SUBJECTS, VOCAB, ID2TOK, VOCAB_SIZE, CLAUSE_TOKENS,
              SEQ_LEN, train, eval_id, eval_para, eval_scale, GOLD_DENSITY,
              CHANCE_CSAT, FAITH_CLASS_WEIGHTS)
    _spent = 0.0
    try:
        for _pb in PROBES:
            if _pb == _primary and _pb in PROBE_RESULTS and \
                    len(PROBE_RESULTS[_pb]) >= sum(len(seeds_for(c)) for c in CONDITIONS):
                print(f"probe {_pb}: complete (primary run)")
                continue
            print(f"\n=== PROBE {_pb} ===")
            switch_probe(_pb)
            _left = max(0.0, SESSION_BUDGET_HOURS * 3600 - _spent)
            _t = time.time()
            _df_p, _ok = run_probe_matrix(_pb, budget_s=_left)
            _spent += time.time() - _t
            PROBE_RESULTS[_pb] = _df_p
            if not _ok:
                send_alert(f"Probe {_pb} partial",
                           f"budget reached; {len(_df_p)} cells stored, resumable")
                raise SessionBudgetReached(f"probe {_pb} incomplete; resume in a new session")
    finally:
        (CFG, PROBE, SUBJECTS, VOCAB, ID2TOK, VOCAB_SIZE, CLAUSE_TOKENS,
         SEQ_LEN, train, eval_id, eval_para, eval_scale, GOLD_DENSITY,
         CHANCE_CSAT, FAITH_CLASS_WEIGHTS) = _saved

# --- cross-probe reproduction check ---------------------------------------
_need = sum(len(seeds_for(c)) for c in CONDITIONS)
_complete = {p: d for p, d in PROBE_RESULTS.items()
             if d is not None and len(d) >= _need}
if len(_complete) >= 2:
    print("\n=== CROSS-PROBE REPRODUCTION ===")
    print("A mechanism claim should hold in every instance family tested. The "
          "bridge main effect is recomputed independently per probe.")
    _rows = []
    for _p, _d in sorted(_complete.items()):
        try:
            _st_d = _d[_d.condition == "frozen"].sort_values("seed")["faith_excess_id"].values
            _st_r = _d[_d.condition == "coformer"].sort_values("seed")["faith_excess_id"].values
            _so_d = _d[_d.condition == "soft_frozen"].sort_values("seed")["faith_excess_id"].values
            _so_r = _d[_d.condition == "no_stgs"].sort_values("seed")["faith_excess_id"].values
            _bridge = 0.5 * ((_so_d - _st_d) + (_so_r - _st_r))
            _solver = 0.5 * ((_st_r - _st_d) + (_so_r - _so_d))
            _m, _pv = paired_bootstrap(_bridge, np.zeros_like(_bridge))
            _rows.append(dict(probe=_p, n_seeds=len(_bridge),
                              bridge_effect=round(float(_bridge.mean()), 4),
                              solver_effect=round(float(_solver.mean()), 4),
                              p_bridge=round(_pv, 4),
                              dz_bridge=round(cohens_d_paired(_bridge, np.zeros_like(_bridge)), 2)))
        except Exception as _e:
            print(f"  probe {_p}: not decomposable ({_e})")
    if _rows:
        _cp = pd.DataFrame(_rows)
        print(_cp.to_string(index=False))
        _cp.to_csv(os.path.join(OUT_DIR, "cross_probe_reproduction.csv"), index=False)
        _signs = {np.sign(r["bridge_effect"]) for r in _rows}
        if len(_signs) == 1 and 0 not in _signs:
            print("  -> the bridge effect reproduces with the same sign in every "
                  "probe; the finding is not a property of one generator.")
        else:
            print("  -> the bridge effect does NOT reproduce consistently across "
                  "probes. Report it as family-specific.")
else:
    print("\nCross-probe check pending: fewer than two probes are complete. "
          "Re-run the notebook to continue; progress is checkpointed.")


## Cell 10d — Deferred diagnostics, executed in the same session

The sweeps in Cell 9b depend on a complete results matrix, which that cell
precedes. Rather than requiring a second Run All, this cell executes them here,
once the dependency is satisfied and if the session budget allows. Each sweep is
fingerprint-cached, so an interrupted session resumes rather than repeats.


In [ ]:
# === Cell 10d: run the deferred diagnostics now that the matrix exists ===
if not globals().get("SWEEPS_PENDING", False):
    print("No deferred diagnostics: they either ran in Cell 9b or are disabled.")
else:
    _matrix_ok = len(df) >= sum(len(seeds_for(c)) for c in CONDITIONS)
    # Charge only THIS session's elapsed time, not the historical cost of a
    # matrix that may have been resumed from checkpoint in zero seconds, and
    # price the sweeps from the SWEEP configuration rather than the matrix one.
    _elapsed = time.time() - SESSION_T0          # (budget checked again per sweep below)
    _budget_left = SESSION_BUDGET_HOURS * 3600 - _elapsed
    _per_run = float(df["train_secs"].mean()) if "train_secs" in df.columns else 0.0
    _sw = _cfg_with(sweep=True)
    _sw_steps = _sw.epochs * ((_sw.n_train + _sw.batch_size - 1) // _sw.batch_size)
    _mx_steps = CFG.epochs * ((CFG.n_train + CFG.batch_size - 1) // CFG.batch_size)
    _need = 14 * _per_run * max(0.15, _sw_steps / max(_mx_steps, 1))
    print(f"  sweep run = {_sw_steps} steps on {_sw.n_train} instances "
          f"(matrix: {_mx_steps} on {CFG.n_train}); "
          f"~{_need/14/60:.1f} min each, 14 runs ~{_need/3600:.1f} h; "
          f"{_budget_left/3600:.1f} h left in this session")
    if not _matrix_ok:
        print("Deferred diagnostics skipped: the matrix is still incomplete.")
    elif _budget_left < _need:
        print(f"Deferred diagnostics postponed: they need ~{_need / 3600:.1f} h "
              f"and ~{max(_budget_left, 0) / 3600:.1f} h remain in this session. "
              f"They are fingerprint-cached, so re-running the notebook picks "
              f"them up without repeating the matrix.")
    else:
        print(f"Running deferred diagnostics (~{_need / 3600:.1f} h projected) ...")
        sweep_df = _diag_cached("lambda_sweep", lambda: run_lambda_sweep(seed=0))
        print(sweep_df[["lambda_faith", "faith_id", "faith_excess_id",
                        "csat_norm_id", "pred_density", "degenerate"]]
              .round(3).to_string(index=False))
        sweep_df.to_csv(os.path.join(OUT_DIR, "lambda_sweep.csv"), index=False)

        dens_df = _diag_cached("density_sweep", lambda: run_density_sweep(seed=0))
        print(dens_df[["lambda_density", "faith_id", "faith_excess_id",
                       "csat_norm_id", "pred_density", "pred_entropy",
                       "degenerate"]].round(3).to_string(index=False))
        dens_df.to_csv(os.path.join(OUT_DIR, "density_sweep.csv"), index=False)

        ratio_df = _diag_cached("ratio_sweep", lambda: run_ratio_sweep(seed=0))
        print(ratio_df[["sweep_ratio", "condition", "faith_id",
                        "faith_excess_id", "csat_norm_id", "degenerate"]]
              .round(3).to_string(index=False))
        ratio_df.to_csv(os.path.join(OUT_DIR, "ratio_sweep.csv"), index=False)
        SWEEPS_PENDING = False
        send_alert("Deferred diagnostics complete", "lambda / density / ratio sweeps done")


## Cell 10e — Estimator extension (M7): Gumbel-Rao straight-through

Trains `grs_frozen` and `grs_routed` on family A with the factorial seed budget and writes
`results_estimator.csv`. Kept outside `CONDITIONS` so the registered analyses are unchanged.


In [ ]:
# === Cell 10e: Gumbel-Rao estimator extension (REVISION v2, M7) ===
RUN_ESTIMATOR_EXTENSION = True
EST_CONDITIONS = ["grs_frozen", "grs_routed"]
EST_CSV = os.path.join(CKPT_DIR, f"results_estimator_{config_fingerprint(('estimator',))}.csv")
assert_fingerprint("estimator", config_fingerprint(('estimator',)))   # REVISION v12
if RUN_ESTIMATOR_EXTENSION and globals().get("df") is not None and len(df):
    _done = pd.read_csv(EST_CSV) if os.path.exists(EST_CSV) else pd.DataFrame()
    _have = set(zip(_done.condition, _done.seed)) if len(_done) else set()
    _rows = _done.to_dict("records") if len(_done) else []
    for _sd in CFG.seeds:
        for _c in EST_CONDITIONS:
            if (_c, _sd) in _have:
                continue
            ensure_budget(f"estimator extension {_c}/seed {_sd}", 20 * 60)   # REVISION v3
            _r = run_condition(_c, _sd, CFG)
            _rows.append({k: v for k, v in _r.items() if not k.startswith("_")})
            atomic_csv(pd.DataFrame(_rows), EST_CSV, index=False)
            maybe_push_state(f"estimator {_c}/{_sd}")
            print(f"  [{_c}|seed {_sd}] excess={_r['faith_excess_id']:+.3f} csat={_r['csat_norm_id']:+.3f}")
    est_df = pd.DataFrame(_rows)
    est_df.to_csv(os.path.join(OUT_DIR, "results_estimator.csv"), index=False)
    print(est_df.groupby("condition")[["faith_id", "faith_excess_id", "csat_norm_id", "acc_id",
                                       "pred_density", "degenerate"]].mean().round(3))
else:
    print("Estimator extension skipped (flag off or no matrix).")


## Cell 10f - Partial-supervision curve (pre-specified plan II, P-family)

At `faith_fraction = 1` every condition saturates, so the registered contrasts of the first
run were unmeasurable. This stage gives per-slot supervision to a fraction `f` of the
training instances and leaves the remainder to the solver pathway alone, which interpolates
between that ceiling and the solver-only regime. Four conditions (both bridges x routed and
detached) are trained at each `f`; the matrix is checkpointed per cell and resumes.


In [ ]:
# === Cell 10f: partial-supervision matrix (REVISION v4) ===
PARTIAL_CSV = os.path.join(CKPT_DIR, f"results_partial_{PROBE_FP}.csv")

if RUN_PARTIAL_SUPERVISION and globals().get("df") is not None and len(df):
    _seeds_p = list(range(PARTIAL_SEEDS))
    _todo_p, _rows_p = [], []
    if os.path.exists(PARTIAL_CSV):
        _done_p = pd.read_csv(PARTIAL_CSV)
        # REVISION v11: rows from fractions that are no longer scheduled are archived rather
        # than kept, so the curve never mixes sampling grids. They stay on disk for the record.
        _keep = _done_p.faith_fraction.round(4).isin([round(f, 4) for f in PARTIAL_FRACTIONS])
        if not _keep.all():
            _stale = PARTIAL_CSV.replace(".csv", f".superseded-{int(time.time())}.csv")
            atomic_csv(_done_p[~_keep], _stale, index=False)
            print(f"  {int((~_keep).sum())} row(s) from other fractions moved to "
                  f"{os.path.basename(_stale)}")
            _done_p = _done_p[_keep]
            atomic_csv(_done_p, PARTIAL_CSV, index=False)
        _rows_p = _done_p.to_dict("records")
        _have_p = {(r["condition"], r["seed"], round(float(r["faith_fraction"]), 4))
                   for r in _rows_p}
    else:
        _have_p = set()
    for _f in PARTIAL_FRACTIONS:
        for _sd in _seeds_p:
            for _c in PARTIAL_CONDITIONS:
                if (_c, _sd, round(_f, 4)) not in _have_p:
                    _todo_p.append((_c, _sd, _f))
    print(f"Partial-supervision matrix: {len(_have_p)} stored, {len(_todo_p)} to run "
          f"({len(PARTIAL_CONDITIONS)} conditions x {len(PARTIAL_FRACTIONS)} fractions "
          f"x {len(_seeds_p)} seeds)")
    for _k, (_c, _sd, _f) in enumerate(_todo_p, 1):
        _unit = (float(pd.DataFrame(_rows_p)["train_secs"].max()) * 1.25 + 180
                 if _rows_p else 30 * 60)
        ensure_budget(f"partial supervision f={_f} {_c}/seed {_sd}", _unit)
        _r = run_condition(_c, _sd, CFG, faith_fraction=_f)
        _rows_p.append({k: v for k, v in _r.items() if not k.startswith("_")})
        atomic_csv(pd.DataFrame(_rows_p), PARTIAL_CSV, index=False)
        maybe_push_state(f"partial f={_f} {_c}/{_sd}")
        print(f"  [{_k}/{len(_todo_p)}] f={_f:<5} {_c:<12} seed {_sd:<2} "
              f"excess={_r['faith_excess_id']:+.3f} csat={_r['csat_norm_id']:+.3f} "
              f"deg={_r['degenerate']}")
    partial_df = pd.DataFrame(_rows_p)
    atomic_csv(partial_df, os.path.join(OUT_DIR, "partial_supervision.csv"), index=False)
    print(partial_df.groupby(["faith_fraction", "condition"])[
        ["faith_id", "faith_excess_id", "csat_norm_id", "acc_id", "degenerate"]
    ].mean().round(3).to_string())
else:
    partial_df = None
    print("Partial-supervision stage skipped (flag off or no primary matrix).")


## Cell 10g - Confirmatory estimator replication (pre-specified plan II, E-family)

The estimator advantage observed in the first run was exploratory: it was measured after the
main analysis and does not survive a multiplicity correction. This stage re-runs the four
relevant conditions on seeds 12-23, which were not used before, so that the E-family can be
tested confirmatorily.


In [ ]:
# === Cell 10g: confirmatory estimator replication (REVISION v4) ===
ESTCONF_CSV = os.path.join(CKPT_DIR, f"results_estimator_confirmatory_{PROBE_FP}.csv")

if RUN_ESTIMATOR_CONFIRMATORY and globals().get("df") is not None and len(df):
    _rows_e = (pd.read_csv(ESTCONF_CSV).to_dict("records")
               if os.path.exists(ESTCONF_CSV) else [])
    _have_e = {(r["condition"], r["seed"]) for r in _rows_e}
    _todo_e = [(c, s) for s in ESTIMATOR_CONF_SEEDS
               for c in ESTIMATOR_CONF_CONDITIONS if (c, s) not in _have_e]
    print(f"Estimator replication: {len(_have_e)} stored, {len(_todo_e)} to run")
    for _k, (_c, _sd) in enumerate(_todo_e, 1):
        _unit = (float(pd.DataFrame(_rows_e)["train_secs"].max()) * 1.25 + 180
                 if _rows_e else 30 * 60)
        ensure_budget(f"estimator replication {_c}/seed {_sd}", _unit)
        _r = run_condition(_c, _sd, CFG)
        _rows_e.append({k: v for k, v in _r.items() if not k.startswith("_")})
        atomic_csv(pd.DataFrame(_rows_e), ESTCONF_CSV, index=False)
        maybe_push_state(f"estimator-conf {_c}/{_sd}")
        print(f"  [{_k}/{len(_todo_e)}] {_c:<12} seed {_sd:<2} "
              f"csat_para={_r['csat_norm_paraphrase']:+.3f} drop={_r['ood_rel_drop']:+.3f}")
    estconf_df = pd.DataFrame(_rows_e)
    atomic_csv(estconf_df, os.path.join(OUT_DIR, "results_estimator_confirmatory.csv"),
               index=False)
    print(estconf_df.groupby("condition")[
        ["faith_excess_id", "csat_norm_id", "csat_norm_paraphrase", "csat_norm_scale",
         "ood_rel_drop"]].mean().round(3).to_string())
else:
    estconf_df = None
    print("Estimator replication skipped (flag off or no primary matrix).")


## Cell 10h — Plan III synthetic experiments (pre-specified in Cell 3d)
Four stages, each behind its own flag, each checkpointed per (experiment, condition, seed) in
`results_planIII_<fingerprint>.csv`. Configuration changes are made on **copies** of `CFG`, so no
stored checkpoint is affected. The answer-supervised and task-weighted conditions use
`run_condition_x`, which is derived at run time from the source of `run_condition` by a short,
printed list of edits. With `task_weight = 1` and no answers it reproduces `run_condition` exactly;
the cell checks this on a tiny configuration before any Plan III run.


In [ ]:
# === Cell 10h: Plan III synthetic experiments (REVISION v12) ===
import inspect as _insp, dataclasses as _dc
import torch.nn.functional as F

# ---------------------------------------------------------------- answer labels (family A)
def build_answer_labels(Y, seed=0):
    """One conclusion literal per instance; label by the complete DPLL of Cell 5.
    0 = True (F |= c), 1 = False (F |= not c), 2 = Unknown."""
    rng = np.random.default_rng(seed)
    N, m, n = Y.shape
    var = rng.integers(0, n, size=N)
    pol = rng.choice(np.array([-1, 1]), size=N)
    lab = np.empty(N, dtype=np.int64)
    for i in range(N):
        cl = _mat_to_clauses(Y[i]); lit = int(pol[i]) * (int(var[i]) + 1)
        sat_against = _dpll(cl + [{-lit}], n)
        sat_with = _dpll(cl + [{lit}], n)
        lab[i] = 0 if not sat_against else (1 if not sat_with else 2)
    return var.astype(np.int64), pol.astype(np.float32), lab

def load_answers():
    path = os.path.join(DATA_DIR, f"answers_{_fingerprint(CFG)}_seed0.npz")
    if os.path.exists(path):
        z = np.load(path); var, pol, lab = z["var"], z["pol"], z["label"]
    else:
        t0 = time.time()
        var, pol, lab = build_answer_labels(train[1], seed=0)
        np.savez_compressed(path, var=var, pol=pol, label=lab)
        print(f"  answer labels computed in {time.time() - t0:.0f}s and cached")
    freq = np.bincount(lab, minlength=3) / len(lab)
    w = 1.0 / (3.0 * np.maximum(freq, 1e-6))
    print(f"  answer labels: True {freq[0]:.3f}  False {freq[1]:.3f}  Unknown {freq[2]:.3f}")
    return dict(var=torch.from_numpy(var).to(DEVICE), pol=torch.from_numpy(pol).to(DEVICE),
                label=torch.from_numpy(lab).to(DEVICE),
                class_weights=torch.tensor(w, dtype=torch.float32, device=DEVICE))

def answer_entailment_loss(clause_mat, present, idx, answers, logic):
    """Rescaled entailment surrogate (Plan III): D = (m+1) * (S(C + c) - S(C + not c))."""
    idx = idx.to(clause_mat.device)
    B, m, n = clause_mat.shape
    row = torch.zeros(B, 1, n, device=clause_mat.device)
    row[torch.arange(B, device=clause_mat.device), 0, answers["var"][idx]] = answers["pol"][idx]
    cm = clause_mat.float()
    pres = present.float() if present is not None else (cm.abs() > 0).float()
    pres_aug = torch.cat([pres, row.abs()], 1)
    _, s_with, _ = logic(torch.cat([cm, row], 1), present=pres_aug)
    _, s_against, _ = logic(torch.cat([cm, -row], 1), present=pres_aug)
    delta = (m + 1) * (s_with.mean(1) - s_against.mean(1))
    logits3 = ANSWER_KAPPA * torch.stack([delta, -delta, ANSWER_MARGIN - delta.abs()], dim=-1)
    return F.cross_entropy(logits3, answers["label"][idx], weight=answers["class_weights"])

# ---------------------------------------------------------------- run_condition_x
_RCX_EDITS = [
    ("def run_condition(condition, seed, cfg, return_model=False, eval_every=None,\n"
     "                  eval_at=None, faith_fraction=1.0):",
     "def run_condition_x(condition, seed, cfg, return_model=False, eval_every=None,\n"
     "                  eval_at=None, faith_fraction=1.0, task_weight=1.0, answers=None):"),
    ("loader = make_loader(_tr, cfg.batch_size, True, seed=seed, with_index=_partial)",
     "loader = make_loader(_tr, cfg.batch_size, True, seed=seed,\n"
     "                         with_index=(_partial or answers is not None))"),
    ('        "grs_routed":    dict(stgs=True,  solver=True,  faith=True, grs=True),\n',
     '        "grs_routed":    dict(stgs=True,  solver=True,  faith=True, grs=True),\n'
     '        # REVISION v12 (Plan III): answer-supervised conditions (no anchored task loss)\n'
     '        "ans_st":        dict(stgs=True,  solver=True,  faith=False, answer=True),\n'
     '        "ans_soft":      dict(stgs=False, solver=True,  faith=False, answer=True),\n'
     '        "ans_st_ps":     dict(stgs=True,  solver=True,  faith=True,  answer=True),\n'
     '        "ans_soft_ps":   dict(stgs=False, solver=True,  faith=True,  answer=True),\n'),
    ('    use_grs = _sp.get("grs", False)\n',
     '    use_grs = _sp.get("grs", False)\n'
     '    use_answer = _sp.get("answer", False)\n'
     '    assert (not use_answer) or answers is not None, "answer conditions need `answers`"\n'),
    ("            if solver_grad:\n"
     "                loss = loss + anchored_task_loss(assign_tr, ycb, logic)\n",
     "            if solver_grad and not use_answer:\n"
     "                loss = loss + task_weight * anchored_task_loss(assign_tr, ycb, logic)\n"
     "            if use_answer:\n"
     "                loss = loss + task_weight * answer_entailment_loss(\n"
     "                    clause_mat, present_soft, _batch[2], answers, logic)\n"),
    ("               train_secs=train_secs)",
     "               train_secs=train_secs, task_weight=task_weight,\n"
     "               answer_supervised=use_answer)"),
]

def _derive_run_condition_x():
    src = _insp.getsource(run_condition)
    for old, new in _RCX_EDITS:
        n_occ = src.count(old)
        assert n_occ == 1, f"run_condition_x: anchor found {n_occ} times:\n{old}"
        src = src.replace(old, new)
    ns = {}
    exec(compile(src, "<run_condition_x>", "exec"), globals(), ns)
    return ns["run_condition_x"]

run_condition_x = _derive_run_condition_x()
print(f"run_condition_x derived from run_condition with {len(_RCX_EDITS)} edits.")

def _check_rcx_equivalence():
    """Tiny configuration: run_condition_x with task_weight=1 and no answers must reproduce
    run_condition exactly (same seed, same code path)."""
    _small = _dc.replace(CFG, n_train=128, n_eval=64, epochs=1, calibrated_epochs=1,
                         refine_steps=5, refine_backprop_steps=2)
    a = run_condition("coformer", 0, _small)
    b = run_condition_x("coformer", 0, _small)
    keys = ["faith_id", "faith_excess_id", "csat_norm_id", "refine_steps", "param_norm"]
    diff = {k: abs(float(a[k]) - float(b[k])) for k in keys}
    # identical code path and seed; a tolerance absorbs GPU kernels without a deterministic variant
    assert all(v <= 1e-4 * max(1.0, abs(float(a[k]))) for k, v in diff.items()), \
        f"run_condition_x differs from run_condition: {diff}"
    print("  equivalence check passed:", {k: round(float(a[k]), 5) for k in keys})

# ---------------------------------------------------------------- Plan III stages
PLAN_III_CSV = os.path.join(CKPT_DIR, f"results_planIII_{PROBE_FP}.csv")
_III_ANY = any(globals().get(f, False) for f in
               ("RUN_PLAN_III_A2", "RUN_PLAN_III_A1", "RUN_PLAN_III_KW", "RUN_PLAN_III_BUDGET"))

def _plan_iii_units():
    units = []
    if RUN_PLAN_III_A2:
        units += [("A2", c, s, dict(faith_fraction=0.005, answers=True))
                  for s in range(12) for c in ("ans_st_ps", "ans_soft_ps")]
    if RUN_PLAN_III_A1:
        units += [("A1", c, s, dict(faith_fraction=1.0, answers=True))
                  for s in range(6) for c in ("ans_st", "ans_soft")]
    if RUN_PLAN_III_KW:
        units += [("K", c, s, dict(faith_fraction=0.005, K=CFG.refine_steps))
                  for s in range(6) for c in ("coformer", "no_stgs")]
        units += [("W", c, s, dict(faith_fraction=0.005, task_weight=5.0))
                  for s in range(6) for c in ("coformer", "no_stgs")]
    if RUN_PLAN_III_BUDGET:
        units += [("B", c, s, dict(faith_fraction=0.005, epochs=2 * CFG.epochs))
                  for s in range(6) for c in ("coformer", "frozen")]
    return units

if _III_ANY and globals().get("df") is not None and len(df):
    if CFG.n_vars != 16 or CFG.clause_len != 3:
        switch_probe("A")
    assert CFG.epochs == 12 or not _study_exists(), f"unexpected training budget {CFG.epochs}"
    _check_rcx_equivalence()
    ANSWERS = load_answers() if (RUN_PLAN_III_A2 or RUN_PLAN_III_A1) else None
    _rows3 = pd.read_csv(PLAN_III_CSV).to_dict("records") if os.path.exists(PLAN_III_CSV) else []
    _have3 = {(r["experiment"], r["condition"], int(r["seed"])) for r in _rows3}
    _todo3 = [u for u in _plan_iii_units() if (u[0], u[1], u[2]) not in _have3]
    print(f"Plan III: {len(_have3)} stored, {len(_todo3)} to run")
    _budget_hit = False
    for _k, (_exp, _c, _sd, _o) in enumerate(_todo3, 1):
      try:
          _unit = (float(pd.DataFrame(_rows3)["train_secs"].max()) * 1.25 + 180
                   if _rows3 else 45 * 60)
          ensure_budget(f"Plan III {_exp} {_c}/seed {_sd}", _unit)
          _cfg3 = CFG
          _curve_at = None
          if "K" in _o:
              _cfg3 = _dc.replace(CFG, refine_backprop_steps=int(_o["K"]))
          if "epochs" in _o:
              _cfg3 = _dc.replace(CFG, epochs=int(_o["epochs"]), calibrated_epochs=int(_o["epochs"]))
              _curve_at = list(range(3, int(_o["epochs"]) + 1, 3))
          if _o.get("answers") or "task_weight" in _o:
              _r = run_condition_x(_c, _sd, _cfg3, faith_fraction=_o["faith_fraction"],
                                   task_weight=float(_o.get("task_weight", 1.0)),
                                   answers=(ANSWERS if _o.get("answers") else None))
          else:
              _r = run_condition(_c, _sd, _cfg3, faith_fraction=_o["faith_fraction"],
                                 eval_every=(1 if _curve_at else None), eval_at=_curve_at)
          _rec = {k: v for k, v in _r.items() if not k.startswith("_")}
          _rec.update(experiment=_exp, epochs=_cfg3.epochs,
                      refine_backprop_steps=_cfg3.refine_backprop_steps,
                      task_weight=float(_o.get("task_weight", 1.0)),
                      answer_supervised=bool(_o.get("answers", False)),
                      learning_curve=json.dumps(_r.get("_epoch_curve") or []),
                      plan_iii_sha256=PLAN_III_SHA256,
                      torch_version=torch.__version__)
          _rows3.append(_rec)
          atomic_csv(pd.DataFrame(_rows3), PLAN_III_CSV, index=False)
          maybe_push_state(f"plan III {_exp} {_c}/{_sd}")
          print(f"  [{_k}/{len(_todo3)}] {_exp} {_c:<12} seed {_sd:<2} "
                f"excess={_r['faith_excess_id']:+.3f} csat={_r['csat_norm_id']:+.3f} "
                f"{_r['train_secs']:.0f}s")
      except SessionBudgetReached as _e:
        _budget_hit = True
        print(f"  Plan III paused: {_e}\n  All finished runs are checkpointed; the next Run All resumes here.")
        send_alert("Plan III paused (session budget)", f"{len(_rows3)} Plan III rows stored; Run All again to resume.")
        break
    plan3_df = pd.DataFrame(_rows3)
    if len(plan3_df):
        atomic_csv(plan3_df, os.path.join(OUT_DIR, "results_plan_iii.csv"), index=False)
else:
    plan3_df = (pd.read_csv(PLAN_III_CSV) if os.path.exists(PLAN_III_CSV) else None)
    print("Plan III synthetic stages: no stage enabled"
          + ("" if plan3_df is None else f" ({len(plan3_df)} stored rows loaded)"))


## Cell 10i — Follow-up study 2: convergence, exact-solver RLOO, shuffled answers

In [ ]:
# === Cell 10i: Follow-up study 2 runs (REVISION v13; specified in Cell 3e) ===
import inspect as _insp2, dataclasses as _dc2
import torch.nn.functional as F

# ---------------------------------------------------------------- exact entailment by enumeration
_ENUM_CACHE = {}
def _assignments(n):
    if n not in _ENUM_CACHE:
        a = torch.arange(2 ** n, device=DEVICE)
        bits = ((a[:, None] >> torch.arange(n, device=DEVICE)) & 1).float()
        _ENUM_CACHE[n] = (bits, 1.0 - bits)               # (2^n, n): x_v = True / x_v = False
    return _ENUM_CACHE[n]

@torch.no_grad()
def exact_var_models(Fm, chunk_f=256, chunk_a=4096):
    """Fm: (N, m, n) clause matrix in {-1, 0, +1} (all-zero rows are absent clauses).
    Returns (Mpos, Mneg): (N, n) bool; Mpos[i, v] iff formula i has a model with x_v = True."""
    Fm = Fm.to(DEVICE).float()
    N, m, n = Fm.shape
    Apos, Aneg = _assignments(n)
    Mpos = torch.zeros(N, n, dtype=torch.bool, device=DEVICE)
    Mneg = torch.zeros(N, n, dtype=torch.bool, device=DEVICE)
    for i in range(0, N, chunk_f):
        Fc = Fm[i:i + chunk_f]
        Fp, Fn = (Fc > 0).float(), (Fc < 0).float()
        nonempty = (Fc != 0).any(-1)                       # (b, m)
        mp = torch.zeros(Fc.shape[0], n, device=DEVICE)
        mn = torch.zeros(Fc.shape[0], n, device=DEVICE)
        for j in range(0, Apos.shape[0], chunk_a):
            Ap, An = Apos[j:j + chunk_a], Aneg[j:j + chunk_a]
            tc = torch.einsum("bmv,av->bma", Fp, Ap) + torch.einsum("bmv,av->bma", Fn, An)
            sat = ~((tc < 0.5) & nonempty[:, :, None]).any(1)          # (b, a)
            s = sat.float()
            mp += s @ Ap; mn += s @ An
        Mpos[i:i + chunk_f] = mp > 0.5; Mneg[i:i + chunk_f] = mn > 0.5
    return Mpos, Mneg

@torch.no_grad()
def exact_labels(Fm, var, pol):
    """Labels of conclusion literals (var, pol): 0 True (F |= c), 1 False (F |= not c), 2 Unknown.
    var: (N, Q) long, pol: (N, Q) in {-1, +1}. An unsatisfiable F entails every literal (label 0),
    as in the DPLL labelling of the follow-up study."""
    Mpos, Mneg = exact_var_models(Fm)
    var = var.to(DEVICE).long(); pol = pol.to(DEVICE)
    mp, mn = Mpos.gather(1, var), Mneg.gather(1, var)
    sat_with = torch.where(pol > 0, mp, mn)
    sat_against = torch.where(pol > 0, mn, mp)
    return torch.where(~sat_against, torch.zeros_like(var),
                       torch.where(~sat_with, torch.ones_like(var), torch.full_like(var, 2)))

def build_rl_answers(queries):
    """Gold labels of the training formulas for Q=1 (the follow-up study's queries) or Q=16."""
    Ytr = torch.from_numpy(np.asarray(train[1])).float()
    N, m, n = Ytr.shape
    if queries == 1:
        var = ANSWERS["var"].view(-1, 1).long(); pol = ANSWERS["pol"].view(-1, 1)
    else:
        assert queries == n, f"Q={queries} must equal n={n}"
        rng = np.random.default_rng(RL_Q16_SEED)
        pol = torch.from_numpy(rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=(N, n))).to(DEVICE)
        var = torch.arange(n, device=DEVICE).repeat(N, 1)
    lab = torch.cat([exact_labels(Ytr[i:i + 2048], var[i:i + 2048], pol[i:i + 2048])
                     for i in range(0, N, 2048)])
    if queries == 1:
        agree = float((lab.view(-1) == ANSWERS["label"].view(-1)).float().mean())
        assert agree == 1.0, f"exact enumeration disagrees with the DPLL labels ({agree:.6f})"
        print("  exact enumeration reproduces the DPLL answer labels on all training instances")
    freq = torch.bincount(lab.view(-1), minlength=3).float() / lab.numel()
    w = 1.0 / (3.0 * torch.clamp(freq, min=1e-6))
    print(f"  RL answers Q={queries}: label shares True {freq[0]:.3f} False {freq[1]:.3f} "
          f"Unknown {freq[2]:.3f}")
    return dict(var=var.to(DEVICE), pol=pol.to(DEVICE), label=lab.to(DEVICE), w=w.to(DEVICE),
                queries=queries)

def rl_answer_loss(logits, idx, rl):
    """RLOO score-function loss with the class-balanced exact-answer reward (Cell 3e)."""
    idx = idx.to(DEVICE)
    lg = logits.float()
    dist = torch.distributions.Categorical(logits=lg)
    samp = dist.sample((RL_SAMPLES,))                       # (K, B, m, n); classes neg/absent/pos
    logp = dist.log_prob(samp).sum(dim=(-1, -2))            # (K, B): log-probability of the formula
    K, B, m, n = samp.shape
    with torch.no_grad():
        Fm = (samp - 1).float().reshape(K * B, m, n)        # -> {-1, 0, +1}
        var = rl["var"][idx].repeat(K, 1); pol = rl["pol"][idx].repeat(K, 1)
        gold = rl["label"][idx].repeat(K, 1)
        pred = exact_labels(Fm, var, pol)
        w = rl["w"][gold]
        r = (((pred == gold).float() * w).sum(-1) / w.sum(-1)).view(K, B)
        adv = r - (r.sum(0, keepdim=True) - r) / (K - 1)
    return -(adv * logp).mean()

# ---------------------------------------------------------------- run_condition_rl
_RL_EDITS = [
    ("def run_condition_x(condition, seed, cfg, return_model=False, eval_every=None,\n"
     "                  eval_at=None, faith_fraction=1.0, task_weight=1.0, answers=None):",
     "def run_condition_rl(condition, seed, cfg, return_model=False, eval_every=None,\n"
     "                  eval_at=None, faith_fraction=1.0, task_weight=1.0, answers=None, rl=None):"),
    ("with_index=(_partial or answers is not None))",
     "with_index=(_partial or answers is not None or rl is not None))"),
    ('        "ans_soft_ps":   dict(stgs=False, solver=True,  faith=True,  answer=True),\n',
     '        "ans_soft_ps":   dict(stgs=False, solver=True,  faith=True,  answer=True),\n'
     '        # REVISION v13 (follow-up study 2): exact-solver answer feedback, no solver gradient\n'
     '        "rl_ans":        dict(stgs=True,  solver=False, faith=False, rl=True),\n'
     '        "rl_ans_ps":     dict(stgs=True,  solver=False, faith=True,  rl=True),\n'),
    ('    assert (not use_answer) or answers is not None, "answer conditions need `answers`"\n',
     '    assert (not use_answer) or answers is not None, "answer conditions need `answers`"\n'
     '    use_rl = _sp.get("rl", False)\n'
     '    assert (not use_rl) or rl is not None, "rl conditions need `rl`"\n'),
    ("                loss = loss + task_weight * answer_entailment_loss(\n"
     "                    clause_mat, present_soft, _batch[2], answers, logic)\n",
     "                loss = loss + task_weight * answer_entailment_loss(\n"
     "                    clause_mat, present_soft, _batch[2], answers, logic)\n"
     "            if use_rl:\n"
     "                loss = loss + task_weight * rl_answer_loss(logits, _batch[2], rl)\n"),
    ("               answer_supervised=use_answer)",
     "               answer_supervised=use_answer, rl_supervised=use_rl)"),
]

def _derive_run_condition_rl():
    src = _insp2.getsource(run_condition)
    for old, new in _RCX_EDITS + _RL_EDITS:
        n_occ = src.count(old)
        assert n_occ == 1, f"run_condition_rl: anchor found {n_occ} times:\n{old}"
        src = src.replace(old, new)
    ns = {}
    exec(compile(src, "<run_condition_rl>", "exec"), globals(), ns)
    return ns["run_condition_rl"]

run_condition_rl = _derive_run_condition_rl()
print(f"run_condition_rl derived from run_condition with {len(_RCX_EDITS) + len(_RL_EDITS)} edits.")

def _check_rl_equivalence():
    _small = _dc2.replace(CFG, n_train=128, n_eval=64, epochs=1, calibrated_epochs=1,
                          refine_steps=5, refine_backprop_steps=2)
    a = run_condition("coformer", 0, _small)
    b = run_condition_rl("coformer", 0, _small)
    keys = ["faith_id", "faith_excess_id", "csat_norm_id", "refine_steps", "param_norm"]
    diff = {k: abs(float(a[k]) - float(b[k])) for k in keys}
    assert all(v <= 1e-4 * max(1.0, abs(float(a[k]))) for k, v in diff.items()), \
        f"run_condition_rl differs from run_condition: {diff}"
    print("  equivalence check passed:", {k: round(float(a[k]), 5) for k in keys})

# ---------------------------------------------------------------- follow-up study 2 stages
FS2_CSV = os.path.join(CKPT_DIR, f"results_fs2_{PROBE_FP}.csv")
_FS2_ANY = any(globals().get(f, False) for f in ("RUN_FS2_C1", "RUN_FS2_C2", "RUN_FS2_C3"))
_FS2_EPOCHS = FS2["designs"]["C1_convergence"]["epochs"]

def _fs2_units():
    u = []
    if RUN_FS2_C3:
        u += [("C3", "ans_st_ps", s, dict(faith_fraction=0.005, answers="shuffled")) for s in range(12)]
    if RUN_FS2_C2:
        u += [("C2_Q1", "rl_ans_ps", s, dict(faith_fraction=0.005, rl=1)) for s in range(12)]
    if RUN_FS2_C1:
        u += [("C1", c, s, dict(faith_fraction=0.005, epochs=_FS2_EPOCHS))
              for s in range(12) for c in ("coformer", "frozen")]
    if RUN_FS2_C2:
        u += [("C2_Q16ps", "rl_ans_ps", s, dict(faith_fraction=0.005, rl=16)) for s in range(6)]
        u += [("C2_Q16", "rl_ans", s, dict(faith_fraction=1.0, rl=16)) for s in range(6)]
    return u

_FS2_DEFAULT_MIN = {"C1": 75, "C2_Q1": 35, "C2_Q16ps": 40, "C2_Q16": 40, "C3": 35}

if _FS2_ANY and globals().get("df") is not None and len(df):
    if CFG.n_vars != 16 or CFG.clause_len != 3:
        switch_probe("A")
    assert CFG.epochs == 12 or not _study_exists(), f"unexpected training budget {CFG.epochs}"
    _check_rl_equivalence()
    _rows2 = pd.read_csv(FS2_CSV).to_dict("records") if os.path.exists(FS2_CSV) else []
    _have2 = {(r["experiment"], r["condition"], int(r["seed"])) for r in _rows2}
    _todo2 = [x for x in _fs2_units() if (x[0], x[1], x[2]) not in _have2]
    print(f"Follow-up study 2: {len(_have2)} stored, {len(_todo2)} to run")
    if _todo2:
        if globals().get("ANSWERS") is None:
            ANSWERS = load_answers()
        _need = {o.get("rl") for _, _, _, o in _todo2 if o.get("rl")}
        RL_ANSWERS = {q: build_rl_answers(q) for q in sorted(_need)}
        if any(o.get("answers") == "shuffled" for _, _, _, o in _todo2):
            _perm = np.random.default_rng(SHUFFLE_SEED).permutation(len(ANSWERS["label"]))
            ANSWERS_SHUF = dict(ANSWERS)
            ANSWERS_SHUF["label"] = ANSWERS["label"][torch.from_numpy(_perm).to(ANSWERS["label"].device)]
            print(f"  shuffled answer labels: {float((ANSWERS_SHUF['label'] == ANSWERS['label']).float().mean()):.3f} "
                  "of labels unchanged by the permutation")
    for _k, (_exp, _c, _sd, _o) in enumerate(_todo2, 1):
      try:
          _same = [r for r in _rows2 if r["experiment"] == _exp]
          _unit = (float(pd.DataFrame(_same)["train_secs"].max()) * 1.25 + 180 if _same
                   else _FS2_DEFAULT_MIN[_exp] * 60)
          ensure_budget(f"Follow-up 2 {_exp} {_c}/seed {_sd}", _unit)
          _cfg2 = CFG; _curve_at = None
          if "epochs" in _o:
              _cfg2 = _dc2.replace(CFG, epochs=int(_o["epochs"]), calibrated_epochs=int(_o["epochs"]))
              _curve_at = list(range(6, int(_o["epochs"]) + 1, 6))
          if _exp == "C1":
              _r = run_condition(_c, _sd, _cfg2, faith_fraction=_o["faith_fraction"],
                                 eval_every=1, eval_at=_curve_at)
          elif _exp == "C3":
              _r = run_condition_x(_c, _sd, _cfg2, faith_fraction=_o["faith_fraction"],
                                   answers=ANSWERS_SHUF)
          else:
              _r = run_condition_rl(_c, _sd, _cfg2, faith_fraction=_o["faith_fraction"],
                                    rl=RL_ANSWERS[_o["rl"]])
          _rec = {k: v for k, v in _r.items() if not k.startswith("_")}
          _rec.update(experiment=_exp, epochs=_cfg2.epochs, queries=int(_o.get("rl") or 0),
                      answers_shuffled=(_o.get("answers") == "shuffled"),
                      learning_curve=json.dumps(_r.get("_epoch_curve") or []),
                      fs2_sha256=FS2_SHA256, torch_version=torch.__version__)
          _rows2.append(_rec)
          atomic_csv(pd.DataFrame(_rows2), FS2_CSV, index=False)
          maybe_push_state(f"followup2 {_exp} {_c}/{_sd}")
          print(f"  [{_k}/{len(_todo2)}] {_exp:<9} {_c:<10} seed {_sd:<2} "
                f"excess={_r['faith_excess_id']:+.3f} csat={_r['csat_norm_id']:+.3f} "
                f"{_r['train_secs']:.0f}s")
      except SessionBudgetReached as _e:
        print(f"  Follow-up study 2 paused: {_e}\n  All finished runs are checkpointed; "
              "the next Run All resumes here.")
        send_alert("Follow-up study 2 paused (session budget)",
                   f"{len(_rows2)} rows stored; Run All again to resume.")
        break
    fs2_df = pd.DataFrame(_rows2)
    if len(fs2_df):
        atomic_csv(fs2_df, os.path.join(OUT_DIR, "results_fs2.csv"), index=False)
        if len(_rows2) == len(_fs2_units()):
            send_alert("Follow-up study 2 complete", f"{len(_rows2)} runs stored.")
else:
    fs2_df = pd.read_csv(FS2_CSV) if os.path.exists(FS2_CSV) else None
    print("Follow-up study 2: no stage enabled"
          + ("" if fs2_df is None else f" ({len(fs2_df)} stored rows loaded)"))


## Cell 10j — Family H: pilot, level selection and main runs

In [ ]:
# === Cell 10j: Family H runs (REVISION v14; specified in Cell 3f) ===
# Re-verbalises the stored family-A formulas with the harder generator of Cell 3f, swaps the
# data globals that run_condition reads, runs the pilot and the main designs, and ALWAYS restores
# the family-A globals afterwards (try/finally). run_condition itself is not modified.
import dataclasses as _dcH, hashlib as _hlH2

H_VOCAB = {w: i for i, w in enumerate(["<PAD>"] + sorted(H_WORDS_ID))}
assert H_VOCAB["<PAD>"] == 0

def _famh_encode_split(Y, level, seed, ood, n_check=4000):
    """Gold clause matrices Y (N, m, n) -> token ids X (N, m, L) with L = the proven bound."""
    L = h_max_clause_tokens(level)
    rng = np.random.default_rng(seed)
    N, m, n = Y.shape
    assert n == H_NVARS, f"family H needs n_vars = {H_NVARS}, got {n}"
    X = np.zeros((N, m, L), dtype=np.int16)
    chk = set(np.random.default_rng(seed + 1).choice(N * m, size=min(n_check, N * m),
                                                     replace=False).tolist())
    lens = []
    for i in range(N):
        for j in range(m):
            row = Y[i, j]
            if not np.any(row):
                continue
            s = verbalize_clause_H(row, level=level, rng=rng, ood=ood)
            ids = [H_VOCAB[w] for w in s.split()]
            assert len(ids) <= L, f"sentence longer than the proven bound ({len(ids)} > {L})"
            X[i, j, :len(ids)] = ids
            lens.append(len(ids))
            if i * m + j in chk:
                g = {int(v): int(np.sign(row[v])) for v in np.nonzero(row)[0]}
                assert parse_clause_H(s) == g, f"parser check failed: {s!r} vs {g}"
    return X, dict(max_len=int(max(lens)), mean_len=round(float(np.mean(lens)), 2), bound=L,
                   parser_checked=len(chk))

_FAMH_MEM = {}          # the most recently used level, kept in memory between runs

def famh_data(level):
    """Family-H splits for one level (cached). Requires the family-A globals to be active."""
    assert CFG.n_vars == 16 and CFG.clause_len == 3 and PROBE == "A", "family-A state required"
    _n = (len(train[0]), len(eval_id[0]), len(eval_para[0]))
    if level in _FAMH_MEM and _FAMH_MEM[level][2] == _n:
        return _FAMH_MEM[level][0], _FAMH_MEM[level][1]
    _FAMH_MEM.clear()
    key = f"{_fingerprint(CFG)}_{FAMH_GEN_SHA256[:12]}_n{_n[0]}-{_n[1]}-{_n[2]}_L{level}"
    path = os.path.join(DATA_DIR, f"dataset_famH_{key}.npz")
    if os.path.exists(path):
        z = np.load(path)
        out = {k: (z["X" + k].astype(np.int64), z["Y" + k]) for k in ("tr", "id", "pa")}
        stats = json.loads(str(z["stats"]))
        print(f"  family H level {level}: loaded from cache ({path})")
    else:
        t0 = time.time(); out, stats = {}, {}
        for k, split, sd, ood in (("tr", train, 5100, False), ("id", eval_id, 5200, False),
                                  ("pa", eval_para, 5300, True)):
            Y = np.asarray(split[1])
            X, st = _famh_encode_split(Y, level, sd + level, ood)
            out[k] = (X.astype(np.int64), Y); stats[k] = st
        np.savez_compressed(path, **{"X" + k: v[0].astype(np.int16) for k, v in out.items()},
                            **{"Y" + k: v[1] for k, v in out.items()}, stats=json.dumps(stats))
        print(f"  family H level {level}: generated in {time.time() - t0:.0f}s and cached; {stats}")
    # the formulas are family A's, instance by instance
    assert np.array_equal(out["tr"][1], np.asarray(train[1]))
    assert np.array_equal(out["id"][1], np.asarray(eval_id[1]))
    assert np.array_equal(out["pa"][1], np.asarray(eval_para[1]))
    _FAMH_MEM[level] = (out, stats, _n)
    return out, stats

_FAMH_SWAP = ("train", "eval_id", "eval_para", "eval_scale", "VOCAB_SIZE", "CLAUSE_TOKENS",
              "SEQ_LEN", "CHANCE_CSAT")

class famh_state:
    """Context manager: family-H data globals inside, family-A globals restored on exit."""
    def __init__(self, level):
        self.level = level
    def __enter__(self):
        g = globals()
        self.saved = {k: g[k] for k in _FAMH_SWAP}
        d, _ = famh_data(self.level)
        g["train"], g["eval_id"], g["eval_para"] = d["tr"], d["id"], d["pa"]
        # no scale split: a small ID slice keeps run_condition's code path intact; every
        # scale-based field is overwritten with NaN in the stored record
        g["eval_scale"] = (d["id"][0][:64], d["id"][1][:64])
        g["VOCAB_SIZE"] = len(H_VOCAB)
        g["CLAUSE_TOKENS"] = g["SEQ_LEN"] = d["tr"][0].shape[2]
        g["CHANCE_CSAT"] = dict(self.saved["CHANCE_CSAT"])
        g["CHANCE_CSAT"]["scale"] = g["CHANCE_CSAT"]["id"]
        return self
    def __exit__(self, *exc):
        globals().update(self.saved)
        return False

_FAMH_NAN = ("faith_scale", "faith_excess_scale", "gold_density_scale", "csat_norm_scale",
             "acc_scale", "ood_retention_scale", "ood_rel_drop")

def famh_run(experiment, condition, seed, level, faith_fraction):
    cfgH = _dcH.replace(CFG, epochs=FAMH_EPOCHS, calibrated_epochs=FAMH_EPOCHS)
    with famh_state(level):
        r = run_condition(condition, seed, cfgH, faith_fraction=faith_fraction, eval_every=1,
                          eval_at=list(range(6, FAMH_EPOCHS + 1, 6)))
    rec = {k: v for k, v in r.items() if not k.startswith("_")}
    for k in _FAMH_NAN:
        rec[k] = float("nan")
    ret_pa = (r["csat_norm_paraphrase"] / r["csat_norm_id"]
              if r["csat_norm_id"] >= CSAT_FLOOR else float("nan"))
    rec.update(experiment=experiment, level=level, epochs=FAMH_EPOCHS,
               ood_retention_para=ret_pa, learning_curve=json.dumps(r.get("_epoch_curve") or []),
               famh_sha256=FAMH_SHA256, torch_version=torch.__version__)
    return rec

def famh_select_level(pilot):
    """Pre-specified pilot rule (Cell 3f)."""
    fmax = 1.0 - GOLD_DENSITY / 2.0
    rows = []
    for lv in sorted(pilot.level.unique()):
        share = float(pilot[pilot.level == lv].faith_excess_id.iloc[0]) / fmax
        rows.append(dict(level=int(lv), share=share, in_band=bool(0.20 <= share <= 0.95)))
    band = [r["level"] for r in rows if r["in_band"]]
    high = [r["level"] for r in rows if r["share"] > 0.95]
    sel = min(band) if band else (max(high) if high else 1)
    return int(sel), rows

FAMH_CSV = os.path.join(CKPT_DIR, f"results_famH_{PROBE_FP}.csv")
_FAMH_DEFAULT_MIN = {1: 40, 2: 55, 3: 75}

def _famh_units(level):
    u = []
    for s in range(16):
        u += [("H_full", c, s, 1.0) for c in ("frozen", "coformer")]
        if s < 12:
            u += [("H_scarce", c, s, 0.05) for c in ("frozen", "coformer")]
    return u

def _famh_do(units, rows, have):
    for exp, c, sd, ff, lv in units:
        if (exp, c, sd) in have:
            continue
        same = [r for r in rows if int(r["level"]) == lv]
        unit = (float(pd.DataFrame(same)["train_secs"].max()) * 1.25 + 600 if same
                else _FAMH_DEFAULT_MIN[lv] * 60)
        ensure_budget(f"Family H {exp} {c}/seed {sd}", unit)
        rec = famh_run(exp, c, sd, lv, ff)
        rows.append(rec); have.add((exp, c, sd))
        atomic_csv(pd.DataFrame(rows), FAMH_CSV, index=False)
        maybe_push_state(f"famH {exp} {c}/{sd}")
        print(f"  [H L{lv}] {exp:<9} {c:<9} seed {sd:<4} excess={rec['faith_excess_id']:+.3f} "
              f"para={rec['faith_excess_paraphrase']:+.3f} csat={rec['csat_norm_id']:+.3f} "
              f"{rec['train_secs'] / 60:.0f} min")

famH_df, FAMH_LEVEL = None, None
if globals().get("RUN_FAMH", False) and globals().get("df") is not None and len(df):
    if CFG.n_vars != 16 or CFG.clause_len != 3 or PROBE != "A":
        switch_probe("A")
    assert CFG.epochs == 12 or not _study_exists(), f"unexpected training budget {CFG.epochs}"
    _rowsH = pd.read_csv(FAMH_CSV).to_dict("records") if os.path.exists(FAMH_CSV) else []
    _haveH = {(r["experiment"], r["condition"], int(r["seed"])) for r in _rowsH}
    print(f"Family H: {len(_rowsH)} runs stored")
    try:
        _famh_do([("H_pilot_L%d" % lv, "frozen", FAMH_PILOT_SEED, 1.0, lv) for lv in (1, 2, 3)],
                 _rowsH, _haveH)
        _pil = pd.DataFrame([r for r in _rowsH if str(r["experiment"]).startswith("H_pilot")])
        FAMH_LEVEL, _pil_rows = famh_select_level(_pil)
        atomic_json({"selected_level": FAMH_LEVEL, "pilot": _pil_rows, "famh_sha256": FAMH_SHA256},
                    os.path.join(OUT_DIR, "famH_level.json"), indent=1)
        print(f"  pilot shares: {[(r['level'], round(r['share'], 3)) for r in _pil_rows]} "
              f"-> selected level {FAMH_LEVEL}")
        _main = [(e, c, s, f, FAMH_LEVEL) for e, c, s, f in _famh_units(FAMH_LEVEL)]
        if not any((e, c, s) in _haveH for e, c, s, _, _ in _main):
            send_alert("Family H level selected", f"level {FAMH_LEVEL}; pilot {_pil_rows}")
        _n0 = len(_rowsH)
        _famh_do(_main, _rowsH, _haveH)
        if len(_rowsH) > _n0:
            send_alert("Family H complete", f"{len(_rowsH)} runs stored.")
    except SessionBudgetReached as _e:
        print(f"  Family H paused: {_e}\n  All finished runs are checkpointed; "
              "the next Run All resumes here.")
        send_alert("Family H paused (session budget)", f"{len(_rowsH)} rows stored; Run All again.")
    famH_df = pd.DataFrame(_rowsH)
    if len(famH_df):
        atomic_csv(famH_df, os.path.join(OUT_DIR, "results_famH.csv"), index=False)
else:
    famH_df = pd.read_csv(FAMH_CSV) if os.path.exists(FAMH_CSV) else None
    print("Family H: stage not enabled"
          + ("" if famH_df is None else f" ({len(famH_df)} stored rows loaded)"))


## Cell 11 — Aggregation, mean ± std, and paired significance tests
Aggregates across seeds and runs a paired bootstrap comparing **CoFormer vs the frozen baseline**
on accuracy and faithfulness, reporting effect sizes alongside p-values. Unsupported hypotheses are
labeled plainly — no cherry-picking.

In [ ]:
# === Cell 11: aggregate + collapse audit + paired bootstrap (exploratory) ===
AGG_SPEC = dict(
    faith_id=("faith_id", "mean"), faith_id_sd=("faith_id", "std"),
    faith_excess_id=("faith_excess_id", "mean"),
    faith_excess_id_sd=("faith_excess_id", "std"),
    csat_norm_id=("csat_norm_id", "mean"), csat_norm_id_sd=("csat_norm_id", "std"),
    csat_norm_paraphrase=("csat_norm_paraphrase", "mean"),
    csat_norm_para_sd=("csat_norm_paraphrase", "std"),
    csat_norm_scale=("csat_norm_scale", "mean"), csat_norm_scale_sd=("csat_norm_scale", "std"),
    acc_id=("acc_id", "mean"), acc_paraphrase=("acc_paraphrase", "mean"),
    acc_scale=("acc_scale", "mean"),
    pred_density=("pred_density", "mean"), pred_entropy=("pred_entropy", "mean"),
    pred_pos_frac=("pred_pos_frac", "mean"), degenerate=("degenerate", "mean"),
    refine_steps=("refine_steps", "mean"), train_secs=("train_secs", "mean"),
    err_wrong=("error_wrong_translation", "mean"),
    err_refine=("error_refine_failure", "mean"),
    ood_rel_drop=("ood_rel_drop", "mean"),
    ood_retention_para=("ood_retention_para", "mean"),
    ood_retention_scale=("ood_retention_scale", "mean"),
)
agg = df.groupby("condition").agg(**AGG_SPEC).reindex(CONDITIONS)
print("Aggregated (mean across seeds):\n",
      agg[["faith_id", "faith_excess_id", "csat_norm_id", "acc_id", "pred_density",
           "pred_entropy", "degenerate", "refine_steps"]].round(3).to_string())

# ---- the collapse audit runs BEFORE any hypothesis is reported --------------------
_sig = df.groupby("condition")[["faith_id", "pred_density", "pred_entropy"]].mean().round(4)
_dupes = _sig[_sig.duplicated(keep=False)]
if len(_dupes):
    print("\n!! IDENTICAL OUTPUT ACROSS CONDITIONS -- the model is not responding to its loss:")
    print(_dupes.to_string())
    print("   Conditions that differ in their objective cannot produce the same metrics to four "
          "decimals unless the output is independent of both the loss and the data.")

print(f"\n=== NOISE AUDIT (gold literal density = {GOLD_DENSITY:.2f}) ===")
print(f"  {'condition':10s} {'density':>8} {'raw F1':>8} {'noise F1':>9} {'EXCESS':>8}   verdict")
NOISE_CONDITIONS = []
for cond in CONDITIONS:
    sub = df[df.condition == cond]
    d_ = float(sub["pred_density"].mean()); f_ = float(sub["faith_id"].mean())
    nb_ = expected_f1_random(d_, GOLD_DENSITY); ex_ = f_ - nb_
    if ex_ < 0.01:
        NOISE_CONDITIONS.append(cond); v = "<-- INFORMATION-FREE (density-matched noise)"
    else:
        v = "learned signal"
    print(f"  {cond:10s} {d_:>8.3f} {f_:>8.3f} {nb_:>9.3f} {ex_:>+8.3f}   {v}")
if NOISE_CONDITIONS:
    print(f"\n  !! {', '.join(NOISE_CONDITIONS)} carry NO information about the instance: their "
          f"raw F1 is exactly what an information-free CNF scores at the density they emitted.")
    print(f"  !! Comparisons among them measure OUTPUT DENSITY, not fidelity. Fix the learning "
          f"problem (steps, capacity, lr) before interpreting any hypothesis.")

print(f"\n=== COLLAPSE AUDIT (gold literal density = {GOLD_DENSITY:.2f}) ===")
_MAX_EXCESS = 1.0 - expected_f1_random(GOLD_DENSITY, GOLD_DENSITY)
CEILING_CONDITIONS = [c for c in CONDITIONS
                      if float(df[df.condition == c]["faith_excess_id"].mean())
                      > 0.95 * _MAX_EXCESS]
if CEILING_CONDITIONS:
    print(f"  NOTE: {', '.join(CEILING_CONDITIONS)} reached the ceiling "
          f"(excess > {0.95 * _MAX_EXCESS:.3f}); their zero seed variance is CONVERGENCE, not "
          f"collapse, and they are exempt from the degeneracy flags below.")

DEGENERATE_CONDITIONS = []
for cond in CONDITIONS:
    sub = df[df.condition == cond]
    if cond in CEILING_CONDITIONS:
        print(f"  {cond:9s} density {sub['pred_density'].mean():.3f} | entropy "
              f"{sub['pred_entropy'].mean():.3f} | excess "
              f"{sub['faith_excess_id'].mean():+.3f}   <-- SOLVED (at the ceiling)")
        continue
    share = float(sub["degenerate"].mean())
    sd_f = float(sub["faith_id"].std())
    line = (f"  {cond:9s} density {sub['pred_density'].mean():.3f} | entropy "
            f"{sub['pred_entropy'].mean():.3f} | faith sd {sd_f:.4f} | "
            f"degenerate {share:.0%}")
    if share > 0.5 or sd_f < 1e-6:
        DEGENERATE_CONDITIONS.append(cond)
        reasons = "; ".join(sorted({w for w in sub["degenerate_why"]
                                    if isinstance(w, str) and w.strip()}))
        _r = reasons or "zero seed variance"
        if len(_r) > 220:
            _r = _r[:220].rsplit(";", 1)[0] + "; ..."
        line += f"   <-- CONSTANT PREDICTOR ({_r})"
    print(line)
if DEGENERATE_CONDITIONS:
    print("\n  !! Hypotheses involving these conditions are NOT interpretable: a constant "
          "predictor is not a baseline. Investigate before reporting anything.")
    print("  !! Try: raise lambda_density, raise model capacity, or lower sat_ratio "
          "(see the sweeps in Cell 9b).")
else:
    print("  All conditions produced input-dependent, non-degenerate CNFs.")

if "param_norm" in df.columns:
    print("\nTraining audit:")
    print(df.groupby("condition")[["param_norm", "grad_steps"]].mean().round(1).to_string())

n_nan = int(df["ood_rel_drop"].isna().sum())
if n_nan:
    print(f"\nNOTE: ood_rel_drop is NaN for {n_nan}/{len(df)} runs (csat_norm below "
          f"CSAT_FLOOR={CSAT_FLOOR}, or OOD >= ID). Dropped PAIRWISE in H3.")

co = df[df.condition == "coformer"].sort_values("seed")
fr = df[df.condition == "frozen"].sort_values("seed")
stat_lines = []
for metric, label in [("csat_norm_id", "end-task csat_norm (H2)"),
                      ("faith_excess_id", "density-controlled faithfulness (H1)"),
                      ("faith_id", "raw faithfulness (CONFOUNDED, context only)")]:
    md_, p = paired_bootstrap(co[metric].values, fr[metric].values)
    blocked = bool(set(["coformer", "frozen"]) &
                   (set(DEGENERATE_CONDITIONS) | set(NOISE_CONDITIONS)))
    verdict = ("BLOCKED (a compared condition is degenerate or information-free)" if blocked else
               ("supported (exploratory)" if (md_ > 0 and p < ALPHA) else "not supported at this scale"))
    s = f"{label}: CoFormer - frozen = {md_:+.3f} (p={p:.3f}) -> {verdict}"
    stat_lines.append(s); print(s)
SUCCESS_THRESHOLD = 0.30      # where the oracle degradation curve shows csat_norm leaving chance

print(f"\n=== SEED-DISTRIBUTION SHAPE (success threshold: faith_excess > {SUCCESS_THRESHOLD}) ===")
_trained = df[df.condition.isin(CONDITIONS)]
for _c in CONDITIONS:
    _v = np.sort(df[df.condition == _c]["faith_excess_id"].dropna().values)[::-1]
    if len(_v) == 0: continue
    _n_ok = int((_v > SUCCESS_THRESHOLD).sum())
    print(f"  {_c:9s} success {_n_ok}/{len(_v)} ({_n_ok/len(_v):.0%})  "
          f"best {_v[0]:.3f}  median {np.median(_v):.3f}  mean {_v.mean():.3f}")
_x = np.sort(_trained["faith_excess_id"].dropna().values)
if len(_x) > 4:
    _gaps = np.diff(_x); _i = int(np.argmax(_gaps))
    _lo, _hi = _x[:_i+1], _x[_i+1:]
    if len(_hi) and _gaps[_i] > 2 * _lo.std(ddof=1) if len(_lo) > 1 else False:
        print(f"  BIMODAL: largest gap {_gaps[_i]:.3f} between {_x[_i]:.3f} and {_x[_i+1]:.3f}; "
              f"lower mode n={len(_lo)} at {_lo.mean():.3f}+/-{_lo.std(ddof=1):.3f}, "
              f"upper mode n={len(_hi)} at {_hi.mean():.3f}")
        print(f"  -> the pooled mean and sd describe neither mode. Report the SUCCESS RATE and "
              f"the within-mode statistics, and treat the pooled sd as a mixing artifact rather "
              f"than as the noise level of a single training run.")
if "csat_norm_id" in _trained.columns:
    _r = _trained["faith_excess_id"].corr(_trained["csat_norm_id"])
    print(f"  corr(faith_excess, csat_norm) across runs = {_r:+.3f} "
          f"({'successes are genuine, not metric artifacts' if _r > 0.5 else 'weak coupling'})")

_co = df[df.condition == "coformer"].sort_values("seed")
_fr = df[df.condition == "frozen"].sort_values("seed")
for _m, _lbl in [("faith_excess_id", "H1 fidelity"), ("csat_norm_id", "H2 end-task")]:
    _d = _co[_m].values - _fr[_m].values
    _d = _d[~np.isnan(_d)]
    if len(_d) < 2 or _d.std(ddof=1) < 1e-12:
        continue
    _dz = _d.mean() / _d.std(ddof=1)
    _pw = achieved_power(len(_d), abs(_dz))
    _need = recommend_seeds(abs(_dz)) if abs(_dz) > 1e-6 else float("inf")
    print(f"\nOBSERVED effect [{_lbl}]: mean diff {_d.mean():+.4f}, sd {_d.std(ddof=1):.4f}, "
          f"dz = {_dz:+.3f}")
    print(f"  achieved power with n={len(_d)}: {_pw:.2f}; seeds needed for 0.80: "
          f"{_need if np.isfinite(_need) else 'n/a'}")
    if _pw < 0.5:
        print(f"  -> UNDERPOWERED BY A WIDE MARGIN. At this seed variance the design cannot "
              f"resolve an effect of this size; report the interval, not a verdict.")

print(f"\nEXPLORATORY ONLY (alpha={ALPHA}); binding verdicts are the Holm-corrected tests in 11b.")
print(f"Achieved power at d=0.8 with n={len(CFG.seeds)} seeds: {POWER_TABLE[0.8]:.2f}; "
      f"at d=0.5: {POWER_TABLE[0.5]:.2f}.")


## Cell 11b — Confirmatory analysis with Holm–Bonferroni correction and effect sizes
The exploratory bootstrap in Cell 11 is now complemented by the **pre-registered confirmatory
test**: for every locked hypothesis we compute the paired difference, a paired bootstrap p-value,
**Cohen's d with a bootstrap CI**, and then apply **Holm–Bonferroni** across the whole hypothesis
family to control the family-wise error rate. A hypothesis counts as *supported* only if it meets
its locked decision rule (direction + Holm-adjusted p < α + effect beyond MIN_EFFECT). Unsupported
hypotheses are reported as such, and the correction controls the family-wise
error rate.

In [ ]:
# === Cell 11b: pre-registered confirmatory analysis (Holm + effect sizes) ===
from statsmodels.stats.multitest import multipletests

def metric_vector(cond, metric):
    sub = df[df.condition == cond].sort_values("seed")
    return sub[metric].values if metric in sub.columns else None

rows, pvals, hkeys = [], [], []
for h, d in PREREGISTRATION.items():
    assert isinstance(d, dict), f"PREREGISTRATION[{h}] is not a hypothesis dict"
    c1, c2 = d["comparison"]
    a, b = metric_vector(c1, d["metric"]), metric_vector(c2, d["metric"])
    if a is None or b is None or len(a) != len(b) or len(a) < 2:
        rows.append(dict(hypothesis=h, metric=d["metric"], status="not computable here",
                         mean_diff=np.nan, p_raw=np.nan, cohen_d=np.nan)); continue
    valid = ~(np.isnan(a) | np.isnan(b))                     # keep the pairing intact
    n_valid = int(valid.sum()); a, b = a[valid], b[valid]
    if n_valid < 2:
        rows.append(dict(hypothesis=h, metric=d["metric"],
                         status=f"insufficient valid seeds (n={n_valid}; metric collapsed)",
                         mean_diff=np.nan, p_raw=np.nan, cohen_d=np.nan)); continue
    # A contrast that CANNOT reach alpha is excluded from the family before
    # any p-value is computed, so it neither earns a verdict nor dilutes the
    # correction applied to the contrasts that can.
    status, p_floor, note = contrast_testability(a, b)
    md_, p = paired_bootstrap(a, b)
    dval = cohens_d_paired(a, b); lo, hi = boot_ci_d(a, b)
    d_note = "" if np.isfinite(dval) else "undefined (zero across-seed variance)"
    row = dict(hypothesis=h, metric=d["metric"], mean_diff=round(md_, 4),
               p_raw=round(p, 4), cohen_d=round(dval, 3),
               d_CI=f"[{lo:.2f},{hi:.2f}]", d_note=d_note,
               direction=d["direction"], n_seeds=n_valid,
               testability=status, p_floor=round(p_floor, 4))
    if status != "testable":
        row["verdict"] = f"NOT TESTABLE ({status})"
        row["status"] = note
        rows.append(row)
        continue
    if d.get("family") == "exploratory" and status == "testable":
        row["verdict"] = "EXPLORATORY (outside the Holm family)"   # REVISION v2 (M5)
        rows.append(row)
        continue
    rows.append(row)
    pvals.append(p); hkeys.append(h)

if pvals:
    reject, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method="holm")
    padj_map = dict(zip(hkeys, p_adj)); rej_map = dict(zip(hkeys, reject))
    for r in rows:
        h = r["hypothesis"]
        if h in padj_map:
            r["p_holm"] = round(padj_map[h], 4)
            want_greater = PREREGISTRATION[h]["direction"] == "greater"
            right_way = (r["mean_diff"] > 0) if want_greater else (r["mean_diff"] < 0)
            meets = bool(rej_map[h]) and abs(r["mean_diff"]) >= MIN_EFFECT and right_way
            r["verdict"] = "SUPPORTED" if meets else "not supported"
# --- example-level corroboration of every seed-paired verdict -------------
# The seed-paired bootstrap has len(CFG.seeds) effective observations. Where a
# metric is defined per evaluation example, the cluster bootstrap resamples
# seeds AND examples, so its resolution does not depend on the seed budget.
# Reporting both is what makes an underpowered null distinguishable from a
# genuine one: agreement means the finding survives the seed count,
# disagreement means the seed-paired test could not resolve it.
_pe_store = {}
if "PER_EXAMPLE_SYNTH" in dir():
    _pe_store = PER_EXAMPLE_SYNTH
elif os.path.exists(PER_EXAMPLE_SYNTH_PATH):
    try:
        _pe_store = json.load(open(PER_EXAMPLE_SYNTH_PATH))
    except Exception:
        _pe_store = {}

def _per_seed_vec(cond):
    out = {}
    for sd in CFG.seeds:
        v = _pe_store.get(f"{cond}|{sd}")
        if v:
            out[sd] = np.asarray(v, float)
    return out

if _pe_store:
    print("\n=== EXAMPLE-LEVEL CLUSTER BOOTSTRAP (csat, seeds then examples) ===")
    print("  Resolution here does NOT depend on the seed count.")
    for h, d in PREREGISTRATION.items():
        if d["metric"] not in ("csat_norm_id",):
            continue                       # only metrics defined per example
        c1, c2 = d["comparison"]
        A, B = _per_seed_vec(c1), _per_seed_vec(c2)
        if not A or not B:
            continue
        _o, _lo, _hi, _p = cluster_bootstrap_paired(A, B)
        _n_ex = sum(len(v) for v in A.values())
        _agree = ""
        _seedrow = next((r for r in rows if r.get("hypothesis") == h), None)
        if _seedrow and isinstance(_seedrow.get("p_raw"), float):
            _sig_seed = _seedrow["p_raw"] < ALPHA
            _sig_clus = _p < ALPHA
            _agree = ("  [agrees with the seed-paired test]" if _sig_seed == _sig_clus
                      else "  [DISAGREES -- the seed-paired test is underpowered here]")
        print(f"  {h} {c1} - {c2}: {_o:+.4f}  95% CI [{_lo:+.4f},{_hi:+.4f}]  "
              f"p={_p:.3f}  ({len(A)} seeds x {_n_ex} predictions){_agree}")
else:
    print("\nExample-level cluster bootstrap skipped: no per-example cache. "
          "Re-run Cell 10; it writes per_example_synth.json as it evaluates.")

conf_df = pd.DataFrame(rows)
_excluded = [r["hypothesis"] for r in rows
             if str(r.get("testability", "")) not in ("", "testable")]
if _excluded:
    print(f"EXCLUDED FROM THE HOLM FAMILY as not testable at this seed budget: "
          f"{', '.join(_excluded)}")
    for r in rows:
        if r["hypothesis"] in _excluded:
            print(f"  {r['hypothesis']}: {r.get('testability')} -- "
                  f"{r.get('status', '')}")
    print(f"Family size for the correction is therefore {len(pvals)}, "
          f"not {len(rows)}.")
print("Pre-registered confirmatory results (Holm-corrected):")
print(conf_df.to_string(index=False))
conf_df.to_csv(os.path.join(OUT_DIR, "confirmatory_holm.csv"), index=False)
print("\nFamily-wise error controlled at alpha =", ALPHA, "; verdicts follow the pre-registration.")
print("Holm families: the SYNTHETIC hypotheses (H1-H5) and the REAL-DATA hypotheses (R1-R3) are "
      "corrected as TWO SEPARATE families -- two distinct studies on different data, metrics and "
      "generative processes. H6 is a dominance check, not a NHST, and is reported outside the set.")
print("CAVEAT:", NOTES_PREREG["small_n_caveat"])


## Cell 11f - Tests for pre-specified plan II

Both families use the estimators of the main analysis: seed-paired bootstrap, paired
standardised effect, the testability screen, and a Holm correction within each family. The
P-family is tested at every fraction, and the fractions are treated as one family.


In [ ]:
# === Cell 11f: analysis of pre-specified plan II (REVISION v4) ===
import matplotlib
matplotlib.use("Agg") if not globals().get("_MPL_INLINE") else None
import matplotlib.pyplot as plt
def _boot_ci_diff(d, n=10000, seed=0):
    if len(d) < 2:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    b = np.array([rng.choice(d, len(d)).mean() for _ in range(n)])
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

def _paired_test(a_vals, b_vals, direction, min_effect, label):
    """Same estimators as Cell 11b: testability screen, paired bootstrap, paired d_z."""
    a = np.asarray(a_vals, float); b = np.asarray(b_vals, float)
    if len(a) != len(b) or len(a) < 2:
        return dict(hypothesis=label, n=int(min(len(a), len(b))), status="not computable",
                    mean_diff=np.nan, p=np.nan, dz=np.nan, ci_lo=np.nan, ci_hi=np.nan,
                    right_direction=False, meets_min_effect=False)
    valid = ~(np.isnan(a) | np.isnan(b)); a, b = a[valid], b[valid]
    status, p_floor, _note = contrast_testability(a, b)
    mean, p = paired_bootstrap(a, b)
    dz = cohens_d_paired(a, b)
    lo, hi = _boot_ci_diff(a - b)
    right = (mean > 0) if direction == "greater" else (mean < 0)
    return dict(hypothesis=label, n=int(len(a)), status=status,
                p_floor=round(float(p_floor), 4), mean_diff=round(float(mean), 4),
                p=round(float(p), 4),
                dz=(None if not np.isfinite(dz) else round(float(dz), 3)),
                ci_lo=round(lo, 4), ci_hi=round(hi, 4),
                right_direction=bool(right), meets_min_effect=bool(abs(mean) >= min_effect))

def _holm(rows, alpha=ALPHA):
    live = [r for r in rows if r["status"] == "testable"]
    if live:
        rej, padj, _, _ = multipletests([r["p"] for r in live], alpha=alpha, method="holm")
        for r, pa, rj in zip(live, padj, rej):
            r["p_holm"] = round(float(pa), 4)
            r["verdict"] = ("SUPPORTED" if (rj and r["right_direction"]
                                            and r["meets_min_effect"]) else "not supported")
    for r in rows:
        r.setdefault("p_holm", None)
        r.setdefault("verdict", f"NOT TESTABLE ({r['status']})")
    return rows

plan_ii_rows = []

# --- E-family: confirmatory estimator replication on fresh seeds
if globals().get("estconf_df") is not None and len(estconf_df):
    _v = lambda c, m: estconf_df[estconf_df.condition == c].sort_values("seed")[m].values
    _rows = []
    for _h, _d in PREREGISTRATION_II.items():
        _rows.append(_paired_test(_v(_d["a"], _d["metric"]), _v(_d["b"], _d["metric"]),
                                  _d["direction"], _d["min_effect"],
                                  f"{_h}: {_d['a']} vs {_d['b']} on {_d['metric']}"))
    _rows = _holm(_rows)
    plan_ii_rows += [dict(family="E (estimator replication)", **r) for r in _rows]

# --- P-family: partial-supervision curve, tested at each fraction
if globals().get("partial_df") is not None and len(partial_df):
    _rows = []
    for _f in sorted(partial_df.faith_fraction.unique()):
        _sub = partial_df[partial_df.faith_fraction == _f]
        _v = lambda c, m: _sub[_sub.condition == c].sort_values("seed")[m].values
        for _h, _d in PREREGISTRATION_PARTIAL.items():
            if _d["a"] in set(_sub.condition) and _d["b"] in set(_sub.condition):
                _rows.append(_paired_test(_v(_d["a"], _d["metric"]), _v(_d["b"], _d["metric"]),
                                          _d["direction"], _d["min_effect"],
                                          f"{_h} @ f={_f}: {_d['a']} vs {_d['b']}"))
    _rows = _holm(_rows)
    plan_ii_rows += [dict(family="P (partial supervision)", **r) for r in _rows]

if plan_ii_rows:
    plan_ii_df = pd.DataFrame(plan_ii_rows)
    atomic_csv(plan_ii_df, os.path.join(OUT_DIR, "confirmatory_plan_ii.csv"), index=False)
    print(plan_ii_df.to_string(index=False))
else:
    plan_ii_df = None
    print("No plan-II results available yet.")

# --- curve figure
if globals().get("partial_df") is not None and len(partial_df):
    _full = df[df.condition.isin(PARTIAL_CONDITIONS)][
        ["condition", "seed", "faith_excess_id", "csat_norm_id"]].assign(faith_fraction=1.0)
    _zero = df[df.condition.isin(["no_faith", "soft_no_faith"])][
        ["condition", "seed", "faith_excess_id", "csat_norm_id"]].assign(faith_fraction=0.0)
    _zero = _zero.replace({"condition": {"no_faith": "coformer", "soft_no_faith": "no_stgs"}})
    _curve = pd.concat([partial_df[["condition", "seed", "faith_excess_id",
                                    "csat_norm_id", "faith_fraction"]], _full, _zero])
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    for ax, _m, _lab in [(axes[0], "faith_excess_id", "F1 excess (ID)"),
                         (axes[1], "csat_norm_id", "CSat-norm (ID)")]:
        for _c in PARTIAL_CONDITIONS:
            _g = _curve[_curve.condition == _c].groupby("faith_fraction")[_m]
            ax.errorbar(_g.mean().index, _g.mean().values, yerr=_g.std().values,
                        marker="o", capsize=2, label=_c)
        ax.set_xlabel("fraction of instances with per-slot supervision")
        ax.set_ylabel(_lab); ax.set_xscale("symlog", linthresh=0.05)
    axes[0].legend(fontsize=7, frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "partial_supervision_curve.png"), dpi=200)
    fig.savefig(os.path.join(OUT_DIR, "partial_supervision_curve.pdf"))
    plt.close(fig)
    print("Curve written to outputs/partial_supervision_curve.{png,pdf}")


## Cell 11g — Plan III analysis (decision rules fixed in Cell 3d)
Confirmatory (A2): one-sided exact sign-flip tests with Holm correction and the minimum effect of
0.03. Secondary: the success-probability endpoint (exact McNemar) and non-superiority.
Diagnostic (A1, K, W, B, D): estimates, intervals, exact two-sided p-values and minimum
detectable differences, without verdicts.


In [ ]:
# === Cell 11g: analysis of Plan III (REVISION v12) ===
import itertools as _it
from statsmodels.stats.multitest import multipletests
from scipy import stats as _st

def _signflip_p(d, alternative="two-sided"):
    d = np.asarray(d, float)
    if len(d) == 0 or np.all(np.abs(d) < 1e-12):
        return 1.0
    signs = np.array(list(_it.product([-1.0, 1.0], repeat=len(d))))
    means = (signs * d).mean(axis=1); obs = d.mean()
    if alternative == "greater":
        return float((means >= obs - 1e-12).mean())
    return float((np.abs(means) >= abs(obs) - 1e-12).mean())

def _t_ci(d, level):
    d = np.asarray(d, float); n = len(d); sd = d.std(ddof=1) if n > 1 else 0.0
    if n < 2 or sd == 0:
        return float(d.mean()), float(d.mean())
    h = _st.t.ppf(0.5 + level / 2, n - 1) * sd / np.sqrt(n)
    return float(d.mean() - h), float(d.mean() + h)

def _mde80(d):
    d = np.asarray(d, float); n = len(d); sd = d.std(ddof=1) if n > 1 else 0.0
    if n < 2 or sd == 0:
        return 0.0
    tc = _st.t.ppf(0.975, n - 1)
    f = lambda es: (np.nan_to_num(_st.nct.sf(tc, n - 1, es * np.sqrt(n)), nan=1.0)
                    + np.nan_to_num(_st.nct.cdf(-tc, n - 1, es * np.sqrt(n)), nan=0.0) - 0.8)
    from scipy.optimize import brentq
    return float(brentq(f, 1e-6, 20) * sd)

def _mcnemar_one_sided(sa, sb):
    b = int(np.sum(sa & ~sb)); c = int(np.sum(~sa & sb))
    return float(_st.binom.sf(b - 1, b + c, 0.5)) if b + c else 1.0

def _paired(a_df, a_cond, b_df, b_cond, seeds, metric="faith_excess_id"):
    va = a_df[a_df.condition == a_cond].set_index("seed")[metric]
    vb = b_df[b_df.condition == b_cond].set_index("seed")[metric]
    s = [x for x in seeds if x in va.index and x in vb.index]
    return np.array([va[x] for x in s], float), np.array([vb[x] for x in s], float), s

plan3_rows = []
_p05 = (partial_df[partial_df.faith_fraction.round(4) == 0.005]
        if globals().get("partial_df") is not None else None)
if globals().get("plan3_df") is not None and len(plan3_df) and _p05 is not None:
    # ---- confirmatory A2 ----
    live = []
    for h, spec in PLAN_III["confirmatory"].items():
        a3 = plan3_df[plan3_df.experiment == "A2"]
        a, b, s = _paired(a3, spec["a"], _p05, spec["b"], spec["seeds"])
        if len(s) < len(spec["seeds"]):
            plan3_rows.append(dict(id=h, role="confirmatory", n=len(s),
                                   verdict=f"incomplete ({len(s)}/{len(spec['seeds'])} seeds)"))
            continue
        d = a - b; lo90, hi90 = _t_ci(d, 0.90); lo95, hi95 = _t_ci(d, 0.95)
        succ_a, succ_b = a > 0.30, b > 0.30
        live.append(dict(id=h, role="confirmatory", n=len(s), mean_a=a.mean(), mean_b=b.mean(),
                         mean_diff=d.mean(), p_exact_one_sided=_signflip_p(d, "greater"),
                         ci95_lo=lo95, ci95_hi=hi95, benefit_ge_0_03_excluded=bool(hi90 < 0.03),
                         success_a=int(succ_a.sum()), success_b=int(succ_b.sum()),
                         p_mcnemar_one_sided=_mcnemar_one_sided(succ_a, succ_b),
                         mde80=_mde80(d)))
    if live:
        rej, padj, _, _ = multipletests([r["p_exact_one_sided"] for r in live], alpha=ALPHA,
                                        method="holm")
        for r, pa in zip(live, padj):
            r["p_holm"] = float(pa)
            r["verdict"] = ("SUPPORTED" if (pa < ALPHA and r["mean_diff"] >= 0.03)
                            else "not supported")
        plan3_rows += live
    # ---- diagnostics K, W, B (paired with stored or new comparators) ----
    for h, (exp, a_c, b_src, b_c) in {
            "K_ST": ("K", "coformer", "p05", "frozen"), "K_soft": ("K", "no_stgs", "p05", "soft_frozen"),
            "W_ST": ("W", "coformer", "p05", "frozen"), "W_soft": ("W", "no_stgs", "p05", "soft_frozen"),
            "B_ST": ("B", "coformer", "B", "frozen")}.items():
        a3 = plan3_df[plan3_df.experiment == exp]
        b3 = _p05 if b_src == "p05" else plan3_df[plan3_df.experiment == b_src]
        a, b, s = _paired(a3, a_c, b3, b_c, list(range(6)))
        if len(s) < 2:
            continue
        d = a - b; lo95, hi95 = _t_ci(d, 0.95)
        plan3_rows.append(dict(id=h, role="diagnostic", n=len(s), mean_a=a.mean(), mean_b=b.mean(),
                               mean_diff=d.mean(), p_exact_two_sided=_signflip_p(d),
                               ci95_lo=lo95, ci95_hi=hi95, mde80=_mde80(d),
                               success_a=int((a > 0.30).sum()), success_b=int((b > 0.30).sum()),
                               verdict="diagnostic (no verdict)"))
    # ---- diagnostic A1 (answers only) ----
    for h, (a_c, ref) in {"A1_ST": ("ans_st", "no_faith"), "A1_soft": ("ans_soft", "soft_no_faith")}.items():
        a3 = plan3_df[(plan3_df.experiment == "A1") & (plan3_df.condition == a_c)]
        if not len(a3):
            continue
        ref_v = df[df.condition == ref]["faith_excess_id"] if "df" in globals() else pd.Series(dtype=float)
        plan3_rows.append(dict(id=h, role="diagnostic", n=len(a3),
                               mean_a=float(a3.faith_excess_id.mean()),
                               max_a=float(a3.faith_excess_id.max()),
                               success_a=int((a3.faith_excess_id > 0.30).sum()),
                               mean_ref=float(ref_v.mean()) if len(ref_v) else float("nan"),
                               degenerate_a=int(a3.degenerate.astype(bool).sum()),
                               verdict="diagnostic (no verdict)"))
    plan3_res = pd.DataFrame(plan3_rows)
    print("=== Plan III (specification SHA-256 %s) ===" % PLAN_III_SHA256[:12])
    print(plan3_res.round(4).to_string(index=False))
    plan3_res.to_csv(os.path.join(OUT_DIR, "confirmatory_plan_iii.csv"), index=False)
    # learning curves of the budget stage
    _b = plan3_df[plan3_df.experiment == "B"]
    if len(_b):
        _curves = [dict(condition=r.condition, seed=int(r.seed), epoch=int(e), excess=float(v))
                   for r in _b.itertuples() for e, v in json.loads(r.learning_curve or "[]")]
        pd.DataFrame(_curves).to_csv(os.path.join(OUT_DIR, "plan_iii_learning_curves.csv"), index=False)
else:
    print("Plan III analysis skipped: no Plan III results yet.")


## Cell 11h — Follow-up study 2 analysis (decision rules fixed in Cell 3e)

In [ ]:
# === Cell 11h: analysis of follow-up study 2 (REVISION v13; rules fixed in Cell 3e) ===
fs2_rows = []
_p05b = (partial_df[partial_df.faith_fraction.round(4) == 0.005]
         if globals().get("partial_df") is not None else None)
_a2 = (plan3_df[plan3_df.experiment == "A2"]
       if globals().get("plan3_df") is not None and plan3_df is not None and len(plan3_df) else None)
if globals().get("fs2_df") is not None and fs2_df is not None and len(fs2_df) and _p05b is not None:
    _E = lambda e: fs2_df[fs2_df.experiment == e]
    comps = {
        "C1": (_E("C1"), "coformer", _E("C1"), "frozen"),
        "C2": (_E("C2_Q1"), "rl_ans_ps", _p05b, "frozen"),
        "C3": (_E("C3"), "ans_st_ps", _a2, "ans_st_ps") if _a2 is not None else None,
    }
    live2 = []
    for h, cmp in comps.items():
        if cmp is None:
            continue
        a, b, s = _paired(cmp[0], cmp[1], cmp[2], cmp[3], list(range(12)))
        if len(s) < 12:
            fs2_rows.append(dict(id=h, role="confirmatory", n=len(s),
                                 verdict=f"incomplete ({len(s)}/12 seeds)"))
            continue
        d = a - b; lo90, hi90 = _t_ci(d, 0.90); lo95, hi95 = _t_ci(d, 0.95)
        sa, sb = a > 0.30, b > 0.30
        live2.append(dict(id=h, role="confirmatory", n=len(s), mean_a=a.mean(), mean_b=b.mean(),
                          mean_diff=d.mean(), p_exact_one_sided=_signflip_p(d, "greater"),
                          p_exact_two_sided=_signflip_p(d), ci95_lo=lo95, ci95_hi=hi95,
                          benefit_ge_0_03_excluded=bool(hi90 < 0.03),
                          success_a=int(sa.sum()), success_b=int(sb.sum()),
                          p_mcnemar_one_sided=_mcnemar_one_sided(sa, sb),
                          mean_success_a=float(a[sa].mean()) if sa.any() else float("nan"),
                          mean_success_b=float(b[sb].mean()) if sb.any() else float("nan"),
                          mean_fail_a=float(a[~sa].mean()) if (~sa).any() else float("nan"),
                          mean_fail_b=float(b[~sb].mean()) if (~sb).any() else float("nan"),
                          mde80=_mde80(d)))
    if live2:
        # Holm over the confirmatory family {C1, C2, C3}; incomplete members keep the family size
        _m = len(FS2["confirmatory"])
        _ps = sorted(((r["p_exact_one_sided"], i) for i, r in enumerate(live2)))
        _run = 0.0
        for rank, (p, i) in enumerate(_ps):
            _run = max(_run, min(1.0, (_m - rank) * p))
            live2[i]["p_holm"] = _run
        for r in live2:
            r["verdict"] = ("SUPPORTED" if (r["p_holm"] < ALPHA and r["mean_diff"] >= 0.03)
                            else "not supported")
        fs2_rows += live2
    # diagnostics (seeds 0-5)
    for h, (ea, ca, eb, cb) in {
            "Q16ps_vs_frozen": (_E("C2_Q16ps"), "rl_ans_ps", _p05b, "frozen"),
            "Q16ps_vs_Q1ps": (_E("C2_Q16ps"), "rl_ans_ps", _E("C2_Q1"), "rl_ans_ps")}.items():
        a, b, s = _paired(ea, ca, eb, cb, list(range(6)))
        if len(s) < 2:
            continue
        d = a - b; lo95, hi95 = _t_ci(d, 0.95)
        fs2_rows.append(dict(id=h, role="diagnostic", n=len(s), mean_a=a.mean(), mean_b=b.mean(),
                             mean_diff=d.mean(), p_exact_two_sided=_signflip_p(d), ci95_lo=lo95,
                             ci95_hi=hi95, success_a=int((a > 0.30).sum()),
                             success_b=int((b > 0.30).sum()), verdict="diagnostic (no verdict)"))
    _q16 = _E("C2_Q16")
    if len(_q16):
        _a1 = (plan3_df[(plan3_df.experiment == "A1") & (plan3_df.condition == "ans_st")]
               if _a2 is not None else pd.DataFrame())
        fs2_rows.append(dict(id="Q16_answers_only", role="diagnostic", n=len(_q16),
                             mean_a=float(_q16.faith_excess_id.mean()),
                             max_a=float(_q16.faith_excess_id.max()),
                             success_a=int((_q16.faith_excess_id > 0.30).sum()),
                             degenerate_a=int(_q16.degenerate.astype(bool).sum()),
                             mean_ref=float(_a1.faith_excess_id.mean()) if len(_a1) else float("nan"),
                             verdict="diagnostic (no verdict)"))
    fs2_res = pd.DataFrame(fs2_rows)
    print("=== Follow-up study 2 (specification SHA-256 %s) ===" % FS2_SHA256[:12])
    print(fs2_res.round(4).to_string(index=False))
    fs2_res.to_csv(os.path.join(OUT_DIR, "confirmatory_fs2.csv"), index=False)
    _c1 = _E("C1")
    if len(_c1):
        _cv = [dict(condition=r.condition, seed=int(r.seed), epoch=int(e), excess=float(v))
               for r in _c1.itertuples() for e, v in json.loads(r.learning_curve or "[]")]
        _cv = pd.DataFrame(_cv)
        _cv.to_csv(os.path.join(OUT_DIR, "fs2_learning_curves.csv"), index=False)
        if len(_cv) and {36, 48} <= set(_cv.epoch):
            _g = _cv.pivot_table(index=["condition", "seed"], columns="epoch", values="excess")
            _gain = (_g[48] - _g[36]).groupby(level=0).mean()
            _conv = bool((_gain.abs() < 0.02).all())
            print("  mean change of the ID excess between epochs 36 and 48:",
                  _gain.round(4).to_dict(), "-> converged" if _conv else "-> NOT converged")
else:
    print("Follow-up study 2 analysis skipped: no results yet.")


## Cell 11i — Family H analysis (decision rules fixed in Cell 3f)

In [ ]:
# === Cell 11i: analysis of family H (REVISION v14; rules fixed in Cell 3f) ===
famH_rows = []
if globals().get("famH_df") is not None and famH_df is not None and len(famH_df):
    _fmaxH = 1.0 - GOLD_DENSITY / 2.0
    _pilH = famH_df[famH_df.experiment.astype(str).str.startswith("H_pilot")]
    if len(_pilH) == 3:
        _lvH, _prH = famh_select_level(_pilH)
        print("=== Family H pilot (frozen, full supervision, seed %d) ===" % FAMH_PILOT_SEED)
        for r in _prH:
            print(f"  level {r['level']}: ID excess share of maximum {r['share']:.3f}"
                  f"{'  (in [0.20, 0.95])' if r['in_band'] else ''}")
        print(f"  selected level: {_lvH}")
    else:
        _lvH = None
        print(f"Family H pilot incomplete ({len(_pilH)}/3 levels).")
    _mainH = famH_df[famH_df.experiment.isin(["H_full", "H_scarce"])]
    if _lvH is not None and len(_mainH):
        assert set(_mainH.level.astype(int)) == {_lvH}, "main runs must use the selected level"
        _nH = {e: len(d["seeds"]) for e, d in FAMH["designs"].items()}
        liveH = []
        for h in ("H_full", "H_scarce"):
            dh = _mainH[_mainH.experiment == h]
            a, b, s = _paired(dh, "coformer", dh, "frozen", list(range(_nH[h])))
            if len(s) < _nH[h]:
                famH_rows.append(dict(id=h, role="confirmatory", n=len(s),
                                      verdict=f"incomplete ({len(s)}/{_nH[h]} seeds)"))
                continue
            d = a - b; lo90, hi90 = _t_ci(d, 0.90); lo95, hi95 = _t_ci(d, 0.95)
            sa, sb = a > 0.30, b > 0.30
            med_share_b = float(np.median(b)) / _fmaxH
            liveH.append(dict(id=h, role="confirmatory", level=_lvH, n=len(s),
                              mean_a=a.mean(), mean_b=b.mean(), share_b_median=med_share_b,
                              mean_diff=d.mean(), p_exact_one_sided=_signflip_p(d, "greater"),
                              p_exact_two_sided=_signflip_p(d), ci95_lo=lo95, ci95_hi=hi95,
                              benefit_ge_0_03_excluded=bool(hi90 < 0.03),
                              success_a=int(sa.sum()), success_b=int(sb.sum()),
                              p_mcnemar_one_sided=_mcnemar_one_sided(sa, sb),
                              mean_success_a=float(a[sa].mean()) if sa.any() else float("nan"),
                              mean_success_b=float(b[sb].mean()) if sb.any() else float("nan"),
                              mean_fail_a=float(a[~sa].mean()) if (~sa).any() else float("nan"),
                              mean_fail_b=float(b[~sb].mean()) if (~sb).any() else float("nan"),
                              mde80=_mde80(d), untestable=bool(med_share_b >= 0.95)))
        # Holm over {H_full, H_scarce}; the family size stays 2 whatever is incomplete/untestable
        _mH = len(FAMH["confirmatory"])
        _testable = [r for r in liveH if not r["untestable"]]
        _run = 0.0
        for rank, r in enumerate(sorted(_testable, key=lambda r: r["p_exact_one_sided"])):
            _run = max(_run, min(1.0, (_mH - rank) * r["p_exact_one_sided"]))
            r["p_holm"] = _run
        for r in liveH:
            if r["untestable"]:
                r["p_holm"] = float("nan"); r["verdict"] = "UNTESTABLE (ceiling)"
            else:
                r["verdict"] = ("SUPPORTED" if (r["p_holm"] < ALPHA and r["mean_diff"] >= 0.03)
                                else "not supported")
        famH_rows += liveH
        # secondary metrics (no verdicts)
        for h in ("H_full", "H_scarce"):
            dh = _mainH[_mainH.experiment == h]
            for met in ("faith_excess_paraphrase", "csat_norm_id", "csat_norm_paraphrase",
                        "ood_retention_para"):
                a, b, s = _paired(dh, "coformer", dh, "frozen", list(range(_nH[h])), metric=met)
                ok = ~(np.isnan(a) | np.isnan(b))
                if ok.sum() < 2:
                    continue
                d = (a - b)[ok]; lo95, hi95 = _t_ci(d, 0.95)
                famH_rows.append(dict(id=f"{h}:{met}", role="secondary", level=_lvH,
                                      n=int(ok.sum()), mean_a=float(a[ok].mean()),
                                      mean_b=float(b[ok].mean()), mean_diff=float(d.mean()),
                                      p_exact_two_sided=_signflip_p(d), ci95_lo=lo95,
                                      ci95_hi=hi95, verdict="secondary (no verdict)"))
        # learning curves and the reported convergence criterion
        _cvH = [dict(experiment=r.experiment, condition=r.condition, seed=int(r.seed),
                     epoch=int(e), excess=float(v))
                for r in _mainH.itertuples() for e, v in json.loads(r.learning_curve or "[]")]
        _cvH = pd.DataFrame(_cvH)
        if len(_cvH):
            _cvH.to_csv(os.path.join(OUT_DIR, "famH_learning_curves.csv"), index=False)
            if {18, 24} <= set(_cvH.epoch):
                for h in ("H_full", "H_scarce"):
                    _g = _cvH[_cvH.experiment == h].pivot_table(
                        index=["condition", "seed"], columns="epoch", values="excess")
                    if 18 in _g and 24 in _g:
                        _gain = (_g[24] - _g[18]).groupby(level=0).mean()
                        print(f"  {h}: mean change of the ID excess between epochs 18 and 24:",
                              _gain.round(4).to_dict(),
                              "-> converged" if bool((_gain.abs() < 0.02).all()) else "-> NOT converged")
    if famH_rows:
        famH_res = pd.DataFrame(famH_rows)
        print("=== Family H (specification SHA-256 %s) ===" % FAMH_SHA256[:12])
        print(famH_res.round(4).to_string(index=False))
        famH_res.to_csv(os.path.join(OUT_DIR, "confirmatory_famH.csv"), index=False)
else:
    print("Family H analysis skipped: no results yet.")

# ---------------------------------------------------------------- completion status of v14
def _n_rows(path):
    try:
        return len(pd.read_csv(path)) if os.path.exists(path) else 0
    except Exception:
        return 0
V14_STATUS = {
    "followup2_runs": _n_rows(os.path.join(CKPT_DIR, f"results_fs2_{PROBE_FP}.csv")),
    "followup2_needed": 60,
    "famH_runs": _n_rows(os.path.join(CKPT_DIR, f"results_famH_{PROBE_FP}.csv")),
    "famH_needed": 59,
}
V14_STATUS["complete"] = (V14_STATUS["followup2_runs"] >= 60 and V14_STATUS["famH_runs"] >= 59)
atomic_json({**V14_STATUS, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
            os.path.join(OUT_DIR, "v14_status.json"), indent=1)
print("\n" + "=" * 72)
if V14_STATUS["complete"]:
    print("ALL v14 STAGES COMPLETE: follow-up study 2 (60/60) and family H (59/59).")
    print("Download the CoFormer folder (or the state zip) and send it; no further Run All needed.")
    if not os.path.exists(os.path.join(OUT_DIR, "v14_complete_alert_sent")):
        send_alert("CoFormer v14: all stages complete",
                   "Follow-up study 2 and family H are complete. Download the CoFormer folder.")
        open(os.path.join(OUT_DIR, "v14_complete_alert_sent"), "w").write("1")
    try:
        if not globals().get("PERSISTENT", True):
            push_state(tag="v14 complete", force=True)
    except Exception as _e:
        print(f"  (state push failed: {_e})")
else:
    print(f"v14 NOT complete yet: follow-up study 2 {V14_STATUS['followup2_runs']}/60, "
          f"family H {V14_STATUS['famH_runs']}/59. Run All again in a new session; "
          "it resumes automatically.")
print("=" * 72)


## Cell 12 — Publication-ready tables (CSV + LaTeX) saved to Drive
Writes the aggregated results as a `.csv` and a `booktabs` LaTeX table ready to paste into a
the target venue manuscript. Tables are saved to the Drive `outputs/` folder.

In [ ]:
# === Cell 12: publication tables ===
TABLE_COLS = {
    "faith_id": "Faith F1", "faith_id_sd": "Faith sd",
    "faith_excess_id": "Faith excess", "faith_excess_id_sd": "Excess sd",
    "csat_norm_id": "CSat-norm ID", "csat_norm_id_sd": "CSat sd",
    "csat_norm_paraphrase": "CSat para", "csat_norm_scale": "CSat scale",
    "acc_id": "Exact solve", "ood_rel_drop": "OOD rel drop",
    "pred_density": "Density", "degenerate": "Degen.",
    "refine_steps": "Refine steps", "train_secs": "Train s",
}
table = agg[list(TABLE_COLS)].rename(columns=TABLE_COLS)
csv_path = os.path.join(OUT_DIR, "results_table.csv")
table.round(3).to_csv(csv_path)

def to_latex(tbl):
    cols = " & ".join(["Condition"] + list(tbl.columns))
    body = [str(c).replace("_", r"\_") + " & " + " & ".join(str(v) for v in r.values) + r" \\"
            for c, r in tbl.round(3).iterrows()]
    cap = (f"CoFormer ablation results (mean over {len(CFG.seeds)} seeds). Instances are uniform "
           f"random {CFG.clause_len}-SAT at clause/variable ratio {CFG.sat_ratio}, "
           f"rejection-sampled for satisfiability. CSat-norm is gold-clause satisfaction "
           f"normalised so chance $=0$ and perfect $=1$; `Exact solve' is the stricter "
           f"all-clauses metric. `Density' is the emitted literal density (gold "
           f"{GOLD_DENSITY:.2f}) and `Degen.' the share of seeds flagged degenerate. "
           f"`Faith excess' is clause-F1 minus the closed-form score of an information-free CNF "
           f"at the density the model emitted, $g d/(g+d)$; raw F1 alone is confounded by density. "
           f"All conditions are trained; `frozen' keeps $L_{{faith}}$ but detaches the "
           f"logic-layer gradient.")
    return ("\\begin{table}[t]\n\\centering\n\\caption{" + cap + "}\n\\label{tab:coformer}\n"
            "\\begin{tabular}{l" + "r" * len(tbl.columns) + "}\n\\hline\n" + cols + r" \\" +
            "\n\\hline\n" + "\n".join(body) + "\n\\hline\n\\end{tabular}\n\\end{table}")

tex_path = os.path.join(OUT_DIR, "results_table.tex")
open(tex_path, "w").write(to_latex(table))
print("Saved:", csv_path); print("Saved:", tex_path)
if DEGENERATE_CONDITIONS:
    print("\nWARNING: this table contains degenerate conditions "
          f"({', '.join(DEGENERATE_CONDITIONS)}). Do NOT put it in a paper as-is.")
print("\nLaTeX preview:\n"); print(to_latex(table))


## Cell 13 — Publication-ready figures (300-dpi PDF + PNG) saved to Drive
Two figures: (1) faithfulness & accuracy by condition with error bars; (2) ID→OOD accuracy curves.
Saved as both 300-dpi PDF (for the manuscript) and PNG (for quick viewing), to the Drive
`outputs/` folder.

In [ ]:
# === Cell 13: publication figures ===
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
x = np.arange(len(CONDITIONS)); w = 0.38

ax[0].bar(x - w/2, agg["faith_id"], w, yerr=agg["faith_id_sd"], capsize=4, label="Faithfulness F1")
ax[0].bar(x + w/2, agg["csat_norm_id"], w, yerr=agg["csat_norm_id_sd"], capsize=4,
          label="CSat-norm (chance = 0)")
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_xticks(x); ax[0].set_xticklabels(CONDITIONS, rotation=15)
ax[0].set_title(f"In-distribution (random {CFG.clause_len}-SAT, ratio {CFG.sat_ratio})")
ax[0].legend(fontsize=9)

sets = [("csat_norm_id", "csat_norm_id_sd", "ID"),
        ("csat_norm_paraphrase", "csat_norm_para_sd", "paraphrase-OOD"),
        ("csat_norm_scale", "csat_norm_scale_sd", "scale-OOD")]
bw = 0.26
for j, (m, sd, lbl) in enumerate(sets):
    ax[1].bar(x + (j - 1) * bw, agg[m], bw, yerr=agg[sd], capsize=3, label=lbl)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_xticks(x); ax[1].set_xticklabels(CONDITIONS, rotation=15)
ax[1].set_title("CSat-norm by evaluation set (H3)"); ax[1].legend(fontsize=9)
ax[1].text(0.02, 0.96, "sets are unordered categories, not a trajectory",
           transform=ax[1].transAxes, fontsize=8, va="top", style="italic")

ax[2].bar(x, agg["pred_density"], 0.5,
          color=["tab:red" if agg.loc[c, "degenerate"] > 0.5 else "tab:green" for c in CONDITIONS])
ax[2].axhline(GOLD_DENSITY, color="k", ls="--", lw=1.2, label=f"gold density {GOLD_DENSITY:.2f}")
ax[2].set_xticks(x); ax[2].set_xticklabels(CONDITIONS, rotation=15)
ax[2].set_ylim(0, 1); ax[2].set_title("Emitted literal density (red = degenerate)")
ax[2].legend(fontsize=9)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(os.path.join(OUT_DIR, f"coformer_results.{ext}"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved figures to", OUT_DIR)


## Cell 13b — H6: ablation dominance

The mechanism semantics do not imply a total order over the conditions:
`frozen` keeps `L_faith` and drops the solver gradient, while `no_faith` does
the opposite, so neither must dominate the other.

We therefore test the claim that *is* implied: removing either mechanism must
not **improve** on the full model. The cell reports, per seed and per metric,
the share of contrasts in which the full model is not beaten, with a bootstrap
CI — one falsifiable number.


In [ ]:
# Run-All safety: skip cleanly if the synthetic matrix is empty.
if "df" not in globals() or df is None or len(df) == 0:
    print("Skipped: no synthetic results matrix (df) available.")
else:
    # === Cell 13b: H6 ablation-DOMINANCE test ===
    # Tests the claim the mechanism semantics DO imply: removing either mechanism must not
    # IMPROVE on the full model. No total order between the two removals is claimed.
    ABLATIONS = ["no_faith", "no_stgs", "frozen",
                 "soft_frozen", "soft_no_faith"]
    METRICS = [("faith_excess_id", "faithfulness (density-controlled)"),
               ("csat_norm_id", "end-task csat_norm")]

    rows = []
    for metric, lbl in METRICS:
        for abl in ABLATIONS:
            wins, n = 0, 0
            for s in sorted(df.seed.unique()):
                sub = df[df.seed == s].set_index("condition")
                if "coformer" not in sub.index or abl not in sub.index: continue
                n += 1
                wins += int(float(sub.loc["coformer", metric]) >= float(sub.loc[abl, metric]) - 1e-9)
            if n == 0:
                rows.append(dict(metric=lbl, contrast=f"coformer >= {abl}", share=np.nan,
                                 n_seeds=0, note="")); continue
            share = wins / n
            draws = np.random.default_rng(0).binomial(1, share, size=(5000, n)).mean(axis=1)
            lo, hi = np.percentile(draws, [2.5, 97.5])
            if abl in CEILING_CONDITIONS:
                note = "ABLATION AT THE CEILING -- this contrast FALSIFIES H6"
            elif abl in set(DEGENERATE_CONDITIONS) | set(NOISE_CONDITIONS):
                note = "ablation is degenerate or information-free (contrast uninterpretable)"
            else:
                note = ""
            rows.append(dict(metric=lbl, contrast=f"coformer >= {abl}", share=round(share, 3),
                             ci=f"[{lo:.2f},{hi:.2f}]", n_seeds=n, note=note))

    h6_df = pd.DataFrame(rows)
    print("H6 ablation dominance (per-seed share where the full model is not beaten):")
    print(h6_df.to_string(index=False))
    # a ceiling contrast is usable -- indeed decisive -- so it must count toward the verdict
    usable = [r for r in rows
              if r["share"] == r["share"]
              and (not r.get("note") or "CEILING" in r.get("note", ""))]
    H6_SUPPORTED = bool(usable) and all(r["share"] > 0.5 for r in usable)
    if len(usable) < len(rows):
        print(f"\n  NOTE: {len(rows)-len(usable)} contrast(s) excluded -- the ablation collapsed "
              f"to a constant predictor, so 'coformer beats it' would be a meaningless win.")
    print(f"\nH6 verdict: {'SUPPORTED' if H6_SUPPORTED else 'NOT supported at this scale'}.")
    h6_df.to_csv(os.path.join(OUT_DIR, "h6_ablation_dominance.csv"), index=False)

    # ---- 2x2 factorial decomposition: the attribution the design supports ----
    print("\n=== FACTORIAL DECOMPOSITION (bridge x solver gradient) ===")
    for _metric, _lbl in METRICS:
        _cell = {}
        for (_b, _sv), _cond in FACTORIAL_CELLS.items():
            _v = df[df.condition == _cond].sort_values("seed")[_metric].values
            if len(_v):
                _cell[(_b, _sv)] = _v
        if len(_cell) < 4 or len({len(v) for v in _cell.values()}) != 1:
            print(f"  {_lbl}: factorial incomplete -- no attribution possible")
            continue
        st_d = _cell[("straight_through", "detached")]
        st_r = _cell[("straight_through", "routed")]
        so_d = _cell[("soft", "detached")]
        so_r = _cell[("soft", "routed")]
        main_bridge = 0.5 * ((so_d - st_d) + (so_r - st_r))
        main_solver = 0.5 * ((st_r - st_d) + (so_r - so_d))
        interaction = (so_r - so_d) - (st_r - st_d)
        zeros = np.zeros_like(main_bridge)
        print(f"\n  {_lbl}")
        print(f"    cell means:  ST/detached {st_d.mean():+.4f}   "
              f"ST/routed {st_r.mean():+.4f}")
        print(f"                 soft/detached {so_d.mean():+.4f}   "
              f"soft/routed {so_r.mean():+.4f}")
        for _nm, _eff in (("bridge (soft - straight-through)", main_bridge),
                          ("solver gradient (routed - detached)", main_solver),
                          ("interaction", interaction)):
            _m, _p = paired_bootstrap(_eff, zeros)
            _dz = cohens_d_paired(_eff, zeros)
            print(f"    {_nm:36s} {_m:+.4f}  (p={_p:.3f}, dz={_dz:+.2f})")
        _tot = abs(main_bridge.mean()) + abs(main_solver.mean())
        if _tot > 1e-12:
            print(f"    -> the bridge accounts for "
                  f"{abs(main_bridge.mean()) / _tot:.0%} of the summed "
                  f"main-effect magnitude on this metric")

    print("\nAblation ladder (means; NO total order is claimed between the two removals):")
    LADDER = [c for c in ["frozen", "no_stgs", "no_faith", "coformer"] if c in agg.index]
    print(agg.loc[LADDER, ["faith_id", "faith_excess_id", "csat_norm_id",
                           "csat_norm_paraphrase", "csat_norm_scale", "pred_density",
                           "degenerate"]].round(3).to_string())

    @torch.no_grad()
    def _noop():
        pass

    def st_bias_probe(n_batches=3, seed=0, taus=(2.0, 1.0, 0.5), objective="task+faith"):
        set_all_seeds(seed)
        model = Translator(VOCAB_SIZE, CLAUSE_TOKENS, CFG.n_clauses, CFG.n_vars, CFG).to(DEVICE)
        logic = SoftMaxSATLayer(CFG.refine_steps,
                                step_size=getattr(CFG, "refine_step_size", 0.5),
                                backprop_steps=getattr(CFG, "refine_backprop_steps", None)
                                ).to(DEVICE)
        loader = make_loader(train, CFG.batch_size, False, seed=seed)
        rows = []
        for tau in taus:
            cos_l, rat_l = [], []
            for bi, (xb, ycb) in enumerate(loader):
                if bi >= n_batches:
                    break
                xb, ycb = xb.to(DEVICE), ycb.to(DEVICE)

                def grad_of(hard):
                    model.zero_grad(set_to_none=True)
                    lg = model(xb).float()
                    oh = st_gumbel_softmax(lg, tau, hard=hard)
                    cm = to_clause_matrix(oh)
                    pres = (oh[..., 0] + oh[..., 2]).float()
                    assign, _, _ = logic(cm.float(), present=pres)
                    loss = anchored_task_loss(assign, ycb, logic)
                    if objective == "task+faith":      # REVISION v2: logits-based L_faith
                        loss = loss + CFG.lambda_faith * faithfulness_loss(oh, ycb, logits=lg)
                    g = torch.autograd.grad(loss, [p for p in model.parameters()
                                                   if p.requires_grad],
                                            allow_unused=True)
                    return torch.cat([x.reshape(-1) for x in g if x is not None])

                torch.manual_seed(1234 + bi)      # same Gumbel noise for both estimators
                g_soft = grad_of(False)
                torch.manual_seed(1234 + bi)
                g_st = grad_of(True)
                cos = float(F.cosine_similarity(g_soft, g_st, dim=0))
                rat = float(g_st.norm() / g_soft.norm().clamp_min(1e-12))
                cos_l.append(cos); rat_l.append(rat)
            rows.append(dict(objective=objective, tau=tau, cosine=float(np.mean(cos_l)),
                             norm_ratio=float(np.mean(rat_l))))
        return pd.DataFrame(rows)

    print("\n=== STRAIGHT-THROUGH BIAS PROBE ===")
    print("cosine between the straight-through gradient and the soft-relaxation gradient,")
    print("at identical Gumbel noise (1.0 = faithful estimator, 0.0 = no directional information)")
    _st_reused = _reuse_reported("st_bias_probe.csv")                         # REVISION v12
    if _st_reused:
        st_df = pd.read_csv(os.path.join(OUT_DIR, "st_bias_probe.csv"))
    else:
        st_df = pd.concat([st_bias_probe(objective="task+faith"),
                           st_bias_probe(objective="task")], ignore_index=True)   # REVISION v2
    print(st_df.round(4).to_string(index=False))
    _c = float(st_df["cosine"].mean()); _r = float(st_df["norm_ratio"].mean())
    if _c < -0.2:
        print(f"  mean cosine {_c:+.3f}, mean norm ratio {_r:.3f}: the straight-through update is "
              f"ACTIVELY OPPOSED to the true gradient, not merely uninformative -- it points into "
              f"the opposite half-space while carrying {1/_r:.0f}x less magnitude. Following it "
              f"moves the model away from the descent direction on average, which is a sufficient "
              f"and specific explanation for every ST condition stalling near clause-F1 0.2 while "
              f"the soft ablation reaches 1.0.")
    elif _c < 0.3:
        print(f"  mean cosine {_c:+.3f}: the straight-through update carries little of the true "
              f"direction, which is a sufficient explanation for the ST conditions stalling.")
    else:
        print(f"  mean cosine {_c:+.3f}: the estimator is directionally faithful, so the ST "
              f"failure is NOT explained by gradient bias alone -- look at variance or at the "
              f"tau schedule.")
    print(f"  NOTE: measured at initialisation on {len(st_df)} tau values; report it as a property "
          f"of the estimator on this task, not as a claim about all straight-through use.")
    if not _st_reused:
        st_df.to_csv(os.path.join(OUT_DIR, "st_bias_probe.csv"), index=False)

    def objective_anchoring_probe(n_eval=128, seed=0):
        g = torch.Generator().manual_seed(seed)
        Y = torch.from_numpy(eval_id[1][:n_eval]).float().to(DEVICE)
        layer = SoftMaxSATLayer(CFG.refine_steps,
                                step_size=getattr(CFG, "refine_step_size", 0.5)).to(DEVICE)

        def random_cnf(density):
            m = (torch.rand(Y.shape, generator=g) < density).float()
            sgn = torch.where(torch.rand(Y.shape, generator=g) < 0.5, -1.0, 1.0)
            return (m * sgn).to(DEVICE)

        def corrupt(keep):
            mask = (torch.rand(Y.shape, generator=g) < keep).float().to(DEVICE)
            return Y * mask + random_cnf(GOLD_DENSITY) * (1 - mask)

        cands = [("gold", Y)]
        cands += [(f"random d={d:.2f}", random_cnf(d)) for d in (GOLD_DENSITY, 0.35, 0.60, 0.90)]
        cands += [(f"gold {int(k*100)}% kept", corrupt(k)) for k in (0.6, 0.3)]

        rows = []
        for name, cm in cands:
            tp, np_, ng = clause_f1_counts(cm, Y)
            f1 = f1_from_counts(tp, np_, ng)
            with torch.no_grad():
                assign, sat_own, _ = layer(cm)
                sat_gold = layer.soft_satisfaction(Y, assign)
            rows.append(dict(cnf=name,
                             density=float((cm.abs() > 0).float().mean()),
                             clause_f1=round(f1, 4),
                             loss_self=round(1.0 - float(sat_own.mean()), 4),
                             loss_anchored=round(1.0 - float(sat_gold.mean()), 4)))
        out = pd.DataFrame(rows)
        r_self = float(np.corrcoef(out.clause_f1, out.loss_self)[0, 1])
        r_anch = float(np.corrcoef(out.clause_f1, out.loss_anchored)[0, 1])
        out.attrs["corr_self"] = r_self
        out.attrs["corr_anchored"] = r_anch
        return out, r_self, r_anch

    print("\n=== OBJECTIVE-ANCHORING PROBE ===")
    print("corr(clause_f1, loss) must be NEGATIVE for an objective that rewards fidelity")
    _anch_reused = _reuse_reported("objective_anchoring.csv")                 # REVISION v12
    if _anch_reused:
        anch_df = pd.read_csv(os.path.join(OUT_DIR, "objective_anchoring.csv"))
        _r_self = float(np.corrcoef(anch_df.clause_f1, anch_df.loss_self)[0, 1])
        _r_anch = float(np.corrcoef(anch_df.clause_f1, anch_df.loss_anchored)[0, 1])
    else:
        anch_df, _r_self, _r_anch = objective_anchoring_probe()
    print(anch_df.to_string(index=False))
    print(f"  correlation with clause-F1:  self-referential {_r_self:+.3f}  |  "
          f"anchored {_r_anch:+.3f}")
    if _r_self > 0:
        print(f"  the self-referential objective has the WRONG SIGN: higher fidelity incurs a "
              f"HIGHER loss, so optimising it moves the model away from the gold formalisation.")
    if not _anch_reused:
        anch_df.to_csv(os.path.join(OUT_DIR, "objective_anchoring.csv"), index=False)
        with open(os.path.join(OUT_DIR, "objective_anchoring.txt"), "w") as _fh:
            _fh.write(f"corr_self={_r_self:.4f}\ncorr_anchored={_r_anch:.4f}\n")


## Cell 13c — Hyperparameter sensitivity: Gumbel-τ endpoint and refine-steps
Cell 9b sweeps `λ_faith`. To show the advantage is **not a knife-edge artifact of one lucky setting**, this adds two cheap 1-D sweeps on the synthetic generator: the Gumbel-Softmax temperature endpoint and the logic-layer refine-step count. Single seed keeps it inside the single-A100 budget; reports a coefficient of variation so “is it stable?” gets a number.

In [ ]:
# Run-All safety + cost control: this trains EXTRA models, so it is gated behind
# RUN_EXTRA_DIAGNOSTICS and requires a usable synthetic matrix.
if not RUN_EXTRA_DIAGNOSTICS:
    print("Skipped: RUN_EXTRA_DIAGNOSTICS is False (enable for full).")
elif "df" not in dir() or df is None or len(df) == 0:
    print("Skipped: no synthetic results matrix (df) available.")
else:

    # === Cell 13c: tau / refine-steps sensitivity (single seed, cheap) ===
    def run_with_overrides(seed=0, condition="coformer", **over):
        # REVISION v2 (M8): scaled() resets refine_steps (and other preset fields) under
        # FULL_SCALE, so v1 silently ran every refine-step setting at S=50. Overrides are
        # now applied AFTER scaled().
        cfg2 = Config(**dataclasses.asdict(CFG)).scaled()
        cfg2.epochs = CFG.epochs
        for _k, _v in over.items():
            setattr(cfg2, _k, _v)
        return run_condition(condition, seed, cfg2)

    # REVISION v3: each run is cached individually, so an interrupted session loses at most one run
    _SENS_PLAN = ([("gumbel_tau_end", v, "coformer") for v in (0.3, 0.5, 0.8)]
                  + [("refine_steps", v, "coformer") for v in (10, 20, 30, 50, 100)]
                  + [("refine_steps[no_stgs]", v, "no_stgs") for v in (50, 100, 200)])

    def _sens_row(knob, value, cond):
        def _run():
            ensure_budget(f"sensitivity {knob}={value}", 20 * 60)
            key = "refine_steps" if knob.startswith("refine_steps") else knob
            r = run_with_overrides(seed=0, condition=cond, **{key: value})
            return pd.DataFrame([dict(knob=knob, value=value,
                                      faith_id=round(r["faith_id"], 3),
                                      faith_excess=round(r["faith_excess_id"], 3),
                                      acc_id=round(r["acc_id"], 3),
                                      acc_para=round(r["acc_paraphrase"], 3))])
        return _diag_cached_df(f"sens_{knob}_{value}".replace("[", "_").replace("]", ""), _run)

    sens_df = pd.concat([_sens_row(*p) for p in _SENS_PLAN], ignore_index=True)
    print("Hyperparameter sensitivity (CoFormer, seed 0):")
    print(sens_df.to_string(index=False))
    sens_df.to_csv(os.path.join(OUT_DIR, "hparam_sensitivity.csv"), index=False)
    for knob in sens_df.knob.unique():
        a = sens_df[sens_df.knob==knob]["acc_id"].values
        cv = float(np.std(a) / max(np.mean(a), 1e-6))
        print(f"  {knob}: acc_id CV = {cv:.3f} ({'stable' if cv < 0.15 else 'sensitive — report range'})")


## Cell 13d — Inference-cost accounting (not just training time)
Deployment cost is dominated by inference, not by training. Co-training is paid once, offline; if CoFormer’s per-example inference latency and logic-layer refine-iteration count are on par with the frozen translator, that is the practical adoption argument. This cell times forward+refine per example for each condition on the ID eval set and tabulates it next to training cost.

In [ ]:
# Run-All safety: skip cleanly if the synthetic matrix is empty (e.g. Part A did not run).
if "df" not in globals() or df is None or len(df) == 0:
    print("Skipped: no synthetic results matrix (df) available.")
else:
    # === Cell 13d: inference-cost accounting ===
    import time as _time

    @torch.no_grad()
    def measure_inference_latency(n_batches=4, seed=0):
        """Architecture-level latency only. Weights do not affect FLOPs, and the inference graph
        is identical for all four conditions (they differ only in TRAINING)."""
        set_all_seeds(seed)
        model = Translator(VOCAB_SIZE, CLAUSE_TOKENS, CFG.n_clauses, CFG.n_vars, CFG).to(DEVICE).eval()
        logic = SoftMaxSATLayer(CFG.refine_steps).to(DEVICE)
        loader = make_loader(eval_id, CFG.batch_size, False, seed=seed)
        n_seen, t_fwd, t_solve = 0, 0.0, 0.0
        for bi, (xb, ycb) in enumerate(loader):
            if bi >= n_batches: break
            xb = xb.to(DEVICE)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t0 = _time.time()
            cm = to_clause_matrix(deterministic_onehot(model(xb).float()))
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t1 = _time.time()
            logic(cm)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t2 = _time.time()
            t_fwd += (t1 - t0); t_solve += (t2 - t1); n_seen += xb.size(0)
        n_seen = max(n_seen, 1)
        return dict(scope="shared architecture (all conditions)",
                    ms_translate_per_ex=round(1000 * t_fwd / n_seen, 3),
                    ms_solve_per_ex=round(1000 * t_solve / n_seen, 3))

    infer_df = (pd.read_csv(os.path.join(OUT_DIR, "inference_cost.csv"))   # REVISION v12
                if _reuse_reported("inference_cost.csv")
                else pd.DataFrame([measure_inference_latency()]))
    print("Inference latency (per example, ID eval):")
    print(infer_df.to_string(index=False))

    cost_compare = agg[["refine_steps", "train_secs"]].round(2)
    cost_compare.columns = ["trained_refine_steps", "train_secs"]
    print("\nPer-condition TRAINED refine-steps and training cost (measured on real models):")
    print(cost_compare.to_string())
    if RECOMPUTE_REPORTED_DIAGNOSTICS or not os.path.exists(os.path.join(OUT_DIR, "inference_cost.csv")):
        infer_df.to_csv(os.path.join(OUT_DIR, "inference_cost.csv"), index=False)
    cost_compare.to_csv(os.path.join(OUT_DIR, "training_cost_by_condition.csv"))
    print("\nClaim support: co-training cost is paid OFFLINE. The inference graph is identical "
          "across conditions, so CoFormer adds no inference-time penalty over the frozen "
          "translator; any difference in solver work shows up in trained_refine_steps above.")


## Cell 13e — Qualitative failure analysis (deterministic case dump)
H5 decomposes errors quantitatively; the analysis also requires **concrete cases**. This dumps a fixed, deterministic sample from the ID eval set showing, per instance, whether the translation matched gold and whether the final answer solved the instance — contrasting a frozen translator with CoFormer on the *same* inputs. No randomness beyond the locked seed, so it is fully reproducible.

In [ ]:
# Run-All safety + cost control: this trains EXTRA models, so it is gated behind
# RUN_EXTRA_DIAGNOSTICS and requires a usable synthetic matrix.
if not RUN_EXTRA_DIAGNOSTICS:
    print("Skipped: RUN_EXTRA_DIAGNOSTICS is False (enable for full).")
elif "df" not in globals() or df is None or len(df) == 0:
    print("Skipped: no synthetic results matrix (df) available.")
else:
    # === Cell 13e: deterministic qualitative failure cases ===
    @torch.no_grad()
    def case_dump_from_model(model, logic, n_cases=6):
        model.eval()
        Xid, Yid = eval_id
        xb = torch.from_numpy(Xid[:n_cases]).to(DEVICE)
        ycb = torch.from_numpy(Yid[:n_cases]).to(DEVICE)
        cm = to_clause_matrix(deterministic_onehot(model(xb).float()))
        assign, _, _ = logic(cm)
        solved = per_instance_solved(assign, ycb).cpu().numpy()
        trans_ok = per_instance_translation_ok(cm, ycb).cpu().numpy()
        out = []
        for i in range(n_cases):
            kind = ("correct" if solved[i] else
                    ("refine_failure(trans OK, answer wrong)" if trans_ok[i] else
                     "wrong_translation"))
            out.append(dict(case=i, translation_matches_gold=bool(trans_ok[i]),
                            instance_solved=bool(solved[i]), error_type=kind))
        return pd.DataFrame(out)

    print("Qualitative cases (first 6 ID instances) -- frozen vs coformer on identical inputs:")
    for cond in ("frozen", "coformer", "no_stgs"):
        def _cases(cond=cond):
            ensure_budget(f"qualitative cases {cond}", 20 * 60)       # REVISION v3
            rec = run_condition(cond, 0, CFG, return_model=True)
            out = case_dump_from_model(rec["_model"], rec["_logic"], n_cases=6)
            out.insert(0, "acc_id", round(rec["acc_id"], 3))
            out.insert(0, "faith_id", round(rec["faith_id"], 3))
            del rec
            _empty_cache()
            return out
        cdf = _diag_cached_df(f"cases_{cond}", _cases)
        print(f"\n[{cond}]")
        print(cdf.to_string(index=False))
        cdf.to_csv(os.path.join(OUT_DIR, f"cases_{cond}.csv"), index=False)
    print("\nUse 2-3 of these as a manuscript figure: same input, translation/answer status "
          "side by side, both from genuinely trained models.")


---
# Part B — Real-benchmark and real-baseline study

The synthetic study above is a *controlled* probe where faithfulness is measurable exactly. The study therefore also requires evidence on **real natural-language
logical-reasoning benchmarks** and against **real, reproduced baselines** rather than proxies.
Part B adds:

* **Real datasets:** FOLIO (human-written, FOL-annotated; Han et al., 2022) and ProofWriter
  (depth-5 OWA subset; Tafjord et al., 2021), pulled from their official sources and **cached on
  Drive** so they are downloaded only once.
* **Real baselines reproduced in-notebook:**
  * **Direct / CoT prompting** — pure-LLM lower bounds.
  * **Logic-LM / SATLM-style solver-aided pipeline** — LLM translates NL → symbolic form, then a
    **real Z3 SMT solver** performs inference (the architecture of Pan et al., 2023 and Ye et al.,
    2023). This is the genuine solver-aided competitor, not a stand-in.
  * **DiLA-style frozen-translator** — the LLM translation is used without co-training.
* **CoFormer on real data:** the translator is co-trained with the differentiable logic layer and
  evaluated on the same examples and metrics.

> Honesty constraints carried over: all label-producing steps run a real solver; faithfulness/
> accuracy are measured, never assumed; any LLM-API baseline is gated behind a key check and is
> skipped (not faked) if no key is present, with the skip clearly reported.


## Cell B1 — Extra dependencies for the real study (Z3 solver + dataset tooling)
Installs the real **Z3** SMT solver (the same solver used by Logic-LM/SATLM for SAT-style
inference) and the HuggingFace `datasets` client used to fetch FOLIO. These are additional to the
Part-A stack and are also idempotent.

In [ ]:
# === Cell B1: real-study dependencies (pinned) ===
_pip(["z3-solver==4.13.0.0", "datasets==3.0.1"], hard=False)
try:
    import z3
    print("Z3 version:", z3.get_version_string())
    Z3_OK = True
except Exception as e:
    Z3_OK = False
    print("Z3 unavailable:", e)
    print("Part B/C decisions require a sound solver; they will SKIP rather than fabricate.")


## Cell B2 — Fetch, normalise and cache the real benchmarks

Downloads FOLIO and ProofWriter from their public distributions, records the
retrieval URL, byte count, SHA-256 prefix and licence for each, and caches the
parsed corpora.

Labels are normalised at parse time. ProofWriter stores the answer as an option
**letter** (`"A"`/`"B"`/`"C"`) with the mapping in an `options` list and wraps
the question in an instruction prefix; both are resolved before normalisation. A
**label-balance guard** raises a loud warning whenever one class exceeds 90% of a
split, since a parsing bug is far more likely than such a distribution.


In [ ]:
# === Cell B2: fetch + normalize + cache real benchmarks, with provenance ===
import json, requests, hashlib as _hl, re as _re0

REAL_CACHE = os.path.join(DATA_DIR, "real_benchmarks_v4.json")
DATA_LICENCES = {"folio": "CC-BY-SA-4.0 (Yale-LILY/FOLIO)",
                 "proofwriter": "CC-BY-4.0 (AI2 ProofWriter / RuleTaker)"}
DATA_NOTES = {
    "proofwriter": ("Fetched from the Logic-LLM redistribution of the AI2 dev split. The "
                    "canonical AI2 release is a zip (allenai.org/data/proofwriter) that is not "
                    "reachable by raw URL; to use it, download it manually into DATA_DIR and "
                    "point PROOFWRITER_URLS at the extracted JSONL. The redistribution carries "
                    "no structured triples/rules, so the gold CNF is derived from the context "
                    "text and self-validated (Cell B4).")}

FOLIO_URLS = [
    "https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-validation.jsonl",
]
PROOFWRITER_URLS = [
    "https://raw.githubusercontent.com/teacherpeterpan/Logic-LLM/main/data/ProofWriter/dev.json",
]

def _try_get(urls, timeout=45):
    for u in urls:
        try:
            r = requests.get(u, timeout=timeout)
            if r.status_code == 200 and len(r.text) > 50:
                return dict(url=u, text=r.text, bytes=len(r.content),
                            sha256=_hl.sha256(r.content).hexdigest()[:16])
            print(f"  fetch {r.status_code} {u}")
        except Exception as e:
            print(f"  fetch failed {u}: {e}")
    return None

_RAW_KEEP = ("triples", "rules", "theory", "premises-FOL", "conclusion-FOL",
             "questions", "id", "depth", "options")

def _keep_raw(ex):
    return {k: ex[k] for k in _RAW_KEEP if k in ex}

# --- label / question normalisation ---------------------------------------------------------
_OPT_RE = _re0.compile(r"^\s*([A-Za-z])\s*[).:-]\s*(.+?)\s*$")

def resolve_option_answer(answer, options):
    """'A' + ['A) True', 'B) False', 'C) Unknown'] -> 'True'. Passes text answers through."""
    a = str(answer).strip()
    if options and len(a) == 1 and a.isalpha():
        for opt in options:
            m = _OPT_RE.match(str(opt))
            if m and m.group(1).upper() == a.upper():
                return m.group(2)
    return a

_Q_PREFIX = _re0.compile(r"^.*?true,\s*false,?\s*or\s*unknown\s*\?\s*", _re0.I)

def strip_question_prefix(q):
    """Remove the multiple-choice instruction wrapper so the conclusion can be grounded."""
    q = str(q).strip()
    if "?" in q and _Q_PREFIX.match(q):
        q = _Q_PREFIX.sub("", q)
    return q.strip()

def _parse_folio(text):
    rows = []
    for line in text.strip().splitlines():
        try: ex = json.loads(line)
        except Exception: continue
        premises = ex.get("premises") or ex.get("premises-FOL") or ex.get("context")
        if isinstance(premises, list): premises = " ".join(premises)
        concl = ex.get("conclusion") or ex.get("question") or ex.get("hypothesis")
        label = str(ex.get("label", ex.get("answer", ""))).strip()
        if premises and concl and label:
            row = {"context": premises, "question": concl, "label": label,
                   "fol": ex.get("premises-FOL", "")}
            row.update(_keep_raw(ex)); rows.append(row)
    return rows

def _parse_proofwriter(text):
    rows = []
    stripped = text.strip()
    if stripped.startswith("[") or stripped.startswith("{"):
        try:
            data = json.loads(stripped)
            items = data if isinstance(data, list) else data.get("data", [])
        except Exception:
            items = []
        for ex in items:
            ctx = ex.get("context") or ex.get("theory") or ""
            q = strip_question_prefix(ex.get("question") or ex.get("query") or "")
            lab = resolve_option_answer(ex.get("answer", ex.get("label", "")),
                                        ex.get("options"))
            if ctx and q and lab:
                row = {"context": ctx, "question": q, "label": lab, "fol": ""}
                row.update(_keep_raw(ex)); rows.append(row)
        return rows
    for line in stripped.splitlines():                    # official JSONL: one theory per line
        try: ex = json.loads(line)
        except Exception: continue
        ctx = ex.get("theory", "")
        qs = ex.get("questions", {})
        if isinstance(qs, dict): qs = list(qs.values())
        for q in (qs or []):
            qt = strip_question_prefix(q.get("question", "") if isinstance(q, dict) else str(q))
            lab = str(q.get("answer", "")) if isinstance(q, dict) else ""
            if ctx and qt and lab:
                row = {"context": ctx, "question": qt, "label": lab, "fol": ""}
                row.update(_keep_raw(ex)); rows.append(row)
    return rows

if os.path.exists(REAL_CACHE):
    real = json.load(open(REAL_CACHE))
    print(f"Loaded from cache: FOLIO={len(real.get('folio',[]))}, "
          f"ProofWriter={len(real.get('proofwriter',[]))}")
else:
    real = {"folio": [], "proofwriter": [], "sources": {}, "licences": DATA_LICENCES}
    got = _try_get(FOLIO_URLS)
    if got:
        real["folio"] = _parse_folio(got["text"])
        real["sources"]["folio"] = {k: got[k] for k in ("url", "bytes", "sha256")}
    got = _try_get(PROOFWRITER_URLS)
    if got:
        real["proofwriter"] = _parse_proofwriter(got["text"])
        real["sources"]["proofwriter"] = {k: got[k] for k in ("url", "bytes", "sha256")}
    atomic_json(real, REAL_CACHE)
    print(f"Fetched + cached: FOLIO={len(real['folio'])}, ProofWriter={len(real['proofwriter'])}")

REAL_AVAILABLE = (len(real["folio"]) + len(real["proofwriter"])) > 0
print("\nData provenance (report this table in the paper):")
for k, v in real.get("sources", {}).items():
    print(f"  {k}: {v.get('url')}\n      {v.get('bytes')} bytes, sha256[:16]={v.get('sha256')}, "
          f"licence: {DATA_LICENCES.get(k)}")
    if k in DATA_NOTES: print(f"      note: {DATA_NOTES[k]}")

# until the floor scored 1.000. This can never pass silently again.
from collections import Counter as _Ctr
for ds in ("folio", "proofwriter"):
    rows = real.get(ds, [])
    if not rows: continue
    dist = _Ctr(str(r["label"]).strip() for r in rows)
    print(f"\n{ds} label distribution: {dict(dist.most_common())}")
    top = dist.most_common(1)[0][1] / len(rows)
    if top > 0.9:
        print(f"  !! WARNING: {top:.0%} of {ds} labels are a single class. A parsing bug is far "
              f"more likely than a real distribution -- inspect before using this data.")
    else:
        print(f"  label balance OK (majority class {top:.1%})")
if not REAL_AVAILABLE:
    print("\nWARNING: no real data reachable. Part B/C will report a clear SKIP, never fake data.")


## Cell B3 — Real solver-aided inference with Z3 (Logic-LM / SATLM architecture)
This is the genuine solver-aided competitor. A symbolic formulation (propositional clauses
extracted from the example) is handed to a **real Z3 solver**, which decides satisfiability /
entailment — exactly the Problem-Formulation → Symbolic-Reasoning pipeline of Logic-LM (Pan et
al., 2023, which uses Z3 for SAT-style tasks) and SATLM (Ye et al., 2023). The translation quality
is what varies across methods; the solver itself is fixed and sound. We expose a single
`z3_entails(clauses)` primitive so every method's *translation* is judged against the *same*
reasoning oracle.

In [ ]:
# === Cell B3: real Z3 solver primitive (shared reasoning oracle) ===
def z3_satisfiable(clauses):
    """clauses: list of lists of signed int literals (DIMACS-style, var index 1..n, sign=polarity).
    Returns True/False/None(=solver error) using a REAL Z3 SAT call."""
    if not Z3_OK:
        return None
    try:
        import z3
        varset = sorted({abs(l) for cl in clauses for l in cl})
        b = {v: z3.Bool(f"v{v}") for v in varset}
        s = z3.Solver()
        for cl in clauses:
            s.add(z3.Or([b[abs(l)] if l > 0 else z3.Not(b[abs(l)]) for l in cl]))
        res = s.check()
        return True if res == z3.sat else (False if res == z3.unsat else None)
    except Exception:
        return None

# sanity check the oracle on a trivially (un)satisfiable pair
if Z3_OK:
    print("Z3 SAT  ((x) and (not x)) ->", z3_satisfiable([[1], [-1]]), "(expect False)")
    print("Z3 SAT  ((x) or (y))      ->", z3_satisfiable([[1, 2]]),    "(expect True)")


## Cell B4 — Formulator interface and shared, self-validating grounding
The reachable
distribution exposes no structured triples/rules, so the previous grounding returned
`gold_clauses = None` for every example: the "full" faithfulness regime never fired and R3 had no
mechanism behind it. ProofWriter theories are rule-generated English with four surface patterns,
so a propositional CNF *is* recoverable — facts, ground implications, and universally quantified
rules (`If something is quiet and cold then it is smart`, `All quiet things are rough`) grounded
over the entities named in the facts.

The derivation does not have to be trusted: an example enters the gold-CNF regime **only if Z3
entailment over the derived CNF reproduces its gold label**. Measured offline on the 600-example
dev split — 313 examples (52.2%) fully parse with a grounded conclusion, and on those the derived
CNF reproduces the gold label **96.2%** of the time. The disagreeing 3.8% are excluded
automatically; everything else falls back to the proxy regime, exactly like FOLIO. The cell runs
this self-test at import and prints the eligible fraction, so R3's testability is a measured
number rather than an assumption.

**Scope, to state in the paper.** Only unary-attribute theories are covered. ProofWriter's
relational subsets (`The lion chases the cat`) would need a binary atom space of
entities² × relations, beyond the projector's symbolic capacity; those run on the proxy regime.
That is a declared capacity decision, counted in the Cell C5 coverage report, not a silent
truncation.

In [ ]:
# === Cell B4: formulator interface + SHARED, SELF-VALIDATING grounding ===
import re, itertools

class Formulator:
    """Rule-based NL -> (clauses, var_map). Deterministic; used as the abstaining floor and as
    the safety fallback when an LLM emits invalid JSON."""
    def __init__(self): self.var_map = {}
    def _vid(self, name): return self.var_map.setdefault(name, len(self.var_map) + 1)
    def formulate(self, example):
        text = (example["context"] + " " + example["question"]).lower()
        atoms = re.findall(r"[a-z_]+\([a-z0-9_, ]+\)", text)
        clauses = []
        if atoms:
            seen = []
            for a in atoms[:64]:
                vid = self._vid(a.strip()); seen.append(vid); clauses.append([vid])
            if " then " in text and len(seen) >= 2:
                clauses.append([-seen[0], seen[-1]])
        else:
            toks = list(dict.fromkeys(re.findall(r"[a-z_]{3,}", text)))[:32]
            clauses = [[self._vid(t)] for t in toks]
        return clauses, dict(self.var_map)

_ENT_PAT = r"(?:the [a-z]+(?: [a-z]+)?|[A-Z][a-z]+)"
PW_FACT_RE = re.compile(rf"^({_ENT_PAT}) (?:is|are) (not )?([a-z]+)$", re.I)
PW_IF_RE   = re.compile(r"^if (.+?) then (.+)$", re.I)
PW_ALL_RE  = re.compile(r"^all ([a-z, ]+?) (?:things|people) are (not )?([a-z]+)$", re.I)
PW_ADJS_RE = re.compile(r"^([a-z]+(?:, [a-z]+)*) (?:things|people) are (not )?([a-z]+)$", re.I)
_PW_SOMETHING = re.compile(r"^(?:something|someone) (?:is|are) (.+)$", re.I)
_PW_COND_ENT  = re.compile(rf"^({_ENT_PAT}) (?:is|are) (.+)$", re.I)
_PW_CONS      = re.compile(rf"^(?:it|they|{_ENT_PAT}) (?:is|are) (not )?([a-z]+)$", re.I)

def _pw_ent(e): return re.sub(r"\s+", " ", e.strip().lower())
def _pw_sentences(ctx): return [s.strip() for s in ctx.split(".") if s.strip()]

def _pw_conj(text):
    """'quiet and cold' / 'not green and blue' -> [(attr, polarity), ...] or None."""
    out = []
    for part in re.split(r"\band\b", text):
        part = part.strip().rstrip(",")
        if not part: continue
        m = re.match(r"^(not )?([a-z]+)$", part, re.I)
        if not m: return None
        out.append((m.group(2).lower(), -1 if m.group(1) else 1))
    return out or None

def pw_parse_theory(ctx):
    """-> (facts, rules, fully_parsed). fact=(ent, attr, pol);
    rule=(ent or None for universal, [(attr,pol)...], (attr,pol))."""
    facts, rules, ok = [], [], True
    for s in _pw_sentences(ctx):
        m = PW_FACT_RE.match(s)
        if m:
            facts.append((_pw_ent(m.group(1)), m.group(3).lower(), -1 if m.group(2) else 1))
            continue
        m = PW_IF_RE.match(s)
        if m:
            ante, cons = m.group(1).strip(), m.group(2).strip()
            ent = None
            a = _PW_SOMETHING.match(ante)
            if a:
                body = a.group(1)
            else:
                a = _PW_COND_ENT.match(ante)
                if not a: ok = False; continue
                ent = _pw_ent(a.group(1))
                body = re.sub(rf"\band\s+{re.escape(a.group(1))}\s+(?:is|are)\s+", "and ",
                              a.group(2), flags=re.I)
            ants = _pw_conj(body); c = _PW_CONS.match(cons)
            if not ants or not c: ok = False; continue
            rules.append((ent, ants, (c.group(2).lower(), -1 if c.group(1) else 1)))
            continue
        m = PW_ALL_RE.match(s) or PW_ADJS_RE.match(s)
        if m:
            ants = _pw_conj(m.group(1).replace(",", " and "))
            if ants:
                rules.append((None, ants, (m.group(3).lower(), -1 if m.group(2) else 1)))
            else: ok = False
            continue
        ok = False                     # relational or otherwise out-of-scope sentence
    return facts, rules, ok

def pw_ground(ctx, question):
    """-> (clauses, conclusion_lit, slot_map, fully_parsed). Slot ids are 1-based."""
    facts, rules, ok = pw_parse_theory(ctx)
    ents = sorted({e for e, _, _ in facts})
    attrs = sorted({a for _, a, _ in facts}
                   | {a for _, ants, _ in rules for a, _ in ants}
                   | {c[0] for _, _, c in rules})
    if not ents or not attrs:
        return None, None, {}, False
    slot = {(e, a): i + 1 for i, (e, a) in enumerate(itertools.product(ents, attrs))}
    clauses = [[p * slot[(e, a)]] for e, a, p in facts if (e, a) in slot]
    for ent, ants, (ca, cp) in rules:                       # ground universals over entities
        for e in ([ent] if ent else ents):
            if (e, ca) not in slot or any((e, a) not in slot for a, _ in ants):
                continue
            clauses.append([-p * slot[(e, a)] for a, p in ants] + [cp * slot[(e, ca)]])
    q = strip_question_prefix(question).rstrip(".")
    m = PW_FACT_RE.match(q); concl = None
    if m:
        key = (_pw_ent(m.group(1)), m.group(3).lower())
        if key in slot:
            concl = (-1 if m.group(2) else 1) * slot[key]
    return clauses, concl, slot, ok

def _entail_local(clauses, concl):
    """Two-SAT entailment used only to VALIDATE a derived CNF (B5b defines the shared one)."""
    if not clauses or concl is None or not Z3_OK: return "Unknown"
    sw = z3_satisfiable(clauses + [[concl]]); sa = z3_satisfiable(clauses + [[-concl]])
    if sa is False and sw is not False: return "True"
    if sw is False and sa is not False: return "False"
    return "Unknown"

# --- generic fallback atomisation (FOLIO and out-of-scope ProofWriter theories) ---------------
_NEG_RE = re.compile(r"\b(not|never|cannot|isn't|aren't|doesn't|don't|no)\b", re.I)
_SENT_SPLIT = re.compile(r"[.;]\s*")

def atom_key(s):
    s = _NEG_RE.sub(" ", str(s).lower().strip())
    s = re.sub(r"\b(if|then|is|are|was|were|the|a|an|that|it|does|do|will)\b", " ", s)
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()

def polarity_of(s):
    return -1 if _NEG_RE.search(s or "") else 1

def _candidate_atoms(ex):
    out = []
    def push(t):
        k = atom_key(t)
        if k and k not in out: out.append(k)
    for s in _SENT_SPLIT.split(ex.get("context", "")):
        if s.strip(): push(s)
    return out

def ground_example(ex, n_clauses, n_vars, gold_label=None):
    """ONE deterministic slot assignment per example, shared by the projector's output space, the
    gold-CNF target, the conclusion literal handed to Z3, and the Cell C6 faithfulness probe."""
    # 1) try the ProofWriter text derivation (self-validated)
    if ex.get("dataset") == "ProofWriter" or ex.get("triples") or ex.get("rules") \
       or re.search(r"\bif .+ then \b", ex.get("context", ""), re.I):
        clauses, concl, slot, ok = pw_ground(ex.get("context", ""), ex.get("question", ""))
        if clauses and slot:
            fits = len(slot) <= n_vars and len(clauses) <= n_clauses
            validated = False
            if ok and fits and concl is not None and gold_label is not None:
                validated = (_entail_local(clauses, concl) == gold_label)
            atoms = [f"{e} {a}" for (e, a) in
                     sorted(slot, key=lambda k: slot[k])][:n_vars]
            return dict(
                atoms=atoms, slot={a: i for i, a in enumerate(atoms)},
                gold_clauses=(clauses if validated else None),
                conclusion_lit=(concl if (concl is not None and abs(concl) <= n_vars) else None),
                coverage=dict(atoms_total=len(slot), atoms_kept=min(len(slot), n_vars),
                              atoms_dropped=max(0, len(slot) - n_vars),
                              gold_clauses_kept=(len(clauses) if validated else 0),
                              gold_dropped=max(0, len(clauses) - n_clauses),
                              conclusion_grounded=int(concl is not None and abs(concl) <= n_vars),
                              parse_complete=int(ok), capacity_ok=int(fits),
                              gold_validated=int(validated)),
                faith_supervision="full" if validated else "proxy")

    # 2) generic fallback (FOLIO, relational ProofWriter theories)
    atoms = _candidate_atoms(ex)[:n_vars]
    slot = {a: i for i, a in enumerate(atoms)}
    qk = atom_key(strip_question_prefix(ex.get("question", "")))
    concl = (polarity_of(ex.get("question", "")) * (slot[qk] + 1)) if qk in slot else None
    return dict(atoms=atoms, slot=slot, gold_clauses=None, conclusion_lit=concl,
                coverage=dict(atoms_total=len(_candidate_atoms(ex)), atoms_kept=len(atoms),
                              atoms_dropped=max(0, len(_candidate_atoms(ex)) - n_vars),
                              gold_clauses_kept=0, gold_dropped=0,
                              conclusion_grounded=int(concl is not None),
                              parse_complete=0, capacity_ok=1, gold_validated=0),
                faith_supervision="proxy")

def clause_matrix_to_clause_list(cm_row, max_lits=None):
    """(n_clauses, n_vars) in {-1,0,1} -> DIMACS-style list of lists of signed ints."""
    out = []
    arr = cm_row.detach().cpu().numpy() if hasattr(cm_row, "detach") else np.asarray(cm_row)
    for c in range(arr.shape[0]):
        lits = [int(np.sign(arr[c, v])) * (v + 1) for v in np.nonzero(arr[c])[0]]
        if lits: out.append(lits[:max_lits] if max_lits else lits)
    return out

# A prompted baseline is only a fair baseline if it is prompted competently.
# A bare zero-shot instruction leaves a large share of generations unparseable,
# and a baseline that falls back to the rule parser on most examples is a straw
# man regardless of the architecture it stands for. The prompt therefore carries
# a worked exemplar (few-shot), an explicit schema, and an explicit prohibition
# on prose outside the JSON object. `fallback_rate` is reported per system so
# that the fairness of the comparison is a measured number, not an assumption.
CNF_PROMPT = """You are a precise logic formalizer. Convert the problem into propositional CNF.

Output ONLY a single JSON object and nothing else. No prose, no markdown fences.
Schema:
{{"vars": {{"<atom name>": <positive int id>}}, "clauses": [[<signed int>, ...]], "conclusion": <signed int>}}

Rules:
- Each distinct atom gets one positive integer id, numbered 1..N.
- A literal is +id (asserted) or -id (negated). A clause is a disjunction list.
- "if A then B" becomes [-A_id, B_id]. A plain fact becomes a unit clause [id].
- "A and B" are two separate clauses, never one.
- "conclusion" is the single signed literal that the Conclusion sentence asserts.
  It MUST reference an id that also appears in "vars".

Example
Premises: All cats are mammals. Tom is a cat.
Conclusion: Tom is a mammal.
JSON: {{"vars": {{"cat_tom": 1, "mammal_tom": 2}}, "clauses": [[-1, 2], [1]], "conclusion": 2}}

Premises: {context}
Conclusion: {question}
JSON:"""

def _parse_cnf_json(raw, formulator):
    """-> (clauses, conclusion, ok, error_message). The message feeds Logic-LM's refine loop."""
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)
    if not m: return None, None, False, "no JSON object found in the response"
    try: obj = json.loads(m.group(0))
    except json.JSONDecodeError as e: return None, None, False, f"JSON decode error: {e}"
    try: clauses = [[int(l) for l in cl] for cl in obj.get("clauses", []) if cl]
    except (ValueError, TypeError) as e:
        return None, None, False, f"non-integer literal in clauses: {e}"
    concl = obj.get("conclusion", None)
    try: concl = int(concl) if concl is not None else None
    except (ValueError, TypeError): concl = None
    if not clauses: return None, None, False, "clause list is empty"
    if concl is None:
        return clauses, None, False, "missing 'conclusion' literal (entailment undecidable)"
    formulator.var_map = obj.get("vars", {})
    return clauses, concl, True, ""

class LLMFormulatorReal(Formulator):
    """Prompted NL -> CNF with one schema-repair retry before falling back.

    The retry is NOT the Logic-LM self-refinement loop, which is driven by
    solver feedback and is what distinguishes that system. This one repairs a
    malformed *response* only, and is available to every prompted baseline
    equally, so it raises the floor of the comparison without altering what any
    baseline is. Both `used_fallback` and the retry count are reported.
    """
    MAX_FORMAT_RETRIES = 1

    def formulate_full(self, example):
        prompt = CNF_PROMPT.format(context=example["context"],
                                   question=example["question"])
        clauses = concl = None
        err = ""
        for attempt in range(self.MAX_FORMAT_RETRIES + 1):
            raw = llm_generate(prompt, max_new_tokens=512)
            clauses, concl, ok, err = _parse_cnf_json(raw, self)
            if ok:
                return clauses, concl, False, ""
            if attempt < self.MAX_FORMAT_RETRIES:
                # repair the FORMAT, quoting the parser's own complaint
                prompt = (CNF_PROMPT.format(context=example["context"],
                                            question=example["question"])
                          + f"\n(Your previous response could not be parsed: {err}. "
                            f"Emit ONLY the JSON object, with a 'conclusion' "
                            f"literal whose id appears in 'vars'.)")
        cl, _ = super().formulate(example)
        return cl, None, True, err

# --- self-test of the derivation against the real data, printed at import time ---------------
def _local_norm_label(lab, options=None):
    """Local canonicaliser: Cell B5 defines the shared `normalize_label`, but this self-test runs
    BEFORE that cell, so it cannot depend on it."""
    l = str(resolve_option_answer(lab, options) if options else lab).strip().lower()
    if l in ("true", "entailment", "yes", "1", "a"): return "True"
    if l in ("false", "contradiction", "no", "0", "b"): return "False"
    return "Unknown"

if globals().get("REAL_AVAILABLE") and globals().get("Z3_OK") \
        and globals().get("real", {}).get("proofwriter"):
    _n = _elig = _agree = 0
    _nv, _nc = [], []
    for _ex in real["proofwriter"][:600]:
        _n += 1
        _cl, _c, _slot, _ok = pw_ground(_ex["context"], _ex["question"])
        if not _cl or _c is None or not _ok: continue
        _elig += 1; _nv.append(len(_slot)); _nc.append(len(_cl))
        _agree += int(_entail_local(_cl, _c) == _local_norm_label(_ex["label"], _ex.get("options")))
    if _elig:
        print(f"\nProofWriter gold-CNF derivation self-test on {_n} examples:")
        print(f"  fully parsed + conclusion grounded : {_elig}/{_n} ({_elig/_n:.1%})")
        print(f"  derived CNF reproduces the gold label: {_agree}/{_elig} ({_agree/_elig:.1%})")
        print(f"  atoms/example max {max(_nv)}, clauses/example max {max(_nc)} "
              f"-> required capacity")
        print(f"  Only the {_agree} VALIDATED examples enter the gold-CNF regime; the rest use "
              f"the proxy regime. R3 is testable iff this count is non-trivial.")
    else:
        print("\nProofWriter gold-CNF derivation self-test: 0 eligible examples. "
              "R3 is UNTESTABLE on this source and must be reported as untested.")

print("Formulator + self-validating grounding + CNF prompt ready.")


## Cell B5 — Rule-based FLOOR on the real benchmarks (honest, abstaining)
The rule-based formulator extracts atoms but emits **no conclusion literal**, so it cannot decide entailment on its own — a single satisfiability check would be a degenerate, unfairly-weak stand-in (almost every consistent premise set is SAT). This cell therefore reports the rule-based pipeline as an **abstaining floor** (always “Unknown”) plus a consistency sanity signal, and defers the real, entailment-capable solver baselines (SATLM / Logic-LM / DiLA, via two SAT queries over an LLM-emitted conclusion literal) to Cell B5b. Pure-LLM Direct/CoT baselines are reported as SKIPPED rather than faked unless a key is configured.

In [ ]:
# === Cell B5: real-data floor evaluation (solver-grounded) ===
import pandas as pd
from collections import Counter

def normalize_label(lab, options=None):
    """Canonicalise to {True, False, Unknown}. Accepts option letters when `options` is given -- ProofWriter stores 'A'/'B'/'C', and silently mapping those to 'Unknown'
    destroyed the entire real-data study in an earlier run."""
    l = str(resolve_option_answer(lab, options) if options else lab).strip().lower()
    if l in ("true", "entailment", "yes", "1", "a"): return "True"
    if l in ("false", "contradiction", "no", "0", "b"): return "False"
    return "Unknown"     # FOLIO 'Uncertain', ProofWriter OWA 'Unknown', NEI

# --- entailment needs TWO SAT queries; the rule-based formulator emits no conclusion literal ---
# so it is reported honestly as a NON-DECIDING floor that always abstains, never as a decision.
def eval_rulebased_floor(rows, name, max_n=200):
    if not rows or not Z3_OK:
        return dict(dataset=name, method="rule-based floor", n=0, accuracy=float("nan"),
                    majority=float("nan"), status="SKIPPED")
    rows = rows[:max_n]
    gold = [normalize_label(r["label"], r.get("options")) for r in rows]
    dist = Counter(gold)
    maj = dist.most_common(1)[0][1] / len(gold)
    correct = sum(int("Unknown" == g) for g in gold)          # the floor abstains on everything
    n_sat = 0
    for r in rows:
        f = Formulator(); clauses, _ = f.formulate(r)
        if z3_satisfiable(clauses) is True: n_sat += 1
    return dict(dataset=name, method="rule-based floor (abstains; no conclusion literal)",
                n=len(rows), accuracy=correct / len(rows), majority=maj,
                label_dist=dict(dist), frac_consistent=round(n_sat / len(rows), 3),
                status="OK (floor only — entailment needs the LLM formulator in B5b)")

real_records = []
if REAL_AVAILABLE:
    real_records.append(eval_rulebased_floor(real["folio"], "FOLIO"))
    real_records.append(eval_rulebased_floor(real["proofwriter"], "ProofWriter"))
    for ds in ("FOLIO", "ProofWriter"):
        real_records.append(dict(dataset=ds, method="LLM Direct/CoT", n=0,
                                 accuracy=float("nan"), majority=float("nan"),
                                 status="SKIPPED: requires LLM API key (not faked)"))
    real_df = pd.DataFrame(real_records)
    print(real_df.to_string(index=False))
    for r in real_records:
        if r.get("majority") == 1.0:
            print(f"\n  !! {r['dataset']}: majority = 1.000 means every label is one class. "
                  f"That is a PARSING BUG, not a dataset property. Fix before proceeding.")
    print("\nNOTE: the rule-based row is a deliberate FLOOR that abstains; it has NO conclusion "
          "literal and cannot decide entailment. The entailment-capable baselines (SATLM / "
          "Logic-LM / DiLA) are evaluated in Cell B5b via two SAT queries.")
    send_alert("Part B floor eval done", real_df.to_string(index=False))
else:
    real_df = pd.DataFrame(real_records)
    print("Part B SKIPPED — real data not reachable. No fabricated numbers produced.")


## Cell B5b — Three real baselines: SATLM, Logic-LM (self-refine), DiLA
All three share the backbone, the formulator interface, and — critically — the **same inference
rule**: emit a CNF, then decide three-way entailment with two Z3 SAT queries.

The Logic-LM self-refine loop was mis-implemented: it fired only when
the clause list was empty or Z3 returned `unknown`, and it broke out as soon as the premises were
satisfiable — which is nearly always true of a consistent premise set. `refine_iters` was
therefore ≈0 and Logic-LM was **numerically identical to SATLM**, a third copy of the same
baseline presented as a distinct system. Real Logic-LM (Pan et al., 2023) refines on
formalisation/execution errors and feeds the error message back into the prompt. It now retries
when parsing fails, when the conclusion literal is missing, when the solver errors, or when the
premise set comes out self-contradictory (a formalisation bug, since these benchmarks' premises
are consistent by construction).

In [ ]:
# === Cell B5b: entailment-via-SAT + three real baselines ===
def entailment_decision(premise_clauses, conclusion_lit):
    """Three-way entailment from two SAT queries. THE single decision rule in this notebook:
    CoFormer and every baseline use exactly this function, so no system enjoys a different
    inference procedure (see NOTES_PREREG['identical_inference'])."""
    if not premise_clauses or conclusion_lit is None:
        return "Unknown"
    sat_with    = z3_satisfiable(premise_clauses + [[ conclusion_lit]])
    sat_against = z3_satisfiable(premise_clauses + [[-conclusion_lit]])
    if sat_against is False and sat_with is not False: return "True"
    if sat_with is False and sat_against is not False: return "False"
    return "Unknown"

def baseline_satlm(example):
    """SATLM: one-shot LLM NL->CNF, then Z3. No refinement."""
    f = LLMFormulatorReal()
    clauses, concl, fb, _ = f.formulate_full(example)
    return dict(pred=entailment_decision(clauses, concl), used_fallback=fb, refine_iters=0)

def baseline_logiclm(example, max_refine=3):
    f = LLMFormulatorReal()
    clauses, concl, fb, err = f.formulate_full(example)
    iters = 0
    for _ in range(max_refine):
        problem = None
        if not clauses:
            problem = err or "no clauses were produced"
        elif concl is None:
            problem = err or "the 'conclusion' literal is missing, so entailment is undecidable"
        else:
            sat = z3_satisfiable(clauses)
            if sat is None:
                problem = "the solver could not decide the premise set (encoding error)"
            elif sat is False:
                problem = ("the premise set is UNSATISFIABLE on its own, which indicates a "
                           "formalisation error: these premises are consistent by construction")
        if problem is None:
            break
        iters += 1
        raw = llm_generate(
            CNF_PROMPT.format(context=example["context"], question=example["question"])
            + f"\n(Your previous formalization was rejected: {problem}. "
              f"Re-emit STRICTLY valid JSON with a consistent premise set and a 'conclusion' literal.)",
            max_new_tokens=512)
        c2, cc2, ok, err = _parse_cnf_json(raw, f)
        if ok:
            clauses, concl, fb = c2, cc2, False
    if not clauses:
        clauses, _ = Formulator().formulate(example); fb = True
    return dict(pred=entailment_decision(clauses, concl), used_fallback=fb, refine_iters=iters)

def baseline_dila(example):
    """DiLA: LLM->CNF, then run the differentiable layer under no_grad (architectural fidelity:
    the translator is FROZEN and never receives solver gradient). The DECISION still comes from
    Z3, because the relaxation cannot certify UNSAT (Cell U) -- and using Z3 keeps this baseline
    on exactly the same inference footing as CoFormer."""
    f = LLMFormulatorReal()
    clauses, concl, fb, _ = f.formulate_full(example)
    if not clauses:
        clauses, _ = Formulator().formulate(example); fb = True
    if clauses:
        V = max((abs(l) for cl in clauses for l in cl), default=1)
        cm = torch.zeros(1, len(clauses), V, device=DEVICE)
        for ci, cl in enumerate(clauses):
            for lit in cl:
                cm[0, ci, abs(lit) - 1] = float(np.sign(lit))
        with torch.no_grad():
            SoftMaxSATLayer(CFG.refine_steps).to(DEVICE)(cm.float())   # architectural step
    return dict(pred=entailment_decision(clauses, concl), used_fallback=fb, refine_iters=0)

print("baseline_satlm / baseline_logiclm (error-driven self-refine) / baseline_dila defined.")
print("All three, and CoFormer, decide via entailment_decision() -- one shared inference rule.")


## Cell B6 — Honest interpretation and what remains for the full
States plainly what Part B does and does not yet establish, so the manuscript's claims stay inside
the evidence. The rule-based formulator is a faithful, reproducible *floor*; the headline
comparison against DiLA/SATLM/Logic-LM at full strength requires the LLM-backed formulator
(Qwen2.5-7B-Instruct with LoRA; fallbacks Llama-3.1-8B, Mistral-7B, Phi-3.5-mini), which is the documented next step and the main GPU cost of the
real study.

In [ ]:
# === Cell B6: honest interpretation ===
notes = [
 "Part B runs a REAL Z3 solver on REAL FOLIO/ProofWriter examples — no synthetic stand-in.",
 "The default formulator is rule-based and reproducible; it is a floor, not the full method.",
 "Pure-LLM and full LLM-formulator results are SKIPPED (not faked) unless a backbone/API is set.",
 "To complete the headline DiLA/SATLM/Logic-LM comparison: enable USE_LLM_BACKBONE with a "
 "LoRA Llama-3-8B/Mistral-7B translator and co-train it with the Part-A logic layer on these "
 "real examples; this is the principal remaining GPU run.",
 "Only after that run should the paper state relative accuracy/faithfulness vs. these baselines.",
]
for i, n in enumerate(notes, 1):
    print(f"[{i}] {n}")

# Persist a combined results artifact for the manuscript.
if REAL_AVAILABLE and real_df is not None and len(real_df):
    real_out = os.path.join(OUT_DIR, "real_benchmark_results.csv")
    real_df.to_csv(real_out, index=False)
    print("\nSaved:", real_out)
else:
    print("\nNo real-benchmark table to save (corpora unreachable or solver "
          "unavailable). Nothing fabricated.")


---
# the LLM-backbone stage — Real LLM backbone: LoRA-adapted Qwen2.5-7B-Instruct (declared backbone), co-trained with the logic layer

This is the headline experiment. The translator from Part A is replaced by a **real, LoRA-adapted
LLM** (Qwen2.5-7B-Instruct by default), and it is **co-trained end-to-end with the
differentiable logic layer** on the real FOLIO / ProofWriter examples loaded in Part B.

**How co-training reaches the LLM (the core mechanism).** A causal LLM emits text, which is not
differentiable into a clause matrix. We therefore attach a lightweight **clause-projection head**
on top of the LLM's last hidden state that produces, for each (clause, variable) slot, a 3-way
logit over {neg, absent, pos}. The Straight-Through Gumbel-Softmax bridge (Part A, Cell 6) turns
these into a hard CNF in the forward pass while passing gradients in the backward pass. Those
gradients flow into the projection head **and into the LoRA adapters** — so the logic layer's
signal actually updates the LLM's translation behavior. The 4-bit-quantized base weights stay
frozen (QLoRA); only LoRA adapters + the projection head + the logic layer interact.

**Single-A100 budget.** 4-bit NF4 quantization (`bitsandbytes`), LoRA on `q_proj`/`v_proj`, short
sequences, gradient accumulation, and bf16 compute keep an 8B model within an 80 GB A100 for a
few-hour run. A `MODEL_NAME` switch selects Llama-3-8B vs Mistral-7B; both are gated models, so the
cell checks for a Hugging Face token and **skips with a clear message (never fabricates)** if the
weights or token are unavailable.


## Cell C1 — Backbone dependencies and an availability gate
Installs `transformers`, `peft`, `bitsandbytes`, `accelerate`. Llama-3 / Mistral are gated on
Hugging Face, so this cell checks for an `HF_TOKEN` secret and probes that the chosen model is
loadable. If anything required is missing, `BACKBONE_READY` is set False and Parts C2–C4 print a
precise SKIP reason instead of producing fake results.

In [ ]:
# === Cell C1: backbone dependencies + availability gate ===
_pip(["transformers==4.46.2", "peft==0.13.2", "bitsandbytes>=0.45.5", "accelerate==1.1.1"])

MODEL_CHOICES = {
    "llama3-8b":   "meta-llama/Llama-3.1-8B-Instruct",   # gated: needs allow-list approval
    "qwen2.5-7b":  "Qwen/Qwen2.5-7B-Instruct",           # Apache-2.0
    "mistral-7b":  "mistralai/Mistral-7B-Instruct-v0.3", # gated on some accounts
    "phi3.5-mini": "microsoft/Phi-3.5-mini-instruct",    # MIT
}
MODEL_KEY = "qwen2.5-7b"           # declared choice; the first ACCESSIBLE candidate is used
MODEL_FALLBACK_ORDER = [MODEL_KEY] + [k for k in MODEL_CHOICES if k != MODEL_KEY]
ALLOW_MODEL_FALLBACK = True        # set False to fail loudly instead of substituting
MODEL_NAME = MODEL_CHOICES[MODEL_KEY]

def _repo_accessible(repo, token):
    """Probe the repo: cheap, and distinguishes 'gated' from 'missing' from
    'no network'. A token is OPTIONAL -- the declared backbone
    (Qwen2.5-7B-Instruct, Apache-2.0) and the Phi-3.5-mini fallback (MIT) are
    public, so an anonymous request resolves them. Requiring a token as a hard
    precondition would skip this stage on a machine that can in fact run it,
    which is the opposite of the intended fail-safe behaviour: the gate should
    close only when nothing is actually loadable."""
    try:
        from huggingface_hub import HfApi
        HfApi().model_info(repo, token=token or None)
        return True, ""
    except Exception as e:
        _t = type(e).__name__
        if "Gated" in _t:
            return False, "gated: this account is not on the allow list"
        if "RepositoryNotFound" in _t or "401" in str(e):
            return False, "not found, or the token cannot see it"
        return False, f"{_t}: {str(e)[:90]}"

# Optional. Needed ONLY for gated weights (Llama-3, some Mistral accounts);
# the declared backbone and one fallback are public and load anonymously.
HF_TOKEN = _get_secret("HF_TOKEN")
UNGATED_MODELS = ("qwen2.5-7b", "phi3.5-mini")   # loadable without any token
if not HF_TOKEN:
    print("HF_TOKEN not set. Gated backbones (Llama-3, Mistral) are therefore "
          "unavailable, but the stage still runs on a public backbone: "
          + ", ".join(MODEL_CHOICES[k] for k in UNGATED_MODELS))
BACKBONE_READY = False
skip_reason = ""
def _import_backbone():
    """Import the backbone stack, with one targeted recovery attempt.

    the failure mode here is a KNOWN dependency conflict with a KNOWN fix, so the
    notebook attempts it once and reports the outcome instead of leaving a 5 h stage skipped for
    a one-line pip. Anything else is re-raised untouched -- this is not a blanket retry."""
    import importlib
    try:
        import transformers, peft, bitsandbytes, accelerate
        return transformers, peft, bitsandbytes, accelerate
    except ModuleNotFoundError as e:
        if "triton.ops" not in str(e):
            raise
        print(f"  backbone import failed with the known Triton/bitsandbytes conflict ({e});"
              f" upgrading bitsandbytes and retrying once ...")
        _pip(["--upgrade", "bitsandbytes>=0.45.5"])
        for _m in [m for m in list(__import__("sys").modules) if m.startswith("bitsandbytes")]:
            del __import__("sys").modules[_m]
        importlib.invalidate_caches()
        import transformers, peft, bitsandbytes, accelerate
        print(f"  recovered: bitsandbytes {getattr(bitsandbytes, '__version__', '?')}")
        return transformers, peft, bitsandbytes, accelerate

try:
    transformers, peft, bitsandbytes, accelerate = _import_backbone()
    if not HAS_GPU:
        skip_reason = "no CUDA GPU: 8B LLM co-training requires one (Part C SKIP)."
    elif not CFG.USE_LLM_BACKBONE:
        skip_reason = "CFG.USE_LLM_BACKBONE is False (set True in Cell 3 to enable Part C)."
    elif not Z3_OK:
        skip_reason = ("Z3 unavailable: every decision in Part C is made by the solver, so the "
                       "study cannot run soundly without it.")
    else:
        # pick the first candidate this account can actually load
        _picked, _tried = None, []
        _candidates = MODEL_FALLBACK_ORDER if ALLOW_MODEL_FALLBACK else [MODEL_KEY]
        if not HF_TOKEN:
            # try the public ones first; gated repos will simply fail the probe
            _candidates = ([k for k in _candidates if k in UNGATED_MODELS]
                           + [k for k in _candidates if k not in UNGATED_MODELS])
        for _k in _candidates:
            _ok, _why = _repo_accessible(MODEL_CHOICES[_k], HF_TOKEN)
            _tried.append(f"{_k}: {'OK' if _ok else _why}")
            if _ok:
                _picked = _k
                break
        print("Model access check:")
        for _t2 in _tried:
            print(f"  {_t2}")
        if _picked is None:
            skip_reason = ("no accessible backbone: " + "; ".join(_tried))
        else:
            if _picked != MODEL_KEY:
                print(f"  SUBSTITUTED: `{MODEL_KEY}` is not accessible, using `{_picked}`. The "
                      f"finding under test is a property of the straight-through estimator, not "
                      f"of the backbone, but the substitution is recorded in run_metadata.")
            MODEL_KEY, MODEL_NAME = _picked, MODEL_CHOICES[_picked]
            _versions["backbone_model"] = MODEL_NAME
        BACKBONE_READY = _picked is not None
        # REVISION v12.1: Part C must run with the pinned backbone libraries. If another
        # transformers version was imported earlier in this process, pip cannot replace it in
        # memory; skip Part C (stored results stay valid) and resume in a fresh runtime.
        if BACKBONE_READY and getattr(transformers, "__version__", "") != "4.46.2":
            BACKBONE_READY = False
            skip_reason = (f"backbone libraries: transformers {transformers.__version__} is loaded "
                           f"in this process but 4.46.2 is pinned; restart the runtime and Run All")
        for _lib in (transformers, peft, bitsandbytes, accelerate):
            _versions[_lib.__name__] = getattr(_lib, "__version__", "?")
        print(f"Backbone libraries OK. Target model: {MODEL_NAME}")
        print("  resolved: " + ", ".join(f"{m.__name__} {_versions[m.__name__]}"
              for m in (transformers, peft, bitsandbytes, accelerate)))
        try:
            with open(_meta_path("requirements.txt"), "w") as _fh:   # REVISION v12.1
                _fh.write("\n".join(f"{k}=={v}" for k, v in _versions.items()) + "\n")
        except Exception:
            pass
except Exception as e:
    skip_reason = f"backbone libraries unavailable: {e}"
SKIP_FIX = {
    "no CUDA GPU": "Runtime -> Change runtime type -> A100. bf16 is required for 4-bit NF4, so "
                   "T4 cannot run Part C.",
    "HF_TOKEN not set": "Only needed for GATED weights. Add a Colab secret named "
                        "HF_TOKEN and enable 'Notebook access' if you want "
                        "Llama-3 or Mistral; the public Qwen2.5 / Phi-3.5 "
                        "backbones need no token.",
    "USE_LLM_BACKBONE": "Set CFG.USE_LLM_BACKBONE = True in Cell 3.",
    "Z3 unavailable": "Re-run Cell B1; every Part C decision is made by the solver.",
    "triton.ops": "bitsandbytes < 0.45.2 is incompatible with Triton >= 3.2. Cell C1 "
                  "now pins >=0.45.5 and retries once automatically; if you still see "
                  "this, run `!pip install -U bitsandbytes` and restart the runtime.",
    "backbone libraries": "Re-run Cell C1's pip install, then restart the runtime.",
    "no accessible backbone": "Request access on the gated model page and wait for approval, or "
                              "set ALLOW_MODEL_FALLBACK = True (default) so an ungated backbone "
                              "is used instead.",
}
if not BACKBONE_READY:
    _hint = next((v for k, v in SKIP_FIX.items() if k in skip_reason), "")
    print("=" * 72)
    print("PART C WILL SKIP -- no fabrication. Cause:")
    print(f"  {skip_reason}")
    if _hint:
        print(f"  FIX: {_hint}")
    print(f"  Gate status: GPU={'ok' if HAS_GPU else 'MISSING'} | "
          f"HF_TOKEN={'set' if HF_TOKEN else 'MISSING'} | "
          f"Z3={'ok' if Z3_OK else 'MISSING'} | "
          f"USE_LLM_BACKBONE={CFG.USE_LLM_BACKBONE}")
    print("=" * 72)
    try:
        with open(os.path.join(OUT_DIR, "partc_skip_reason.txt"), "w") as _fh:
            _fh.write(f"{skip_reason}\nFIX: {_hint}\n"
                      f"GPU={HAS_GPU} HF_TOKEN={bool(HF_TOKEN)} Z3={Z3_OK} "
                      f"USE_LLM_BACKBONE={CFG.USE_LLM_BACKBONE}\n")
    except Exception:
        pass


## Cell C2 — 4-bit LLM, adapter lifecycle, and the clause-query projector

Loads the base model in 4-bit NF4 with bf16 compute and wraps it with LoRA on
the attention projections. Only the adapters and the projector are trainable;
the quantised base is frozen.

Each clause slot owns a learned query token that **cross-attends over the LLM's
token states**, so clause *i* can attend to the span of text that expresses
clause *i*. Mean-pooling before projection would be order-invariant while the
target is ordered, and would leave the LoRA adapters almost no route to
influence the output.

`fresh_peft()` / `release_peft()` guarantee that exactly one adapter set is
attached at a time; `get_peft_model` injects adapters in place, so calling it
per (system, seed) without releasing would stack them on the shared base.


In [ ]:
# === Cell C2: 4-bit LLM + adapter lifecycle + CLAUSE-QUERY projector ===
if BACKBONE_READY:
    import gc
    import torch, torch.nn as nn
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    try:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                 bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb, device_map={"": 0},
            token=HF_TOKEN, torch_dtype=torch.bfloat16)
        base = prepare_model_for_kbit_training(base)
    except Exception as _e:
        BACKBONE_READY = False
        skip_reason = f"backbone `{MODEL_NAME}` failed to load: {type(_e).__name__}: {_e}"
        print("=" * 72)
        print("PART C WILL SKIP -- the backbone could not be loaded.")
        print(f"  {skip_reason[:400]}")
        if "Gated" in type(_e).__name__ or "gated" in str(_e).lower():
            print("  This account is not on the model's allow list. Request access on the model "
                  "page, or leave ALLOW_MODEL_FALLBACK = True in Cell C1 so an ungated backbone "
                  "is selected automatically.")
        print("  The rest of the notebook continues; Part C is reported as not run.")
        print("=" * 72)
        try:
            with open(os.path.join(OUT_DIR, "partc_skip_reason.txt"), "w") as _fh:
                _fh.write(skip_reason + "\n")
        except Exception:
            pass

if BACKBONE_READY:
    LORA_KW = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                   target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")

    _ACTIVE_PEFT = {"model": None}

    def fresh_peft():
        if _ACTIVE_PEFT["model"] is not None:
            release_peft(_ACTIVE_PEFT["model"])
        m = get_peft_model(base, LoraConfig(**LORA_KW))
        _ACTIVE_PEFT["model"] = m
        return m

    def release_peft(m):
        global base
        if m is None: return
        try:
            base = m.unload()
        except Exception as e:
            print(f"  [release_peft] unload failed ({e}); trying delete_adapter")
            try: m.delete_adapter(m.active_adapter)
            except Exception: pass
        if _ACTIVE_PEFT["model"] is m: _ACTIVE_PEFT["model"] = None
        del m; gc.collect(); torch.cuda.empty_cache()

    _probe = fresh_peft(); _probe.print_trainable_parameters(); release_peft(_probe)

    HID = base.config.hidden_size

    class ClauseProjector(nn.Module):
        """LLM token states -> (n_clauses, n_vars, 3) logits. Each clause slot owns a learned
        query token that CROSS-ATTENDS over the token states, so clause i can attend to the span
        of text that expresses clause i (mean-pooling would destroy that and starve the LoRA
        adapters of gradient)."""
        def __init__(self, hid, n_clauses, n_vars, d=512, n_heads=8):
            super().__init__()
            self.n_clauses, self.n_vars = n_clauses, n_vars
            self.inp = nn.Linear(hid, d)
            self.q = nn.Parameter(torch.randn(1, n_clauses, d) * 0.02)
            self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
            self.norm = nn.LayerNorm(d)
            self.out = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, n_vars * 3))
        def forward(self, hidden_last, attn_mask=None):
            h = self.inp(hidden_last.float())
            kpm = (attn_mask == 0) if attn_mask is not None else None
            q = self.q.expand(h.size(0), -1, -1)
            c, _ = self.attn(q, h, h, key_padding_mask=kpm)
            return self.out(self.norm(c + q)).view(-1, self.n_clauses, self.n_vars, 3)

    N_VARS_C, N_CLAUSES_C = 32, 56
    projector = ClauseProjector(HID, N_CLAUSES_C, N_VARS_C).to(DEVICE)
    print(f"Projector ready: clause-query cross-attention, hidden={HID} -> "
          f"({N_CLAUSES_C},{N_VARS_C},3)  [capacity sized from the measured ProofWriter data]")
else:
    print("Skipped C2:", skip_reason)

if BACKBONE_READY:
    import hashlib as _gh
    GEN_CACHE_PATH = os.path.join(CKPT_DIR, "llm_generations.json")
    try:
        _GEN_CACHE = json.load(open(GEN_CACHE_PATH)) if os.path.exists(GEN_CACHE_PATH) else {}
    except Exception:
        _GEN_CACHE = {}
    _GEN_STATS = {"hit": 0, "miss": 0}
    print(f"Generation cache: {len(_GEN_CACHE)} entries at {GEN_CACHE_PATH}")

    def _gen_flush():
        try:
            atomic_json(_GEN_CACHE, GEN_CACHE_PATH)
        except Exception as e:
            print(f"  [gen-cache] could not persist: {e}")

    @torch.no_grad()
    def llm_generate(prompt, max_new_tokens=512, temperature=0.0):
        """Greedy decode from the FROZEN base backbone, memoised by prompt."""
        _key = None
        if temperature == 0.0:
            _key = _gh.md5(f"{MODEL_NAME}|{max_new_tokens}|{prompt}".encode()).hexdigest()
            if _key in _GEN_CACHE:
                _GEN_STATS["hit"] += 1
                return _GEN_CACHE[_key]
            _GEN_STATS["miss"] += 1
        msgs = [{"role": "user", "content": prompt}]
        enc = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                            return_tensors="pt")
        if not torch.is_tensor(enc):          # REVISION v12.1: transformers >= 5 returns a dict
            enc = enc["input_ids"]
        enc = enc.to(DEVICE)
        kw = dict(attention_mask=torch.ones_like(enc), max_new_tokens=max_new_tokens,
                  do_sample=(temperature > 0), pad_token_id=tokenizer.eos_token_id)
        if temperature > 0: kw["temperature"] = temperature
        out = base.generate(enc, **kw)
        text = tokenizer.decode(out[0, enc.shape[1]:], skip_special_tokens=True)
        if _key is not None:
            _GEN_CACHE[_key] = text
            if _GEN_STATS["miss"] % 25 == 0:
                _gen_flush()
        return text
    print("llm_generate ready (greedy, deterministic, frozen backbone; memoised by prompt).")


## Cell C2b — Pre-training contamination / leakage check
FOLIO and ProofWriter predate current LLM pre-training corpora and may be partially memorized. The check: run the **frozen, un-adapted** backbone zero-shot on the holdout. If its accuracy is near the majority-class floor, the backbone is *not* silently solving the task from memorization, so gains from co-training are attributable to the method rather than leakage. If zero-shot is already high, we report that honestly and lean on the controlled synthetic study (Part A, no pretrained text) for the causal claim.

In [ ]:
# === Cell C2b: contamination / memorisation probes ===
if BACKBONE_READY and REAL_AVAILABLE and not _reuse_reported("contamination_probe.txt"):   # REVISION v12
    from collections import Counter as _Counter

    def _ask(prompt):
        try:
            txt = llm_generate(prompt, max_new_tokens=8).strip().lower()
        except Exception:
            return "Unknown"
        if txt.startswith("true"): return "True"
        if txt.startswith("false"): return "False"
        return "Unknown"

    def zero_shot_label(ex):
        return _ask(f"Premises: {ex['context']}\nConclusion: {ex['question']}\n"
                    f"Answer with exactly one word: True, False, or Unknown.\nAnswer:")

    def premise_ablated_label(ex):
        return _ask(f"Conclusion: {ex['question']}\n"
                    f"Answer with exactly one word: True, False, or Unknown.\nAnswer:")

    def _probe_pool(real, seed=0, n=80):
        pool = []
        for ex in real.get("proofwriter", []):
            e = dict(ex); e["dataset"] = "ProofWriter"
            pool.append((e, normalize_label(e.get("label"))))
        for ex in real.get("folio", []):
            e = dict(ex); e["dataset"] = "FOLIO"
            if e.get("label") is not None:
                pool.append((e, normalize_label(e.get("label"))))
        rng = np.random.default_rng(seed); rng.shuffle(pool)
        return pool[:n]

    ho_probe = _probe_pool(real, seed=0, n=80)
    gold = [lab for _, lab in ho_probe]
    maj = _Counter(gold).most_common(1)[0][1] / max(len(gold), 1)
    zs = sum(int(zero_shot_label(ex) == lab) for ex, lab in ho_probe) / max(len(ho_probe), 1)
    pa = sum(int(premise_ablated_label(ex) == lab) for ex, lab in ho_probe) / max(len(ho_probe), 1)

    print(f"Probe set: {len(ho_probe)} held-out examples | majority-class floor {maj:.3f}")
    print(f"  full zero-shot (premises + conclusion) : {zs:.3f}  (gap {zs - maj:+.3f})")
    print(f"  PREMISE-ABLATED (conclusion only)      : {pa:.3f}  (gap {pa - maj:+.3f})")
    if pa - maj > 0.10:
        verdict = ("MEMORISATION / CONCLUSION-SIDE PRIOR DETECTED: the backbone answers above "
                   "the floor WITHOUT the premises. Real-data gains cannot be attributed to "
                   "reasoning; lean on Part A (no pretrained text) for the causal claim.")
    elif zs - maj > 0.15:
        verdict = ("Non-trivial zero-shot competence but no conclusion-only signal: the backbone "
                   "can reason on this data, yet shows no sign of having memorised it. Report "
                   "both numbers.")
    else:
        verdict = ("LOW leakage risk: near the floor with and without premises; differences "
                   "between systems are unlikely to reflect memorisation.")
    print("Interpretation:", verdict)
    with open(os.path.join(OUT_DIR, "contamination_probe.txt"), "w") as fh:
        fh.write(f"n={len(ho_probe)}\nmajority={maj:.4f}\nzero_shot={zs:.4f}\n"
                 f"premise_ablated={pa:.4f}\nverdict={verdict}\n")
    send_alert("Contamination probes done",
               f"floor={maj:.3f} zero-shot={zs:.3f} premise-ablated={pa:.3f}")
elif BACKBONE_READY and REAL_AVAILABLE:
    print("C2b: reported contamination probe reused.")
else:
    print("C2b skipped:", skip_reason if not BACKBONE_READY else "real data unavailable")


## Cell C3 — Real-data co-training with a solver-decided inference path
Earlier revisions predicted
real-data labels with a trained `Linear(n_clauses, 3)` head on top of the solver output, while
every baseline used two Z3 SAT queries. A learned head can absorb the label prior of a
three-class benchmark; a sound solver cannot. R1/R2 were therefore structurally biased in
CoFormer's favour, and the "neuro-symbolic reasoning" claim was false at inference time — nothing
symbolic decided anything.

| | training signal | inference |
|---|---|---|
| before | trained `Linear(24, 3)` on solver output | **the same trained head** |
| now | **parameter-free** surrogate of the two-query rule | **Z3 `entailment_decision`**, identical to every baseline |

The surrogate compares the soft satisfaction of *(premises ∧ conclusion)* against
*(premises ∧ ¬conclusion)*. Nothing in the decision is learned, so no label prior can be
absorbed, yet the gradient still reaches the LoRA adapters and the projector through the
soft-MaxSAT layer.

The projector previously emitted abstract slots
with no mapping to anything in the text, so the gold-CNF loss aligned two unrelated permutations,
the emitted CNF was uninterpretable, and nothing could be handed to a solver. `ground_example`
(Cell B4) now fixes one deterministic slot assignment per example, shared by the projector's
output space, the gold-CNF target, the conclusion literal, and the Cell C6 probe.

**Worth stating in the paper.** On FOLIO there is no gold CNF, so `frozen` has *no* route from
labels to the translator at all. That is not a weakness of the baseline — it is the gap the method
targets: when gold formalisations are unavailable, the solver gradient is the only supervision
channel that reaches the formaliser.

In [ ]:
# === Cell C3: real-data co-training -- solver-decided inference, no learned label head ===
LABELS = ["True", "False", "Unknown"]; LAB2I = {l: i for i, l in enumerate(LABELS)}

def _augment_with_conclusion(cm, conclusion_lit, negate=False):
    """Append a unit clause for (+/-) the conclusion literal to a (1, C, V) clause matrix."""
    B, Cc, V = cm.shape
    extra = torch.zeros(B, 1, V, device=cm.device, dtype=cm.dtype)
    if conclusion_lit is not None:
        v = abs(conclusion_lit) - 1
        if 0 <= v < V:
            s = float(np.sign(conclusion_lit)) * (-1.0 if negate else 1.0)
            extra[:, 0, v] = s
    return torch.cat([cm, extra], dim=1)

def run_condition_real(condition, seed, train_pool, hold_pool, cfg,
                       epochs=2, accum=8, return_model=False):
    set_all_seeds(seed)
    solver_grad = condition not in ("frozen", "soft_frozen")          # REVISION v2 (M3)
    use_faith   = condition in ("coformer", "no_stgs", "frozen", "soft_frozen")
    use_stgs    = condition in ("coformer", "no_faith", "frozen")

    model = fresh_peft()
    proj  = ClauseProjector(HID, N_CLAUSES_C, N_VARS_C).to(DEVICE)
    logic_c = SoftMaxSATLayer(cfg.refine_steps,
                              step_size=getattr(cfg, "refine_step_size", 0.5),
                              backprop_steps=getattr(cfg, "refine_backprop_steps", None)
                              ).to(DEVICE)
    trainable = (list(proj.parameters()) +
                 [p for p in model.parameters() if p.requires_grad])
    lora_params = [p for p in model.parameters() if p.requires_grad]
    lora_before = float(sum(p.detach().float().norm() for p in lora_params))
    opt = torch.optim.AdamW(trainable, lr=1e-4)
    scaler = torch.amp.GradScaler(AMP_DEVICE, enabled=(HAS_GPU and AMP_DTYPE == torch.float16))

    def encode(ex):
        return tokenizer(f"Premises: {ex['context']}\nConclusion: {ex['question']}\nLogic:",
                         return_tensors="pt", truncation=True, max_length=384).to(DEVICE)

    def emit(ex, hard, tau):
        enc = encode(ex)
        out = model(**enc, output_hidden_states=True)
        logits = proj(out.hidden_states[-1], enc.get("attention_mask")).float()
        onehot = (st_gumbel_softmax(logits, tau, hard=hard) if hard is not None
                  else deterministic_onehot(logits))
        return logits, onehot, to_clause_matrix(onehot)

    # audits
    n_full = sum(1 for ex, _ in train_pool if ex["_g"]["faith_supervision"] == "full")
    gold_applied = 0; n_label_signal = 0; n_no_signal = 0
    cov_atoms_dropped = cov_gold_dropped = cov_no_concl = 0

    model.train(); t0 = time.time(); step = 0
    for ep in range(epochs):
        opt.zero_grad(set_to_none=True)
        for i, (ex, lab) in enumerate(train_pool):
            g = ex["_g"]
            with torch.amp.autocast(AMP_DEVICE, dtype=AMP_DTYPE, enabled=cfg.USE_AMP):
                proj_logits, onehot, cm = emit(ex, hard=use_stgs, tau=cfg.gumbel_tau_end)
                cm_s = cm if solver_grad else cm.detach()
                _, sat_with, _ = logic_c(_augment_with_conclusion(cm_s, g["conclusion_lit"]).float())
                _, sat_ag, _ = logic_c(_augment_with_conclusion(cm_s, g["conclusion_lit"],
                                                                negate=True).float())
                y = torch.tensor([LAB2I[lab]], device=DEVICE)
                loss = LAMBDA_LOGIT * logit_penalty(proj_logits)
                if g["conclusion_lit"] is not None:
                    logits3 = soft_entailment_logits(sat_with.mean(1), sat_ag.mean(1))  # (B,3)
                    loss = loss + F.cross_entropy(logits3, y)
                    if ep == 0: n_label_signal += 1
                if use_faith:
                    fl, used_gold, cov = faithfulness_loss_real(
                        onehot.squeeze(0), g["gold_clauses"],
                        g["faith_supervision"], N_CLAUSES_C, N_VARS_C,
                        logits=proj_logits.squeeze(0))                 # REVISION v2 (M1)
                    loss = loss + cfg.lambda_faith * fl
                    if ep == 0:
                        gold_applied += int(used_gold)
                        cov_gold_dropped += cov["clauses_dropped"]
                loss = loss / accum
            if ep == 0:
                cov_atoms_dropped += g["coverage"]["atoms_dropped"]
                cov_no_concl += (1 - g["coverage"]["conclusion_grounded"])
            if loss.requires_grad:
                (scaler.scale(loss) if scaler.is_enabled() else loss).backward()
            else:
                n_no_signal += 1
            if (i + 1) % accum == 0:
                if scaler.is_enabled(): scaler.step(opt); scaler.update()
                else: opt.step()
                opt.zero_grad(set_to_none=True); step += 1
    train_secs = time.time() - t0
    if solver_grad and n_label_signal < 0.05 * len(train_pool):
        print(f"  WARNING [{condition}]: only {n_label_signal}/{len(train_pool)} training "
              f"examples had a grounded conclusion, so the label signal reached the translator "
              f"on <5% of the data. The solver-gradient contrast is not meaningfully testable "
              f"at this grounding rate -- improve grounding coverage before reporting R1/R2.")
    assert step > 0, (f"{condition}: 0 optimiser steps for {len(train_pool)} examples at "
                      f"accum={accum} -- the model was never updated. Lower `accum` or enlarge "
                      f"the training pool.")
    lora_delta = abs(float(sum(p.detach().float().norm() for p in lora_params)) - lora_before)

    # ---- INFERENCE: hard CNF -> Z3, exactly as every baseline does ----
    model.eval(); correct = 0; per = {}; per_ex = []; abstain = 0
    with torch.no_grad():
        for ex, lab in hold_pool:
            g = ex["_g"]
            with torch.amp.autocast(AMP_DEVICE, dtype=AMP_DTYPE, enabled=cfg.USE_AMP):
                _, _, cm = emit(ex, hard=None, tau=None)
            clauses = clause_matrix_to_clause_list(cm[0])
            pred = entailment_decision(clauses, g["conclusion_lit"])
            abstain += int(pred == "Unknown")
            ok = int(pred == lab); correct += ok; per_ex.append(ok)
            ds = ex.get("dataset", "unknown"); per.setdefault(ds, [0, 0])
            per[ds][0] += ok; per[ds][1] += 1
    n = max(1, len(hold_pool))
    gfn = lambda k: per.get(k, [0, 1])[0] / max(1, per.get(k, [0, 1])[1])

    rec = dict(condition=condition, seed=seed, accuracy=correct / n,
               acc_proofwriter=gfn("ProofWriter"), acc_folio=gfn("FOLIO"),
               abstain_rate=abstain / n,
               n_full_faith=n_full, n_gold_cnf_applied=gold_applied,
               cov_atoms_dropped=cov_atoms_dropped, cov_gold_clauses_dropped=cov_gold_dropped,
               cov_ungrounded_conclusions=cov_no_concl,
               n_label_signal=n_label_signal,
               pct_label_signal=round(100.0 * n_label_signal / max(len(train_pool), 1), 1),
               lora_param_delta=round(lora_delta, 6),
               solver_gradient=solver_grad, faith_term=use_faith,
               train_secs=train_secs, opt_steps=step, _per_example=per_ex)
    if return_model:
        rec["_model"], rec["_proj"] = model, proj
    else:
        release_peft(model)
    _empty_cache()
    return rec

print("run_condition_real defined: parameter-free entailment surrogate for TRAINING, "
      "Z3 entailment_decision() for INFERENCE (identical to all baselines).")


## Cell C5 — Unified real-data comparison and Holm-corrected confirmatory tests
Every system shares one backbone, one formulator interface, one Z3 decision rule, and — per seed
— one train/holdout split. Checkpoint/resume, coverage audits, and the real-data confirmatory
family (R1–R3) live here.

The separate `EqualBudget-lora` system is gone. Under the corrected
design `CoFormer-frozen` *is* the equal-budget, no-co-training system: identical trainable
parameters, identical step count, identical inference, differing only in the detached solver
gradient. Keeping a second near-duplicate invited exactly the "two identical baselines" problem
that `SATLM-lora`/`LogicLM-lora` had.

Symbolic capacity (`N_CLAUSES_C × N_VARS_C`) is a hard bound. The cell
now reports, per split, how many examples exceed the variable capacity, how often gold clauses
were dropped, and what share of conclusions could be grounded — instead of truncating silently.
Abstention rate, formaliser fallback rate, Logic-LM refine iterations, LoRA parameter movement
and gold-CNF application counts are all printed as explicit audits.

In [ ]:
# === Cell C5: unified real-data comparison + Holm confirmatory tests ===
import pandas as pd, json as _js, shutil
from collections import Counter
from statsmodels.stats.multitest import multipletests

def build_pools(real, seed=0, hold_frac=0.2):
    """Grounding is computed ONCE per example and attached as ex['_g'], so the projector's slot
    space, the gold-CNF target and the conclusion literal handed to Z3 all agree by construction.
    the GOLD LABEL is passed in so the ProofWriter derivation can self-validate --
    an example enters the 'full' faithfulness regime only if Z3 over the derived CNF reproduces
    that label."""
    pool = []
    for raw in real.get("proofwriter", []):
        e = dict(raw); e["dataset"] = "ProofWriter"
        lab = normalize_label(e.get("label"), e.get("options"))
        e["_g"] = ground_example(e, N_CLAUSES_C, N_VARS_C, gold_label=lab)
        pool.append((e, lab))
    for raw in real.get("folio", []):
        if raw.get("label") is None: continue
        fol = raw.get("premises-FOL", raw.get("fol", ""))
        if isinstance(fol, str): fol = [fol] if fol else []
        if any(isinstance(f, str) and f.count("(") != f.count(")") for f in fol):
            continue                                   # LINC-style bracket sanity filter
        e = dict(raw); e["dataset"] = "FOLIO"
        e["gold_fol"] = dict(premises=list(fol), conclusion=raw.get("conclusion-FOL", ""))
        lab = normalize_label(e.get("label"))
        e["_g"] = ground_example(e, N_CLAUSES_C, N_VARS_C, gold_label=lab)
        e["_g"]["gold_clauses"] = None                 # FOLIO gold logic is quantified FOL
        e["_g"]["faith_supervision"] = "proxy"
        pool.append((e, lab))
    rng = np.random.default_rng(seed); rng.shuffle(pool)
    n = len(pool)
    if n < 20:
        raise ValueError(f"build_pools: only {n} usable examples after filtering -- too few to "
                         f"split into train/holdout. Check the data cache and the label parser.")
    n_hold = min(max(40, int(hold_frac * n)), max(1, int(0.75 * n)))
    train_pool, hold_pool = pool[n_hold:], pool[:n_hold]
    assert train_pool and hold_pool, "build_pools produced an empty split"
    if len(train_pool) < 40:
        print(f"  WARNING: training pool has only {len(train_pool)} examples "
              f"(holdout {len(hold_pool)}); real-data results will be very noisy.")
    return train_pool, hold_pool

def report_coverage(pool, name="pool"):
    """Symbolic capacity is a hard bound; report how much of each example fits and which
    faithfulness regime it landed in, INSTEAD of truncating silently."""
    if not pool: return {}
    cov = [ex["_g"]["coverage"] for ex, _ in pool]
    reg = Counter(ex["_g"]["faith_supervision"] for ex, _ in pool)
    ds = Counter(ex.get("dataset") for ex, _ in pool)
    out = dict(
        n=len(pool), datasets=dict(ds), regimes=dict(reg),
        mean_atoms=float(np.mean([c["atoms_total"] for c in cov])),
        max_atoms=int(max(c["atoms_total"] for c in cov)),
        pct_over_var_capacity=100.0 * float(np.mean([c["atoms_dropped"] > 0 for c in cov])),
        pct_parse_complete=100.0 * float(np.mean([c.get("parse_complete", 0) for c in cov])),
        pct_gold_validated=100.0 * float(np.mean([c.get("gold_validated", 0) for c in cov])),
        pct_conclusion_grounded=100.0 * float(np.mean([c["conclusion_grounded"] for c in cov])))
    print(f"\nCoverage [{name}] n={out['n']} {dict(ds)}")
    print(f"  atoms/example: mean {out['mean_atoms']:.1f}, max {out['max_atoms']} "
          f"(capacity {N_VARS_C}); over capacity {out['pct_over_var_capacity']:.1f}%")
    print(f"  theory fully parsed {out['pct_parse_complete']:.1f}% | conclusion grounded "
          f"{out['pct_conclusion_grounded']:.1f}%")
    print(f"  faithfulness regime: {dict(reg)}  (gold-CNF validated on "
          f"{out['pct_gold_validated']:.1f}% of examples)")
    if reg.get("full", 0) == 0:
        print("  !! No example reached the gold-CNF regime -> R3 is UNTESTABLE on this data and "
              "must be reported as untested, not as a null result.")
    return out

@torch.no_grad()
def eval_solver_baseline(fn, hold_pool):
    correct = fb = ref = abst = 0; per = {}; per_ex = []
    for ex, lab in hold_pool:
        r = fn(ex); ok = int(r["pred"] == lab)
        correct += ok; per_ex.append(ok)
        fb += int(r["used_fallback"]); ref += r["refine_iters"]
        abst += int(r["pred"] == "Unknown")
        ds = ex.get("dataset", "unknown"); per.setdefault(ds, [0, 0])
        per[ds][0] += ok; per[ds][1] += 1
    n = max(1, len(hold_pool)); g = lambda k: per.get(k, [0, 1])[0] / max(1, per.get(k, [0, 1])[1])
    return dict(accuracy=correct / n, fallback_rate=fb / n, mean_refine=ref / n,
                abstain_rate=abst / n, acc_proofwriter=g("ProofWriter"), acc_folio=g("FOLIO"),
                _per_example=per_ex)

# REVISION v12 (Plan III, family D): pure-prompting baselines on the same frozen backbone.
# They answer directly, without a formula or a solver, and are added only when
# RUN_PLAN_III_NL is True, so the stored comparison is resumed and only their rows run.
import re as _re_d
_ANSWER_RE = _re_d.compile(r"answer\s*[:\-]\s*\**\s*(true|false|unknown)", _re_d.I)

def baseline_direct(example):
    txt = llm_generate(f"Premises: {example['context']}\nConclusion: {example['question']}\n"
                       f"Answer with exactly one word: True, False, or Unknown.\nAnswer:",
                       max_new_tokens=8).strip().lower()
    pred = ("True" if txt.startswith("true") else "False" if txt.startswith("false")
            else "Unknown")
    return dict(pred=pred, used_fallback=not txt.startswith(("true", "false", "unknown")),
                refine_iters=0)

def baseline_cot(example):
    txt = llm_generate(f"Premises: {example['context']}\nConclusion: {example['question']}\n"
                       f"Decide whether the conclusion follows from the premises (True), its "
                       f"negation follows (False), or neither (Unknown). Reason step by step, "
                       f"then end with a final line of the form 'Answer: True', 'Answer: False' "
                       f"or 'Answer: Unknown'.",
                       max_new_tokens=512)
    hits = _ANSWER_RE.findall(txt)
    if not hits:
        return dict(pred="Unknown", used_fallback=True, refine_iters=0)
    return dict(pred=hits[-1].capitalize(), used_fallback=False, refine_iters=0)

# `CoFormer-frozen` IS the equal-budget, no-co-training system: identical trainable parameters,
# identical step count, identical inference; only the solver gradient is detached.
SYSTEM_SPEC = {
    "SATLM-frozen":      ("solver", baseline_satlm),
    "LogicLM-frozen":    ("solver", baseline_logiclm),
    "DiLA-frozen":       ("solver", baseline_dila),
    "CoFormer-frozen":   ("coformer", "frozen"),
    "CoFormer-coformer": ("coformer", "coformer"),
    "CoFormer-no_faith": ("coformer", "no_faith"),
    "CoFormer-no_stgs":  ("coformer", "no_stgs"),
    "CoFormer-soft_frozen": ("coformer", "soft_frozen"),   # REVISION v2 (M3): completes the factorial
}
if globals().get("RUN_PLAN_III_NL", False):                   # REVISION v12
    SYSTEM_SPEC.update({"Direct-frozen": ("solver", baseline_direct),
                        "CoT-frozen":    ("solver", baseline_cot)})
REAL_RESULTS_CSV = os.path.join(CKPT_DIR, "real_comparison.csv")
PER_EXAMPLE_JSON = os.path.join(CKPT_DIR, "per_example_real.json")

def _load_per_example():
    if os.path.exists(PER_EXAMPLE_JSON):
        try: return _js.load(open(PER_EXAMPLE_JSON))
        except Exception: return {}
    return {}
def _save_per_example(store): atomic_json(store, PER_EXAMPLE_JSON)
PER_EXAMPLE_STORE = _load_per_example()

def _run_one(system, seed, train_pool, hold_pool):
    kind, spec = SYSTEM_SPEC[system]
    if kind == "solver":
        r = eval_solver_baseline(spec, hold_pool)
    else:
        rc = run_condition_real(spec, seed, train_pool, hold_pool, CFG,
                                epochs=getattr(CFG, "partc_epochs", 2))
        r = {k: v for k, v in rc.items() if not k.startswith("_") and k != "condition"}
        r["_per_example"] = rc["_per_example"]
        _empty_cache()
    PER_EXAMPLE_STORE[f"{system}|{seed}"] = r.pop("_per_example", [])
    _save_per_example(PER_EXAMPLE_STORE)
    r.update(system=system, seed=seed)
    return r

def run_full_comparison(real, seeds=None):
    seeds = seeds if seeds is not None else tuple(range(getattr(CFG, "partc_seeds",
                                                                len(CFG.seeds))))
    REAL_REQUIRED = ["system", "seed", "accuracy", "acc_proofwriter", "acc_folio"]
    real_fp = config_fingerprint(REAL_REQUIRED, fields=_FP_FIELDS_REAL,
                                 extra=f"C{N_CLAUSES_C}x{N_VARS_C}|{MODEL_NAME}"
                                       f"|seeds{getattr(CFG, 'partc_seeds', len(CFG.seeds))}"
                                       f"|ep{getattr(CFG, 'partc_epochs', 2)}")
    assert_fingerprint("real", real_fp)   # REVISION v12
    real_fp_file = REAL_RESULTS_CSV + ".fingerprint"
    if os.path.exists(REAL_RESULTS_CSV):
        _stored = (open(real_fp_file).read().strip()
                   if os.path.exists(real_fp_file) else "(none)")
        _cols = pd.read_csv(REAL_RESULTS_CSV, nrows=1).columns
        _bad = [c for c in REAL_REQUIRED if c not in _cols]
        if _stored != real_fp or _bad:
            _arch = REAL_RESULTS_CSV.replace(".csv", f".stale-{real_fp}.csv")
            shutil.move(REAL_RESULTS_CSV, _arch)
            for _f in (PER_EXAMPLE_JSON,):          # the prediction cache is tied to this matrix
                if os.path.exists(_f): shutil.move(_f, _f + f".stale-{real_fp}")
            PER_EXAMPLE_STORE.clear()
            print(f"STALE REAL-DATA CHECKPOINT ARCHIVED -> {_arch}\n"
                  f"  reason: {'fingerprint mismatch' if _stored != real_fp else f'missing {_bad}'}")
    if os.path.exists(REAL_RESULTS_CSV):
        dd = pd.read_csv(REAL_RESULTS_CSV); done = set(zip(dd.system, dd.seed))
        records = dd.to_dict("records")
        print(f"Resume: {len(records)} (system, seed) rows already stored "
              f"(backbone {MODEL_NAME}).")
    else:
        done, records = set(), []
    todo = [(s, sd) for sd in seeds for s in SYSTEM_SPEC if (s, sd) not in done]
    pools = {}
    def gp(sd):
        if sd not in pools:
            pools[sd] = build_pools(real, seed=sd)
            if sd == seeds[0]:
                report_coverage(pools[sd][0], "train"); report_coverage(pools[sd][1], "holdout")
        return pools[sd]
    _n_solver = sum(1 for k, (kind, _) in SYSTEM_SPEC.items() if kind == "solver")
    _n_train = len(SYSTEM_SPEC) - _n_solver
    _hold = len(gp(seeds[0])[1]) if seeds else 0
    print(f"This session: {len(todo)}/{len(SYSTEM_SPEC)*len(seeds)} runs "
          f"({_n_solver} prompted baselines + {_n_train} trained systems x {len(seeds)} seeds).")
    print(f"  holdout {_hold} examples; prompted baselines decode from a memoised cache "
          f"({len(_GEN_CACHE) if '_GEN_CACHE' in globals() else 0} entries already stored), so "
          f"repeat prompts across systems and seeds are free.")
    print(f"  trained systems: {getattr(CFG, 'partc_epochs', 2)} epochs over "
          f"{len(gp(seeds[0])[0]) if seeds else 0} examples each.")
    send_alert("Real-data matrix started", f"{len(records)} done, {len(todo)} remaining.")
    t0 = time.time()
    try:
        for k, (system, sd) in enumerate(todo, 1):
            ensure_budget(f"real-data {system}/seed {sd}", 25 * 60)       # REVISION v3
            tr, ho = gp(sd)
            if not any(r.get("system") == "majority_floor" and r.get("seed") == sd
                       for r in records):
                dist = Counter([l for _, l in ho])
                maj = dist.most_common(1)[0][1] / len(ho)
                if maj > 0.9:
                    print(f"  !! seed {sd}: majority class is {maj:.0%} ({dict(dist)}). "
                          f"That indicates a LABEL PARSING BUG, not a dataset property.")
                records.append(dict(system="majority_floor", seed=sd, accuracy=maj))
            records.append(_run_one(system, sd, tr, ho))
            atomic_csv(pd.DataFrame.from_records(records), REAL_RESULTS_CSV, index=False)
            maybe_push_state(f"real {system}/{sd}")
            open(real_fp_file, 'w').write(real_fp)
            if "_gen_flush" in globals(): _gen_flush()
            print(f"  {k}/{len(todo)} [{system}|seed {sd}] "
                  f"acc={records[-1].get('accuracy', float('nan')):.3f}")
            if k % max(1, len(todo) // 4) == 0 or k == len(todo):
                send_alert(f"Real-data {k}/{len(todo)}", str(records[-1]))
    except SessionBudgetReached:
        raise
    except Exception:
        tb = traceback.format_exc(); send_alert("REAL-DATA CRASH", tb); print(tb); raise
    if "_gen_flush" in globals():
        _gen_flush()
        _h, _m = _GEN_STATS["hit"], _GEN_STATS["miss"]
        print(f"  generation cache: {_h} hits, {_m} misses "
              f"({_h/max(_h+_m,1):.0%} reused; {len(_GEN_CACHE)} distinct prompts stored)")
    dfr = pd.DataFrame.from_records(records)
    send_alert("Real-data matrix complete", f"{len(dfr)} rows, {time.time()-t0:.0f}s.")
    return dfr

PREREGISTRATION_REAL = {
    "R0": dict(metric="accuracy", comparison=("CoFormer-no_stgs", "CoFormer-coformer"),
               direction="greater", decision="supported if diff>MIN_EFFECT_REAL AND Holm p<0.05",
               rationale="REPLICATION of the synthetic headline: does removing the "
                         "straight-through bridge also help on real benchmarks, where the "
                         "translator is an 8B LLM rather than a small transformer?"),
    "R1": dict(metric="accuracy", comparison=("CoFormer-coformer", "DiLA-frozen"),
               direction="greater", decision="supported if diff>MIN_EFFECT_REAL AND Holm p<0.05",
               rationale="co-training beats the frozen-translator architecture at identical inference"),
    "R2": dict(metric="accuracy", comparison=("CoFormer-coformer", "CoFormer-frozen"),
               direction="greater", decision="supported if diff>MIN_EFFECT_REAL AND Holm p<0.05",
               rationale="EQUAL trainable budget, steps and inference; only the solver gradient differs"),
    "R3": dict(metric="acc_proofwriter", comparison=("CoFormer-coformer", "CoFormer-frozen"),
               direction="greater", decision="supported if diff>MIN_EFFECT_REAL AND Holm p<0.05",
               rationale="the benefit should be largest where VALIDATED gold-CNF supervision exists"),
}
PREREGISTRATION_REAL_EXPLORATORY = {
    "R0b": dict(metric="acc_proofwriter", comparison=("CoFormer-no_stgs", "CoFormer-coformer"),
                direction="greater", decision="EXPLORATORY (outside the Holm family)",
                rationale="if the bridge is the bottleneck, the gap should be largest where "
                          "validated gold-CNF supervision exists, i.e. on ProofWriter"),
    "R4": dict(metric="acc_folio", comparison=("CoFormer-coformer", "CoFormer-frozen"),
               direction="greater", decision="EXPLORATORY (outside the Holm family)",
               rationale="FOLIO has no gold CNF, so `frozen` has NO channel from labels to the "
                         "translator; a large gap is evidence that the solver gradient is the "
                         "only supervision route when formalisations are absent"),
}

def confirmatory_on(df_long, prereg, label, min_effect=None, exploratory=None):
    if min_effect is None: min_effect = MIN_EFFECT
    rows, pvals, hkeys = [], [], []
    for h, d in list(prereg.items()) + list((exploratory or {}).items()):
        is_expl = h in (exploratory or {})
        c1, c2 = d["comparison"]; metric = d["metric"]
        if metric not in df_long.columns:
            rows.append(dict(hyp=h, metric=metric, status="metric absent")); continue
        a = df_long[df_long.system == c1].sort_values("seed")[metric].values
        b = df_long[df_long.system == c2].sort_values("seed")[metric].values
        if len(a) != len(b) or len(a) < 2 or np.isnan(a).any() or np.isnan(b).any():
            rows.append(dict(hyp=h, metric=metric, status="not computable")); continue
        status, p_floor, note = contrast_testability(a, b)
        md_, p = paired_bootstrap(a, b); dv = cohens_d_paired(a, b); lo, hi = boot_ci_d(a, b)
        d_note = "" if np.isfinite(dv) else "undefined (zero across-seed variance)"
        rows.append(dict(hyp=h, metric=metric, mean_diff=round(md_, 4), p_raw=round(p, 4),
                         cohen_d=round(dv, 3), d_CI=f"[{lo:.2f},{hi:.2f}]", d_note=d_note,
                         family=("exploratory" if is_expl else
                                 "confirmatory" if status == "testable" else
                                 f"excluded ({status})"),
                         testability=status, p_floor=round(p_floor, 4),
                         note=note, n_seeds=len(a)))
        if not is_expl and status == "testable":
            pvals.append(p); hkeys.append(h)
    if pvals:
        rej, padj, _, _ = multipletests(pvals, alpha=ALPHA, method="holm")
        pm, rm = dict(zip(hkeys, padj)), dict(zip(hkeys, rej))
        for r in rows:
            h = r.get("hyp")
            if h in pm:
                r["p_holm"] = round(pm[h], 4)
                r["verdict"] = ("SUPPORTED" if (rm[h] and abs(r["mean_diff"]) >= min_effect
                                                and r["mean_diff"] > 0) else "not supported")
    out = pd.DataFrame(rows)
    print(f"\n=== {label} (Holm over the confirmatory family only) ===\n", out.to_string(index=False))
    out.to_csv(os.path.join(OUT_DIR, f"confirmatory_{label}.csv"), index=False)
    return out

if BACKBONE_READY and REAL_AVAILABLE:
    real_comparison_df = run_full_comparison(real)
    if globals().get("RUN_PLAN_III_NL", False):               # REVISION v12: Plan III family D
        _d = real_comparison_df[real_comparison_df.system.isin(
            ["Direct-frozen", "CoT-frozen", "SATLM-frozen", "LogicLM-frozen", "majority_floor"])]
        _dsum = _d.groupby("system")[[c for c in ("accuracy", "acc_proofwriter", "acc_folio",
                                                  "abstain_rate", "fallback_rate")
                                      if c in _d.columns]].agg(["mean", "std"])
        print("\nPlan III family D (descriptive):\n", _dsum.round(3).to_string())
        _d.to_csv(os.path.join(OUT_DIR, "plan_iii_nl.csv"), index=False)
    print("\nMean accuracy by system:")
    print(real_comparison_df.groupby("system")["accuracy"].agg(["mean", "std"]).round(3).to_string())

    print("\n--- Coverage and integrity audits ---")
    for col, msg in [("abstain_rate", "share of holdout decided as 'Unknown'"),
                     ("fallback_rate", "LLM formaliser fell back to the rule parser"),
                     ("mean_refine", "mean Logic-LM self-refine iterations"),
                     ("lora_param_delta", "LoRA parameter movement (must be > 0)"),
                     ("n_gold_cnf_applied", "examples that received VALIDATED gold-CNF supervision"),
                     ("cov_ungrounded_conclusions", "conclusions that could not be grounded"),
                     ("pct_label_signal", "% of training examples whose label signal "
                                          "could actually reach the translator"),
                     ("pct_no_signal", "% of training steps carrying NO gradient at all "
                                       "(no gold CNF, no grounded conclusion)")]:
        if col in real_comparison_df.columns:
            s = real_comparison_df.groupby("system")[col].mean().dropna()
            if len(s): print(f"\n{col}  ({msg}):\n{s.round(3).to_string()}")

    # ---- audits that must FIRE, not just print -----------------------------------
    # The Part C dry run exposed two silent modes that produce a complete, plausible-looking
    # results table from a system that is doing nothing:
    #   (a) abstain_rate ~ 1.0 -- with Z3 inference, a CNF that is not precise enough to refute
    #       either side yields "Unknown" every time, so the reported accuracy is simply the
    #       frequency of the Unknown class. This is a much higher bar than the learned label head
    #       used in earlier revisions, and it is the honest one -- but it must be visible.
    #   (b) lora_param_delta ~ 0 -- the adapters were in the optimiser but never moved, which is
    #       the exact failure mode fixed for the equal-budget baseline in earlier development.
    _issues = []
    for _col, _thresh, _msg in [
            ("abstain_rate", 0.95, "abstains on >95% of the holdout: its accuracy is the "
                                   "Unknown-class rate, not evidence of reasoning"),
            ("fallback_rate", 0.50, "fell back to the rule-based parser on >50% of examples: "
                                    "this baseline is a straw man as measured")]:
        if _col in real_comparison_df.columns:
            for _sys, _v in real_comparison_df.groupby("system")[_col].mean().dropna().items():
                if _v > _thresh: _issues.append(f"{_sys}: {_msg} ({_col}={_v:.2f})")
    if "lora_param_delta" in real_comparison_df.columns:
        for _sys, _v in real_comparison_df.groupby("system")["lora_param_delta"].mean().dropna().items():
            if _v < 1e-4:
                _issues.append(f"{_sys}: LoRA parameters did not move (delta={_v:.2e}) -- the "
                               f"trainable budget was never used")
    if "pct_label_signal" in real_comparison_df.columns:
        _p = real_comparison_df["pct_label_signal"].dropna()
        if len(_p) and _p.mean() < 50:
            _issues.append(f"only {_p.mean():.0f}% of training examples had a grounded "
                           f"conclusion, so the label signal reached the translator on a "
                           f"minority of the data")
    _floor = float(real_comparison_df[real_comparison_df.system == "majority_floor"]
                   ["accuracy"].mean()) if (real_comparison_df.system == "majority_floor").any() \
             else float("nan")
    _sys_acc = (real_comparison_df[real_comparison_df.system != "majority_floor"]
                .groupby("system")["accuracy"].mean())
    if np.isfinite(_floor) and len(_sys_acc) and _sys_acc.max() < _floor:
        _issues.insert(0, f"EVERY system scores below the majority-class floor "
                          f"(best {_sys_acc.max():.3f} vs floor {_floor:.3f}). The stage has not "
                          f"measured reasoning: with abstention this high, each system's accuracy "
                          f"is the frequency of the Unknown class, and the contrasts between them "
                          f"are differences between abstention rates.")

    print("\n=== PART C HEALTH ===")
    if _issues:
        for _i in _issues: print("  [FAIL]", _i)
        print("  VERDICT: these real-data numbers are NOT interpretable as published results.")
    else:
        print("  [OK] no system abstains throughout, all trainable budgets moved, "
              "label signal reaches a majority of training examples")
    with open(os.path.join(OUT_DIR, "partc_health.txt"), "w") as _fh:
        _fh.write("\n".join(_issues) if _issues else "OK\n")

    if "n_gold_cnf_applied" in real_comparison_df.columns:
        ng = real_comparison_df["n_gold_cnf_applied"].dropna()
        if len(ng) and ng.max() == 0:
            print("\nWARNING: NO example received gold-CNF supervision. R3 is UNTESTABLE here.")

    real_comparison_df.to_csv(os.path.join(OUT_DIR, "real_comparison_full.csv"), index=False)
    confirmatory_on(real_comparison_df, PREREGISTRATION_REAL, "real_data",
                    min_effect=MIN_EFFECT_REAL, exploratory=PREREGISTRATION_REAL_EXPLORATORY)
    print("\nR3 >> R4 would localise the benefit where validated gold-CNF supervision exists. "
          "Check the gold-supervision audit above BEFORE making that claim.")
else:
    real_comparison_df = None
    print("Real-data comparison skipped:", skip_reason if not BACKBONE_READY else "no data")


## Cell C6 — Human-aligned faithfulness validation on a FOLIO sub-sample
On synthetic data faithfulness is exact (gold CNF). On FOLIO there is no propositional gold CNF, so the LLM-backbone stage uses a *proxy* faithfulness signal — and it must be established whether that proxy tracks real translation quality. This cell validates it on a small, fixed FOLIO sub-sample by comparing the model’s emitted CNF against the dataset’s **gold FOL annotations** at the atom/polarity level (a semi-automatic structural match), reporting agreement. It is intentionally small (single-A100 friendly) but it turns the proxy from a hand-wave into a calibrated number, and it is the contrast that makes the R3≫R4 result (gold-CNF ProofWriter vs proxy FOLIO) interpretable.

In [ ]:
# === Cell C6: human-aligned faithfulness validation on a FOLIO sub-sample ===
# Validates the CO-TRAINED translator (not the raw backbone) against gold FOL on SIGNED
# (predicate, polarity) atoms. Grounding now makes this exact rather than "conservative":
# slot k of the projector denotes the k-th atom of that example, fixed by ground_example().
import re as _re2

def _fol_atoms(fol_strings):
    atoms = set()
    for s in fol_strings or []:
        if not isinstance(s, str): continue
        for m in _re2.finditer(r"(\u00ac|~|\bnot\s+)?\s*([A-Za-z_][A-Za-z0-9_]*)\s*\(", s):
            atoms.add((m.group(2).lower(), 1 if m.group(1) else 0))
    return atoms

def _signed_jaccard(pred_atoms, gold_atoms):
    if not gold_atoms: return None
    return len(pred_atoms & gold_atoms) / max(len(pred_atoms | gold_atoms), 1)

def _atoms_from_clause_matrix(cm_row, grounding):
    """Signed atoms implied by an emitted clause matrix, using the SHARED slot grounding."""
    atoms = set()
    inv = {i: a for a, i in grounding["slot"].items()}
    arr = cm_row.detach().cpu().numpy()
    for slot, name in inv.items():
        if slot >= arr.shape[1]: continue
        col = arr[:, slot]
        head = name.split()[0] if name.split() else name
        if (col > 0).any(): atoms.add((head.lower(), 0))
        if (col < 0).any(): atoms.add((head.lower(), 1))
    return atoms

def _raw_backbone_atoms(ex):
    f = LLMFormulatorReal(); clauses, concl, fb, _ = f.formulate_full(ex)
    names = {}
    if isinstance(f.var_map, dict):
        for nm, vid in f.var_map.items():
            b = _re2.match(r"([A-Za-z_][A-Za-z0-9_]*)", str(nm))
            if b:
                try: names[int(vid)] = b.group(1).lower()
                except (ValueError, TypeError): pass
    atoms = set()
    for cl in (clauses or []):
        for lit in cl:
            nm = names.get(abs(lit))
            if nm: atoms.add((nm, 0 if lit > 0 else 1))
    return atoms, fb

def faithfulness_validation(real, n=60, seed=0):
    if not BACKBONE_READY or not real.get("folio"):
        print("C6 skipped: backbone or FOLIO unavailable (no fabrication)."); return None
    tr_pool, ho_pool = build_pools(real, seed=seed)
    folio_hold = [(ex, lab) for ex, lab in ho_pool
                  if ex.get("dataset") == "FOLIO" and ex.get("gold_fol", {}).get("premises")]
    if not folio_hold:
        print("C6 skipped: no held-out FOLIO example carries gold FOL annotations."); return None

    print("C6: training a genuinely co-trained CoFormer translator for the probe ...")
    rc = run_condition_real("coformer", seed, tr_pool, ho_pool, CFG, return_model=True)
    model, proj = rc.get("_model"), rc.get("_proj")
    if model is None or proj is None:
        print("C6 skipped: co-trained model unavailable."); return None

    rows, raw_js, co_js = [], [], []
    try:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(folio_hold), size=min(n, len(folio_hold)), replace=False)
        model.eval()
        for j in idx:
            ex, _ = folio_hold[int(j)]
            gold_atoms = _fol_atoms(ex["gold_fol"]["premises"] +
                                    [ex["gold_fol"].get("conclusion", "")])
            if not gold_atoms: continue
            raw_atoms, fb = _raw_backbone_atoms(ex)
            enc = tokenizer(f"Premises: {ex['context']}\nConclusion: {ex['question']}\nLogic:",
                            return_tensors="pt", truncation=True, max_length=384).to(DEVICE)
            with torch.no_grad():
                out = model(**enc, output_hidden_states=True)
                onehot = deterministic_onehot(proj(out.hidden_states[-1],
                                                   enc.get("attention_mask")).float())
            co_atoms = _atoms_from_clause_matrix(to_clause_matrix(onehot)[0], ex["_g"])
            rj = _signed_jaccard(raw_atoms, gold_atoms)
            cj = _signed_jaccard(co_atoms, gold_atoms)
            if rj is None or cj is None: continue
            raw_js.append(rj); co_js.append(cj)
            rows.append(dict(example=int(j), raw_signed_jaccard=round(rj, 3),
                             cotrained_signed_jaccard=round(cj, 3), used_fallback=fb))
    finally:
        release_peft(model); _empty_cache()

    if not rows:
        print("C6: no comparable examples."); return None
    raw_m, co_m = float(np.mean(raw_js)), float(np.mean(co_js))
    d_arr = np.array(co_js) - np.array(raw_js)
    rng = np.random.default_rng(0)
    bo = np.array([rng.choice(d_arr, len(d_arr), replace=True).mean() for _ in range(4000)])
    lo, hi = np.percentile(bo, [2.5, 97.5])
    print(f"Signed-atom faithfulness vs gold FOL on {len(rows)} held-out FOLIO examples:")
    print(f"  raw frozen backbone   : {raw_m:.3f}")
    print(f"  co-trained translator : {co_m:.3f}")
    print(f"  delta = {co_m - raw_m:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}] "
          f"({'improves' if lo > 0 else 'no significant improvement at this scale (reported honestly)'})")
    pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "faith_validation_folio.csv"), index=False)
    with open(os.path.join(OUT_DIR, "faith_validation_summary.txt"), "w") as fh:
        fh.write(f"n={len(rows)}\nraw_mean={raw_m:.4f}\ncotrained_mean={co_m:.4f}\n"
                 f"delta={co_m-raw_m:.4f}\nci=[{lo:.4f},{hi:.4f}]\n")
    return co_m - raw_m

if BACKBONE_READY and REAL_AVAILABLE and RUN_EXTRA_DIAGNOSTICS:
    ensure_budget("FOLIO faithfulness validation", 30 * 60)          # REVISION v3
    if not _reuse_reported("faith_validation_summary.txt"):   # REVISION v12
        _ = faithfulness_validation(real, n=60, seed=0)
    print("\nThis grounds the proxy regime: a CI excluding zero is human-aligned evidence that "
          "the co-training signal -- not merely the satisfaction objective -- improves the "
          "emitted formalisation.")
elif not RUN_EXTRA_DIAGNOSTICS:
    print("C6 skipped: RUN_EXTRA_DIAGNOSTICS is False (enable for the full probe).")
else:
    print("C6 skipped:", skip_reason if not BACKBONE_READY else "real data unavailable")


## Cell C7 — Seed-budget-aware statistics: example-level bootstrap
The pre-registered confirmatory tests pair over seeds. On a single A100 the real-data seed count
is small, so this cell adds an **example-level bootstrap** on the held-out predictions: it
resamples *examples* rather than seeds, giving a CI that does not depend on a large seed count.
Where the two agree, the conclusion is robust to the seed budget; where they disagree, the
seed-paired test is underpowered and should be described that way.

The previous version called `per_example_correct_real`, which
**retrained every system for every seed** purely to recover predictions Cell C5 had already
computed and discarded — roughly doubling the GPU cost of the LLM-backbone stage. It now reads the cache C5
writes, and if the cache is missing it says so instead of silently launching hours of redundant
training.

In [ ]:
# === Cell C7: seed-budget-aware statistics -- CLUSTER bootstrap over (seed, example) ===
def cluster_bootstrap_contrast(per_seed_a, per_seed_b, n_boot=10000, seed=0):
    """per_seed_*: dict seed -> 0/1 array (paired: same holdout order within a seed)."""
    rng = np.random.default_rng(seed)
    seeds = [s for s in per_seed_a if s in per_seed_b
             and len(per_seed_a[s]) and len(per_seed_b[s])]
    if not seeds:
        return (float("nan"),) * 4
    obs = float(np.mean([np.mean(per_seed_a[s]) - np.mean(per_seed_b[s]) for s in seeds]))
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        drawn = rng.choice(len(seeds), len(seeds), replace=True)
        vals = []
        for j in drawn:
            s = seeds[j]
            a, b = np.asarray(per_seed_a[s], float), np.asarray(per_seed_b[s], float)
            n = min(len(a), len(b))
            idx = rng.integers(0, n, n)
            vals.append(a[idx].mean() - b[idx].mean())
        diffs[i] = np.mean(vals)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return obs, float(lo), float(hi), float(min(1.0, p))

def per_seed_correct(system, store, seeds):
    out = {}
    for sd in seeds:
        v = store.get(f"{system}|{sd}")
        if v: out[sd] = np.asarray(v, dtype=int)
    return out

if BACKBONE_READY and REAL_AVAILABLE:
    store = _load_per_example()
    seeds = list(range(getattr(CFG, "partc_seeds", len(CFG.seeds))))   # REVISION v3: real-data seeds
    contrasts = [("CoFormer-no_stgs", "CoFormer-coformer"),
                 ("CoFormer-no_stgs", "DiLA-frozen"),
                 ("CoFormer-coformer", "CoFormer-frozen"),
                 ("CoFormer-coformer", "DiLA-frozen")]
    missing = sorted({s for pair in contrasts for s in pair
                      if not per_seed_correct(s, store, seeds)})
    if missing:
        print("C7 skipped: no cached per-example predictions for:", ", ".join(missing))
        print("Run Cell C5 first -- it writes", PER_EXAMPLE_JSON, "as it evaluates each system. "
              "This cell deliberately retrains nothing.")
    else:
        lines = []
        for a_name, b_name in contrasts:
            A = per_seed_correct(a_name, store, seeds)
            B = per_seed_correct(b_name, store, seeds)
            d, lo, hi, p = cluster_bootstrap_contrast(A, B)
            n_ex = sum(len(v) for v in A.values())
            print(f"  [{a_name} - {b_name}]  diff = {d:+.3f}  95% CI [{lo:+.3f},{hi:+.3f}]  "
                  f"p = {p:.3f}  ({len(A)} seeds, {n_ex} predictions)")
            lines.append(f"{a_name}_vs_{b_name}: diff={d:.4f} CI=[{lo:.4f},{hi:.4f}] p={p:.4f}")
        print(f"\nTwo-stage cluster bootstrap over seeds {tuple(seeds)} then examples: it does NOT "
              f"pretend predictions from one trained model are independent draws.")
        print("Report BOTH this and the seed-paired Holm test. Agreement means the finding "
              "survives the seed budget; disagreement means the seed-paired test is underpowered "
              "and must be described that way.")
        with open(os.path.join(OUT_DIR, "example_bootstrap_real.txt"), "w") as fh:
            fh.write(f"cluster_bootstrap seeds={tuple(seeds)}\n" + "\n".join(lines) + "\n")
else:
    print("C7 skipped:", skip_reason if not BACKBONE_READY else "real data unavailable")


## Cell U — Completeness protocol: the measurement that justifies the design
The soft-MaxSAT relaxation maximises satisfied-clause count and **cannot certify
unsatisfiability**. This is not merely stated — it is measured on genuinely random, Z3-certified
UNSAT instances.

A three-way entailment decision needs a proof of UNSAT for the "False"
class, so a relaxation that essentially never signals unsatisfiability cannot serve as an
inference oracle. This probe's number is therefore the stated, pre-registered reason why every
decision in this notebook — CoFormer's included — is produced by Z3, and the relaxation is scoped
to a *training signal*. Earlier revisions asserted a caveat here while quietly letting the
relaxation (via a learned head) decide the labels.

In [ ]:
# === Cell U: completeness probe -- the measurement that JUSTIFIES the design ===
def sample_random_instance(n_vars, n_clauses, clause_len, rng):
    cl = []
    for _ in range(n_clauses):
        vs = rng.choice(n_vars, size=min(clause_len, n_vars), replace=False)
        pol = rng.integers(0, 2, size=len(vs)) * 2 - 1
        cl.append([int(p) * (int(v) + 1) for v, p in zip(vs, pol)])
    return cl

def completeness_probe(n=100, seed=0, probe_vars=6, ratio=6.0, max_tries=4000):
    if "z3_satisfiable" not in globals() or not Z3_OK:
        print("Z3 unavailable; completeness probe skipped honestly (no fabricated number).")
        return
    rng = np.random.default_rng(seed)
    n_clauses = int(probe_vars * ratio)
    verified_unsat = layer_flagged = tries = 0
    sat_margins = []
    while verified_unsat < n and tries < max_tries:
        tries += 1
        cl = sample_random_instance(probe_vars, n_clauses, 3, rng)
        if z3_satisfiable(cl) is not False:
            continue
        verified_unsat += 1
        cm = torch.zeros(1, len(cl), probe_vars, device=DEVICE)
        for ci, c in enumerate(cl):
            for lit in c:
                cm[0, ci, abs(lit) - 1] = float(np.sign(lit))
        with torch.no_grad():
            # REVISION v3: use the step size of the trained layer (v1 used the default 0.5)
            _, final_sat, _ = SoftMaxSATLayer(
                CFG.refine_steps, step_size=getattr(CFG, "refine_step_size", 0.5)).to(DEVICE)(cm.float())
        m = float(final_sat.min().item()); sat_margins.append(m)
        layer_flagged += int(m < 0.5)
    if verified_unsat == 0:
        print(f"No verified-UNSAT instances in {tries} draws; probe skipped honestly.")
        return
    rate = layer_flagged / verified_unsat
    print(f"Verified-UNSAT instances: {verified_unsat} (from {tries} random draws, "
          f"{probe_vars} vars, clause/var ratio {ratio})")
    print(f"Relaxation signalled unsatisfiability (min clause-sat < 0.5): "
          f"{layer_flagged} ({rate:.1%})")
    print(f"Mean min-clause-satisfaction on UNSAT instances: {np.mean(sat_margins):.3f}")
    print("\nDESIGN CONSEQUENCE (pre-registered, not post hoc): a soft relaxation that flags "
          f"{rate:.1%} of certified-UNSAT instances cannot decide the 'False' class of a "
          "three-way entailment task, because that class requires proving unsatisfiability. "
          "The relaxation is therefore used ONLY as a training signal in this work; every "
          "reported decision, for CoFormer and for all baselines alike, comes from Z3.")
    with open(os.path.join(OUT_DIR, "completeness_probe.txt"), "w") as f:
        f.write(f"verified_unsat={verified_unsat}\nlayer_flagged={layer_flagged}\n"
                f"rate={rate:.4f}\nmean_min_sat={np.mean(sat_margins):.4f}\n")

if not _reuse_reported("completeness_probe.txt"):   # REVISION v12
    completeness_probe(n=100, seed=0)


## Cell 14 — Health check, evidence-bounded claims, and the final alert
The first full run finished without a single exception, so the completion
email said "ALL DONE". That was true and useless: three of four conditions had collapsed to
constant predictors, every `acc_id` sat below the floor, and H3 was uncomputable with n = 0.
Twelve seeds of A100 time were spent before anyone could tell.

The cell now runs a **health check** before it writes a single claim — degenerate conditions,
zero across-seed variance, no condition above chance, uncomputable hypotheses, emitted density
far from gold, insufficient power, missing the LLM-backbone stage — and changes the email's own subject line to
`COMPLETED WITH DEGENERATE RESULTS — DO NOT REPORT` when any of those fail. Claim sentences are
**withheld** rather than printed on an unusable run.

In [ ]:
# === Cell 14: health check, evidence-bounded claims, PUBLICATION BUNDLE, final alert ===
import glob, zipfile, shutil, datetime

# ---------------------------------------------------------------------------------------------
# HEALTH CHECK -- the alert must not say "done" when the result is unusable.
# ---------------------------------------------------------------------------------------------
HEALTH = []
def _fail(m, blocking=True): HEALTH.append(("FAIL" if blocking else "LIMIT", m, blocking))
def _warn(m): HEALTH.append(("WARN", m, False))
def _ok(m):   HEALTH.append(("OK", m, False))

MAIN_CONDITIONS = {"coformer", "frozen"} | set(CEILING_CONDITIONS)
_bad = set(DEGENERATE_CONDITIONS) | set(NOISE_CONDITIONS)

# one statement per degenerate condition, and its severity depends on WHICH condition it is
for _c in sorted(_bad):
    _why = []
    if _c in NOISE_CONDITIONS: _why.append("information-free")
    if _c in DEGENERATE_CONDITIONS: _why.append("degenerate")
    if float(df[df.condition == _c]["faith_id"].std()) < 1e-6: _why.append("zero seed variance")
    _blocking = _c in MAIN_CONDITIONS
    _fail(f"`{_c}` is {', '.join(_why)}"
          + ("  <-- it is part of the main contrast, so the headline is affected"
             if _blocking else "  (an ablation; bounds interpretation, does not invalidate "
                               "the main contrast)"), blocking=_blocking)
if not _bad:
    _ok("no condition is degenerate or information-free")
if CEILING_CONDITIONS:
    _ok(f"reached the ceiling (solved, not collapsed): {', '.join(CEILING_CONDITIONS)}")

if float(agg["csat_norm_id"].max()) < CSAT_FLOOR:
    _fail(f"no condition exceeds chance on the primary metric "
          f"(max csat_norm {agg['csat_norm_id'].max():.3f} < {CSAT_FLOOR})", blocking=True)
else:
    _ok(f"best csat_norm = {agg['csat_norm_id'].max():.3f} (chance = 0)")

# The expected row count follows the SEED ALLOCATION, not the naive product of
# conditions and seeds: the factorial cells run at the full seed budget while
# the two faithfulness ablations run at ABLATION_SEEDS. Using the product here
# reports a complete matrix as incomplete and blocks a healthy run.
_n_expected = (sum(len(seeds_for(c)) for c in CONDITIONS)
               if "seeds_for" in dir() else len(CONDITIONS) * len(CFG.seeds))
if len(df) < _n_expected:
    _fail(f"results matrix incomplete ({len(df)}/{_n_expected})", blocking=True)
else:
    _ok(f"results matrix complete ({len(df)}/{_n_expected})")

uncomputable = [r["hypothesis"] for _, r in conf_df.iterrows()
                if isinstance(r.get("status"), str) and r.get("status")]
if uncomputable:
    _fail(f"hypotheses not computable: {', '.join(uncomputable)} -- report as untested, "
          f"not as null results", blocking=False)
else:
    _ok("every pre-registered hypothesis was computable")

if POWER_TABLE[0.8] < 0.80:
    _warn(f"underpowered: achieved power {POWER_TABLE[0.8]:.2f} at d=0.8 with "
          f"{len(CFG.seeds)} seeds")
else:
    _ok(f"achieved power {POWER_TABLE[0.8]:.2f} at d=0.8 with {len(CFG.seeds)} seeds")

if not globals().get("BACKBONE_READY", False):
    _r = globals().get("skip_reason", "").strip() or "reason not recorded"
    _warn(f"Part C did not run -- {_r} Real-benchmark evidence is missing; see "
          f"outputs/partc_skip_reason.txt for the fix.")
else:
    _ok("Part C real-benchmark evidence present")

if not CFG.FULL_SCALE:
    _warn("FULL_SCALE is False: this is the demo preset, not the full configuration")

CEILING_ABLATIONS = []
_max_excess = 1.0 - expected_f1_random(GOLD_DENSITY, GOLD_DENSITY)
for _c in CONDITIONS:
    if _c == "coformer":
        continue
    _e = float(agg.loc[_c, "faith_excess_id"])
    if _e > 0.95 * _max_excess and _e > float(agg.loc["coformer", "faith_excess_id"]) + MIN_EFFECT:
        CEILING_ABLATIONS.append(_c)
if CEILING_ABLATIONS:
    print("\n" + "=" * 72)
    print("HEADLINE RESULT: an ablation reaches the ceiling that the full model does not.")
    for _c in CEILING_ABLATIONS:
        _d = df[df.condition == _c].sort_values("seed")["faith_excess_id"].values
        _f = df[df.condition == "coformer"].sort_values("seed")["faith_excess_id"].values
        _diff = _d - _f
        _dz = _diff.mean() / _diff.std(ddof=1) if _diff.std(ddof=1) > 1e-12 else float("inf")
        print(f"  {_c}: faith_excess {agg.loc[_c,'faith_excess_id']:+.4f} vs coformer "
              f"{agg.loc['coformer','faith_excess_id']:+.4f}  (diff {_diff.mean():+.4f}, "
              f"dz {_dz:+.1f}); csat_norm {agg.loc[_c,'csat_norm_id']:+.4f} vs "
              f"{agg.loc['coformer','csat_norm_id']:+.4f}")
    print("  H6 (no ablation improves on the full model) is FALSIFIED. Removing the mechanism "
          "named in that ablation is what makes the task solvable, and the effect size is orders "
          "of magnitude larger than any pre-registered contrast. Report this first.")
    print("=" * 72)

# --- design-completeness gates -------------------------------------------
# These fire on the DESIGN, not on the numbers: a study can produce a clean,
# plausible results table and still be unable to attribute what it measured.
_fact_missing = [c for c in ("frozen", "coformer", "soft_frozen", "no_stgs")
                 if c not in set(df.condition)]
if _fact_missing:
    _fail(f"the bridge x solver-gradient factorial is incomplete "
          f"(missing {', '.join(_fact_missing)}). The headline contrast "
          f"confounds the two mechanisms and supports no causal attribution.",
          blocking=True)
else:
    _ok("factorial complete: bridge and solver gradient are separately identified")

if os.path.exists(os.path.join(OUT_DIR, "negative_control.csv")):
    try:
        _ng = pd.read_csv(os.path.join(OUT_DIR, "negative_control.csv"))
        for _, _row in _ng.iterrows():                                   # REVISION v2 (M8)
            _base = _row.get("base_condition", "coformer")
            _ngf = float(_row["faith_id"])
            _rlf = float(df[df.condition == _base]["faith_id"].mean())
            if abs(_ngf - _rlf) < 1e-3:
                _fail(f"the shuffled-label control on `{_base}` reproduces the real run "
                      f"({_ngf:.4f} vs {_rlf:.4f}): the model is not using its "
                      f"supervision.", blocking=True)
            else:
                _ok(f"negative control on `{_base}` passed (shuffled {_ngf:.3f} vs real {_rlf:.3f})")
    except Exception as _e:
        _warn(f"negative control present but unreadable: {_e}")
else:
    _fail("the shuffled-label negative control did not run; an apparent "
          "advantage cannot be distinguished from a supervision artifact",
          blocking=False)

_probes_done = sorted({p for p in PROBES
                       if os.path.exists(probe_results_path(p))
                       and len(pd.read_csv(probe_results_path(p)))
                           >= sum(len(seeds_for(c)) for c in CONDITIONS)} | {CFG.probe})
if len(_probes_done) < len(PROBES):
    _fail(f"only probe(s) {', '.join(_probes_done)} of {', '.join(PROBES)} were "
          f"run; the finding is not shown to hold across instance families",
          blocking=False)
else:
    _ok(f"reproduced across instance families {', '.join(_probes_done)}")

_untestable = [r.get("hypothesis") for _, r in conf_df.iterrows()
               if str(r.get("testability", "testable")) not in ("testable", "nan", "")]
if _untestable:
    _fail(f"contrasts excluded as not testable at this seed budget: "
          f"{', '.join(str(u) for u in _untestable)} -- reported as untestable, "
          f"never as null results", blocking=False)

BLOCKING = [m for s, m, b in HEALTH if b]
LIMITS = [m for s, m, b in HEALTH if s in ("LIMIT", "WARN")]
RUN_HEALTHY = not BLOCKING

print("=" * 72); print("RUN HEALTH CHECK")
for status, msg, _ in HEALTH:
    print(f"  [{status:5s}] {msg}")
if RUN_HEALTHY and LIMITS:
    _v = "HEALTHY WITH SCOPED LIMITATIONS -- claims stand, bounded by the items marked LIMIT/WARN"
elif RUN_HEALTHY:
    _v = "HEALTHY -- results are interpretable"
else:
    _v = "BLOCKED -- a condition in the main contrast is unusable; DO NOT REPORT"
print(f"\nVERDICT: {_v}")
print("=" * 72)

def _num(x, fmt="{:+.3f}", missing="not computable"):
    """Never let a NaN reach a claim sentence."""
    try:
        v = float(x)
    except (TypeError, ValueError):
        return missing
    return missing if not np.isfinite(v) else fmt.format(v)

co_r, fr_r = agg.loc["coformer"], agg.loc["frozen"]
claims = []

if CEILING_ABLATIONS:
    for _c in CEILING_ABLATIONS:
        _d = df[df.condition == _c].sort_values("seed")["faith_excess_id"].values
        _f = df[df.condition == "coformer"].sort_values("seed")["faith_excess_id"].values
        _diff = _d - _f
        _dz = _diff.mean() / _diff.std(ddof=1) if _diff.std(ddof=1) > 1e-12 else float("inf")
        claims.append(
            f"HEADLINE: on in-distribution uniform random {CFG.clause_len}-SAT (n_vars="
            f"{CFG.n_vars}, {CFG.n_clauses} clauses, clause/variable ratio {CFG.sat_ratio}, "
            f"satisfiability verified by DPLL), the `{_c}` ablation reaches density-controlled "
            f"clause-fidelity {_num(agg.loc[_c,'faith_excess_id'])} (raw F1 "
            f"{agg.loc[_c,'faith_id']:.3f}) and chance-normalised end-task "
            f"{_num(agg.loc[_c,'csat_norm_id'])}, against {_num(co_r['faith_excess_id'])} and "
            f"{_num(co_r['csat_norm_id'])} for the full model -- a paired difference of "
            f"{_num(_diff.mean())} at dz {_dz:+.1f} over {len(_diff)} seeds, and 0/"
            f"{len(_diff)} seeds in which the full model is not beaten. H6 is falsified.")
    for _c in CEILING_ABLATIONS:                                       # REVISION v3: derived, not asserted
        _r0 = df[df.condition == _c].iloc[0]; _f0 = df[df.condition == "coformer"].iloc[0]
        _diffs = [f"{k}: {_f0[k]} -> {_r0[k]}" for k in ("factor_bridge", "factor_solver", "factor_faith")
                  if _r0[k] != _f0[k]]
        claims.append(f"`{_c}` differs from the full model in: {'; '.join(_diffs)}.")

if os.path.exists(os.path.join(OUT_DIR, "st_bias_probe.csv")):
    _st = pd.read_csv(os.path.join(OUT_DIR, "st_bias_probe.csv"))
    for _obj, _g in (_st.groupby("objective") if "objective" in _st.columns else [("task+faith", _st)]):
        _cm = float(_g["cosine"].mean())
        _desc = ("opposed to" if _cm < -0.2 else "weakly aligned with" if _cm < 0.3
                 else "aligned with")                                     # REVISION v3
        claims.append(
            f"Gradient probe [{_obj}]: straight-through vs soft-relaxation cosine {_cm:+.3f} "
            f"(range {_g['cosine'].min():+.3f} to {_g['cosine'].max():+.3f}), norm ratio "
            f"{_g['norm_ratio'].mean():.3f}; the straight-through update is {_desc} the relaxed gradient.")

claims += [
 (lambda _sup: f"Confirmatory hypotheses supported after Holm correction: "
               f"{', '.join(_sup) if _sup else 'none'} "
               f"(coformer - frozen: faithfulness {_num(co_r['faith_excess_id'] - fr_r['faith_excess_id'])}, "
               f"end-task {_num(co_r['csat_norm_id'] - fr_r['csat_norm_id'])}).")(
     [str(r['hypothesis']) for _, r in conf_df.iterrows() if r.get('verdict') == 'SUPPORTED']),
 f"Degeneracy audit: "
 + (f"`{', '.join(sorted(set(DEGENERATE_CONDITIONS) | set(NOISE_CONDITIONS)))}` "
    + ("collapsed and is excluded from interpretation; "
       if len(set(DEGENERATE_CONDITIONS) | set(NOISE_CONDITIONS)) == 1
       else "collapsed and are excluded from interpretation; ")
    if (set(DEGENERATE_CONDITIONS) | set(NOISE_CONDITIONS)) else "no condition collapsed; ")
 + f"emitted literal density {co_r['pred_density']:.2f} (full model) vs gold {GOLD_DENSITY:.2f}.",
 f"Achieved power with {len(CFG.seeds)} seeds: {POWER_TABLE[0.8]:.2f} at d=0.8, "
 f"{POWER_TABLE[0.5]:.2f} at d=0.5.",
 f"Mean training cost {co_r['train_secs']:.0f}s per run on {GPU_NAME}.",
]
_ood = _num(co_r["ood_rel_drop"])
if _ood != "not computable":
    claims.append(f"Relative ID->OOD degradation: {_ood} (CoFormer) vs "
                  f"{_num(fr_r['ood_rel_drop'])} (detached); lower = more robust [H3].")

print("\n=== Evidence-bounded claim sentences (edit; never inflate) ===")
if RUN_HEALTHY:
    for c in claims: print(" -", c)
    if LIMITS:
        print("\n  These claims are BOUNDED by:")
        for l in LIMITS: print("   -", l)
else:
    print("  WITHHELD: a condition in the main contrast is unusable, so the headline itself "
          "cannot be trusted. Fix that before drafting any claim.")
    for b in BLOCKING: print("   -", b)

# ---------------------------------------------------------------------------------------------
# PUBLICATION BUNDLE -- every artifact a submission needs, collected, captioned and zipped.
# ---------------------------------------------------------------------------------------------
CAPTIONS = {
 "results_table.tex":
   ("Table 1. Main ablation results, mean over seeds. `Faith excess' is clause-F1 minus the "
    "closed-form score of an information-free CNF at the density the model emitted, g*d/(g+d); "
    "raw F1 alone is confounded by density. `CSat-norm' is gold-clause satisfaction rescaled so "
    "that a random assignment scores 0 and a perfect one 1. All conditions are trained and differ "
    "in exactly one mechanism."),
 "coformer_results.pdf":
   ("Figure 1. (a) In-distribution faithfulness and end-task quality by condition, with "
    "across-seed standard deviations. (b) CSat-norm on the three evaluation sets; the sets are "
    "unordered categories, not a trajectory. (c) Emitted literal density against the gold "
    "density; red marks conditions flagged as constant or information-free predictors."),
 "confirmatory_holm.csv":
   ("Table 2. Planned confirmatory tests (H1-H5; plan fixed in code, no external time stamp), paired "
    "over seeds, Holm-corrected within the synthetic family after the testability screen. Effect "
    "sizes are paired Cohen's d with bootstrap CIs."),
 "h6_ablation_dominance.csv":
   ("Table 3. H6 ablation dominance: per-seed share of contrasts in which the full model is not "
    "beaten by an ablation, with binomial CIs. Reported outside the Holm family."),
 "confirmatory_real_data.csv":
   ("Table 4. Real-benchmark confirmatory tests (R0-R3) on FOLIO and ProofWriter, Holm-corrected "
    "within the real family; R4 is exploratory and excluded from the correction."),
 "st_bias_probe.csv":
   ("Table 2. Straight-through gradient bias. Cosine between the straight-through gradient and "
    "the soft-relaxation gradient at identical Gumbel noise, and the ratio of their norms, swept "
    "over the annealing temperature, at initialisation. For the task loss alone the estimator is "
    "partly aligned with the relaxed gradient; with the faithfulness term included it is aligned."),
 "cases_no_stgs.csv":
   ("Qualitative cases (first six ID instances, seed 0). frozen, coformer and no_stgs give the same "
    "outcome on every case; the residual errors are refinement failures, not translation failures."),
 "hparam_sensitivity.csv":
   ("Sensitivity of the straight-through conditions to the Gumbel temperature and the number of "
    "refinement steps, single seed. Exact solve rises with the number of refinement steps at a "
    "translation held at the ceiling; the final Gumbel temperature changes no metric."),
 "objective_anchoring.csv":
   ("Objective-anchoring diagnostic. Clause-F1 and two solver losses for gold, corrupted and "
    "random clause matrices: the self-referential loss (satisfy your own clauses) and the "
    "anchored loss (satisfy the gold clauses). The sign of the correlation between fidelity and "
    "loss is the quantity of interest."),
 "capacity_probe.txt":
   ("Escalating capacity probe: fidelity at the production epoch budget against 3x, used to "
    "separate an optimisation ceiling from a capacity ceiling before committing the full matrix."),
 "real_benchmark_results.csv":
   ("Part B rule-based abstaining floor on FOLIO and ProofWriter, with the label distribution "
    "used to verify the parser."),
 "completeness_probe.txt":
   ("Sensitivity of the relaxation as an unsatisfiability detector on Z3-certified UNSAT instances "
    "(threshold 0.5 on the minimum soft clause satisfaction). Every decision is made by Z3 because "
    "Z3 is sound; the relaxation is used as a training signal only."),
 "lambda_sweep.csv":     "Sensitivity of the result to the faithfulness weight lambda_faith.",
 "density_sweep.csv":    "Sensitivity to the anti-collapse density hinge; no setting, including lambda=0, collapsed.",
 "ratio_sweep.csv":      "Instance-hardness sweep over the clause/variable ratio, frozen vs coformer.",
 "negative_control.csv": "Shuffled-label negative control: both shuffled runs emit empty formulas.",
 "contamination_probe.txt": "Zero-shot and premise-ablated memorisation probes on the backbone.",
 "faith_validation_folio.csv": "Human-aligned signed-atom faithfulness against gold FOL on FOLIO.",
 "requirements.txt":     "Resolved dependency versions for the run that produced these artifacts.",
 "run_metadata.json":    "Device, platform, full configuration and derived quantities.",
 "run_health.txt":       "Automated health check gating whether these numbers are reportable.",
 "partc_health.txt":     "Part C health audit (abstention, fallback, adapter movement, label signal).",
 # REVISION v12: Plan III artefacts
 "plan_III.json":        "Plan III specification with its SHA-256, written before any Plan III run.",
 "results_plan_iii.csv": "Plan III per-run results (answer supervision, K, task weight, budget).",
 "confirmatory_plan_iii.csv": "Plan III tests: A2 confirmatory (exact sign-flip, Holm); A1, K, W, B diagnostic.",
 "plan_iii_learning_curves.csv": "Plan III budget stage: ID excess every third epoch (first 512 ID instances).",
 "plan_iii_nl.csv":      "Plan III family D: direct-answering and chain-of-thought baselines (descriptive).",
}

# REVISION v3: run_metadata.json was written in Cell 4, BEFORE epoch calibration, so it
# recorded epochs=40 / calibrated_epochs=None. Rewrite it with the configuration actually used.
try:
    with open(_meta_path("run_metadata.json"), "w") as _fh:   # REVISION v12
        json.dump({"device": str(DEVICE), "gpu": GPU_NAME, "platform": platform.platform(),
                   "versions": globals().get("_versions", {}), "config": dataclasses.asdict(CFG),
                   "n_clauses": CFG.n_clauses, "loss_version": LOSS_VERSION,
                   "run_fingerprint": RUN_FP, "probes_complete": _probes_done,
                   "epoch_calibration": EPOCH_CALIBRATION,
                   "per_family_grad_steps": {p: int(pd.read_csv(probe_results_path(p))["grad_steps"].iloc[0])
                                             for p in _probes_done if os.path.exists(probe_results_path(p))}},
                  _fh, indent=2, default=str)
except Exception as _e:
    print(f"  run_metadata rewrite failed: {_e}")

if not globals().get("PERSISTENT", True):
    if globals().get("DRIVE_API") is not None:
        print("\nThe FUSE mount was unavailable; pushing the final state through the Drive API.")
        push_state(tag="final", force=True)
    else:
        print("\nThis session did not use Drive; writing a state snapshot so the run can resume.")
        try:
            snapshot_state(download=True)
        except Exception as _e:
            print(f"  snapshot failed: {_e}; zip {BASE} manually before closing the session")

BUNDLE_DIR = os.path.join(OUT_DIR, "paper_bundle")
os.makedirs(BUNDLE_DIR, exist_ok=True)
manifest, present, missing = [], [], []
for fname, cap in CAPTIONS.items():
    src = os.path.join(OUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(BUNDLE_DIR, fname))
        present.append(fname)
        manifest.append(f"## {fname}\n{cap}\n")
    else:
        missing.append(fname)
_captioned = set(CAPTIONS)
_swept = sorted(f for f in os.listdir(OUT_DIR)
                if os.path.isfile(os.path.join(OUT_DIR, f))
                and f not in _captioned
                and not f.endswith(".zip")
                and f.split(".")[-1] in ("csv", "txt", "json", "png", "pdf", "tex"))
for extra in _swept:
    src = os.path.join(OUT_DIR, extra)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(BUNDLE_DIR, extra)); present.append(extra)
if _swept:
    print(f"  swept in {len(_swept)} uncaptioned artifact(s): {', '.join(_swept)}")

# the raw results matrix belongs in the artifact so every number is re-derivable
if os.path.exists(RESULTS_CSV):
    shutil.copy2(RESULTS_CSV, os.path.join(BUNDLE_DIR, "raw_results_matrix.csv"))
    present.append("raw_results_matrix.csv")

header = (f"# CoFormer -- publication artifact bundle\n\n"
          f"Generated {datetime.datetime.now().isoformat(timespec='seconds')} on {GPU_NAME}\n"
          f"FULL_SCALE={CFG.FULL_SCALE} | n_vars={CFG.n_vars} | n_clauses={CFG.n_clauses} | "
          f"ratio={CFG.sat_ratio} | seeds={len(CFG.seeds)} | "
          f"health={'INTERPRETABLE (see run_health.txt)' if RUN_HEALTHY else 'DEGENERATE'}\n\n"
          f"Every table and figure below is regenerated end-to-end by a single Run All of the "
          f"notebook; `raw_results_matrix.csv` contains the per-(condition, seed) rows from which "
          f"all aggregates are derived.\n\n")
with open(os.path.join(BUNDLE_DIR, "MANIFEST.md"), "w") as fh:
    fh.write(header + "\n".join(manifest))
    if missing:
        fh.write("\n## Not produced by this run\n"
                 + "".join(f"- {m}\n" for m in missing)
                 + "\n(These come from optional stages: RUN_EXTRA_DIAGNOSTICS sweeps and Part C. "
                   "Their absence is recorded rather than hidden.)\n")

zip_path = os.path.join(OUT_DIR, "coformer_paper_bundle.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(BUNDLE_DIR)):
        zf.write(os.path.join(BUNDLE_DIR, f), arcname=os.path.join("coformer_bundle", f))

print(f"\n=== PUBLICATION BUNDLE ===")
print(f"  {len(present)} artifacts -> {BUNDLE_DIR}")
print(f"  zipped -> {zip_path}")
if missing:
    print(f"  not produced by this run ({len(missing)}): {', '.join(missing)}")
    print(f"  -> enable RUN_EXTRA_DIAGNOSTICS and/or set HF_TOKEN for the complete set.")
print(f"  captions for every table and figure are in {os.path.join(BUNDLE_DIR, 'MANIFEST.md')}")

print("\n=== Limitations to state in the paper ===")
for l in [
 "The synthetic study is a controlled probe with templated language, not naturalistic text; "
 "sentence segmentation is supplied as preprocessing, as in SATLM / Logic-LM / LINC.",
 "The anti-collapse density prior injects the known gold sparsity; on real data that prior is "
 "estimated from the derived gold CNF and is therefore weaker.",
 "Part C symbolic capacity is bounded; examples exceeding it fall back to the proxy regime.",
 "FOLIO carries no propositional gold CNF; the gold-CNF contrast rests on ProofWriter.",
 "The relaxation cannot certify UNSAT and therefore cannot replace the solver at inference.",
]:
    print("  -", l)

subject = ("ALL DONE" if RUN_HEALTHY and not LIMITS else
           "DONE -- claims stand, with scoped limitations" if RUN_HEALTHY else
           "BLOCKED -- main contrast unusable, DO NOT REPORT")
body = ("HEALTH CHECK\n" + "\n".join(f"[{_s}] {_m}" for _s, _m, _ in HEALTH) +
        f"\n\nVERDICT: {_v}\n\n" +
        (("Claims:\n" + "\n".join(" - " + c for c in claims)
          + ("\n\nBounded by:\n" + "\n".join(" - " + l for l in LIMITS) if LIMITS else ""))
         if RUN_HEALTHY else
         "Claims withheld -- blocking failures:\n" + "\n".join(" - " + b for b in BLOCKING)) +
        f"\n\nBundle: {zip_path}\n")
send_alert(subject, body)
with open(os.path.join(OUT_DIR, "run_health.txt"), "w") as fh:
    fh.write(body)
print(f"\nAlert subject: {subject!r}")
